# Chapter 19, Unit B: Break, repair and transfer acceptance

> **Learn with Prof Rod** — *Build Your Always-On AI Agent From Scratch*.
> **Read the full book and get the latest learning materials:** [https://profrod.ai/book](https://profrod.ai/book).
> **Join the Prof Rod learner community:** [https://profrod.ai/community](https://profrod.ai/community)
> — bring your questions, compare experiments and share what you build.
> **Original source and updates:** [profrodai/sovereign-agent](https://github.com/profrodai/sovereign-agent).

**Instructor worked edition · 90 minutes of dedicated work · 2026-09-09**

This is one of two practical units for Chapter 19. Unit A constructs and connects the
mechanism; Unit B investigates a controlled failure, repairs it and transfers the invariant.
Each is a complete ninety-minute session, with its own setup and required conceptual introductions.
Basic Python variables, conditions, loops, functions, lists and dictionaries are the starting
knowledge. Libraries and specialized concepts used here are introduced below before the main task.

By the end you should be able to:

1. Explain the chapter's mechanism using a prediction and an observed intermediate result.
2. Repair the failure: Forcing totals to match hides a real accounting discrepancy. Create a real approval, corrupt a retained ledger amount, then call the actual report and inspect both structured evidence and displayed exception text.
3. Solve **reconcile an independently supplied order ledger** using changed inputs and an independent expectation.
4. Retain your implementation, failed/corrected observations, causal explanation and limits.

| Minutes | Dedicated activity | Evidence you produce |
|---|---|---|
| 0–5 | State the problem and make a prediction | Initial prediction in your own words |
| 5–25 | Foundations and library examples | Values, explanations, revised predictions |
| 25–35 | Trace setup and the main interface | Input → learner function → observation |
| 35–60 | Reproduce, diagnose and repair | Source, visible checks and runtime evidence |
| 60–80 | Implement and challenge the transfer task | Function and a new counterexample |
| 80–90 | Retrieve, explain and save | Exit ticket and retained submission |

Installation is preparation time. These are planning estimates, not measured completion times.
Use the reference primers when a term is unfamiliar; in Unit B retrieve an explanation before
re-reading it. Run All checks that the artifact executes. Unfinished student functions deliberately
produce NEEDS_WORK. Keep your first attempt before opening answers.


This notebook belongs to the nineteen-chapter edition. Its supplied teaching runtime is embedded, so it can run without the textbook or another notebook. Where code uses `REFERENCE_LESSON`, that is the frozen runtime exercise identifier; the reader-facing chapter and saved unit identifiers use the current edition. Building against a supplied runtime is not proof that you have constructed all of its dependencies.


## Run the self-contained setup

Use a **Python 3.14** Jupyter kernel and **Pydantic 2**. If needed, run
`%pip install "pydantic==2.13.4"` once in a separate cell and restart the kernel.
Package installation needs internet; the lesson itself needs no repository download, API key
or prior notebook. The complete source runtime uses Python 3.14, so this edition does not claim
compatibility with a hosted notebook service's default interpreter.

The collapsed cell contains 102 frozen teaching files. Base85 represents compressed bytes
as text; `zlib` decompresses them; SHA-256 checks that the decoded files match this edition.
These are supplied packaging operations, not learner algorithms. `tempfile` creates an isolated
working copy; `Path` handles file locations; `sys.path` tells Python where the supplied modules
live. The code is available for inspection below and performs no package installation itself.
The subsequent lesson teaches the libraries used by the mechanisms you will implement.

Run setup on every fresh kernel. It writes scratch runtime files separately from your retained
`practical-work/ch19-b` folder. Rerunning setup restores the frozen support files and keeps
your saved work. Restarting a kernel clears variables, not saved submission files. Source basis:
Sovereign Agent `444c5f6`. Some tasks use reviewed local subprocesses; they are not an OS sandbox.

<details><summary>Supplied offline setup and teaching files</summary>


In [ ]:
import base64
import hashlib
import json
import os
import sys
import tempfile
import zlib
from pathlib import Path

minimum_python = (3, 14)
if sys.version_info[:2] < minimum_python:
    raise RuntimeError("Use a Python 3.14 kernel for this complete-book practical course.")
try:
    import pydantic
except ImportError as error:
    raise RuntimeError('Run %pip install "pydantic==2.13.4", then restart the kernel.') from error
if pydantic.__version__.split(".")[0] != "2":
    raise RuntimeError("Use Pydantic 2; the tested version is 2.13.4.")

# Frozen, reviewed course files: data until explicitly loaded by the lesson.
COURSE_ARCHIVE = (
    "c-ri}jdR;ZvM>6t%;COzCKHGz^<|5`+;bIK+4x1Vd?h8hTcQdD0z(li5Mb~jnd{R3{;IoY1~b3_BxTu~obuU"
    "Iu?P(2dwTlg*WI5w(<Hfj=|vCTDsz+gr9bEX-7*Q|JbUTS2gCky)fsgr_`j36x6FCUh9j25obiv#DD=aeEs}"
    "soEaiU`Jj)&S5ax5{#Vnc4qA=zx<=HZcGtOq=$Gk{+pK6=LlQ>)~lQd^udb{+}jH?HJvRtVjb1$1m;Z*(pSC"
    "+(+c$Ov$R-{oBPWz&1CJOv?b@kzb2Vu(n{3kCCBAz<zLUwOjgi+uoOCIwS`f&Nv`3E*;pC&P5j7>Uzk>$yPr"
    "<2ZzO*&_VznXNM@@1Ox7#_jPUoYQk@0Mv26n>t;yK8y~|MzLq$?gjJ;PU5RcYZs5|L*MUc+zp$q!W7!POpCR"
    ";xLLldYr`W+=~NvvkSjdo}>X!U0fJ=`ufpnZsFvo^ON(l<EztZ8z=L`Pa-epwN{4qW)9n$xxBhK{{4s3i;I6"
    "&+qukB?_tW*bX9N2+RlM&=f;`DkFY9}IN&q3;92I~@~oT9lVxv2Wm29OX>3lh`jeI-O(M~iY?bAFp;oCZ;wN"
    "#=sr5<c_cY8o^H?^A{j*5YIE-&umM8ulo2DV3v00Kb*b99IJJ+o0*Nn3yj#ez6bCwm$Wfby2d_D90JnOUf-h"
    "wl^LD-Z>$pe%QQzqtu(V4SV|2AQ9lCx!z`g1Si%!^m~9Qv1}%#XZq!ScD6<B7wX_{T8Iv%WFr$~u<wQ)IBy9"
    "dkn9g!O}Bv80K2SeB%@d&gJV_$n=UPpwwDQcG_YC0-!PBSjcbI)6OycyH3Ftem!7lMWVzM)42FcqLXJ-x@o?"
    "D3;|Z_ZCp-M=#2_@h<0!B~LvdJ5XR?*6)3E^W=`l*uw6?q4755UY5W<aQSSOq++MUNzB#EYCARMUf|+Z=>~~"
    "kEO?yDQHL{Jzc35qEcarcs}c?iLSL1`8S=u6v){a^;HPPtq+M8-#WLb~n8b|7_dH6LoGpqhXH$-AKK&Q>^GT"
    ";imH6`{^m)d{Og7TL<vCPCFI2&LuS5xlMPZirV0T44mQQ=^zs79X-cz^~wB7<$4pSaf`o_kxlh=cr_P&Vu!H"
    "td&#8_rw9A<NuR%p_}zQI07mXprLlNp2own~+;I7t^?6s=Tm#g0?m6>W>haArO|HqNCer42wl5w+w=61lz?M"
    "X(_C*D0SBnHQBOr(rlATSi_O%US1vpjHoF2G=w9b8Y&=S+y0U0%G>OEDNFY`MN1o1&y^%B3@tcoPW&OxYDYk"
    "GmVo6pE)ecQ=A!i2~YZ2o`%crni+xfxme~aF%h*c2@(VqmWLNfwTq;B%}vu?UW;4s#$oWcy4r3Ww(vf72lTj"
    "d!`-l*O6gv&tQ78+B@1IDvz%z?wa!o6IP8wE#*w#}1|D0ESlNqSjS?dVCrFW)No`0`^1xFzrWwT_T~Wm{O{N"
    "iF0O`2a>0q)5^9;U|_ZORV*!S24y76ypBdHCcEELWvbHV<Xea^f%(3_FnrR@^aP}J~M^Yaug$uMT!NoN^(J`"
    "ePJIi*XIt~~Kt-kjp0xMu_N^g*vx)!6jNyr|{;^}Xei#{smSWLY?kI4fdk9T%5*J{*{lm_)46+?ui;ZF|PvG"
    "Mgv48wR%RDfr`lHurW94!i3v?>>+HB;eg%uRrG>gYcGTIqU)18uG_RJF<Nsg~@W&4>(`KA5^GvwRdD`xG#;K"
    "i@$p?$<ri29$K*t@}+U5-iQ1_>}2Q_+?VR+Bu+ec^^S(r`}|t(#SPFmy(6|S8>k5sVgB5UZ-Jo;W1r(*lJW+"
    "SS#0o)dgQTL%Ck9<pDH1=F$v9_QI>mokrCVR{`B<irTgRY$<?pNXYR%6Z|_cjHyFH0CrG^+PC6cX{mm@$Zn2"
    "&^X)xxZ!c{4nh%2d)NZUtZ-TcVQGIjyi7ndSf*0l&^a^h*B>6I%cv77N|=D?IXY_@b5Y*&XZvRjAExfk#>b6"
    "CtDiZoiYe6mf3_bCtFup*-s^pkiN-WDkj7>@&_IE=~)4nS_gD7@!*QFmd>%J%-oj#-xYcRb%okiUdGFb@|zD"
    "RKwtlgIKT_agS-MbVBQfhcfWc&QiXocEcOr1xZXVVonW<QaDF4n#qq9C_-+849(~e>No<!!H&^<mKT#XWlKa"
    "cybfb9vn+yx2rGcJkOV-moKBl_o8`{<)h(ne{cWg{qUtf_wtwJWu6%uEAVn}tkD#l21P@(f?{dLpC+9Xy5e`"
    "PR!bpfctCu8oTry4zjvezAr1g44Rq1GTiryxULW3cm7YP&_jqs6V_v4z491ptz*9#r`f&WZhX?M#-m8-L03J"
    "8>9(wHW3|}3#Zz-kB;2(X?197?dw4N_AHeKbMMd2dMt1Ji3ArQL^+oLUZ{^O5l@7|xf7pIpW&fj028j1+Xv5"
    "#71-3l{2Qk-y2<W;Qes9H-^YSx%R$5zr1o`vQD?jg3z@?x6d&7CPF0KOTHYN9RGHBCTWl=7{lV)|6+GpGe>+"
    "+p4{x#tYH7NeRzd!NunpRu`j4-N8|5f9L>4OEwy9~GCO>n6M~R@^A6uYj+Ts;vHTdW!`91HS17JoD3V2?t>8"
    "x<TT*ZcqQT?*)PDiEp}-&dv@{9^gobVHo48rrgcbf~&37^sNx89U!k>5#^dFP9J*dEiiiGd;AA#$Rsm}w;CG"
    "@W&l)V@<kG?#B;c2=^no+-(4$-{)T2F>(g(#2h){yyyP9o##J&42NS=A=Zr@gXVsIUol@iz@SwLb4B8;z>@I"
    "V5eu<~r(N4J5J25x>W2n<n_`hN8Wc`8~Q0PCKd11tZKD*4lG-u~g<SjhnWQ!;wPLxL|n32ZgjYZ>w_cWGODp"
    "!{AyjVK$5G3n7U6tDz`H&3nfk|v^WX0NtxG~P&{dUTJIe&Y4#?&-+EOR~Sh}l-f*pKi2@71ptr{&JWDrv^A7"
    "4vaZ(pB|Ja344N)QbN6?vbJmi@ZTyS>Z);L8a_Af{k3OP7u&=##|Q!1lMI_kYimJ4yNlWF_zBhB<?tCbltl)"
    "UH9&Y<*Oxdf4~ZY*q*WEo~N{upw$cGodsVc>8e@3ZK$(JeJy^ZMGUV{nZ5LK;Nzth`-7LyCsNV9Tm@d7hrWF"
    "IgO_pKbPhW~R_1N!=MMWZ<WZnL%Alh>yp3Jr5^%q{co~Umu974=@uH~V1JqIkVYc*ge=fgyD}IAg{q!UEiyS"
    "x=s7ceN<GA{15xX#|ZkDHA7*E%AXJN!$x7UXY9~TaFQ)p<Ibb7ts^`rxJAW`{n@$TgG67NJ)bq?CPfm2s^kJ"
    "#QoJ=gWqBewhMz~E8eCz3*ET{WAMoYPDZlgP%}6nFW@JoUzt&MZl%VE{W5)DnJ9vX$Pz2a<iuR<stCWOY{nY"
    "Yi6%E?svK#%{#p+k8GAqC!Fa8Vrou-@<__>#jB?8~CGuZIiD?z+tLZZu9ZLVG)N~2UXF}#Z7JqfJ$*aW}njJ"
    "K`9?b6?rgC9(pK*r^$m|3-D<0UGfF0px{CMgEbs}P!M0Kl<O=EIF49w@lwV>`K9X`)z!2+$X|W;U79?gJp=h"
    "WR6dhphlXxZ?a&KNx$|h&2~*-aYF0?gBL~?JbG`ufw*2PC=($i_bb&}$G!jU+v%5m7L}Z;6)dWr4&wM4Pt%&"
    "d9<Uy5JRNXF$3W1PpKeEZ%9Vq)rOkq^`s~wazY_^w1D_U8mh{N3VlPq_A*w9dv$icPj!HvnGl{1EyQEj?l_Q"
    "z_5%38`2wBVqy+;+X|iy{vm>};ynx+cbER^J5SYzAB^5@k|2EF%tVKC(+tpRr_r6iE(>YPD5rv?seO#&X2Hw"
    "Tu1TcUWU>O=JhvK9CP;B~kb-SI(T0@2JDqmPNj)Z!1j;^7i8R$1BsWCD)y9W<zypUQXR=#bD)dkt|qt-v#$b"
    "vJ-bYa&Q69dAa~`3vObuTg0eBp4DXtjpA3{V7oe_=4{;W@|@Xw?a4j<A}f`i(0!O}V7+`)Vg7pgmf@t=yItK"
    "q)xL{LpVoamoUYWjR^Og@kzb%h=g~1H#emv6{10Fd^F+*kpS|b6j7goXWiaMNS$T;~SjKT<NbV)a)5UG(FfR"
    "fPa4?>w$-jA=@f=_Dft^8O55HiNv3Rf7AT-K7t6#gCltKpNF%|2F@qKc~yOLw;hQU}U>3|7=mdBMn&7??}jX"
    "yp148Hxkt6$@i(#Z?&#=OL#y6Z{j9!S-sb0ewUZIT4l-jy?H_EU%ls-Na2&^y`&me9%1p|rgj<I?n0(7l@OT"
    "MDt>#yy{U!L#-oy0Nb7Q_tG-XBKz%Y~s$|2&rE35c7aVNwTE#2ltP+LXD(UfMsBt^H6YSc+V5lyaBjpTLI|o"
    "^!Vca>BUA;Mar`zy64><=rmwuu7@}5B~W|rr7=(Cz(LfQ#Al$V%N(fJ#leo5XzY5@p_h0(%7?NfiYK7f7D5>"
    "QgZH>mhbmF2rYw#X;m@lgjUBc7jB3>TN;OJ+>)c4$%e^{3Kf5%(2Bw;*3pI9nd3pQ~6syE#X;>Xvp)(cD-%m"
    "VRA}WS0Qw|eeVN~$AWPSEym}WVtM@kB1D)lCvACr`E&z~z!9(SI&W&h3w4ppEXK`OyNdZ4u@5R5!()yf5ihO"
    "QJP&gd<S*`UuZi^alA!+(>J0I7J&vm(kf^UAE$e$4K9Iz`QepDY$U4#*-9Bs{}*@%;Fl`6>5u(0D-g)R}yqC"
    "dKVsHCXF8&LJvDmua$0GA}~&N5i!z)s_uwD&RhiEBc{j9f=(k#vdlu@G^(X8fL~@B!$6FKv!-|vk*`b+{`1X"
    "<c9wp7x_E^Q%p%)a<F4S&(b7fvnY9Z0~54jK_c(~nS>n%+z&IbT4<C<3IeVh0)y!agA!ckhDZR9`}mZNS(wB"
    "9jz#o3B(y@*A#q3!6Rf~@PHFf7c|9reWswV9k=~f8`_$Nrn13XbS6{X+3kW^sWu7eG<-*Kl6(dS|qMIlr@5("
    ">Uy&@wh2rc4!Krtz%1k5UcbHe7k&EN+`hPsGc8QYkVo#AHeh||+~#<FH}bfTTQ?q9WBe}z(*Vmfy2ttJc<_i"
    "T5%z37t|6iZz_=W~XVec7Q@@tpc-_BR<#uR$WvR#%X-5xW;gqeEe{(X2)L_~?x=pza<H21Ip%%AnF)lD+<<)"
    "2ky9X%hH7D1v<6CHmd76C5wj9#GH*o_!u>^2odqHr!jVM0ycIoxPZdItNDB%ac?Q9JvBnX}y!1_j=L+Eg@*i"
    "=+~@>(VjEu+%y4bB=9VZ#pTmiKuDt;UWZ}af-G~`wdB@?Mk<+RJiW*3?pg!QG&gDLo5a&=d8OPCb`07yU}HG"
    "^B+kRQ;N?@4AlO*9gq9VmYp9}4!H>k)k~^SaA>FCK7TXCb!a$8maCNYfvOzUbZb&i@^3%o<8d;=8Hl{9V5|L"
    "R-PO-uP)y_>9E7HU-`MW%E+CxK?qCGE_X!?w4a)p=;p>CAj!g1<|HrwIfci3=9nsn68>8H6{BYcMr8y;9n-~"
    "10TmNtEe;0)~Ox~A8eTZxq!GbJx{2|-=jPi-bqji2gP?r>3Wpfne3%u4Y947a7uVbZzj8t;VtMQtM>2qb->>"
    "_dHXw8|YOg$jqT580Tcr*5xeE=1ZwDm_P$8Zx#3$x_csDrT#I$hKOYEJK23C;X-bl#Z}0L-*W=<I77jgNmwz"
    "4fMz3cV{|E6=%O(^O`+Vli({v)L~%^W{L4`&pew0vXpAGLR9GF{Fe`Br&p(MCmpgKYaJVIstF+0@DioCQ?O#"
    "wA>2eauy4~1?2p)zrvMCrxdT{M3do!=i5=|ZEJ<&9F2oxz7r42M%?ji~k-&e<xAp$9X$!<J0QZC7zk&TU<ue"
    "Y{lDkQI>&4-}@h;E$cvkyp?Udi0liSkeJXs3iKI1%a!Jt7<TDRQz4R%xfFKHCYj+5S?;RA`cam6e#H5D-d^="
    "NV(5VXZe@USfSV%ppdRqI2N-l;0VWyE1T{-fXpcLe^up$b5MO%e1ejKAXd#unyM1EL9n=i)BXML~<aw$M$oN"
    "?wz+QiGMKcQa5lxboPsu@ZDPmgc^@d$8EqY?X%@)?VQe?A<WUr1C}G4pR=WuX@_t$8Xo`hSw|Vc_g*yUvb!)"
    "Ub=dV$igIDb>UQnAIHLMNpOxG;5EGrzr*4NHYlI7F_M_Bdf!7uBOQe7k0=B;I}PkKWn*=Gx==w+-@Jmtbc3m"
    "}E9iW8gQ+8Is&r*(HXu+xWp~8I3%|91!+J^QJNW9Q*hDXfhTSlj_WdZyc(=D<KU1E2VT`w}Mu=+ND?89rK0d"
    "fkS;m1FidujJPKwk=K?3ivYG3P+q2n1`MJX5jj=%<~`Xo#K^hxxffj9x}UGp^V8q;X@t-KbobDPi8mffF$!%"
    "|Lv=kt_vHZ7uv=b5Fydq#f+=@&I7QLvMvZrQ9~ssg<VmK|lBhZrdg-X)?QBzik=)4-JK?k#x1U@80#3co#nF"
    "T7Jxee7j-S;Or2&ARX!Y*|X<PzjvfSYD&G@7mOzeWW@yL!WLv7yEhAv27ZO$&lTk6)weGOEr~c-0W+)`+iK)"
    "+a%9<EadDku5ex>8nd;pEiKi@3uGgSW7_0>j`m)Fab2FCot|7VAN0W8?|SUVi}PQY$%ITgz5a~n{yd3!7dAb"
    "d`(b@B=Mq&fl<H}qPB{)kpsnmXI_3|pJ9<;)wy~Qe+hOo772&>Fd;=n7x|5EjFKI=1*Tp>Z4Z&b5+^J5w4U)"
    "v~!vw^M?V)(mKBiv^x*S>Ifz2z+V0!4#Rg@^k5tRJ|BbN{%K{GZqw_te=%VzuHk#+d87uCH?_m-{2YQgV8RS"
    "0NW_I+b*nyOFi8<Qf!KD5xojRa~7k0?$bv4?eL6vz#+yPMDzTP+(<s5U~kU6o2~*A~Q-;6X282=SH6L`-^lo"
    "zzA6AI{ECeg-L8DHaUXflhHwgy5Be4N|yuEJ!}7(;NBGTSJ{N+CrbOR;TbZy+RXL_b1mk9N6>?2LO#lCu*T`"
    "=OhK((_esS!oUj>e;`;wFq8T`*euY<UqQRjUb~nZuuO<1PL3}@5$Ahx5C&e(U7YN!oW?J%wGbp<E|!~tKbue"
    "rOO1xqgAsS@NIY~HMXq$#ln-GOs5*LINmKDntlms(jXt5^{V-<QP2VM7_0bbqA`6i;`5`MWRCg%cT;|8^mHn"
    "Az{-?`(BxK9JmwaCZB1Ka$>EI7ROm~oW?A#Cgd9sM~u_^Kb9ASAvvD?(3O+xn3X?4nH3E&bG>B5^69RLgh05"
    "fwvyfFdGa9A^8ztpB$Mn0e_dNs=+p7F7wfn8(sIz)NKL}Zu`n3^%t(d*G()x0LUAnkkfvuCwLR%viqxD6L#("
    "oKs2F_>Dc@v1F^p|%#d2Q_{YTgu4k8DoDf1*B7!OLJ7p#j{E06FkG#A4h{pM;NIoVnR8Yj@4MH!CkE0cpLFc"
    "R-Wn8q*HD*1d@vde6+Jq22-duxmXhn-EeXG<FA*%EA^_PO`Y-5HiJdXE$heWRL4N{P;k%i3IuV78PADHGbQw"
    "`HF}kk=~>ik)-!oMxRKlFNpj*d3#iL9SuFEr@ov~8h6FqgTU10;uT2m1$%HpOTJ=ISb+<(rcqq_FQHfezuha"
    "P@%ranU<*6e`BLC<|MZhy!8Jm1!k;2=@RhO$b8cvBm$)2X(TK{!}v@9uix3^YqR+8o>g3ngM=<|qy!X<tDRJ"
    "|H~U(>AByZJ%FMX`tU*=LEPjieQM6phauRbfSsPZM3mOu|aZSsSy#*OOBZZF1`2hy_K8;P_>V7q@0t%y^bjf"
    "UE*7<Ir^#I8*9SSP}a?1@&gW+E!>yvTyS1{{;EfZmcKT=_*y1Fy50E43%1-jCM;1%~z0)?XKcH;<u@{Z~^rZ"
    "^GJD^i|3Bq>~0cshb@x`<?Ig^=f8ZoV&mt6o|m$Fv4p+TC3oN%xT`U|F~-B)y|#Fi^c`XXM@;+L5}48wNM+K"
    "2p3;jB)q$-7K@2l@N@1}Op!isKi=BJ9{nK0p(h(G<jGv>&y8M`vn=H(!CHSY||1J*l4~eAmo{tv{u0;+?&@j"
    "Gjh11ha)dBV5IUCComG{YPmT@J*8Lu)@qSv0oo!|ma-i6%(x(0#q)Q}wjp_ZtQd=AcEu<3hM04uzri{X~%zl"
    "siaY$y)VPbud?ZQ%8bPwSw|zX>=V5P5#jW9IqbV;%7TVu!)Ucmk$i*W%F%&`}Zi_VY}rF9R-}+2QuLN)Y<=T"
    "4qk~X|Il7sZ646$c^({cR9Esrw0`k(g)cMeK-~g4scqK-^+?cN!H+EE>Ds;ZbyRw`=0awa!h(aIM{um7lr>8"
    "QjA0nHOV&?4T}giX%%Tke<qDds}7(h|F_Ei>*RhV^NaYBBvsUP7tPv&u7xYuT%S_a2t2Eb!NgkzQ!Tt;5xb?"
    "ePTjBu=xj52wn5qwRrUo7SU%WT_^Qi36??Q}6TO|f9H>2_7=WTHW2vS!`HZe1iKve2^C~$HAiJuTIy+*{?V#"
    "sWJYSzp3Giivyl1sG%J`o6jVz*hVbK{<_J%DL*n;nB(_kZ^)QISMRkn6grx^dU<Qlryg2NgP_9Sa{bad2{d_"
    "i|O80tTEd-}PjsEG-Ry_!yCxV!hL-&_IO9}=;Trb+IUU)B{AaB+k|h8_;PHWin+qNJ22A_<miH~84l(39*&9"
    "1-!cBgq<#hP!(@bvN|(>d0@VwwD6oj$$mgQt1WLeuGx;tqPA5#<_P_>i05;PNv4)G{Y91)-=gW25!u*2M!w!"
    "_HU|dRtc&D@dXAq^`JN+#6W}M!xHgNs6xWP1}<X1|8#nB%IN5iUtF9Xzy0S6c7E~p^n(5HPjbHHfs-Z=z-~0"
    "Ep>=mGbX+!39ZS%#C|rl5$q3^x07X`P(x@6MMkC{X5(jQM{Pu+fR4_+Hg6veS0>s1N!{$y7EuAc@7RY8RMq("
    "C)J{FRP1ABoxNmbBm8D4!aoAgjzcIAZQ+&LE9Q+))QcF3x1tmvScb&k~Q?bh8krJ|lTaE>`x`oKC`t48lc#e"
    "F9^8P{Rgq}mpy9<;?403q&DsgoL!ZCT0eR{Bt{<dTN`#9BUw)*TmlvhW}>53nI|uw~QQ9RibvvPb125k)=`H"
    "QRuINphw1e0D_SBeYXfB>ejR=lAEozqbU*xjcPei-J^MLp~3ed5bzZwo+76xeNkqo(Xz-7skQ(h3a)*a2hy8"
    "FLXM(Rsns1NKOc*$m2kDRbaU3$DdFyu2Ac8X2)z;RY5mzVX40UN~jV|OSQ8jnHFQSSP>sa9Oq~`#z+xtl@ux"
    "HC6I*(DCz?6fiQTKRKNnJF;qf@q^jGvqiK>Liv)VEF|ehM?1u6<%c~uMYo5O;O=EBzuhf9I>_96L$I<5maDk"
    "+-0hOqL1?z1yI@Rkt@aF*W2#p9?ek7A?hu>7qK_WmdX@D%u8cnNQp^TLqlFd0d*(;=<j476tu*)6Hs02w&(r"
    "|$nbR)wij+hdlOpi_2TNY<6v-jFzDf{bbTTa`Da=Xgd{t)SdjaicQd3+zHNi4k8FVBBFy*PdMkN58JKTh9Yx"
    "mTxWr~kM({>8mI|M~PivH#^V`9d06Tg~~0(~IM)^NUMWLqA(a5Tv-{RFkSu-$W7>5XZ5ZewYQ}ZJ2j^c#?q#"
    ";&X_BJ|H&?_z4#qb*tAnZ`IgzJjWFTBH^bTIV3M4p*zV!gc!udf~TRcZsC%WK!Zqf!~|5AU4@_qE>Rx0E{;*"
    "WvWu140BwWQw1}E;T<)Dlv3L{lRNgV~`GeH=w91__8%~f};w8U_xOZhelI)`C%1hG_0y_jnDgz8nlOl%O7J6"
    "7ZGk~Jn3zW$cs1+Dv0a75L2GBX?1&c7xDd_;j6bvD`3P5YK4E=j$Z4hi80whaN7hcbBQM0SL9vvBaA(?!^5k"
    "n`@aicGz)$)Q|x!15Rn+bw~Q5pX?JAVJRDi)eimh3_zJA3!byDPRkFfPchrVT*<0N@wJ)#oS;7cm`h<_u+CR"
    "*AeJ7)TnqPIr}b0TNL~e73tAkbqKJtZPDw{L{1z$ja>gto0}!(h=s*>-*9!P*ya(Riufs_k5ak=q5MJfUK@F"
    "9Elbmzbv$02oWK#ZXp+CTky5-?hbTPt;CM;IyuPbxcbb#Bc>_$?rM!d{8MRu65sb~yIjn$8I<>0_k+CIR&Hq"
    "%tpphegYkdESr9QVX;E*S-WtJ05M5PBTdPXMXlpJF8fpoB$*hSqk|LibA6*L0Qo?A;Nwt2J`H8d#OB(Mdah9"
    "hAsgTkle#62zg9LC$WL02&kpB~L%ObsK&3l0xLTtmjO+{UtR)I&Gq=B)$x{?qksWwhs-XiHjELTlKkMAw}W~"
    "?M*%a*fLFQO4uK2yGv)~-pQkr@6L#~Zz};f-FEcq2+I^shvvT!Ei_8D<H<1L-c4adhE8Jj_E(N9KW_Xn<)?o"
    "(F7{6;r_*l#f=K1&bsfgo`f>OxmmlmCH~=3?;D7Hu;)zVu8LZ#K*Fd%=co4J%BZCE3i)nj%P*0A)=sv=Rn#x"
    "@#HJ|yvR6FE@>3YU=^b;8R+c!BAf}pP-A?vO#_>a%fM!Mww@=1*!CgN+QOh&LZEFPr>vYTd2xSET(X`u=or="
    "C`%mb;ghuB{dN1wDtg3o+Q3;Bbw3PYtFbX!(ef=e)qC9wayv@3dvU1Ho>Uj|VhZdLgsfqS4MR&B~w+ilm2{D"
    "EosRJYza9#G{KiwYNVS7iKlDrJ)Sbp%mQAA!MR!4jhGsX;+piL`*3Gv{o`p4=oI1M>)u8~Wot86?RSVx1KEt"
    "vMh#exSRKmu?hm5hE}?yM{PfBo?G`0A9IxLt@4e0h4smL8-?9Gektgzg^hQ<Qt+<Y8<^LaaL~bn9L{E!GD#t"
    "ihyL1Q1}#tkEvi<TV6t5H<-I4J?FMUTEQ8$<*fT@zthwU`(yhl;G~P&92q7CO+zwMwHE#1`Sf?y}8jUHA4tQ"
    "=GhJGUH?j^eGy<Jcu`YBHyM|#AqKpNXOMAt5u&9!En-pvV*AJeh!WPgE4ETFQG426m?sg(#H>Q^kzl_0X<ge"
    "P>MTFFE_1C2?uT$rwNI)XqEUqij(@G7otb2eDv*%2xT94>U}MNpy`ar~N(RJg_RrB~q-3|d723EVL1EuB=zM"
    "kWq97ou6k(5dqU50(oyHhpuZ}?hslxQ3)6NwZ(oU5uT<r^L4uTE$1{HqtzEGdH?1|=;y=ej{7(#B-T0SRD3}"
    "ZNVpuz@PD6&3{UrV&nT`_*4jp2oDyEHAwm+$6qn{IA*^Lgol*VOM-nRs2mr#0XY%@V|F7>PT+8j+AwN0#03l"
    "~XEnG^3ixwpQcV?)VByeuB+(^^-V~|Gd09{pI5P{EB{G{qo`My9*Jw?D+jZ#9t@w@!8p=^JoQRv=~se!3k8q"
    "<-)3(|1f<g{ucx6li9jj=<MtlDmboYV71Kq-!P^+InYW`0|=XD!OJM#N}QA#rDWRdkw}0HvX55fq5v#WnE5J"
    "MrnI1<u99{KR_c2(Lt8JU<=`s(n(o|-R;RJ1X6k>yWYFfIuCrAp_9>{xU~zU9B0`+03lX8v%0H2G(Vrz80R$"
    "8HyH`z1#^d|3wtBVXUIFB#);s&|!)d(`Pt!Gpz_N$=e`+6lY3|1S!4)vPYN1B_)Ox|F9*-d4UH}q{6hszDoa"
    "9Ly`rRJ;o^_7~me|y<do={83dgEHCyivS+W%EHN{DTmBDA&k8VnO<Gn+Dl6;q-k0#ggak<FrD%R^1I5SdNV@"
    "HULcFDMs=+UO(g-XHD0u+^fI$@*Ie+mt+Ls_*hjCZbgaR8%PR$C16S6$6f1cUXfg;(Xa1^as0@PDl_GXw^$N"
    "qqRdCkKiy{^a{wMim5`ys04f-D2}2GI*7)ysVi<5*Y#F3NdZW{iObb*3V6E>2-(;)Qn6UsE2WY~T)!|J7YtD"
    ")99h7S8`5+L>j&x;9b4Lv%4lw^%hjmNZD77$Ic)gqz+k@2#lM!8EwRv+`FSC{;j=tkfTP-IGKqIsErIVVk@("
    "~P8NTWt3^2FHWD;M2QgNg|e9^iGBy0F<sfH7v+cgZLA#1Cfi3A05Clz0oU8#L<=bE;WMvc8h$wqMZuVt#*=|"
    ">G~5C3OkSuJz9hF7)w6^LoZ4URK5sMPw!QJh;KIrWoa!)|UFWX8UpjdtQSHe+*7DA{ww*2~w<y4pzRB-RS*Y"
    "?sZ})>mr1^-3+?X7<m)S1s~ZJa9$42#Y86cGE(RR_nXIZaTKB$36`eGP$Ov$CmeP?<{utma|>cEyrF|a{&g2"
    "woMr=S&%Z~qU2g(U2AM)$r?L?`=xx$SP$gQD`KQ6U6D_1d{C)U#D~&YX6c0daS0IMmTsaGZSR@;V-r}VD~hO"
    "w%waxINFX&1>(<}M)@>1YF0al%d_c!V%)|<vt?Na8N1qf0#PcKW#nGy&DmH=WUlg|R+6G&AJ)%^N%v&zg<lc"
    ")|R;2ggJ<p^+nj@rvWSS-5)xtc?(*=2deJ@78Z5~@N3kF1iB$~bFKqd%hUJaZeo-DlByH)W`o(aI9ywx|u9!"
    "NP{e0?c?pF|<ZwCi98;`);xgZVd%Z$Bha2yt9!g&^=M-O`RoJ;}ia2I+GD6A4G)fB(C>0AN#=Q{jqrCXro*`"
    "0#?2BZD_8=@%cP--&!8YpH~B+(`FO5kD$gh$?bzXJPT_*=)5fAMWhPDXYB%^#wXhSw4cvwbVASwjgJ#UOm1s"
    "wD8*Bbv?-2)8F3zdUj@(si;BBG7@BOIx@E^By<ZT5fn<n>IJJl{}^VFmP(kOo<Zx-EH(=$x7n~Yedk3HLp3s"
    "%hA$|lI^#1hj4=2cDSiw=WEJu#s2u<9q%+U+<>=+h;qI&cpg-sj38_qdf&aP~rl8lOqoKUrD_lYAEnmOkW$_"
    "0u%&oWPmN$6Y?cLaxitoT@V72TLL_+0UH#$x4WYwZ~A7HSkRjUS}6jx~JK1s`WjJLHckHJt-iYsDpUxAzo(W"
    "*?HqAoRDM%mla(ZwKNXdti?I4~4+<ts-kq=iO#9{pcKq}qDo1+j5okF2PIrVvj`!mDTo;R-Pk?y~+J%wAUst"
    "53ovsNcMZ7&*AH@JsS>u$h{25AunO74avecfK+rw8{`eL3qovyqn0}6g`^)QU)4%f&zf*w+p~kEVJr_)7wB7"
    "P!WNZr|h_L^;>#qM!dwOeC#<)Lw}8TG5Mpam3l1S4F-dD4GuP3@5zSk+EqFhv;PTZqYLCD3+gbkD&QpPu&<Y"
    "jnL=Dri<FNkZ^V{_{r8Y2bSNlXh6cF7F^`vm|Kdmuv#}2Q`JNpPHW&4rO|)i~SwcXP<hI2CzxW(Azf_ag^1V"
    "2eqGYe7sGN0)y=yLMu19mZriF5zYo(g!&pK}{^R>Y|Rd%ICnx<({C*4+ud}lLTZ82pVhF0G8a%<oH7aR<bfk"
    "q(F)eeH)ZbmTVwwDCIynISBP+r-_F6*t?34dcJgvbI;5Fw*1jQ=H20%%|*>H0PWI6KX9AoyRx{(E5#h=22v-"
    "4cM%1VU)I*lMUUahU-^L2T{71?D#z4{A4@oUz_o<45y~3cgaz%9mS1wPSjJ#RJ_)=bm<0|Hh6*^4EYZ=c_CP"
    "2(`$pE?|S;Tn!*oK*7bFgA+3D;mlib+%C5Si}*6T^*JCjv}I@n8*!VjTDeVQ(W77sBer56mGv4lG{J^!WWyn"
    "Awnj?;T(OLJH;D+U39&s+e>)e!&xW6CCzB(R=vU(RYF079Fs`u;3ES6TBRk>+tSGwB@rZk{wn`aEohj2*R8Y"
    "7^q>yL04O(lik8m@FtC86fJGtTQhCJ3<a<eIGwFO(%dpfmMgO=kHzkAQbpZ}Tr@@F`afHY|BWs+0i%_ibn!w"
    "zpqUt5I4Vo^jMhN%F(G>+%1iHa>^P;Q3xY&xG%Y!d;f^??4V<R8+1kx55x%&w{G0uF$J4qI-2MK|I7I?zD_)"
    "WC9ksAy+WLo04q5d%Z0`2H;-U0HDvBAgVYB&uBxiWoHA4h@$B)>y}49iN<>e!%dj*6*IsKGZ9kt8qlh9aTk}"
    "ww-)E^J=Hb1G<+rtahn0BZ}fh)^+#l%4Q8|4{c~bd-w@~1ZkUyiL-vJ0n+40!<(MzT?uu(s)SqcB9uSJg5`Q"
    "9u#ri~9iVjiI-<S}Xwg1_YA|Eg9SE#h1-o5F+?Axhgf>F_$q-TJ8HQLFs7geVA(oH8T5kZTOqa(}0D}6B;IH"
    "ZtrU<FF6RcLhk#`hsAn1Sr6UuNx@#F8r(AR=6bWMlVYj#VwO=)eHuA6}@H`2{A%J_hOn5s-~;~)pr#SO3lM>"
    "cH0(TI5vk4XgR#igKudKf((w(By9GcE#F1spadkISfhbyUDNAix&{0(=Gl;Q0*y+CFgQG}Mj%y4hcrk``@|C"
    "qTbpY{mbFVFJt>^(%q{RPWMcxq~3Q9eJ>g3ci5|J|9G&6G=rQf-u19rrb{!oQb7-PBekBxBe%>3M4xOws1*i"
    "HgY=o4A!gE+6`4;(HiF^EYHLxf_ueg;DTl%`sW2095n$9E+nu(vW8or3XW84@2Yd49*Xa?VFebh_%rJSj;!o"
    "sg_|U6_Y<_q+kg?Y_;|ey-Qz#k^gpPX{-uruW>F8=M+14X9<90=D9ngeY}Z@eN^%#K)+Kip1Kcigi0?4aW4("
    "$7F(s{uDJY-lbnQ>7gc;T|ZUNJUh9emf4fAcK(ydThl#+*-r`bGQ(!qzWcwwA@gK$_iiT;fVJaxc<9Yi4?GJ"
    "E+P(Mo_{@>m8yCh&ie63lR*QQ7{NGhVl;jh?*!o=<bMNf#QGM>1#6Q%zrAmRO17BQvP@N$iJF=uwO_$2NzK="
    "+~d-Aj$-PpLG%&&$p<(c24y%XQsA=(uNdq-r$O+O@-1KnV_$)qeM|G!k<HlG91{HD8s=>QYz*#U~WN`0+i1I"
    "N*J>l$daIF0RuC>z(CF*lVo`qMa%>L?dOXL8Q#WT^d<B${}DyYXU2;BtEp+qB(uVa_(^&jICvmR5y+>D+gPT"
    "#$;Pi$p1oFCToe~4cU`~$?MTXKi`Z<}l&bOF$eL!e5>mu2oHszfRm^pvBhn)KVnqQ17cL$7okW34k_WN?lF_"
    "$kw`^w;EW%9^>1m+yJQ~XZ$N*5d1F9K{)OH<Ll>wr6dc1wgTJ<cF(K>l$6X~Sk#(ExULzgm2d(txr8RGT&;m"
    "d;dd_uO-tB8!cev>u^Y4G~_Vky}?Rk;6<hB+tg&Ql{{FDbWcg27XlVbJZ_BVo(Q#KLN3ZAE2RcKEj+2G%mAG"
    "rcNVKC*rx<`jyHKdX_kn<8MB(<QFou6>5_+Eiz?q>5e5Ee%meI|&35?FoHfNiT;|={1G8&j>t|9{S*jyr8^K"
    "MmlRfT<A#Q5d=xs&gv%_`4)nTs~tsJ6&05j-etK5E}BKY1UmNEGZJI_l#fizx@@i0rbqSZADUH`YVNW^3!th"
    "zicMmNV}uC?Ya!;!6+S?i4@AtNj1e_z(^;9WLmRmdX)7l~?m`zT@x`sm-P%1PmI0Z#tnhu#;nwlMXPAA@v}$"
    "XK20n?8UA(G1h;Z%-6bsC;=pc~5SyW{6v4(T2velGH{C)DCKU~Pzs3R>Ux5!Q>(sF+XaTOf)-FJ5n@RvTl4T"
    "IF>ZVV_Tvr!1VbC_W&LWv|ssVoiJ{Hh%GorA1#sk=?`{^BkOQ@NB%o)RXfr^Ks$E#uW*mm64mZ>d1crf-+b9"
    "Q;L9uMLV;L&IrmT%V+9{Tn-nfL<yvm;(Lc6V$+96n^OlE?_fgm4F2;V(;DyBT(q{|0l(Qqsdqw#>^6@pv`<J"
    "?QNAkner@&?s>Oozi@X-zR+GB?!<;q8;y_p3F^xjag{PRUp-!0bG)`zyF|@?Y`M-W8k+J}FOy;$HbV$&58FU"
    "92ujwwNjax~?})E$@hNueil!trgXY#vb`2JZFiFc?DW-)%m^G<{&S%+_v{E;ui6nWF>~JrQRw9ahJt!O;LC8"
    "UG(O}`=0g~gAdZA1I^e^B2<L7s0XT7FRZ1c6(Yu07qM`rYn9kZ!=Wbe_M>P!=GRqK3CJedqUk38t=4a?_YMk"
    "(%tglEDSjBGh<rJ0xIJF_Hx@X~-%6QcgRzjjyZ63E}xYAPQ1hJDWl{lP&^`>dr@!tk*h`mvZif>i`SUS@1u`"
    "mNMO$&k%{El-s6oK2u?G!;#O?}XVv6tNzoN@Q1+(ratww<gz+w$H8&w?@?4%*{h|j&!iQ4y!Jb8g`|UlEY;6"
    "SYiU%&~VT<J~$#+o?U>-lfs{Ok#@St-S|f@%J?QxfSPH>!FDubAS<nZWlWAnqNy{z>ozvMSqZP#3ULuTX1)k"
    "hL~{)J>gKCj{YuqQBL>$JHC(N@q*$8T;Kt@Dc+;}2|0+=02Ghr9IdU|G+tatp=et#DNzPTU@!C~Tr|7*pS$+"
    "5R?6f*<LS1XsI{65G1BC?{LLh<`aTPM$D4IN6ua+mKtV7t;U7^8}TW(1~Iju@88E=z+3{hWFr3oA8ssypE@>"
    "Y$SFc%Xcn_reDNNf5yd5Gnq8?m9iV7ZmC5X5<~mTS`|sHfIXA%-#Kr4kvPMr_cmcWaDWXS*jF*+(1hYjcXf*"
    "0%WEX^nJAt;~?U=8KZ7uPIn&*8A){(ik_Bx~k2+X5Y{XI8<7l_-c@yr==(&@vOy2t0yTc4Yai!punzFG)Kr@"
    "NLoTVBvOqz%vKVQgepjkhnZNi^D|j!sngPOORff@8{i{i9`c(?TH!NrRU=)Xsz-UNk;j6lYIxJmQS>S`SN$T"
    "wm~qdavoK2{%=-cqmRuVsz_Nv|+!z6tWhzEXX5@tkI}$~iOkfPUJjm^cO=QJFHc}%Pt&PZCS?2zzL{F?`V69"
    "|ceP+Y`&}z6Jj@Ueb`v??NDvf96s0_}(oP3}N{0UMGOy%iCs1Ef);VJ)Qt9@x{e@^52`$SSbG+=x^OFRFVr="
    "AO``AU1a7YEbiqobyeUT<X`f2#MwhM5>K@Xl3{#tyqmlIX;XqK0oX8Vn9WpP=O`pC@s{N0_M)!tO#Y8u`_!Q"
    "Yb=`u*g51UtYaCJ9XclpZxmE>HDiPKbl4mrbPq-L_AXdSCo<^%vcmIFvtjdcN>F77c>$w%kT^FI_s+_`jgJD"
    "8D}QhfGBs<Vg;FW$gfKUdx%z$QasEk-+7oR{W0wQ*f2?)BnwDr0ls)Ru_M_Y{O`vk^&!Y84?+aQ<cT_cQ1ct"
    "M;9dryDtJ7DqB0*wF342MX@B@SRHnh^QKoao9cQ<G=-Gea=2=kAN<CX8=BTu1r@S!}ac1?Lc5*{?!(cq=1c_"
    "frxgChwSR>r}JIN(v<4=!Om55oR@8w>U+-enyC+@;uZd{cm=>b$OJ@~9VdbR%|>833dO4$XG-*QyPC+NPIBi"
    "Mn}0L(qR<~W?;RjaQfAW|^l87xW9jx$kbo=T1BpX*vxH%_m{A88+#Vb7%nlLuz{XCxIucPDHud0LS14E2kc7"
    "`m?vXLN`z_#Xw&@(T>nt(o;)7X%~M6<a(@iqz-&5aD^B9)qz1|M=6K5utX{>sz-v7CC~}%dQwO4+lj!*Ukpf"
    "X}5fCZcUmkh)G_~;$v^C$QQK%=!3o9K~B)fHT$*17{RXy;OAG&T}n)An8a-fOig2IC6DQo?5YBfWm;6lT{F^"
    "Yt!Y~INR^*eFI=NZ`8gX`?(uHb-_Azet7WiJ4?>nAVF0)1dh3<1bPmsaLq-PXLQsEgN0(iBDJy+Bq(Q^Q3q`"
    "67$fKR$uCqhbY!Fq3pk{&4xW;i;PL#vGGdV`emgg^Balw%U*l;UUNtW5584jS#zfgDF(6ZTYZ5#NE{4^Q=!+"
    "Mi;5pmOEB)O>oFmdpD!fTWKl!V-vu#JfMMz)w*aq1Nb*D^QuwcLaFX>}XwEEX=jsj?Rp4l5OwgQT`Z1NAunp"
    "Hh`{n^qfUL$ve={rO~JRp7zNPv<A+XUA8k<s&^P&STTicW*?k2?FU*JpmPP0N&71MTsF!au-7&f{Ng=o_XLW"
    "!8b1}3<wJCs;z5P@r^Ajs+iGyo+icZoFv;{PCiHot<q$O3F+16?|D<pRMgt`+%ki@k1i@2$}4VU_xd=$)W!A"
    "@dw+h#E`ELghC#kx{fZRv4`Ngac&I8`hA~8t4i`u#HVlzj=6z88h40x|L&987I@J_{m&fnl{&4<(-FLqn|Kr"
    "p+beW%`6e-kb8`W}oa&dfh@{_p(czjEkZw-<|8SbmqyoHoz7cLx2^WG&{7(l)BR~N_T$UZuvjjx&+bhx{>Q3"
    "OGgEs%P(QOOue=5a?P`BTs<)Y<@)N*Ij|IozBe$%WUr0!=C8B-vyVLuiw+B5Ga4CB1kV#>*mq3F-~RyS(W2t"
    "|__Qi<`-$27j4!>Tp9*d=w+P?fQk7rWYEi!?->5x*$E=^^5GTcyTi}=dtI!D8k^yh(N^!fQlb`J-vqQm<3bT"
    "=?am?F}MN|@vzPhh)p;a|Mu+Z+))OrU~$B)rIHR=@(d>l_;e-+AHZ+}^gz31DtF$=`7a;NPAfip^$<k$dL*Q"
    "mGV>xz6nE2z4R`lo;}v8EJ3Khp+qc{WCt@>_GDuQ$RG9!io>Wf)J*21{($e9sj>Hi<B(Yv3E!B1}d4T};HDX"
    "H-K$R7K<Wp62N5U{B@=!sR5v>?Yno-#u0EbOo;M&JTOsby?aQ2{oVCMs6@F*HreZ{Wc{c`G_z5C_emB5XZB{"
    "7vxu~bEU830LNT)6=6VgOoWq)L+5;oht_>R9Xl%9pC`vif=niiW#ZHT>>~aX1j**9$G9uMPyt$k4`<WSmW-5"
    "S=xXJQFZDE>K%y);G|@&%F#3>~_6;-R&(|L|FPcmGkGb8MsPa&XeUYlpA;x#X8Lmg5{~2ljF<NOGi!GMzw?s"
    "1ZyvJKRoZAOMS>XmAnLHOU*u**EUwN|8#Zr;TIxER(*S^`}#2|c%1(p=5vT5Vj7ROgtph!b*Et0%IBbc3p22"
    "3^FI5bSOF6W5372(-yv%NQCEM}tu9^fNxi~80T6k~({8V?KCFJyG7k9J7b1>we3^6z_=4#&QKBf)1gL?-s5~"
    "gn#NTw`xOsBN<E%d7W6Z+&=NWGWX>1=oKqOdY3Xbsq^ruzfS(_Tm3uB(lD@NO}NjeRKfX9N98Ev3$ZJ94A(!"
    "{H7gA`uZziz7KHua-3*>O!pDjYiMrh_$laL8jWr8Kti0^T=#LB6zlQ)XjtQ*1VUD|1r7ia~Jy@uc&|1CRICe"
    "kj>C4I1LGIggg8dj>9xL<O-4#5G0n)<X>>v9anEdx};jE0J_edgau1rl<yTS{3B%cr7oGuHY5{T9A#qc)eT?"
    "!2ChG4b5+`<FE@ih94_zC;~^r_yDE_F9M}fQNxxE_iCT3Pbm<gho~3}i+yqW|DB#7hrLnN@+oOTaBNB?gnUU"
    "xt;C6|cCQk+C38n}?J!Isv*U}|MIJ<*pq-RvSjO7VL<-}wzsh|OxR|lJxVsv`wavaU9Q@drU+9F5%)yk%(&n"
    "(0#AB^_?ezcl?(}!};`IOg`tG8Xb5u4=7z-$mKpllcBAMS$QrPNbyTO@R+n)3YlR|XG!3J_!#R#n-$Pe%$*`"
    "$L%Kzt5irC@<~?uUKg<5a*w#37D}dl7sJhN3~-88Lhk4UBCY9vkWa;kh+sdxh^;tPY*2F)F9T@k%B5(&e*Rk"
    "^-%X0axAWnMy7E28o*CplIP9HHk-@25obRjyg2#Y~00vogQ>#$vr^bD;k$kIt9S{kXF2Mib)Nw4yxQ$RY(ys"
    "@8~o^2Mm{+s)7UYvIIv}E2`9VthR>0Wv!zS=2=nY!K4Ip=AyHKIPeu1Qz;?oGzn)xCljiw1q#%eOR;Ks;Hp!"
    "#&0PH;s$;U69jb<j>PKtmYD-&ODMsup(<GU-ndFvuo2j4xnxH^fLc09G5vF3UlS@?>YlZaH&*t9l!6D|yq|-"
    "HB<vi>5di^>7C<3aZCWQWO)1{Pkh!a}fAQbV)wsq=K{G~^ptaysV$nSXyE+c_0?}R+3G!>(!(8MD7pr#sFmL"
    "O6oc|&EE`x6iS@GFSrd%9Tu0a!ptMHV1kWVb~aVDRWLQ?w;e5!d}Z6j3%PDt8o3J^!vQjsI1JZ;RMWDR$hfM"
    "Feh?oADzzi@jwwPtfzlTT|7R2sen&ohtwydAqy#aC=CcDd&JWDm#(eqESAil|N<yrzoS~EAGo6Ac8?6RF|g!"
    "U&)L`UVK}4x11Pq_DA7GVZPc1==B6CwE1J>oGaZw2L*y;ZUC4ILUx}RE?Jl0FO>;$dyvcX3i#{LD!E$dny98"
    "_^)#6yjqmw<)~u(KaD%1U7Hm=;)wa7$hoYAk*VBFbiEw9TmTP=eZd|!}@wj&(!wH7WFhR#Cgv&Vxxx`x_Dxr"
    "F}7{Rh<lg=k7__*^)RQqVKyp`J%&K+Kx(yzH*w>|cZ<~c&Tg8HlP^#C=M02^hQNoO*NMOW8XBMLUxq)*}c>h"
    "#jhb%HpD;eyuIr>&O!sE#AiYOeW#R!kZ~KGALkv~+1KLyI}p<_oCN|7}xKY2j%4*p<GBG!-<au=&)Su|{s^5"
    "e>4BF<{z?i<K|Gkf)~YBgM72#Ad?R6~?pqbF~8EPJ|_PVX?ctK1VOIZm;=kgMV%9<c#wGm?T$6Xo;KgB~Y@L"
    "(=_C>ondRLAlh-jA>;QybG{Te6Z(w&difSPvW0iYMfi^-1I>5F;bQSYGZp(GX6R^bCG(S|=;OISl<(mF3|J0"
    "4vP|jnru37iq>pHLRY|v-n|WN1u@qO8V3YLt^FU4~9R#y!XHh@qRvTNl>V#HubX`4pa7l&X%Ee}?jF6OgGQ)"
    "Y&tY5@*y?XcCam7H~s}E+keK3-gtYJyCil<(HuGJfjakqVpwV9lHL7SqkR^DWQ0k#S7t-*~v-c-TRVrQ)(Cv"
    "Q`LXT-NkLaAp8tMsl0zeMqcGev703q5Y>tkb%Xdz&pJp4>GjRIB|qNw!$dy)Bm$YPW?WwrpL6&+=#$6zxSRw"
    "_1_TFi|eu@$(!zM+u$dii9q8LD-ySlb)ZQoxXK{I6nE=Hmp-GMHsG-aCsni2`Qv%m0C8y{-!4xtp!5c2?Hmu"
    "zmLnvi-j<1f{S#rCrH2KLN%%`tvd9I(gSKJS&`&Vak{~@FZc`%bt$}hVu5&~WbWPSVpLfVnS^#s2uHof?M;X"
    "k(^$1LebLy+e!DakXJb_8G$hn)^zOIb7$W1EG1H%QUu<@FtjX{vFU|F)S7T>IeDjlu8@xKhEz||8ub;%rtK%"
    "~nyG0gJS}+UX3BWkFg@|H7T~+R@rIwQ%SgS48J9XX7h#O^@r0Zn%!c%Cy0&Xl3GP;e#tI7>_Z5u@|q>Y*ETp"
    "48@XWgk65NVYZNQ|(andU)?xC-qi@r^e<L+@%86DvG(>OIudvnBRhaN0L(v+sEiy;ds|g?_7<nM_mK<9fzG="
    "v*a70Vu~hYw$#vmDFfdU{-ccpr!`apgiJ`C^df+V)j{l*uRaEX?N23PKGY+^^97H<K8r4f5VuBrjQV`cVQfi"
    "Uo;7PFASEoDLRRfRRy~@W(O92LA@7daw!inRNutP=%XsC-dc4Cn;q5$3MZuWTWE8hLKI|6UzaqLrZVJVeS<1"
    "yMS36J^GwWw&43|$ZAR62MxXnA-Kua*(bar)Sz!1cFZJhw>;rg}V#@lk1EQ$3DoAg9E%;U4z|iDn|DO+cU)7"
    "X9n_E6;IJ0@O+}uY$;1RzypoD@}yo*75B~uQI9-nwQhf9@uEmiPXgVf#pQ6%bb*)0d1!nX;4j|=Y90^tjcrx"
    "teN$+7S;%uH7gJ}d7>i16v9xA)h(k~c7-vZ~j6;^@KPYv1n6Z-rsB`u^aiw_Q|uDlJfXUYN^O=-QS>zx};9P"
    "KwyKGvC`Y-h$~ypIBA3h1E6-G&0vPc@(oZD`vCsV|UW&%Vc<l-kEIl3?$){2Q{A?a^ncrrx2{C62vYvB=plp"
    "U90!+4D-=8^66xLmHF>PAZ65$iP(4|grv{I`-TzKlO7Bk6k(h4^mP{uVCIbt)zgy)jmtlc4Xv+DEpkh*Axib"
    "<CVwf3)z>uVG`JKCdw1<_7is@&uA@(K9L??uM|kQuDp3*gYo*gu=_(4Yi<|g^LVs>SS&SyaY32yH7|0YA4rJ"
    ")Cp~DX9;W)d)LG!Dj!*(5r+)#hF58pU!*Y@4+VBoOb0exw_u7g^|PR1Jtx3Rdhy5Q|q^V6i!j_IF<XuG9s1a"
    "pF1S^1<4PAXk<y4uWHnEYdb<DnkP_-iYI^&tbMM>K;Kp?lAz=+v+j!XFeFwNcfzd&<Eygf<QzAgdUDO|L=3*"
    "ONq{mb}grflf{TH+oEoMX?dnEB-?Y8P`NgH)PV#I8o2*T}>sUvTW=Ps+s(7+1Pla1>7IDWrdX7h_I5S2-Oq~"
    "II2ry`G>=JmWK<_<{>*Fg3rF%Jh<Z*X-}wmnnKi8O4q|>8$H|7$|nY4f4R{D?KWAv>eFE_XP=|>Rw0r1;Wu!"
    "lv!#5H$i|HVMmJ<aklI3^$H{D_Ql;0nzhDc+{nDl53JTu7xc+NjC|1$(jghfL3#ML=zmGBaN9?}-6eI9E?0%"
    "Ej`!e|V^F`mU1dwE#rtfZv#lKNJ{-?y`7q(*{qK#I{w!r)*S!$^OHc7ALJ#7hF30pu>5nSJCimSXPX8*>){o"
    "4ZkZ$z6KIrW3JjCzx-u%3ah=l_=B{?{_`6)GTW(u0)tdaafNOPVwu2V^1;S`WIjWnyp?eVdf~?J~i7gT*${l"
    "{ZN5iK6?)$vrnp@@W`5E`ELg{vD8(wKsrnT9A4Wyqc>j7s3XwiBj^XnM`pffA&N+4X}9@>xaf3_?>v(^1>Pk"
    "-Y`gFE{?F;=#uP1GK9>cht#79YwT>pDWxJB|J7^KGU(a_nZL*?cH7SP24UyrXO+P7Hba~cT{K&W5UZOLf^0q"
    "`Yhv*n|9bV)`Ng}df4Y}f=O3C#%zBIGw)_5z5t5D1fI0^!$L~*0&$cq2euX(%9|^u1BdQgth4QpMd(QJr-{8"
    "+QXU|1Lx8Flew6tX|%0Nq+JwpaA$dn>h8b9a&k6I{XgXr>vfLtbd5{_eaOYxevgu@EzJ47fi(xaf|PiAnh7c"
    "x{UP!@DUo3#atev1*D+(JZcqo^q@_irW4{cbDVQ4>a!Opn38-vnRWtmD#qY>fw6Q#HE!)@A}|R^zUn9=TOU4"
    "cQ*n3{>rM<Y#f2N!f+r1#v%+@(+`sz;wP5{)Z)tKq+z@)q$!u)tY^vk<V}8@nvCKCW38Pfp%RQWpfIu0VYv#"
    "W(>}*%c@67lA5MX^fAMprid~%afANQ9wJGXFNOye2;JvyKg%Y2TnEbCUUmKqOh)Ua(8x?!w85ve2UroKXeaU"
    "w2*>cUj*a)!v<Pl_{%l*yrW&q1+;FqmwZP-54!BZIBY@*-4iu|Cup!1%f_WXQiMJ(T5lBq}L6(oKJC%}j5Bk"
    "IU_J>AlnYsmMUolV<SS|`oWq<Pl(=)A_<EUi%+H@F}!>cu+jY3|}3Q<vJy@8FMM1lU~dp*CPEn%Z|BDCtw3E"
    "3!RZ;75N2jGK^nQo4pmCOLcV3`UzRN<dC&+cohks7msgrrUIFKQJ08u7-EN!u2urV$6n#*bCI5_n;>lA#|dW"
    "aIb+5XnZ|T0Pz$t{C=~Y<*zt+&x6^%JR*#Us+9)tB**VjlU1r`3190$=?TUDF$sf*|05U-sg|Yi+JYqE(Da&"
    "2GDZ`z3+{ew!uItW@AGqo%JjKUzVlcWIhJ1*ZFzOSvL3JMxTaJgxRlsTs?}UNZc)^f$(DF{B{o=U7TP-lIXP"
    "rxYdwwbjoFz{7T$gWpPgZd6;t_9LHF|{Se&~m==9#R>YKL9cT~69h!w7F+@k?7m-MD)_|kj3=JrNRWcQWL_7"
    "grW=P|a7^tC)rAT%bGFNX5COoi44m=pid_~LyY$5CmAl^~pdl8cWP3GNt;CmrvMh~(+4*?jr+A>}N<|QxTgp"
    "{lCOG652ySXwyQw6{$;h#KkEBRSPLJC(@Ai;W6jfT~G9z>%<n)ovA=(A)VssR2}c!SDq1Q;Q4FKzf8N(Wv>6"
    "6iY2A{@!kS13ckEEh`%`sGCsE+yqS7>5;*rUQeHf1#vBX5>V`SyfV@>8a~Uq*dY9C=kO~%u|1pRK2=8ezjgQ"
    "ioz6jRn~E4W-A>%9gHRNI)W@?a^7_ov;nt8Fv?Ky*Y!h!e5&lZN}S%axhyRS3L`>8ImneUnYhO4v82mlu*3S"
    "gXeX<&p>2S1=5d(K1=iW3s+cEHP*z+@4zk*H(t(;{l@C_!2?6`Z>ZqB@U>%lw=`GL4G_!qbAYWOkS22X1l;T"
    "x<ZDl>HLcJP@P3gN<*(-)FA|N)SXv+DL@sTj@V3Ll@bre~!=^uxQ<lYsLc(^;*$TBA&pK=|_yt|bhDt!zb6_"
    "tFfb}T3y(^&ZwRAl+XY7_!_ZH@(rBUw%!R#BDhxawcKaD%dO)s;xTZjSx538Fyjm#okZ4*D^S_$guqz7#@;y"
    "25~FKcoQs!*sQ*Oz&w2m#KLGtyQ<z&vF!=+5gK1$>7y1i%PiJv?BJdJX5;Ap4ttLpn{d%*|RzOo}7NLxEfaB"
    "OVp?UG-IQxsYxf`5Wg;6?Hr51oXrQM8rTBJ83feM=9Q61GX`l#8gSGQ%OA*9o_Y^co+8e$JO<t@=V_eCkp9)"
    "Y9S+;?ea58M+J5#)uTG9=QJ)#_fzaJdE<qi5R$i(W=U)@5iB@eU-fDX_jFXxM2$-L?BU$Pj>Xjk>5n8jmBO-"
    "@|&X=GN6`4~(n^ku+5T8_T#fm<y0oq_EHdqx^>k8+kuAD>dP7|wPr$g<UuIHdb%arekMbL6*lp>Fc#L+~No?"
    "j0%d`dehzfbO@dh0382ziG%nu_p)dqJMbZK#sWCBWluS^u@5L(38)zFSdYMxNFS`UIZ2AR|VUS=ibT6vBuBL"
    "cK?3L*wGc+!3S{6o;X{p+z++J-SDUmUM)+`N7|qg0U~>t1JXD+DH!t+$rSg1+f8`>STtB2C?&kQX$#5_Cz0$"
    "9n4x8C8jb{!yK{K5=58YQ=ap|qtjH0mZhm$&2V>%dh{!zR?*se9<380sB_4(#+eY;DYjVNcO#^E)T$7v$*iV"
    "dxsO(?r@WSGHL5&Rl`88M*#|=sDu(@0!x6^{+fugo1IIM6E9?4T@c3wRVb`3Epsl7rOiIdF<G8*FxfhxdgH)"
    "v61N3|_>9FtE;r@2C<7?v7m39a@Yxck~MrQ+N{Ry{g=}^`*OB)GA_KwKP8|Z>=L&ks@r&d7Jq!Sd&2yOS}-n"
    "JE!SBf1HGU;SXfSIU0x_dYnw4a7%X7Mv;>)IGo!_qX|>Y)#}Uh9<`29Di!t}fjs6p12Hlg``Ivv-g@?5%ZCb"
    "m9ZKlHULL?&23q*~*NPGJIUsDZ*auhcPe;mdQ5r9Efa6DjE*NS#Z@uBhA4k^LZ^ILzg!%h{ocHNQ0~%#heFt"
    "n0k6-Xlz`|0c8srKwfpb4jX{#-<Az>=&=2!cdzZ=-E2bsgh~g8(w%f>Nb>e}9oEG6P{30etf3zMP>{d#%P&R"
    "yz&x)M`h1xHxIQVOfR!n%ZDN!Ovp~skN0QGf&O++tymlYB7P!pzQ1lirw5O=jgr&n(2`QdrMBP}QR@Yx_6lr"
    "F?dI)MZbWE1Gn8G3>XQ^xIqd{4n@&y9b;zryayk92;wFo7<R>y7oy$1F*5`HCjXT7c>u#w7TcT+y2o4Vx_$("
    "6fk4|B^4Z`e>AqlZO2a6kWUq;leda19RLe!0sF!Yv8ui0xYo&r+VvT|y0xO6`ZOXm#7Svjn8fbBkE;Qw_7L-"
    "BNaJVmK?cEbX30r6R$WLv*c6eQ70igD|b9%20l9A~Lfu2~t#tZsqHjXq_Fr?k}5stnAHAT5Pw9Mpg|+6qI1O"
    ">SMM+jFyq7?L9)mS*b<|&|3x<fLY8b*pQ4api=UTEIBMS0~?N674BJVZS*~u0ok~;25J;qOxw#Wy{Cw;a364"
    "t=3RRw{718wNz7mN?|FRxavH`jm6dP@t!HYU%|sZ0p2~7B&BsZ+BNF0JymIUvWsng;D=zb7dCC3p(V#40=B&"
    "7smI0|%dQ-*-FO8b0G?X~^K;-X`xO9!yRWo9oM*)~OVqJZO2m<qN7lhBOw~07b+K9`eeufYMX*Y)3Un()!OI"
    "A`sGjNy&k+)LT8++LrS(W7~GjKud2LuZHh;~^(c2p)%C%iWbQ(dO3stxovVGc@S^kQZ&b>+CQ{x*!VM!l`$v"
    "yyK62(@ng2sB)QVlCC`L|$)q(s?788PrZupculLNh04+WBo2@SO-d?5im$TT3XyPl{A`7SA?bkYdkn2YlKl{"
    "K6}PUT8CEf%i##JRD*OMK@S1eGEbH)@K!_&bv2zjL$Cv2PI)T9?mYY4+?iUmtqL+Gyp6r65h(LNF*KJv>l6v"
    "AieuR}I@h~0v;-l!H>yYM*!b3!DbmC`Ri1BD845f=TZ51^z;dO0FLu}i2;`g9BX^+8AS!;JpuD*$UPq==q0H"
    "gU7@Mz)5Q6`@QIag>=Vy@VH+vrVPYz3;dJ7kL(ijr)(Pu*6>n0Fq1iWA10DL$<dw25B&EP%5-913}I81ZM8t"
    "*|Io=JT2)4Q{`?&<sAO4aloUyT^)EMgU3g0CEU))r*~B1`bqXU3L5G5nubKb^m)8XhO{W62bQ(bFyBJdB8ib"
    "X0oi*Ja{TGeC_Qia-<>yd-)sTb%f19cIQ(Y_V%k!`a{jFp#O&notEI<RK7q&HjRwV0z?AuG$|x;4hGdJuO=<"
    "c0=fBx?rdI7x&>xq3bm%a&)#c+I*k9=MNV$r`=~XTN^z}9*D6;I1a=LT@+cau%3e4_(>XM)rp`y3O^(=C6}="
    "LVd$j5TkVKM3uaOw*M$t;u8Td^JBV@ZiX{YY#*CRal~)-7%iy^-eo#Kj^hEg_qV>X=ujF~9SHv3J^88n7L@@"
    "`|kZ)Q?EPd!xrjIVRQ_J!5Y0?qtiCoGhk5FCtZBIK#qFC4Pi{T%Yh8j}FkBsDHpLEWQIw%^khD1?4A=6M?73"
    "`CRuo+Pu*eV?TFmtC#uB7+JS}1>i?&bDRAwMhW36?2_a<7}UAX48){2T{czLW(YtDz(bWm!TV^$AlPqar<9z"
    "#f}$eYAUtb;GQ)c{JW_hxthr8`W6zU`lzmOyZ1}w?$o*s6zlX<ePbUYsgROED2tS!iGUtl1(_39jNO=e|`V+"
    "`}5!5OU)x9=vEtm2$h~rVD6sg3}{tuLePC6Nm`mG+zC4vlct|9mMY6Am28Na_zB1x3<q`_{pC<{EP?CmjP#i"
    "{&0cC=6ew3VCi8?@J7<KG((a+%LHjKvToSW6s?#w!5g7Z0{6Io8JXk`6<Sa*b$GbRrh~dKUlLa@@K>JZBj?H"
    "(<>{1~nS|Fmd(8EGqR{}q=NoQw4OhJWT5-2G7XW0~fvjZKV!0;lTlP+4~MP3ba<a<lFFkA}RQGII*jHC{M?z"
    "9NeniN-6ik)(D0&gWI$5esI2~s*H$pLSnK07MaXPQi?Akd}?t=f<V@K%lm&j;{UF5qLit3D6uO?~dCe8Cq}V"
    "Q|$AIWmDW{W_%_da}q%@*Ndfl|HX(m8TCY7CDeZOH?;O3uX07B}HW|E%eCfCM}49XkOOfgF#V^$GB64OW`$>"
    "`Ewo=5w{wfteWx?P;7fE`fPjEWJ<FXo`q?ayFtOnfz`kP8r<~Wi(JL)AMV%e5LI}x7-=AH*gW*_>KjSJd#Nw"
    "b^fXi&6!j&VZ!0FS8S>-Rz`Gik4j%InN{M3<7Q0rr=iP?TrBr|Y;{4(4{N(34jI<=Zn4F#roT?6?W9wC2v~h"
    "*2W2oCRKWiYOmCa0|Qv{`#EPmBoGEH)wS+DOH$=(iIjk?Yk_z>}CD2-Ffu6!=w_{3Kq9UZN))@wTwLtVgHOk"
    "qVenSQvt_t+z_#<1CtWIo@HUh5FW_FmXdMGHXYdA3-4N}a}i)c~C-6)bDU32f*4E0MtlRtPpCidL4Tz$_AX1"
    ";t*j`eM`phc?F|)Vhi50sp4(z38J;UKrV3(MHw07>F04R>{sH9$D#Gk}?<5nqvZ}-QH;1xOr-Y+d@ONgyykq"
    "isr(SKN(kiA+0fCbTOr++(tXF;rk78J++Z<%O2H}z*tZPdfFX?BI)=X=nMo2xET>VEKMGI_V}>^v9tz1_FW@"
    "xY!kTLr|hm63BVNPF@!rnoDxTNL5)|c79SgNf))Jm@Sp*kP>1#ohZ{otK*2*juHIS~$3Frg&?c|!IkIb6E!5"
    "wi1+r52H<Gb1#%<xHt*5rU*6RkmZ34{f4{b-PA<E35ZJq50)l!fm?8pLZ*ho!o`NF}Bg`nH?vM&8LX7gS1UJ"
    "hwTMB{1<NG5hgbAP3fy?rF*oBwDq<0dh^VGnH1Z4TJ?XpQ~oK)&vm!pW3<0uu5f3|`zkioXfu_TuKz>hcRjM"
    "N}hb3y+Hh+J3uhsm$lGle=Gx9e{>SOG3A*J9JPwPLs$@;%G%_J~zBjHw=NRyUdDtthw9N#b!zn^k<A^zEdiS"
    "pBBy>swnkCO_hiO%58p4e;O+i(=TodG#}iw)FmJ$Yb$H2=PyNNP=ni~btc7bWoMSA(8ya%18+oiC_EoaQn7t"
    "4!O@#MGxQ1^`ME7j;;!A>G%({IQ-#gpkN+$@(a?74L9`280=IPOpZ?{$fBgLJ?5t-kZ=0vRUZdYCPb)#Ccg#"
    "lCuD8w>{avlVb!mCH!Dx_!gQnHiK`i9bt5m;4k~hS#lVYx_LaLMdy1ZuI01ZC$So_9Gu)nb@2#G}6BL>=#jD"
    "jSAnz6Ko!QI9^f6ik5P+DU8HLGm8a=mQ=Td2+F-FsmKB;O?q$_6$nHB!k<Zxgn1-3A$}(R}gkfJD|2G?YRPB"
    "hP6)A;UqVn;3mmiM%fBU6+TJda+@bDVfvQ{(VEBWYHvkx+Xrdt;`1Dk(+}rsB2FcjFnfYc&~?JKmbttN&9LI"
    "$s^sOgCbPUhFZCH)P3FM=yNX|df`H@Vcgf|z0>ZRQ<{4v_o?4YWn@RqM%?I;<JAKjOFc^&=?6%hW@9E!z8pZ"
    "4YK1f&xa#c+jwgCaLOekGRMoRc%TPqnm+vg^5p}PJ@NSS2P?AAs%*rHe6r2YR=iy+mhJh3D)vNH!Myrzw8m{"
    "2tB8sx+Hz_>NHLuGN8$PyImkBUnbI^2|om=<KZuISu2>-I^pxMvXMY5K;3j~a=b*r&;U-Vu=!ihwfQRd0l5K"
    "yP8K!X=&R`KD?svewK-G4KqEh1-=d`SOe?wi%=k=LS&cS)vD?tz$!VRc)^0lFHDE7n1}(jYJkTyxeo(bmj%5"
    "Bme*DxxYQhZ1-@sJ(2cYF|1bm{HH#zLmk1nj%PKenOD1M#_7vq2>dgLv)rHFo!;y7FigBe^TJB-oOn8X{%e|"
    "^A2kYnLp`BZu3fjco}B*fGO^?_X&fmj4kH~;Z3<0DVJ41DYIe;tPn+o?ptZT-8;@JB07)PP2Dn4osFWlayn-"
    "UwAWa+h(&~%e*<cCZSgGkBJ>gxXa^=B5<F=8NG^k7)}sVmz#2agOC{x|&W6eWJ5EM(iUhTpc31Q^uVXz~1(+"
    "2NT<W^ZKZb})tjUE9B$-a{NRS0Pbi<1aN-*PUcp-AYg-<Ikx%RoO>7+_`T^nzrIq<d<{n7?aBk1QJ^Gb4PNf"
    "UZaUrO>Iac>PsJXy>^rD_bS3ruD;nbc7_<QXJeM&Rcdd}gf0Y?J=Z;*2w5yMo7aa62%O9)os(q%#897rbc*i"
    "Ci91$$~TColj~Gz>C9Vv>3hZPONa{NoP$kR#Sf*C4co=6LM^jac|q~9c+UP9&F2mUTZyU(GLn_Rp}GeKNg6>"
    "5BR1Va5_xjFz>oS;=687|FrJ~fh$pnT|+m^0Gd4hPzGLW`l3|DZcz2wLW0RzR|@ftwq62?A5`V^A^roO>i!<"
    "^vrpqR+=l@=t-W)oV5v?}@5G<j)f%Av%lSSZeG%VhIUzKqTAkX{{^BkOQw)|q9we_IUk-IUBPBA!F1g9w7y<"
    "Bl&~9(l)hkltu3<?9n%a!6v29j%Z`GELU3RFFkKNLh&-IbyLE(FOlD<qqx3%CqrHgj9b3a7l-x+m~K?Ig?&K"
    "KTq08}Q7HHAPh;nl)=FW+D!-IHXopeuONIob`Td%^zRVc;JgO%D!__Br>bdtTu2Y4CdY)vLkm;Ar<?psLWir"
    "14{lXGO0CaON}vEdcO3pHq{SK}Xz6W1hbB=Yw6`s@g~84L<4c*(^Bn4i2V6@9<!MZ-00CYX9K%%pdGc_q?N5"
    "`$yax@WbG6Ha(i|P4@%tdk6bd?;toD9{ID~sZlZIxuA+Q+Kqco%Hiz4#*D<Jk?}36IBTEc7B&k~a;y;>9vTy"
    "oCR5R}=6UK6_SsR}mK+Zp<zd?5CAzIs-u6zVr3>lWh~lX06`zTCG>*sxyY^e59g^;phNGV&lrr2AE>tj%%!`"
    "9*@(}>R8=UruIPbUuI~bo}2OjpIrkt`};~^K-_Td=AOA}H^MPD-b)Ib%=sthnU-M<(}j$k1Z(+WF*>qu_7@u"
    "7-C4KJi)*F;k_qD5;oq5X6lB~vdVM@~hTp*TI6bns#bV^I*PzmDJ6!Uavl&KyY?G&U;dto(H$C0aRL-7sV({"
    "E<~k?m+v%h$F5C=cbF&9-5o}kyWe<!#&wL<o2ZIJX1A72xdV<)Q0}*PEwV~XU5bS(a%5<ZgFGPKCJq(g5JDm"
    "jlcc<;Xuxe@H{EI2JHVNUep~dpoul)SS*@~8nU>?kBTq~+&oF541V5p>G9=H=O6I5ie*O6RdjzW3qj0akETg"
    "{yQ8$8lQN>&WYP&zZzk4=-e_UX-~pihCUB+NjuZS>uXpqcoQozg$ufrLl4?qY6r7N(NdXonlTLyM>4s{B5xe"
    "wKFG$rRrBJ-LqE3B3;$B=V@tJW!z+DRZ0k=%0I5NEtD}C#X*bMF&S>(}LS(@DOc!R2PE#RJB6KNpef;uaFA$"
    "mc2Pmg8%p2s_RF=bdAdzkZBrFQ}78_!r4e(bZ$Ifukn7(<+xbD&JwoI|5o4pvGpr%L+ODT%yTxs6Ram&oOSM"
    "|K#Hsz1xYYl{W~31-k~;+Ch>yFQvkO$C^`(UD^VkqnHa=sAu3Sa^5BaH>>gX%TT&LRsFR9{nCvdSNaO7);3U"
    ")xltgGC1L293;4&QhJ$#LrvLCG7l#hkg4okp0F7x&p;uU!^8rU+-DzB4v<!`75Q@(c`Fp4As9?XM%^$57MUH"
    "hX_7?HPsj`dQEOq!0Gb~QhFBjT%>)jL2WBrRwz#$gkSR!dZ+}?j^F;S*15Y`Yr=b_UDY<8M`G3yBoMZk%{)W"
    "W~TY91LyiS?)`v?3YY7K>AAk2iOK#BHpU)mYWy)7Bcy*37OZ|Hl|z1Ob?fyWOH{MQGA-J@WC-`_jjKlJwx{n"
    "vZDfq$^SzyE4?cQ<&w>mR*7+CAcj!`c4P%nJsM3?~1$OfpcJem@qF>%I-b*~Db(pAYN|=I;JC2J;((`HjK+#"
    "$f(O8O-rk?B$-uUg{w~<PNIG-Jq}=TGWkd*bU+~(U}GvzKP$|b~;&CBu_i0^3Y!hbwj186WyLeFELJU$>_Fm"
    "zqX<!M=a_Sb|fm|2oD^3B|_|AjrNSOEnIAtCrdX)1R8RtX`j?U4mPSK=cbo>LQ2Kh>IKRN_`Vw9oC?`e9ey!"
    "h*b-#@zzoA-W<?(wFZKDsyYZLvx2I?BkH_zf%*Wes)RqnnzG#OkjCqOoD>J)_*f7ATg0Vw5?Rq&}Ul1@LOGK"
    ")29xxF+F=J7Z+_728`5ImvIBQXrFvO&a--P_TBFNC3D;E_Oq{%V^*7n~#O@KEK(Tq~2En||s5v((?^Y;nlq$"
    "67mW@GSj3d)csVBktIVI>a;>PRC*PO^pBz9?e0N{W<0C<jcxyFxM{SmTF;=1|m>ImCf`grwYoXRIVEeRfeMq"
    "!jjaI4{yjj4`<$I7lfOO;q-mWwH0IEOwUqFDq=;%UVdYHfH<v{%d~V`QGek@9@xv|DPQm9R-KM?yIALH{3hi"
    "_hy6PaBuH$aP<0M_G<TVw*PwO9q#TP3}5^EM~%!jP$o0v^MAAB_itgD!h1d*;%zyp*jYtUr{^>=;g)K*sYw#"
    "Fqro>u{TrkHjZy!`sQ)J!^)-i2#^X@|Kq)+oqhXseR0)@1V~io9QdOpElMnRGr18Nohoy#jo9fE6chw4aRfW"
    "kzHm(Gp(JOA|r6Kfk`Jqd*C{)8<yy~XNu2ZNQ;MT|E$_NQdjA%c?QcWmGNyzU-HP@f+M(On(p^on;a>8WN0X"
    "m~bq#D3qWV^+VKbrO5d@+{0X(B#H58YDdh6-tctIJAs5M7Ydg4o0uHm?YJM2WJ|+Y;GSc%Kyz-3tu6+Kyhs`"
    "ktt8M<P<V&^!}cp+|2{lLzDvVX+~YGemiug%R+|Vg80?Ns8P#b@Txd=j5&vB%JU?q?IKPQ4e&#NxEW?66CfB"
    "1FzQCidLCAN(1#41|ZRc5YK6V(}im0BEyg`<kBQqmp*$(<YFaM&VLjz#eM{7lvtQ05xNJX>KR#950j1mHM!e"
    "^zZiE5MidV0Ll{MUX>#fH!z?SN@InYNy+*`BI772S15n}U@Ed#jjXnLwo_=Fb|2pg`gbU>^QbdQOe2CX8e#?"
    "mJlXyAC<M5Ke13*6WWPyMLc&X#5Unmd)1V0cvY%-Zl;(Nl{;k%)F8pc_k7PzPxzB)nMs))=Rt@^5@B-rC?+H"
    "%xYfp7;#5=*@h$6SsImTw8q%)v=^R_mmrI_3cHQFu8YlYcD3Pez|Ahjj#Z!J{Mk1-j;pXXDBeYvqa2lv=K_j"
    "HhYi*nfZsa!hcu;##WKtR=2fmD@S;Fy>Ks8-j<iWP)7G#}elX27*IWuaT&Pu~Nxlc*)76gMFs2Zgkza+$fXI"
    "NPhCg*y^^?d=6{9rrr(N>st=63WA&mU^7AetCwZPf@iFVm9sa*k@cow6y_nQJmY{Zyp>WfM=QmH!%59D=of{"
    "V+(64rTOHVKLMqH5?Pr()L5V^?%%c@%3Xz$8#UAamQ=9<k)2d{s@lq?(MUMy+fp`d<NJ%EjFDO|S7AbQyl8P"
    "8MZ5soA8id4s`^hpy5hIy$(17ESMtyTq<Z6WRt<*J()6K+TCCxPG+E)@>gZQTs5h8n?5xdQ!l?)?)8$t#fa`"
    "OL+a^{B{bLQ1-5^aq6;j35tV0U-_=-_C7@73<%{_eqRfA`4S=l-65Fx)#DO!s#W4vwaLIy(w>r~CY1kG~%5^"
    "MQA?cf@Cnj5+Lnh}Yh~<vCvI7@{2*2}FUH<Q^}B2Id?p!7QkDkjLIP=KLFT{*5{R#+?7@nDeHDKA6MDmdC_M"
    "WP*dlGHH@51WR7eJR%a0h`b}Uj3C!I<T70caB{=JP}dFPGvaAeYKVTx(m-J+DlrE<l9@RsA{$53Cz9GDBYK-"
    "A1%gTFL&aBC+2Y4X*b8Bhjb*$#Jr1-}ivZHd9b<!T80Uav#iqcUHw=X4IVe($%_EQVXzgA%gGrfW84sYjqrV"
    "uVY!0l3uP_D%Z=b?6*V-7iwlQ4KK~E;d?qGP7F}nS{0AM}LlcESQX<o@G2{vwy9{9*4?7(Idrya6QM@7K14c"
    "KNJ3xs0hQL-fO*(n#SA>=&CScd+-OtLHN-P?@uKM24emAe9KK|O(;O*}YgwY{kH`=`dBf+8!x*_H+2Y{t_!q"
    "<bc%vO-+zsr!~`lGEmscxl4HKp!IR-E%F76`)KY561>bY6zvs=<o`j8lNCYqk!KzI?e_}TKfzzA*jYi8$m`~"
    "J8;M#3j_PUdIPa|=A^?X*!h&pY@L5u&id68blxboweZ!m{n!4hSuow(<<sH*>~*kza1`v#X0MNC2e0^yPkAu"
    "hJ3QPw4EB7^{oru+dT{t^cyO>ge7(m9UL#)(wj7soK47e#N%;sxc=kO5ZTND@1NJ=|_Sko-Ts4;hX;>Oq?+x"
    "17yk1Gmb`OW&SoUu$`!|;T8_WJD)~D+SS7wF*A_Q95Sf$YsI^HUbqgm%$rn-R;^tHq`L+mwyBgfakuZrX|PM"
    "kc9hkHl8UNt83e($D{1;!}5ON?tr0Phi1?WmXd`-`x@PW84}--5bsDlPmgnPy*;!8WI@8eeH>1hyHp{VIQ$i"
    "LJG%Z(^R;G3eX$vGvzxecW$n@Q)d?$PeBfXMV~N3!xCz6aos9=K#l><=hKcGDC1QrMk-LEn23sVgX1HLu62}"
    "6^ZCx!}b3F{Hd@@WvJKAi5E{rz}JuB4DJv5>|C<L8MFe3kYvV$fJ0=Q4h><lrAwse-$lvo8)D@pbOqz<U@|T"
    "OlSDAYMF_#eSlFr`|A06W;_;BmU{=&PfKdSGBu<zY+{4g8-Q8y=$-+oYr1Ca1z6>*nl!}USkNFY2mV^eSkQE"
    "|aNGyT5Pq~-koP&GkUl)`8`UxDo2z%PX!yg2E|Fyq242G|d!2R*?Xu5ka+Z%cZhX)4-!@bvoJ$`f;3=fB|X5"
    "Oy<+Mn*fI@+5JUJdv6_h&~3ZJ-s(L_uCMONbt(yhwnv*4|bkFp2_RAeQ{^Y!SxY7H~^j$ub_t(KojI8(aR3E"
    "&s-r|0mgU(S?m0uC+Uv#I%o%!N6n8Fe<hrmXNz#OAZCou<z79Eo0nckgmxjrU<m`0|Zn+gedwzjm)yBC@0YM"
    "c<lx-iM*rT2-QuY;`>&3;$}BU?2aq81S#q1nPb0RMxfRaJbdM9!dJsD$9z}&Dxn_+b6O8cT*>}rcKfMXa7$l"
    "uGn}!lE0u_!n{}lryhs)<iYWJDo)nol(~u1AVV-2@`nL!%<i$sy^8l6Qcpx_BlrO`NDk%;Gt(UtH=TpY1oPm"
    "oVU?(1?JOyAm6!d}s^QZ~;#WDg~i5C%@obfP+o*^ykvr~Ku+{oMmwm3uYL*jN|+`v$yOJ<M|>I7&tV6-!~C~"
    "|_o5ND0&AjA1?^>Rc(q{M#nkDi~axkN5Ij5F8?*c+Mw!ihlGlvv_kR7?{5Lc+l%WL?nj)Wfmna}O}_5Lo0Er"
    "5d<_|Eiqz(O-+RmQhj1W#S_2jecp?5+_cy)QRYs-fQBg8>=shFzCa7_PZDeck4zZJ>2`|5c!Q2|Hg`cW5xf}"
    "8gldSiD`uGEJL7Ubp8#ZpJ<VEU&_$Du=K*NSP!l)P~kD<axt2t!-i37RaO^AL~zICE^<FVhn!(<%rTu=vG>&"
    "7Kn3zugHVCtNC5mU_hB$~a8GL+MGeHPf)qy-)^_s$v-d9EZ6jBn@V`={>^Ua8A+{(#B@dHT+j6(IZA&A`$>x"
    ")^G)NRhL?FNhK+B9*|ND3Ds~&ielpmQS>(0!vNEQlJx2kU4=kH3b7Oz^PY{15%H!fA&k~a<#GR7fQYusF}uC"
    "7+M)5}x={x^=uri^UKvLGSD#diNnLPHN;yf{2LKYYniLTR-k^1!)W*B)uV;Lf^3z3NQ6EUaC=koS2E@AFonP"
    "P?zQBF)NTl8wP5#VbO(B7Y!#&?}SRc#!ZfYQn>X?A|afSVMN2dh_{qs$|nhYq3m~`Bq&ims{`N{`U6xFK;VV"
    "C7cbq)#gK4Nr%t%{Itz#_v_5ceE+$VN*778NxpIjO5f6|WnNVnRo(*H4PobCxDigsVD>P=j#{K+okQmF2614"
    "V)&Rj?0H=eNl@N8eqnD^~0^-kNt`J-Ykaps0^yk=mTH`PU`DhZSktnqW(^9=5BuwsvO8$kYO=2yXMG}#4RYK"
    "gRqRPNh(*ZhPU<?}Sx%Q%C2N)1&`{*UyHnVwF>J$^cArpFBI=DNbK_Jo}L%;}A49DZpW0j_SFd^_F2=>Apj0"
    "k1oMDkZKyE~$gF{fCLxaUaQH4AF#1`4xq#OGGoc7op^W1qEc9BZ7a+v!q;qXI1oaWT#?r(F_7TtZL`nt@vV_"
    "4xJ64ODH)KyfteBwVE9sV2}S$sXZt_v`ujiJh-JfiqKjKGjg1If130gJV!p-Jf&Hsuvnjemtw}tVB|R5WzwD"
    "<tdAvLNBJNK12wpM)fD90$hN|Aa$e`b2I@^sUhB#C^%A4oDFsT{?5NrARJe??df?oo&q#I`;&+GMVkhn<pfO"
    "Mtb9NE?g#s>e1BKIzboJWD&c8d_Zgg0c8?%DB``rcK<h)(U_8+4!0ZW;<SRP@dmWJ92rE{BytYjYm$Fn)(TP"
    "UAcsA2f3{{!Bg!w1yF8jJ=C%ij(d2oKH{N(mLOVrunxuVyzo`xTSE*<+Gu-hKId{`_dllUWc)-xHdqzE}V?f"
    "7)XGswg=YB(-%o+UPs#SF`k(aE0PXQU+Mz4fkZb9)eIelstkX)S)lHjvXW(RAYtadrgB#1B)p4TyPE3Jz<Ye"
    "o?@m?d%^Jlvkf+qi3)>)6aJOHNCFp{A+kye_f;;J7Fl_1#&HNsyHbdhITmWN8ud!jxB>(W`10pM9*!MKTZIn"
    "aE=lR$}2%V_#5Llk&mk&U5iGLYfh6VQG=B+$d~~!Kf6tVjb7{2mGTn}GR!3YsH2`bNWlaVry-WAGrYpQ3rZ7"
    "!+&qajWZ{HHPl7t+kW~|6@Ep92X+jw{Sl)e+Bu=d<H{Cl<DK&^IZL+W)Y2XeU=F2#<ieX|I>Uxu07gO@W`JO"
    "?1h6#e4`RFCihSYE9!srtSCIL=RtK@*hPL)W-aT=z@E#@OFSs5+B`v~x{AoBgIx)J;R&9m15XIU@?R%sSHyL"
    "$K8UNm_$emL3dKb-W(lfAGHpixhrjw47@fA;8c7(RZy`|xr9$)laUr~1j$$?ne6iGH-#fB0xscpie+a-~9~1"
    "I!<!Op((-k?%2z$0}?ox1(l^smD*h>m1*8j_*3hcb(&(FN@t}#fDQD_?Lj=c_3OI3R<;wLx4i`TI$1=zq&WK"
    "d7DXX)`E^xI=HyvgTyJi9a9>sYlHH1+9Qz)7a#aJ1;bV7^2r6Fa)F^-_lE-^iR%EbH&hu|0(CBY0(F{Et^zt"
    "bVI&h!?%8%G9Psbms|En6uB}Qou%nT(V?uGMH8#oeF%cjz=w1yYbBc_u%D0y%r(qf;;FvIV6CjTXh~EvGs)c"
    ">yVqwk0`dqc3wYN25{|PTDRKrkk@1lH9Y26`X6G6pk#T~0i7jqEG&{qOFt;pSTqbq_sfas&^s)ZdWT4|JE6l"
    "9JUN~ic8%Hs(91Ze{5I?F`xlWJPi4D9P@F9$CN$liqN59!~S_-)1xP_t27NX-B*cRnsedOrYHhw~KnMsj~Oo"
    "1DEU;8Bp!zf}_ad5-v3UhlB6IqzMHf0_M^4s%emc)*oGxl($JCB>}20_P%LC@3=~0=9s^@?4-)YT-!#gt*I8"
    "=j!GEvhw_Ht32Pr%YQ;%{u9IS!D_A$?ROuAd(TFra5CO~^6bge-Es8r@!sQT{HXur@nq-mZa>=F8|^&V2_Nr"
    "`cgIh|XOBmtC;joW{xcmtd#Kw>+jdZRD5s#!fvIMjbEtkaA=s_4e=UzThw(lA9>w=vsQ)h1e;4Zi`-S?|hu-"
    "pIIfO3`&JNE4C3vZy!8?v86X1hq(`=3<f*`FuEc5_!Z#!eWdB?Ct@lCLQ$9V1r11}-Pz@U;`3q22HP@U>40s"
    "?C7z(2tXK~+S>;S9xN&T7HV2@EylxTh-vkBYawQ|heJ!%!eMhh({{;;8bTj&(s#-2aWX2irGfeseO9)0h4B+"
    "u7o46l}7Mw@kgMDUW#t8tXc_ylF&r=fGzTsTFv(A)qOYvN<3(O~OS|E)}^3Kr-gxfg21g$-s-EFptzCO+buu"
    "L`%SM6#!S-O&~dgstT6owX*)HROFR2#KO%e*WpdUflX|apDa=`{*z^2ssWOqXsINzR$O6%6Ml+mc3YtRTGqx"
    "47gYnef<{^_=|F=eXU;PzMjD2Bore+l3Qj=1kXKN#VXmPTTJbs8A7XvGfe~NOt&oavx<DBbGrp7Hqrw?Dxd{"
    "0yOvVd?^u;Fl(GoQ>-(a6>@ehD!LS~PJNec4oAkr-kC32{YI*!h9I6y+s#Qt@q|A+sARU!}!yGX-%G0gy>8?"
    "JW<8<I`A%8HLp*p?CzPrsKM{x0f&7xll3`u{E^B79B>eP=n^(o9jVYJ~+7K_x5Qq}hb!lxs|M;PD?ZBu9!&b"
    "e&f9ViOO3bR>5?sXsorPsNFMg;wv*UN%*laB50$7LHd}0+KQKTtg@VMOjdNp{@<iX{tNnQDIf(iLX?3qUyD%"
    "b?8SoeO9pyOM=fC-8T$*8i#2VBeJo~E2@V+Q7y6|>>twfE(&tu!;YrdjJj0NzxvjJOig2N)LFdo*V8!chI11"
    "(K0o5AWn4A=Xw9%>OI5Q=HXm+9yOPoPMK~zq0HUa@2?Nt0A`QCN2*c(frZ0nf>88MPgD0UeV+3Sm4Cr^?Zh^"
    "C?B1JjPv&Hq4adIJQ$(HFjI5~3aKO5SDuubZZ9I~jV-i9+xl@1iS#0o_SUy&~1u%vNO#$(Hn#!J!I2rt+~iD"
    "eV3!a>Q8brqmPXJCvPmf~3O4Dw+1DUZaG`Fa8RE2?_rtx<7l0L-SEP~S)f!-Dl;9>XD4uVO+qLl4$N%vQwI2"
    "tpth!l8?dlUFLCUe6Jw0r`S9AyyfpNkePy`d?SCdi0rIl?WiH*Lqd-uAW5UWUn7R9_>C2bu<q9PxZscqrIKI"
    "$+K{xAL_}|M>~&3;dt_BccLFY+kHG4J?uZ5JRbEY&z`obR|pC+P@Oo1D`MHZUZ9`xN43{gKmDZoJ8h~JKUjI"
    "L{)k0K_^xYx*R{UuTHkf8e{#2qWyo>}kBG;?YGmkD;ZPm?2o&H3t$Ul8<DlPP;obm#>IBwYxZhoI1dgFWt)c"
    "%DBY=Sgc_;?u>Lc}Ey%KUJ3TWq>Nz^#IS_ZT`6-;cbpmMq6``mCKd2@f&n;(D1i(6`EK&1xA%%u%P@k5lbia"
    "W#BQNt9}^+N7}8tXWL4&y>HXd`tC(K37wg9u91C;*2nhMZyuN!Ka60Wj2`heN?Gf7f}&ViC@fVunCkyCz3~7"
    "3IvlN}U?_13n}{{3^@Aox^q80+3Y4I!SE8fty!=B3r>2w=!@lI#-kwNe?4td;GdcqJ99$uxfW_2Vb<i(-)eE"
    "#LuS{e?|)Q)S0*{03v)Al4uMm4&oKaLFg2)-G5yn^>a!@1A~aidX;p#x3kxOw)5!e!|~+dZa?hrjdmuFc1EM"
    "84<Coov#`GxjmDFmFnszneE29Dg<*eJPo725Wc>7T813|1rPEE*%7QrnF-L>Gy~_HliqqY<-<?(c3Uhn*?7M"
    "{fUBdk?;eMBJ|B1Z`%+2NWfthSE55QUk{2)+M9VW<ZDoM%2mA9$=W@*V{njO$X(334&P2W|{6psR1^1~=F6B"
    "DMPp`n)|W)>1L(<)SRhtbeB-lspES{ksGK~}^W4#PzhGcb~yQSAsUR0%bvoIg0I-%LWT4!_Bw+dhXcslMFs="
    "DLl0<*PR<(CpNTpcg7xco2s2vLJ@T21*F?$SpVv8Xg7Nbv|7dgl~g^GgAoBA88GCO0d1gNwvyZqv@E0Id>Rv"
    "lF-LRc1px)F?TKnx}y!#$Z|-ro;qTLTq2;FB;mCr@)lfi0#8Fv9Yfx^z}n{eL&nHL0+568G>CMBv_{jpit=z"
    "$$|juW85HK2FfNh-H(C{O!<8stP5_`oK`(SF0pWKlGzZ#-3@}78gBMgInZDB)e#8YJv2}sv0U3G;a{~ZR7hQ"
    "co4&+i6WrtvOJ_rA2uJcKj&p2(EaDP~w=Gl<Gfn&$CU}vo&kse^Zu*eB&F=?ilPXcEgWm$>eK~6=$wZU&N=U"
    "1h=q*@~&nQthBKlw~cF#|KD*PP|E5~X+V@#CkDqw(JO+2hCk@o4f0EX7(!;binI+IjqN=jjL{@}G>y<8bfs="
    "<$;$JG+m=(Zg_WGJ5nlYL&<_^bQt$pgIVU9oX=DfB-BP^F(*>GAml+@tsauOGNK|uMquRn*J_Lf0w5J+3m$u"
    "wrucyD&Lz57u|1xwIjtP(0bt)HyHFE?d?^f@vDR5W7jN26JjOt^yWx-x3mRCzBfNZCT2)Byv{WW)BJvx&2a<"
    "DOcEVkb9rCmYsPMAbqgi^V5kteq|JyZUImuPzPG7h#;Q3q)NlG`qJ9It8fETg+I>L?e)EfKYDVjOH;uD5r7@"
    "^La<RxM+?_5xNM!{Z`1O?P%VC~pHnHLGrZNVWpMcTK=Nct-Jfa|i(|*}V$Ojn`g4P#^<tRH1L%|&gPzVZPxt"
    "y%wJX(s2(AD_=-xoTMt9uj=I>xr}@W}p7aw<r~kh+Svpjcd*WPXb%1KJcrftET?9r#WMPbUhOGBh9#51k>gq"
    "9GK~h-bPe!`b{funY}~Kg8gG{5%QAWkI1Rqh+bLEl}ufm?Ya&hz_H_S7;Mwx8WS95r2rIg@?ULk@|u{MCX{{"
    "Mq{aZ!T}?5a3hGe7SrSUqt3@MXE3KoHpD-&DZMo%!izrv(iCHzhIyPJClQ!ZXdxj%V=iM}Ta<rhLqGg;b(!8"
    ">rdvT3nq5bWaad;gPx&H+r5`5Ys2G0e_vXv3{jE#<--~R%q>WNSb;UTnmey*VMX2~7JS}Y_y75T)!Q2nrWdJ"
    "zqKsfa-)5{dMH>8axVNv86oXvEShot$Ls3*t{87~kwOXN|^>dr@TiG(70_AXNhbwZ#f$|BJbFO%$y0@Aq0(>"
    "RH!T;e>}B~c9CbA09|Z^Hm-`sroL*EHz5Njx&2;qo&dvcmjXOc!OG*pKmb8af}%y4$ao_HU82S12UuWr|W2i"
    "_^m)v>y(;y<8Vr@<DgHJzzR%d6~XAI6ruO{EHg6l}bAJ^iN>Bp=I2*wS9{15B(laYzz7eBZ}AUb}#6D;8tKs"
    "OW}XGc2H6t`gt9W_E~z=6Ez&hX<QD69Y9eERGvY7de&~c`wqrBfGh!Euo__DPN<}UOAP;F0wK*Io=`1vt05z"
    "jy!2S@Fk9qfqjJE@yXq(Vt_xA9uPV<OQ0Rf%k4p;sKs=E>(Ibtz<8{%~DV&c^7mLM^_}b}0Hi|n8(Z#R!eMI"
    ";?j0>&)3>OUs&~`AE4^JJ>jhK>?kTj1|kbLZUvMUB6oj*~_X*N>tCam3!r=OVO`#ZbQ-S#+}FSqNx1Hp~H=e"
    ">^AKGPgmT7$#)1IM0wWI%IACCU^n4Xq_6);CUcn5VFT)-}ZVPrt)%Wtr%(_kav#u61Sf@SnE<Wu`&Bou%8#X"
    "7P2J<$A#HrB6^@051KabGbDf&X?nGJk`S?DwSJ5^ybU)<yN=SS8YF_`X}3-$Fcsrkc8wvb-87>;1GXD`%}|{"
    "Zcc#rF0$fW)wY!$XL$tCgcrzwgxAoC7=kFD&#5bSp+Jr<WRNsb_eA%Kc@me{$UWX@R#pQw3qN*t0*@9Xm|v-"
    "F_0V~MHA93*z1-^kEsN96<Z|m%tbSDeyM6TuUvUv%-Cb_szK?A!4}4jG)ms5g2U~Un9s|T!cDwF`G23%U)2>"
    "YylR#Q$Yuk~vE?_y<XG>zqyYeT_rnxJR38KtLV50OGGPyp8DL542c)}RhbGn~lda5Y(+`_(lkp1Iq4gxe0;5"
    "m~iWB|t`95wJeB80QyaM<jv>3FmVRNp5z2{3Y#SqG0xb&?m(%7rUz795@eSpB8;f>~e#An;9?-{{DezMJIXH"
    "HjcBfZTUk!*o2&s_WUfaA>|N(3qW$b;lVU?7X}SD+q0zA9_WZ$Ma5C7MItZ-bw4eetA#w3}V?)flIR#3O+0w"
    "X|qV<-xsE^gHYz)i%X;PR<6Fpi<H-Q4jz#}{ScVcUoi_hw!IH=o)OH?z}1rBzngva!T$!Pa}%f!aNM(^$4~d"
    "-QqMe&!^;BYyw#`6t&@ZEUx|mG{q^kp@XhJ*@i~1ye{=Hk=#>6`eem`d{_Dl?;Pq>ODl<LaHIDh>a_i);=f5"
    "7mr8iu`UMXpNTON-0t~0nlsJ8&;ibW4KYJec!sF5yj!Fh&2(nb7tBc8~HCe0RRj%631;AYRi{_(ns4B#e=pq"
    "GP9wR)YkT)z1XuU_h=>y|f~%|HzqsEcA*uvH9T+JIQwKLgRswDHF2UU^lBP&peo5_|6i(q1ZeJin!Ky^e3Af"
    "tgL^H=TYMIIB_lwJ4)(0rjQqTaQi->y31tuW1Bl0JE}c4~Ka<O!e)s(3qh$$T(#RDl=`(vm*ZJ3q1}Kf?TtP"
    "@di93GC1RifG~>`s-Nf%$gMCdFr!we7is4ix<cVgXX8>rnKRdO>g^Eiv{7X|e_#U)>wre<7tL>~b4n$N3w51"
    "mi+NA|Mrl5pJ_ptQfCZvriFqlgTA`CQL~BX34FQ>`zl#%7VBd@drgp@e2)<tz_OhY}BIx|OBU7KpQNSu?j|+"
    "cPD5^1wz1Ed<fO&!4FId9I%`1rYzMRto=3(hqr_liJGGBM1w^fE}9@Kem9}E<zCj&Ue^pn$*V|J(wqrO}H){"
    "a(8W0mX86A7VcS+}+yJN8No)KMDgkN9_$M*+@=Ll;CH0heeK5eGI_Sg!KOy`Y`JgFj?kcI(XzT!j;MQf~zyk"
    "xp;DYh8A3^b#GjWZc9%OD_Zd7^kLZxwpEF41z@T`mK1mh4(7h!{~yo{>*$78Z@(1FtZ!(fUTl>FSLyCGjG0z"
    "20G8V0c^5r1BWco%B6X=sZa#k$=?kx!~%YV5_xcT1`ddcYy>#0>T>JV!O?4|Ggg0?6dQ5=H|S$+kBIM}L2YS"
    "!#D)blnKfLI6K?;aV%VtlidX8IPDu;gq|du%RZ#F=>0^b#G#hX>(?{6YA|mUtXdYlbFJdr_h@c}OF%dZ`NHy"
    "8U6WwBfVMUR~><Xb!?@O~7S}aDKKpk*Pm$k6;ZW~o6AvxpT{(F0TK0JH(^P8h{ENgey#zpCU$Q3FF0tNq1&V"
    "vNGSz~B+VBCwhlabG?G%g6EYU_u#a;+cU;T!bh%xZX}ml&w$*|0b6Lq|j8k=_!+e}jRl^4JZA(ZJJ304rR3)"
    "bm&csDF0l{%YgRyeU?&sI&9acQ4N0ogNPVeDM0{<?#IQ;Ki?pr^D03KOY_bWq5k{U+<1iVe`Niq{Uy$#LUQ%"
    "A=!OhYX}uy+4}~r2Axy>G4RTQRp}7ZEu6VOS0+e^>r8-KA;k(Y<G50m9r=hoZ{(P*+D|G5OY5s`a-fKF0V><"
    ">{sf?**`h4OL$WR2uWjF_a>M~{pyzT6svKdw6$DZ>E*I9?!F>kgDs)|249wN5`7Asi#b&;R&yQZZXP{Kd!3~"
    "=P7m@MV5ED)-I(NC<zAd(EZ2sm)J^(FYMt+cy9AQ7oh=OWn?9!bty)Q7bOg7aFm}jLRa*O#HHw;=v<(&W>J-"
    "`?ABCY4}M`w0%xpnsI@rgBtLNhXvMvF1JVh~esD@zfyn$&qltDa!644C+#pV1;tq9Gt37ludTCpD0k9q653g"
    "kaM2q=9mm$3RKIM+JJ^(YghId<(e)uqFZ@-@-^#iVT8p9oX^9t>R|E{KtvjLaru(N{G07$7ADv*hQVg^lxD1"
    "aX|nJ7z%QMLjcAM^zoAR&~?;*`go_QrLaS)RJ`@_^x)OG7~()ImJy-5;z^*M1^CiH?F4GO4`Y7n+bVdNl`FD"
    "^ejXj`UsHV70OI1K8n2cA8VTOq^fOjL{=RGIvT}QMWo0TFaKQx{tGCofn|p6>&s%<nN-Jn;_az?TZ?Y|0rp&"
    "vtY3n+Jlq2}bo&UbNxVr1gjO-lHX6j5LiHnlaRQ);Jq3>39w|(1yd??O{d}X<9G>t11G%wth(1lfIt1<8UDl"
    "xb6Ef*w*A0V9sMyPYSB_(yWlFI8UjAn6(dYIw$Cg!RsQq2MXL=2_w`m6oX_pANi_iDc%t{Q&~;4Zk<W#AMFD"
    "L_@5>Dfr<A`$R8ciK?+?pCUAS%HyrD35XjG$|6Zi8N%B<V*e(-m=7xC56q*Hhn(LL}MUx{JlGv#ZekxPfPk#"
    "PUF0_5AdqR*J)EP{L?m=R_PmrZ{1)tkj4@6+lwhg%<|WZ9JNb+9`I`i4U~6!dUN#h?a?p4qFZGaevD^}*`V)"
    "pXmH0uSGZJtk{7bug6<{Zt0XB{x>TBx(rVf<#cHo<9cmX}zH3AfPykn~Q3Dse_c38_BX9QAJEEyh4NY7sXUf"
    "e}zxCio`yhez_)-wDK2Ru#v(@dU^>?4Yi@IH;yxA7gsdv%A3wW{dH9G(V(Dyv(HM{YOT_pR4|E*h!8?U{(ZM"
    "ea3-*N>mQVAx^b(zNi+Vu^uzQ^CMzVBBb&9aIjYOX>DU{fx>IeL|4Joppgf->KmyY70s?H{l}Yr}?JK+vr|e"
    ")ge2x8A)y`mc9~!(ZMVy*zmP;t-oyEKu+pgwYU-%+sGE$^?GkafJ#-sG7G5>P;^+KQbq4<(nhOeQsu&pPZ2@"
    "JJh>2X4-AU{Io-Xgyd>1+et+gM2b3}>o}%hLv~K3;T(Z$S8PQM^&h<xi~fHnZP(KCeuMhQF1CNFPz$`zsajy"
    ">d>)h6YW>MNd~<UCS26m&IedG5RykbvI7b_9Q-6s6Y+JRNB%iwp+7|okU*b=`U*i9dUE(F6)8*l8D5)Ym$$4"
    "h)XbRu|IV)Xb<rqWy)WEwkE**bZLv;UQ4(PH)EkFxADg2unNULgJXw{;AX1bhaQGmD!-Da1j+o$Wxp|3h|pl"
    ")Ie8{txpc=9l3^jZHYW+0UCm)m=rUyQmO#(2`=n!9SI3xF=DB+A*}VOi+ME$RY?0W2XdhNBGm6PiJr45^ssP"
    "<oHU7sy+d<Ryc7Hb06g585pa)JoO<d)MR>8)4M`&zXC5T=>s1nV&tXI{f8_owgqyqPo7A>m0+Z>LKaUa90*="
    "Z*Q-yO`lhCZ^wII<4umLb;j4x2+~j-w*RdE^seja*ZfpV;<KbZXOl^xYiZAX)`sInUSwd(@Kc=OEzcf*1?^m"
    "{N~>q(8kaVk6-D#@(5=o3Gv1qKvuievbIvbhm7a71*uLI5!W-+V$>h9dHecVAECoZ`4axtYz3^=p=j%%oPrp"
    "kO|IeBPTnG{X!HgCJr!u~gTSUG!qOj3BeZkndn-p5IhtWb0c6W9HO3?feCPT7N7lWt%frQs#_i1)J=<n?{u{"
    "vP;Uc23gkBKMt+xvnx_h*}sw+Mr;p3J{x03X|QG9T@(9xI^Pt{p5k`Ui}*zq7-Pstt|8;k9Gug)5OW1&OoDZ"
    "2k54yea<ku*zruDy&kzh-csV=C|ORtKpXz^~T^Z-J2C7GEg9>L&AzZ5TdPV97Q^<@!^*c0AXfUM;h8<gdu6c"
    "Fgt@)C_99)Q%Bf&vLhKFVIwwCEz-ilY%+*VGLX@VF3!$R5B~D=;pypLp_fN3y;$`z%0PAWfP@5dsFwc4N3az"
    "fsiG&|6DSE`k^(oPq=J)V$H7oZ{tG>O8d@kTpLzK4M1~X>m?l7hoQ$(ndjvq2cvUMu`b4W_qtvmK)Q{@Pj-O"
    "q`%aC=#Ztc&yId%>2_rZ9o^fur<jThbHot+giF;UhT1)sC)AMfo|^p{$Btj|_r92A@nI-exjZL1pd-$O&7+J"
    "D11fYq#C%g`L~%dOiy!+yTeB4PJ(+@yTjxAWQdcfKoV|K@@QOa{%j+Q<}~-Yo3&vxSF15TWiRMf^-|j*X=Ve"
    "ur~$gG($+I03*6lzKNF68Md)6#}ER5go>u<6($M3OK!i!1}?*RA`-UQwcNt(zizXPi42MhRe^yGPO6Ay3tuW"
    "fG>LXGhjd%m(u?F6h#I=q)FX_Hj^wL@dDk|Ls8l}+Ly$mARW0LH<;0G^Q>;G{7j7vPa2k4Z@z6f?mk5HE0oB"
    "n*l1i%hnNHMv2(fAGh?&c&jS6>!*wtP{(g}raeC8fyhXlX6AK<Py~VE9SHFAw7u9bco~GkB7TSFINhOrJ+Ij"
    "t^TjE=(^8NnzOZDIUQoRRV-ew09WWVCNrBYA?7)#HYRGJl@+bm0lQC!R+ug%bvKLa|=p+T~M-i+B5U^uGCNR"
    "Ty==)4o=*M*<src=qCz1-qfO-ZVsF1Kj=AtL!x<0^zfszGHa9gz-(z)XoBobq1|!2%hUpJ^ndp$rGtY|POsm"
    "A<_!2amB>Vs$QHpjXYo7j9Yw8X0}O(;MTU^`cNT@Kr8t2`iOm<&b+fs_~sS99o&04RfPWeyU9St}^UO5VAJ|"
    "ci~o!%h>@uM7#|)THn9qF*(_r78pYW3;p{79Bg5NXe6z=pi2*Y|K}2fSIEzH0VND>I3o%SB^F3xr@EUE#~v*"
    "#kW24uJy_Vru{3?XNe2wyHN=Mo_@XKys>lX6jYXMP<|j2#0al$ShWbEd>x{yvc9<zyZT&gE+$xr7In`x62G>"
    "unS&m2ndT3jZr<oNZs6-)f{VzO6qcE!E6LQaRoIEz*pMDn3hi3ejTX!6O?v7W&79H`!<+AW9-(K?71()x6$="
    "?MRLzm;p1W*#rrrG=rNGBWj;?xG`u3jKenjn=2CL+^>m*LLwup!_PdyAMX#>=g#P7p~6&EH-4e^<+{JQiK+h"
    "Nn80TZb=yIfTFWFd26*Q2mEbyTO|7Oli}`F5e#h1^(>!1NAt7>fN?a`|GoA(5(cF60%u=&^btmdP1;Jt^>cL"
    "00L>8_^Bs&GM+?+R38{RcdBa{(3=L>Mc|@97|up0_J)aE`ON<h>ZMNNktUdCf^4^tLMUhySg(Ezhg)TNI8HR"
    "7mKX7Co-Bzc5LN@7enMsFiR%c8#H)d<0DW#I{L7<PuMSTS-(qrB|1mmo`p*Kj*EiJnnqah8*XQJh!t^5YBv&"
    "7N+V$nK_2i}A0!*#-DCuN1e|<Lo@H-p-o`}<&h=czOr4(6XJ`V6B#GIqdx^b9Bam4YlHuciAGdZaS9enafb-"
    "Bga1`ca!NjV^ylO;a&qbvR(1ho25J=h?%r_k!%*~|8<gDRGBL-v6c*@bL4f&leSDE-wC3*4Hmz#djbJy1@9O"
    "+}C%gO!N}(*^@=*Ig%Ow6Id2?$F#OkC=#I|I7Z}YMZi|cxB-wb%J5uiUIwz&maE*xr}jY&~Y6Bqh_QOb;R4}"
    "SmCjcdT+;DXQ&K8Hpfm;aoh_Tc*9!~?E4fYrrcx>skAW$5s&C?1h*smR_0zZ#(T7+q11?Z!61~d<8ZSzQum7"
    "0KPK3fpUu;@p;l1yv)<m!uD(^eO#eIF!@v7gJ6}cs;4+rU{25gF&>SSsiMpDG5hU7pdJ)%+52$Fo%h;*Fe4x"
    "t3h}e4vs?dWp3#I7kkEem!g+F$8_>cbH9+au4@33Nh*T1pY4uWsPTaO1oCGn~eRWybH{nMx;K!1Rkv=|8r5{"
    "{S4WAcwRI3;ZJgEHYCO<guYD-Hka-SPRMU?$j>+4#m$&8p(HrKaTY@9y*ki0N{xUx8q%Uy1jiQPVBxZkDfh6"
    "^aKNM`nHL>w|8gR*DB_TfshRimXALR&=ciY7a-D0Yx4-1bI+%vo)g&Vs(~z9*?!NFVsV^J+}l0o0n5PIvCa9"
    "PwgWKl*`=n?6#)!-R~Ie_EC$gZ|A+o{jcGyr1-u@N9E49y1l+5HvIeJ5SnUK8>x>i#oVb-x+&`Zj|@8Y!-Y%"
    "{?SZm2;GINw1VyaS#*`?n0t<6r?Xbf31{vB4EW7S?!6mIXiYIBV!r@e?SGGib^<s1@)k~fF3XGu}LPfOPHjz"
    "{kxm)Y^p2G%6WUly98{vJ43!IJqwqypE5Zux{EO%u&I>j#T%1hg&s&(6ZTm6(SWm^|~z9_4j)aqsYR&4&s_t"
    "cW_KAw9Q7yhTsRpGPLRB-shDY}agtxdIPE9dQ-1g*wjoN=;AaqaZ$<c>57?e&}yFJHvX6{%%{v}S>D2FNCv5"
    "w06<O>&6w^RwHlu@rSnScP$P-Fx2%=z2>Jl~T`)B;2p$8hmM<<jNx5{hhu8lDH{x?n@Cz*EZNUpHp$$tGDe|"
    "+pb^&evRG<_hMD1Xq-~`XgDT=jW^GKM0RTnd;7--4TM9vU53Stsr0(FzlA|#0KY%gvk=`00e|BZ24EO${)LN"
    "x5+*Q>^^+}tY)Z85t6irq@??>YZBb+oj@&I69hVYXe7>AR2`2zXuebXh6nN9yfkGT-AYTZMo$(^!LOE1Lc)r"
    "NTRDj0m0)o>}Frq$Q$Vb0wABH+I9`e4*_20nzZDEMnfFURnF`&T4MV7#Cp{Y~|<}76$bowLQA^bHJm*6^8I8"
    "myfJPU6$Ps?DA>)%mCBOiiFwT^n~ggJyF+$?n>woQplqCySq4oD;Ls9}ozwa&LGvntHvayo;!t6r5taCC$|S"
    "k7G0Mgf5YiZKMHMEG{Tz|!6LHP31cg`_5n#9fn~dO_VPXAh_{L$X65IVYq`!MRYQCI0q;_LX|5k}RA1BaPzl"
    "I?alhx9dSFGqkiDhp8IL@V7~>$5CRGI*2aY$Y%VGr?Et{gEJCQWpDJ71|`TL0hW?jK%GkHMuiFx{1MAP;HvC"
    "#0jJ(B35EvqLYLEA>q@Mp%Jp?zlsQXzD{88r=S$dVT!yB))526yfyHrHhDmlE*sU8mdzQ*Vk=iK)>#0++65d"
    "YZ@sui<L&acRVNf2z@zft!63=2L*4xbP=|W>=s<<#k*9*;9?WRn5u!@U3YULshFl!qZdAZ1esA{HXS-xD!L&"
    "jO2(@jB@M~JAnf`FR9_1D=MzlxRo3>#b3U|LB^wl{_|`kp$|37iQmHHYQtxI7;%%@`{rLtVRqDy-*WdV^Zrc"
    "AOSEEn;kN3+n}YFbU@Y)rnQOzN4i|^)06RnoSEG&xLuQT$n<ku@x-fltGyTAr<X>uDGC~J|dT{f37IJQm946"
    "%u*=SQ@Yw_u`EhGQw*j>0jGT2k=iRy`N^WhQ$-7O>%jC9qKJwzU2qN4S@wYjlZQ|{4eDZKNu;h9aa5nQf9OL"
    "f`xnljUVQv_9jV_J8r(Yo`h!R%1@9RyX01O!hXDpk<+C`2qd!*8DPwjFWnT$2c-8TKL4;Q2Z48|f{4-Y5ywO"
    "XhYKeCctO9yQ&CVeuSGcM}9dNzE#Q`))Geu70ELWW4##P^Ptin^s61JZ8S2!kIiOB*~9kb|Mg|5(J_*Um3Fi"
    "Dk7*vwbJJLh3h;_izhtS^d(a#POmxRo<Bol=J@{FwqzfMZt6d{_CxFT6#hzT-I8aJi4ec{qv_G^cGMRG5QM6"
    "nZv-@H%7$bg*?2U&p|uXf!AxB=I=*pyT<J5ewKhb&o_+iPSU(YrwLn4%0~n#mQjp!Ahq_iwL%jm8qTcF6PcK"
    "isUeffUzj-lB{40XApK{`4ovhNj8S_eM8kX^K5ZF_3y%ilOwzxQ^Oa8M7VFEJNN_YYeMNw38^80%u;`y9KaO"
    "DqLvYCzgf7H&Y*}V^ybc)bbhngVUo1*2FPtOoH@HKo||r=zsVv=Z2js&m(LX?2Go{R?g_J^ypGQ)m0w}SMY<"
    "Tcm@f4Hzs`=|x{JG-iLmGf3l$++dcY&!M)6>q2g>;y^Du{mjby3jz`_7c07nXQa=)Y9+EWKW-r2mQYZJrYVF"
    "#0N1E3N{p2N_5h;wb_CWm+waB1nCWImCt1$=Eh)#IB&MHzAz+{p^?==p*M8JRZ5CM@JhD+?I{B;645f1@@wa"
    "zZGRe1D-@y+Xd2gtIt_Lz0MeZm%a4^+PH^t1NP5D_1#!6OVEv)<uUAB7UB7wA+1i&k!5nO-5*ZYJ}NkK;J}T"
    "95-z>++=2rfTb*EVJbtCYnqBW+x<$@`3WEy9q}U4R!gFsQkHka2yaL^bw!qQ5&DQlwKy%qn^khiJ2qM8I!tK"
    "dvU2zW5<#SI;wEMvjk5B&$SSxw#Y)U(pCp8RrxqzH10d+bG*@z&7qB0J5G=fDSzx4OVOCb8oH#9^ey~uLuj@"
    "88bL2=(xLCrlG4PUt_h_9Rd49B1`eQgQjT!{7p#d;-p9v4yP^zXnnS)@5LLpp^1-)?^Ur#}7WRyc%DEQ%ank"
    "Bl4I2G1IGBo4L27-dh+Yo6?V4etQN{TO&DbeY5IR)-QTDPnPa~U}LhEJh*=WHklzo&Z|Gx6J)4aa!%jw4o5s"
    "8q)&N1Zc>E+)4FX~)EWQV?6fvZ|yG#)iD+X)CWm1icm}b4G}ItGUaY7)#3DhBq1nt74ihl1PoSFww<WuUA%Z"
    "*D`cp-oi)F`vvKJ8Bzd@R#bI3g<~73NfKTc0UPT8Vh%9ngtZ6*BBMD=Arah!Iam>dDHwP<lnYb9v%zzTnrLw"
    "o&*xYyo&W-iH9SHlL~3>n#+K_4VvL1FxLOJA7aE{CdRYW8d~{pUBtMX;WSnOONF#G<jW)PqwGe_+gVQ!zqU0"
    "{aXLeIS-}zF_+$9v2nBAfuu8ldoV7m&4kl`F3$8%{YWE?H{=4?!24yY0+dI{M_OGDgwvM9V^iq=kQgtIkPo%"
    "?r~IufF-Oe=CFPI7t<;HT{PLWQ$W03#<H6{w&vFD4ShYqxP7U<pxG289ebJZ*2(JV2U81F@p%f`mA($%oF_K"
    "L;h-T)~ji7^bC$dpKXDDav6PFg~#7$eLi@3r*`bOp@(!0_GN8piMZyvsMnku9-F_aELFZM(Dq4V+3b2$fsuK"
    "&=`k1)*!#B@nVK1I8cOjT;nILnv1*?6gLD4qh&?^ki#9wwhhc3Jv(3(b3|&kHh}XC)B<pVKDX9~MLO0wa)fM"
    "Aj<cCYVVZs;nFJMlvWA8;xJsTY&05YVLk~4*u?Y*RY<r3fo;go=6~j0o&mIXCL_g#0gzZ1`xP@^|&@kCfr%t"
    "t@mbs;5BB;Xi)+Qq<Hc52IcbDs%Sfp&yTP=B@VDE_@XsIF+srWk0K>L_wDHsb92-{d1j}mNca0iub%;h(*V#"
    "02+;4Y-(Ax%;DBuN}Nx#Ku_jymb0B%P9MBk1?fV|Zn0nxIYsBOs<@28N%^vtk;CjSZR^E#*p^C`c^0O}FvDb"
    "4CH#WwKOn`;cjYt>R5iPd26NT+j|Zek3d0Q*TkzK>&l0_Kg@lTe%Ksq_RWe!rY36!s4+o*3(>%bv!RO<S+f5"
    "b_RnksG)&lY34J3A@vt7TrTj{uFn*U{6qW!_5o_wMS;5Y6t;5$tkVfNYUeXxWOf}ds5i3`9!fmUKHL%UWXX;"
    "-j<>*T`D3YbxKZpLB}kjV>@ZKlNB2HKK@|b1Fpj#;jGx483k;4j4lx@i-aFD^gwi9jwh&p}81`7gj*b<+6y("
    "t(<_nH>1UC`EFHo!$-qH*+MWQ<l9c=Omm;m)$Y&Li~dsF2_SuLAbEskl9X&FW_bTF;NB$Df5jL~UmK$XZS9M"
    "(!p33=7XN1C%-Cria{wJ=2~2!|Aqoz-{7(dwCnVMCExq!dhGv{|GYGACpzVLqO2%L?Ic3*)q0=~f}$rzW{Br"
    "u5nnZn8W^7Y)-N`8aC8ly(C&;OozhrHa4=n~>QJz$k$`40yH0H6kw)$$umnh9!h5m8(S=GPq5&W2($G@ut)s"
    "6Sn4=%RtZ~MXBViHlu4{BbL)|WjW*nX!Y-c>5Y7)RK>oH)0{fKFtI7mP0G}I-NvBla3mG>1`=m>vzVLXXUrb"
    "L{w*{`z(*2sFFczaTHz>d0WN>=j0q)i^62z~4a)2Z{09t~f&IV<R(W{q`e;eb0!auYE`*#|CAS=r<U>9ate_"
    "(DIx-W-nF=oyrXz!ZQL<W#w>nBaK;XZDY(jZ&V05iH0BVfnCe3bBvCgLKRvm$C7k&ui1dk0~<#9oZJ18zWvz"
    "=D`2G1~gor-BZH^%C^Z#!_Ty0lQ3gzqBIh<iO>qzY70>I=;r&YB3T*XcNeHO?00d;ux$f7k0Y9$ad4Vj&$UO"
    "_V)80!sqR5(VfaPBGFF_amB`<QIR?e1^%fSKt<+A+_=Ycg-U{CUS6&&p_@=b%|0rFyt^!ft1hX0~&9^84XY6"
    "dIlOZNF+w<Vm~|}1LX02IK_%^5Kh%^RewxI;|e};4#rRdB|ynqgnx3(v1}=2vNwzaaU)-WH|jJlrnvKD$l~("
    "{P=kH6u+68-A_itWi&Iyu#a9EY<2YGFqS%lhG}04r_Y{HSpK;?#Sn~dm!9Eg4bj>&6@r75Q^%=<kMrT-^WGv"
    "GKS_L|wD~g3D;#)ehI1n5kfH)5sDPM*ZBq#GhV_wq~!cbuzN1&$wt8|qqw2Bj!1PH@fp`&C;KodH*uOx6?x{"
    "uMZe9dy0<v=+UjBVpEud*CN88~zTBoi}t!?)-+_NRhKkR{P`6V#$H5(f-mXoL`wTS7e-Uj-a+<sp0k4d=#x%"
    "fZzfu$f9!2kGQyY_N@h`9wCs3YG?pxfO-=K!HDTl;~}QE8#m4HydU&lAj@DUI4cN>V4i6@UTt7IBC~<&`ySS"
    "8mJrwmN}h<c??k^@KG-HJOdlUM1}b%E+HUWC1HBK0Bi6hjLYd{k)$NjZp3dMwktzkhJPE^c|qbTSQX+K8Uki"
    "j)v2lzmAWtxQq!;?W<H6p7c9O=7|nXA#c_B6Mc8?#;@OPCgvk50Z*Dza?~Fr1Fv33&xE7N^t>9_8E@hi@I7U"
    "nX63$pf1hvWJur11L2F#6k7kMy<QI;A-BF1!B!I=(){Xq6Y9MSV6VL2v{0Ill=jmEl`CCUL^%W-OYu<#3>SJ"
    "|1K$zNG5$;e)n4w0Kg8l>|PDL$hjBYniBSG$dNP9993PZk;?(eXOBtHUCR&^*1iPGvgc7@|e^49~q4G)YuU5"
    "!WCi-4?gu92G{#YrQVEOw<?NPZ%&OaVO|m%2EW=sP8Wm!$8`R)!9P_2*KIrPADTxh=is^N|3UVX2#jdV&tNd"
    "Rddo?=+&%zK8N0y3g{-LmQhj0CD^M#Qb;J$>N@8r8V!*t6e6?{(`k=ZqlEAlUh^|+6xP77g3lZ#;Es?K=oD}"
    "I+e}f;_FBeL2~7eIE#AI3K6{Dr<x-1U;>6mXI)x!Ch-1e4dNIEZ^C<BB&V^1AvJhG0Ct<9lc7u>z7q3?gN6J"
    "`l=WA-E@`F)kPDB|^0_6`kc$CS?23s%)pn=7+=~iVfuH%4hj6Ifyw~~XrO0q0hQB09K3f48pVHIfMV6P>t)>"
    "?xJtLye6XIGd*WyA61Kw45S3i~aaO`TX@-0h`K?F!Dp^;aELoh1Nj0RR>}#h2{v>=<MN8;B~ywS$pdI^I62n"
    "1*u*d_4R(Pe9_moi4#i55`bCL>=a7q5iJ(%n^Ev6>^4MXWCTa%Vo||@a_T!rcucYCqjJFQ{EnVCh4>-=lobw"
    "00i-%MA3Jp`GZVsC}xq@)fj7v#Y}8>o9Q%SucPQQN2!CWq{J9B3eGboSKL6x_)p=}ko~1eMrIQk+rGw$4s$d"
    "NB5ppY>>!KcAr-qf1{f{CG&Wz1lDHttXVreoTF87!;SzNdXPjoysvwb{;pSe<NgqL<pmjbI4M?0p(;Dntv@N"
    ">E*fh&$p)p1uoE#bJ13ZYrk;}#4WTs<?@PW1a2^R+#&p58jL)A#uknD&U&I1-JGf8l+U<&Uz#?;3N8$>Cuz}"
    ")#I6vl<ti<C^{ZJgzV*Nx68x^6;Dgv9(u70<$JaLjQ8NJZeQ+o5T#o+GDWpE_+HyJ{xZ#3tfAcbr}<D>seW="
    "RBvR7<2cKnXFM=2Cs}PX;9HyF=qm8{S*QK!a2H=k@si0k#0C374#ucWHd$TC7?L+xf9VO;W+D?Rk82rInMPc"
    "%g}*rtgo(D$Jr~eaHwAJb!Du%BEf9RmD+>ZYMG!d?CHRG%2uj5xekkBfl>qAksu;ZLGCo>Ztyv7Zf7y$lcY9"
    "eP~dJxomqDJzH5+4mh2<oN*(O8r3A7-%q@#TqEjVW$GV%#<#TzTWR=Y*#~E&Xv}hKNn;K$%U<Zw}x%SM!5$w"
    "%X#+VlrjKp3-;e-@Nzd-h59LkdaHlu(ka%G5tmxUZ2vn#xyA%+OLKzj{~tu*tvfJhTG<8TVzdXw&}EFYT%Mq"
    ">}T%V3!cGZpI$j^bhr+$9|^SIRXU#0qQcdc2X>VA>i+(IfXOtA1j>GufWTmlp@d(gxTjVV?%=h8W;B5Kj_V5"
    "k#(&tzgaOK-{(dWn+Xfv2CQG8Hbu9!Ma{&>)>K%CU1Qvfdh$0LpD*}=dg#wfFBJZ7^R?S71z?%%9dUuM}b2K"
    "=H_&~6swEt9ukvjxQOBsDzV3a>KYbP;7^knxfvE7%Cl?+T%7DdbsY)*xw=g6Hc3ewVoKuBM=dyAq}0F8w~Y}"
    "}cpBbY3^tohth2)jabW<19weBb>JPC7(>rlhI{`58qaC)V&K4tfn8N0^$|2+~z%^2YX*9|{_Ab+N_fk8h*Sr"
    "o#An_ESOo}ty8qv{s&t+A%Y0exv0mCym3dr(?LrS6;4h`HV`0~hJ=b$Zw0I^YpgK0uy$QS8+>EJSaEN}C0z8"
    "Z=FbIKXcU^oPfgy9e{Ly9c<pgY|j`BE=aXZ8bnQ7>PlL+G_rlz9jG;l>DgprgeYVuS>;Vwi*@Uiix`gkrKiV"
    "k+<X!P#$TKDZTjioG=@v{tZ9pi7&riU|IfDXzad!>7-~;-+I?DgY$B2U*?;tp}Q{Y_bmn;V-ZiC`*AIHV}P9"
    "*@amiUTX&}LtrY0LokI7haC{z0+nZwug7EcY6>joI`4FQroG?NEk2M;dg7cJNQ27nKrA)DC?UYlBHu`M3+IV"
    "w9Zwj`%g7a&<X(Rrh~HT6-A7V^ul;Wjdln8F$eKmjyA6|@4*W3>a{$DEVVR;s-AeUJ5Na{DD;?_&^b%knuXq"
    "7o>PZ*rjLHDUE(`{CrWdXlbk$t`V1o6RaW)8r({F@^vCuLdN0PyN6oDQ{N+ip{goCdW3iq@ls8A2q0bc82s^"
    "Qq>ks-~<&aTofCdU+@>B0Z~Kn=%PGNgEI+Z@b}uv#9qd&NA7OZ=$=o@folww)>+c074_J(D+GZuS0_#c2nhy"
    "TCWXpJ}GHRbTDX7<3v9>D#MrZO{^`H(>lCdnoufJ`;4n-NO)1sGx$~6VwKM_FeRBLc$q-*8?~HywhF3p2cE>"
    "QDXL(n|uOr%%yQ{`EcvP1nfQ*&f1G~gxIq*_^vnj3N#<=JnE`9%AzIhze+*4ygFm&fs381`gy25?>|7N<Ua-"
    "tJIz7%+n8u3@G#QjBt!#jm57zfLMFgO4a{}KJGqUBvXlryeR+vcF3th)Y3y%tg68vf@s}KPl$kJC8M4#U`1g"
    "hOrbmelQSGEjr9f{WflCcQ0CfSlWM6TQ-5Mi@@wn*QEe+k{wlR7AGYP?2L(q#CInAJQBrES>T(b29j-AkzcW"
    "GRzgR0BX#L~bJ#6rR&@#xM{)xaZ8BQ_g#8L$C|9{?Td;&O{&(6+7u?`v%8T_7-NVluS9=)p}l7S8o&n;nS8C"
    "KN2hdt%e=?M*VS%(!_M@UUdRd9#Hr=0Rn%@0Z!_91Ng?%dHp3Z{HriI6r*p1G3=@=Xo~L-uoOn;g6SFS9RoU"
    "Y?Uw44!$EW(F6ncarts+qd~5`;ffX}<Av9)A0bMy)YcOX+Xs2Mb$)vA_UzT+DdM}!SQ{W$+|d=`2Sdq_fEB@"
    "1UpXuErH6+5A8@Ol9Gsn9ZedLJXP%}oHOip}y<yC5(3MG}MdLq5K4p`Mx3!{dykG|Xsfy`e#ffo$b4*dLFj{"
    "LT(m%lpLh~xnpvOBY5UYYTSnJg19A*rLkZT@TGZ@p&hNE#ba_G{my7&E6?J*Z~=g&uHM?b$l9Kz0kv*<Lwvw"
    "za=4$EoR-SZAmkSr#MQ+YYO(Myu?0!5eSVAtXM-i8K!%h}M~{l*>QsAkSsnh#oC0+Nn;*ShT3-PIxqRa%?<)"
    ";01>P3viXQs~^J!&fEsy2q;<7`wUu9$qKuRo+_N;a!q?*A2F1pI4r%Pe8i4sK5DY-F!BXADQvk2seNV1xQf|"
    "mXhH7xKMVXn}wT|W34lKX-(z;<lWW~)oIkk`ANrUP*ub0b}lww{WPBL?7F6{mqw>3l#wwpt<hi{?LK$HD+`X"
    "o6ayu{vyQ+C?y{9eAhh|6A)<-QXuWVWw)MOc40hvI4JF3SSKI`0ip`I2D754x&ps{zxy!ae+Xyu}NSDUYRmI"
    "@{IV?06T6%$DH7{cTn0W;Sq=F`d{<B|?PwHl;%jznip>3W)z(1@6+nO#ex8QkW6FNCPdU1G$7oLw5zOQy4?+"
    "D7{i(ijl9KSvQ!1{glaA&6p{c&IIK7D*gy_)E|f%-&NmEvY`xplRVOVtCIDEI@B%w1Cq42|R|eEf3j>drM?I"
    "mI?RhHSD^sp>36CPA$&L(7i-SmxnCs)VBsg@Nv(^4k}j(2MO`oHxpQikrngbf5;bMx9xl4k>4Q(1+BTkLJ_P"
    "j@SE3@XXu3t1pum2oM$cd`k0{>vFIIJ|5^*y2Z;XF3KUgd^*F*p>-@&pMLluhlDRE(g-1&uw1k7BUBKe^~p7"
    "1BI1St;cT0K$kHKjEZ`^3?Tawdfh-$busZ6<vh@(T!(4z@4!dDLpjVkUhPcP;qF3B3q-vz}R4e@Qn_dbZWj~"
    "d)b-*FHW{m?Vio0*$yvD0+D`O(z17UB`T<!@Tms^}byyR%qvzL|M7ht3uXGJL(nfr>))`h(F%9hC$dSI*uA)"
    "0z5p~?nA1=|Dt%2Bt2`>&u+a03zd9|P{QMsM%tGVP;yG66%c37PUE#Gx;<92A!{uuES%V^F9-X4q|A#2<dZo"
    ";;e%fXv#;7hWr`%0Z=}`hf;`(XfD5{x)erg*5B=Kg~747_h=#=<xeEcT8wdevz4o@JX3anE7YEHR(9}c=KEs"
    "&d1w<b2#?RO?Gi{kS?zRxOKRRmsqi{MpVX{K|mc%4yDkVrK7$WOjx+?kRaCLRg>&pcYLJ>TP)3JEtomd)XE`"
    "^Y3zL0_>6%sE6gK?mRzJ9goPnbyQ<9G`HBbpk1!jJ?Y~+l+Sx9aE!%?N4)CQ3p^qBQPOny2+Ubn|cZ22uILy"
    "qv)XAx(vlyocg^9ZyUsXMkfqA0sJ<dcO7Dg|%&<r8*x95zEB%x;XQvWgxTvll3=sB)TapkXg^&>lsHoVVw@#"
    "6K)?M6ZXs*&Jm9^{u>|K}py{`<l9|F^TfH{8DZ@qbvTTzkv`sqdmFb2-9}2w^gu_FFw_ob2_3DF|Jun*A0dm"
    "tlw$gG2=l2K^_)htD3>%u2H5t8#AV6a3O6N)T9VMmVl0uESGz-I}|}c8RJZd_|wu>rE!JtRSBgrr6c&(CjGb"
    "?><xiIjBt%zUvQn9z7dAe)?n+J1vE7mV``-0b2+?#^D83t<2CHKe<E)k4e*FR63}Qw?2hBzb@JsHr56233oH"
    "KFaxtnEVtQ3F_Ih7k?Ur<Q@ipF<XB0Zc3rh3d`WkV(>G}4n7f6QFuf(C0g=HELoKA42!!q!YSl5MMqzu=a_E"
    "1UX62B(xZJv1SvXRTBDfBKhSh6Bf{<xGPQGlNyKG!QYm<326<@CrvgfB!QqDznKekv`Hy8t1+f1C{P%Nu!Xx"
    "ccq{ra_C8!FPcVzeQkPF@_Ae>#RJmBy0KI&iD8Vh_wAhiRA&aH@4LJa8!Pz}kM)ADfSB!*n*GD0fk2*)RdWh"
    "DXV!PNd!&P6EEbKkio4g$We!Z2nV}sGwSaGf?O8OlOOdKOLWa?ncD}M-ins8&P;1=<5z3oI487y^RI%e9c(k"
    "EF1J`Zkc-H?rLSH73zFhu@+Z!?`Hk`Tj=1edN`&x`8v4K!)4S|FV59DTU2rAtnsGYKowu5JhZFi_fX3>QO)n"
    "Go*QvrNQIJVjZ~zf19htE`PWd<H&)ZXrmBts9+<ZxWi>a_)!SG3haa%J+G1>?w{N7oWB2QH_jUTax8M#cffw"
    "Rc)3i2M@S0URZ0f3<o9J>zlcQzYK$F9DT=|->s=&#ct~@i2)ml21zD7N9%7h!0!c!*fGu?ImhKKj`H$42x{)"
    "QO_k~T-6+CpHfeGHc=TvkI-yw=(IE8Z5qoi0;Rmq7?O{aq0J418Y+3q^&R*awG&;s-x4^Z>$v;}Vm1DK8B}6"
    "R74ECY#Mcypu>3hjMG}WBC5_MV{^(lU1qHQnYO92Y-M}xP4VF=83+L$pos`>p|gFamEo+$gs?AbXtfdeaB_%"
    "47b{#JaJ{=PnTN&RAf-ZsCogN5gwiO_|xaRdUy~Gw^q<k>IrAb5L-~$*o(RV*<dqcH$DmwsDQ*#(AH`L8(o{"
    "N&RFJ*HU=|yRF8dEp!PMP?p?!6?&@DnSlKf=9udQ9cU8M)h<>!KsN%uE?jIaUG3Xd`Q5+4pvhsExRu5(}s4{"
    "Rf^1-J&Dy|FvGr-S-TD>2P=P=uY>N!+h26mHNZvCp0xq7`AFHH)ee-k77EPCn{&If5?N7z%YZ{7j#5ZRP+h)"
    "L-9quL>L+_kAu3kj+!0eILJJn=h!Np5J$BQ>5p_YDKJ1BnB(SqN#QgyCpSB!z!3ReC>w{K^sL@{F_DOs9<g<"
    "kYO7;B&;KmC!dDjGLs0S4DN?NJGqG5XenHb*f)pElFkmnnD$TRV#Y3VjW4|l3agY!!yj#0m@L4Q)MVu8}Z54"
    "V0tQi!zXb>;&IRD7t2}XhyD|(v^dVvsMuGNBn#n(Cp!+u{V1v|KlN~T`1HxM6+E>++}V93?*w73vDKsD&K|!"
    "X66$6k7~$|pU~?F>ofhHy`wg(<oNM`<gV5)gKRs=RQo!N#R-$i6d$+_38h*AEZ2HwBR#O9y49)%5s1>x1(xp"
    "YvScn|7g>=~geow}YuH-oZ<+7Y@CjU85JIHnEV{8gWZDNA;Q||LHAw#H*u4^*If=JYh3j)C5uu0m2T@&@kU*"
    "PEOF$@OPi3GT%Z5T)td{NWps}icUq598FMwdd|;dSWKiG(HT)F0?3hd#^Zr^sBL!mEqIbrK?glJkXJUHf>AG"
    "_#9FR?&9WYp2(!Tl}-wctBKzN@X4ArGZlnBQRq_Ck1UG_DUeIG?m7jWocGsX*>=X)g7+F`5*~rqbS@L=cVjg"
    "-iQPFBg>gnia_Y#Ob#lBB^izsDiwUw0SyP^!>L=@sJ`22K_UG}M)LvarUq2<Zh=*`cTA<*6b<7TIA(bnZ02I"
    "xp_L7IK?flBHkg*|hNwIcI1>%r5eCix10G*shG#}XfMV{<{!`}e?jB#xwaZh_I-&9M<Sw_)&W}%qr-%Rb?(m"
    "EPt-YRv-hUjJRan#C`O%xh;p?L}N9VpfhCCxsZPx6&p14cuZYf8Oy>RKrPh539+b`3@zb{u0^TqMo^TYpr-Y"
    "`;9IYG`D0B&A~JNNGed1b?$p|*K3yOnTj>9&cdR*880+u_@Wu{B!p&5kyQqLEkr(cd=SW!`EgHzRs;{POVi@"
    "Wt_26Un%WE{|fcY8d`Q?zaB17H5ki7;ybHN#^O)$Lcc=yIr~sDi3d<hF?muv=0oo*_Pek&=?#I-l`AG`c_)k"
    "y-rrKzn6KxvJ|flj$VJm^D37#H1_v?^>&6ocVJW3o5Ht-E^Eed^!CpOua901-<=)&a_Db*>g&g*H#h^1P$^{"
    "f+tXw5F=UhVbjvt$ToSs(&nyOMsMBl9>fujoCUJUra(Hll_;UE-;Pva_QEhpgE9po3`afE3lGXWnGk-XK-Na"
    ";J%#?kFJ-duU4n>cOf6>B<$)#JD5-ri1$I@^P9p?Jx<n<w5CjoavdK$X1-7ieSmj*O9)4}1Jlk>k0PY+LC|F"
    "y<wx_;1OeUa-Na@Bo!&pUDBb9RrjW@Ww>MrbTx&fUqr{sv!ceXZ|d#Io8pjA(^v%IwwQW_B+}Q+F<RnpxfLy"
    "WUvk$0b(pDL8(X(AKDmiS$e4z$@Z)7I|9K<8I(vHGnF3d54;~(xSPas1GYq+vp#@VS@VY;l*jx?s%SKYO>tq"
    "?Ujq=tjy*|C1ZFr>P3!C8NJ+W9OjB-`r|ZQ0LB~Sk)XXqvZ6_kfKp`u+=H+0w8u811<+YRYYFtn#CE~@QX5N"
    "Q^JYT(A)0lzKeX;pHY)ngl;)<6snNJWE5Y|c#Yy(0$(-7>Yd4=t@olH#W(HJe^JtUVxJfn5i_7Ldh<-@pBnj"
    "0M1cJmMobYy}^L(iu;aATYtZDbj&JI~6!3x(?Z!;JF&hcH@6a4G>`H6de$)j<cYQ|JKPLdF-g45=HZmK|nmk"
    "pt!!HYvUW$VE(Q1%>LZvB3%(}zGf#z92wxW*z_v!?V-u5U09eVf1dMm$5DJ_GOVzVKlhHOCs^i7x;|9BS_l5"
    "x=@OEz9}Pd6FGz{#=@dQ5(TLTtaeQQz|D?$hdBZUbQw%91PMBLe;&~S%NKyfOVI1uEUuPk*U2S{3voIg@UZK"
    "23~FfTph4m%mLq5JN5ih9Tpk#Og)((*clwhH1qJ~y{oN>`!ky!So(IFqN`!4_J8^bj{)Gm?f3hS9zOc%Lmvx"
    "b{=}ON(R-TM2N=8t&ehE8Wi{=FXQny?s54+TFX(>XJ~#fYa83a`fI&!zICm@5F4^d<OvmWaM2;SrSwME*-Pv"
    "gpzUns~8U28RnDQeM5>*RP`=Ap5SM$<J&ASmJqjH%|vv>?=m%y?HrVB){!QU{Q-}xF_dfEKV$Reg|FdU+C0;"
    "&Ul+nDO#GJgo>v?Xlt`DPQXoe<*p;EOzP7{{P#T3nbz+B7HAhD5a?9S!iZa$M6V^a;-Cc5?l&cZiQjJxkgv)"
    "h+WaaVsY7V40y;e1dpqvYfT4Td><f<;Cu}0(EiKUAK0otIFz`??L^0xVzaj1mzD#u}#%D^4wODZdgMYDKHN3"
    "&3~4l1yO4k<lDYlC7y6Q2ju8N{MA=$WfkX$HTL7UxUZVnP`BBS4Si9#3e$QeZ(#+cw&X@}X>AWO1a5KuM&hP"
    "Xr<KPm+8$e@kp%#UFwEIpK=mWkuu4@<K`(1c$L5r5GV0U}IgM?tWye@^YIkCVq?;Z`3gZ`oWw~Nr*lpB&oKO"
    "OpAo&67mR44U*pe1_5w|wX_9%_?$9f#@L0RVUXi@4$GhEw!(j0N))KM~bOp{VYQoZ07vO?T-($NZEVuFdwf!"
    "vo*jmOTnlh~TiQYOksRE_qOjrbl5E(Yympf32et6%B6EBM<Bu|6;m2^k>v>!&VG7fzmrgq}N0#CB*7ZzYZsA"
    "L6+|(6l)V;N@#pzuCX#vU*q@?8Ls%H;Y_#0R^t2x>dTTNQwas>B7X2SB}4sxZR}F!H!I@Hpl;6xSF2xr5DHP"
    "d~ZW{uez%YE*#k`!@_wwCV)<p()lP*{XXTYV@$Bi$~!fM`$?esj~iNFHANg^4UXa`woq(isy!evmf0-X+TUX"
    "IIz%5vaU2_f=Ep@s@W(!UVX%;3KDD4yKLpfU)~m}jwlH0Vgvf1OrY8oXe3lVRX{CTu9_tB~Y>k`{BQm4(R5i"
    "3HxJ(gFpX)6Ef(|6S?Gr$SDW^yHsfAy2%HX<Etw5|Fkm3svj+pzS6LpP9!GO&ZrU-oB7Q=+GOBl#>nV$Xj4q"
    "swJnOcIErcmM5dtXfO`dA4F1Cn4ZzCuLOE0j0YXbJscg#o~c#X?IKn+P9%P?&KOO1C{D96<$xqxZzcWlF%T#"
    "$ch2pgIo6U4evINW5;q^_JY|BS92nhK7Y?4Fu~}uvO&apK_>54%fmk%df*U{yTxM{v<{UH@5a99Nz%5B_Q<P"
    "hCLxzd1HTPVh8}UVy>agG!_voZgir{ELE3_-JSj;g$bRCyorH2VJwZGiZTK?SA?|vOki0rj<@(diA()(!+70"
    "BQ-;N|rzc+W=a%MF?@$__vc}QUmTI!gbdW_4@ypf+7!+GZkH9rU*120A0v8vzPxb9EjsmqP$HO$cZFvCY6Z6"
    "bje4WA!{P30v(nbYSqcJPK65Q_uqhJ(#Ee2&xEX|9sm>#Cq@L?@_GcI%m(8>to9uU%W5nls9xfGX9CE%XTQP"
    "3f<lTaYc@p0;^kh;dBFu{WJ$UJ)ru$1Mim_?%ZXYby0=p`JO^d7LQpaaC!25MtQT!H6aDzwPYfJFcMBGzTH<"
    "d7KPc{q=rI)j4o6d1|DQ_pQgB*8v&%#xYbdkA^H#uACcIC@W5{klMok~xeBUp#^$^;Ff3Y99y$G;nK<P_mUk"
    "<5F#JtM`DfJ(P+7x_GaUiFq-qNu1t5r&QAxOMN1~K85r^iy4*=K+GgY6!dR9Y;+q!2EdF;zV_U8m)UX>I1kl"
    "X1ZTUE4Z&$=pkV$lX<&luintd7;L`A0(~fRx=dD2Fo2cFYglJctmrEx%w;0L-QCy_j{v+Z(>^H;zxl@KXx3+"
    "V8lmeT{^5qHcBjfn-PcY{*d9)*$!28O{hN{Gn6<SA~XtZy*CXU~+B1_=&KsX7eCXrNket6)-c^?!-I8&t#;k"
    "ydu2mM{h&bMr2@W|x&qp%T;dJa~<=go~qJqt0|V}pOEYLY#Adv<twu8vO?xR(!J9IB(Y=f_4efl(X4MD^!`*"
    "Y6I`ROe5D`ct=NH5GgvOqX-Ln)3Z{2*3fuVYk~HBKk!mj%DFHRhT$b0Rma`z$|s(uW<7;&Ht^N@1p?1+8=ro"
    "w@xRg>D@P(KJqG^mU*m;4qS!qfoYvu!=x^^psQ`x3}8?8cZNHlc=!_&gXO+eBQCdE=jO@)ooIx*1ISUmf>TW"
    "c)k3Td?@TO-)_iTA{r2ub0aXCRW6O68uu8X#EEIJ{xu5D7g6vL-R?I;_Y2w;zJ@uZq(|f$*-g`UgJzZov--Z"
    "QQ!d)VlV4TX;5EoQw3#=q@s@?-3hTQ#oHPT6Ts}c>7qGk#tC_{<4q<|U>47P$5eYpDKgQDWZV_NPD8*1l&&C"
    "Ko(C<-7jZojXL+^lAcq>Q(LMpBNha}5Fj>2nJ#Y4&TDMDXu!cVEFR`v)w&USdhS2<tSAOPCtOd4d9h91VkGZ"
    "&Pn&oQVUup(h@%!twSLvsI(-jq{A8VeZXqE;XW{pWSU>xvGFlh#k(lE7Z<*e~0%BM*{@5(p<n5Y;76Z4>9(I"
    "YxrJ;qwIt3KF2Y=IDYl&@X)WAS(ITe2`iy`dGzYl;pySqb6FIbq5G3Jd=$D+p}WnHd=G4a*+NRpkEjSH81|t"
    "vL6yZ#oGzejH<UCQXK^}~5+Ui!!&e9IUY`$N9GoA#KK?~=*xbWq^VmeTw7+%SEM44egRg)0IqZe$HFy-bB7?"
    "Jy2U2HNp4xD<l`ID{c3Mnl_`m6N8A8F6$wcd>DeU$uQ+V=tr#WswCez<(oy`+xHeFv75yrtocLfu0vEi+-ay"
    "ctYf0?2@Lk}=nsIA9c&!E6%sIE0A><v5tj{WL0?}##M4t`KQ1aMSL@x|=c=miH+MA;W1puuC!Rrg>@zE$WU-"
    "jjc1pIbg)jRy*5C3^ckZ`}7(suYR|A!2g^8U<j2FZtq-A|Ixv8K^_0=7F(=LwKpnZJyF}u-@)8OCUW=Z$rcF"
    "exIwOK|SVfK+J*skzsmG<xL<Nf^6wn(h|Cv!3d=S9-x#jN-D^MrRZX??O?Sm)-@`(PfUsmuWPbYZx8>%t1A?"
    "zn~m71OoaM+24j}l#6^~nwQ3fYdHfMnD6>*<Im?xdwT!IB33nspxQEEd9m1H~CR9``a2ghJr-6<KskXrX;M1"
    "IL@ou9sw_TLWM9Tfhy<zky8oiYKA|)I40wj;5RG1Ej%Dgx(tTpadhc~k1cKWcv!gNVe01~%)TT%(J#JDdDx!"
    "uii*!LuW%7K8mBD81}fw~QEY=J|G(>Pgx|BZl;i-0S3xti#B8pbIp+hEuP`Q_}lch6~Q#0cQsWLO|B)v{Pnh"
    "lq-Ph>je>4QwuD&~o`A^3E4=j6zowUk~91Sy)&7r`lZ&ixQe00@l=G1p_kPwc=)h=55^4JxBP&!O}*LQ(>c5"
    "-oPM68lLaksA>4LYK#*Xy9EY<78_xhpWyjBGq!a~UegGx4wVhmim$E<qAF1%sHa2tzc%WI4Q}Z#o=v-+_$_r"
    "sQmq>O{&gy?Y9}LWZnLxUS$_)6EgQAh+7R!$_luCcw+h(s#(#fBneAN^o#IiT@tiv{AdPwmT1>|m{>1JT_^{"
    "0^s#XouDD{zH+!OTPEHE|PoE?UcfqNmfdWBatQf!?tFe*g|CV08}&!=2KG@DfXKvUc&g*tkBcJ%U)goxQpM="
    "@59v~M5>ArLb)LDo1dkh$Q63A*c0g$Ca5l(sL>*i3~vwN^Ss7n#AvVQGzi2Je5_a4`vEHe>5_6&i76U)G?QH"
    "rTRfhp!J`oUi3R>ecD-8)q^8f|d8+!WyVQ!A3+*?FK0KCb}F$mDEaZA>wkasAIxzo<{!qjzP7~etXwz<abcC"
    "wgjIMtE4nwJMOlp?8GoLxVaIk9teWN7MWT*f2PKh*_y(=YOdg(uw4ker?@Iwr%z@Do<67zC`nG^DAFmFM)oV"
    "#H{n4w{`jr{_qoY4H}2(CM6l+H0UY&C4cSX=JKxLJtA&afD*Vy*_<8iag)AflK7xAOwt6g{!z@?Vnd9_nro*"
    "`v(#6M2kI(aL7DG@Y1fAVZb*?3Xi#U>B(t2aMpK3Q2VHLH-nA1*c6Nge?p`p~kl3v!Y3|?z-ge4@w7s}U@pA"
    "UzOz0#_PuNxYv0bc`^<%^?%>7_FHn>dB+4ST%VTvnqJDG7ji6DGR;>tcju&2l{%)P<n}6z#ZN4lcKDsm3kE8"
    "dgd7p97C#KCB(5*TGmPS#1DLA>RS}lWW1=?3f;spSoe3e;_q~Pn0q+`i<fQuV2zE(UvK{5>og`i#xX3S)@f}"
    "fV=fZFVWeG!Q$krJ%0U?8fhpbgqjxXVAZai71{@7<8J>G@3K3%ItWH5L+vjcbNCa5NoOgr@9phsk8rzMQzc!"
    "l=+&gFoV>2Kusn(iauh0Ek+{R0{p{e)A<UI@)%P`d?!D@unY;^^yj3b~Zv=)wOt^>>s!SefjC;m<btp5y&ur"
    "`E)gWt<-<PXBTQhiJmvFR1oDblyar7L%r^<IZgd<c42E+lcG>Irs;e0;VVb128kiG{IE!PCmhf|QU(6woX6P"
    "OQ`1h-=Y1f<C%{wM|q8Yj;*2AA}WE~<U89ZOu%_956+)*eK_)@Wd{o1}IG*i1c70JDylj~p8?c{0O6(VIBZ6"
    "?)sPJeaK5u=}43GrIH6vC9}6#ICJ(Cod1q4;>Ne?C{)S*?(#fsf<uoL6+QH6ZyL82Qz#t*Q+0DGJT6P5>?#G"
    "t=}ozO`dP{-TgtGI4ejTs&*LnsY<JC8#;1$FmT)6y4{f`xDCNjfi269L2Gd_j^_wB;JV`Gd6rGKM>@=nyw4|"
    "z&RfbRIEiQygA)h@J@wx1ruUqGU#NF)kN)f3q1#ty7GE<?Prpl&ni#g>gq08vh;bot>FnQ0jAbz0j^<z?hFp"
    ";}yQQ-ZQ-cFE4s%#{W9s&qtlNj-w}BchFg|B-O?~D1I)`HD7^N}_#~6MR@;SnYH1I0AeUY3+LK(-6^^OquCo"
    "U-hT%mxJhe&J;a1?gw_Cxk^c*QgZ&3zdF8q0Vp=3<EJ-X23#Po$za=h}w2GL!hD+Y?r4hulMTy$JI#Ew#%4?"
    "s=lv>WPQRuJnD567>Mt+T2;s?T5c2Yz{a|Sd^o)k*B+AyQ-()g$a}@<#SUT+NTx^m7L<HpDV|@v5k>L_f@+="
    "$rUyCTLjR?$11`Y0^|B$H9p21jk4#7xiwB-SUwWLEM0qTluD$w6ZVHmr!GUTMf89MdSN2+t}0dNLs3DCspBh"
    "%G6ZR*{a1L$Di3cFlDX%a5*S;2omI=22ik58G7Gr#QOBn*4^P$4e}#7d!PtRF<^V`^V%iuNVP(FR3<ieru0U"
    "?{+x^r)-YM{luc+{=ngjREn;W!7P(8|=dyd<7pG%m5-Lvl3El-l2j^2UeHpz8lzYCOTETC|-cMbTKqF`KZbs"
    "JH4ZWaUY9eAXhTK--x{g!yTn_5;~&)!ot{OLy0xzT+7^WIPOr@L*3jThU&*9Y-$$O>7@aVyIFYX9?&1_*48#"
    "xarY5P`?j2HTE=;5fL5|J6fqRbeCS6eJ?oX9tlrZvC{4r}g6a;Pv6zi^EPW0yj`QU3GBA;kGpfP??@%0OaY}"
    "ZHzrgVdu30S8ND5VH8u3Fa8C@l#qZMlO(=I1#+He5aXR=ATr@<!~K77_8Sn@I`3`638b))_z<T(byOPhDb18"
    "A@3GH<BE~I~Tb|NgB)xI8RPQ168VZ5nnBS}L8brX-b@8%q-gs!^LVWjt!cc<uS>l9Y0wWAF=8*pqr6(#tPk8"
    "e17o4_$1EA@KpoQzH7a0KPW{}DTPJ0OmvxOY!=mi|cEc9SAVi_hBXzXS#Pz=~STpIa!2GbcWjWs+}qxc$TO2"
    "I#kF~)w3tmiMFP4VX7e-BR&&(4ou{5E{|_UIhyaz5U%hhPZCGtE|!7Doi0grdy7E%Oqe;LvO^luklOiwQZd8"
    "l-u2P{^T%^D|Et1rZjvuH8Z@`5F{v{8++)sXt`FR=uxkfbYSM0y+eZglt!Ztw)$fn7jdhV-27Q4i-~&>^R9&"
    "4<UQfL4)Vuo@f87Is@u_fCxQkEJs<bMCnM48076c5a`Zpw03&ExI*Oy-Ly!XHLKo+omDJ>K6)~6PE*UFk)wA"
    "5FVj=%7JKO`9i*w_og13e0>Hh{Cc?~d78%9th50UM2(J&sJj&-350kCN=G>f|1(j!Pb+dh$L<ds3Ax;PfMa("
    "k<C{%<xr2vsce+V7F9ULxdjgSGLdX6!}I82vVTnbnXDY%vV3=_3T99Jj+s%bG5MfvRb&7nI~fhmYMph%i{fh"
    "$sFl8udo5Gnz;8Vs*VTuj-hIAQp58a|lC&`8)*=eL<EbIGJ-{*&iu!semJV6IC4#>vHyN0h*IRAe(vFPlJ4E"
    "SLzy&dGcg<9yj88fhe-iNtc0;ao$qh@tiAB896DBiKmIND_$fZT5#6`pWXs#X!tY0?-tvnxj#1q{~~tdZkhl"
    "nhS$qs1;=KG)Lp01=8D>)lipazYqCpc<)diH(cYUr(Q!%oP$&?(omAnBj%}+!*iSENd)0vo;;vNdKyDoXJd~"
    "hBWxQp!c>d1=W^?Gkyeil?mlyp3-{>snv9Pdczq3%4|}Xc9h^C@3{>HKUqEdp1YQN%0lYW|XUh3hLEwj5!FC"
    "0A)c-m@dJDZ2s!+#oRnbF7P}nOna0T(j5z@DR@@CZWRy+_USienLUCOJr#suSlh0LG4cimY~28VexE$tz$1c"
    "$HLRIbqynVL%qvS5yuunjS9E!AJNSHKyIVG0<{5T^A_V<P*)v~jCA4>V}e_E*0|vOPPjM1t@Ix%&tg;DAnkt"
    "#$ZRdF2Cq<Ad%UQPjM#iJE`O@*D9A(TWoA4IFFskISN=I8{68U$jJ=^jta74PM-?t+O=Ng2lbb2{tl2!7$<R"
    "<30*GS}sA2bXGZ9t4<aV>1&=YW}n=33lp~4lHwhdp(J`z!e^7eiz^(L^__lS{;Rk18Vhb_vrX0~8eW413eZo"
    "<z{3q>1Xp})jKp!dMffGIm4@5an{K`q6!=8c18YKcK`1A`P@iOo59qBAuI?IFhtX7=HL4${wDFp{_YwMNto#"
    "d9l|ej&RcbjwAq4;2^Zd}c2`OxdPEF%c0j>y*2vua|si9#wwOge^=xvtY6!UNl20gxEoGpZpc^gPfVIpZT^2"
    "<GrA&$gZ@QVjBi3eAADhV+V-3T(6RZMJf6%RZx4_K1|h2BB9JUYmTJ>w74@B@XMZ*)9~c8yV3!d39T!)AnGV"
    "aGp*+MOlKNkHtDeZnHlXYCYzwqdWZ4r;`smD?oJ<0Q;=g^TdRO`CwVFa*k(m>Hn5X|to_iR~Cpi}~;$1Ea>>"
    "gKt<xjcp6+6ZdjQPC_&a@leo61xX*D<gIsY%$@qgn6{Z{i#AQ6+Tj=Q5I6#0h$Ax(y|&C+N9Y?WKlzKfudFY"
    "?TjVa*05iOS+O;u`u2cEd)JzsT;Seybb+n<bF7=%DtEN#el?o$o<Fu?gk|1;qPSPSR0MpL&20^X_wA`tf@B@"
    "KkG2e8Z;99VmlDb*u2w|)cD=*BKXh_XTvs5R`ateb-`<gkrR2l~C+kw<>iA=OcN)@!&T*;PN$ZZkW8jftqvK"
    "zIS8(`TcRnkndEjDw_ksu0F-U&wKnSMUix;A=5d(3A0X&_CLH4y9BhN&iDJS)EFUgcS+&<-E82uWM%Hy(;c>"
    "itpG?6Y4$%xuy~MgWw9yc!?{?kQ|odvPPT`I+$=9pk-Y-=2j_o|GnjhRIZMIR!R`6#>4C_ZULM$`)nQpwS>x"
    "G*aB+Io$u2GB8hd+pMARN+T^THx+rluki1LX*&xFaa4^*pebTIG;d$8G!tu7J+m%9xqgaXG*S-=oxmBx1S^n"
    "9ggg&co(>p4Lu;YEOlqJug@0SJb=Cxe@oM$gThaZMYKf>Xoa_G{lHuE|;@d#YVC`Z_G;$dIZBd|!M}Q3&dIq"
    "?d#6lt2zKBy*L+)GjXU`eU@z0J_#jEW}RgD#ryH3$QfCe>P{!r(AY~SScQF`aS0*bztqqTZ|9mgu~3bQq-r$"
    "5T5H{UI?>_Ch{eZqe3kYl2U>JxW(CyARj_@-(3&#SIP38XJGRe(E-5RKrrGbygq@PqzJmtHYLt9DBYgD(S`E"
    "x^K$72c|{TgCU-Dx9qq%}y4hBp$oyrBj{Ebxzqwg$-yXhe6fTEY9dcjs;wFAXh$D1X>7GhBR#m7z!2RIGYsp"
    "iaHpg3h#Ili7c&>tLJ01RB${~-f$bi16oPf;+~S=tVT4bin}U9?ZqefpLs3;kT&IzQp+6!+|)K(UUN#3I(~c"
    "TWrv9@d<fA2qRxaRW<^ID5EnItWg%I`XA?(yHqS~x{3O{fuyN&f4+J*qi~J_r+tswSP`&^8@Ry^v>gdgz!<R"
    "<~=ZEiM(_Z6CoeVPLZ7F)_TMKh6pDr+*1zg11ohtBMn7no4V48;uFd!-QsC-u)M1Ce*t|?9olilWF!A>!DTY"
    "(@C@4{cvaSHfY5=wK6_U%mdAMsRfZM6y$<t58-<PbJ4)O!lQtr$Nsg~YfxyYCSfha~diy_XpQL!-n86Z31>m"
    "htJ)3O+ZkK*u{@#vsHvXbK@*xd`tFak?`dLL$7Hgrgkuv?I?cX%JSxj}?JH3B`WuI*wnw;653H1l^3;>erj3"
    "4tkfERF=R?ipkezf>sA_Un1Gf2c~|_2h`#VLPxyojNP(40|b{^s92-mWJ>q(lv!D8$V+rh0zdrV$QAPJ#pRa"
    "3MzS@)9BN1^f_g__7Jz7`yK2FXKPYQWdv$brb{@F?SLE{v3S33aO=^PZJn!qE!S_FI^YB*)=l5H^`=IMKzTd"
    "+1eI)$#>0%p+PV53(olhZu{n(IM(Xhj6;@z4pRnOINfg|4@{&H}B^ykBUiPH@~Fq`nio-$LT#nPYve+Ci0MV"
    "iM7+<^8NY4~p&cggxB|K<iOYNA!Exlm%eF#gs`Ots)5PoG-%49B$BH^#aLYih;5bCT9QT#0(!uN>F*$M2ed`"
    "V7zn@K>3v&DC6Sqh0D9SI(Tk!|J)wh5&0K9?b{;qbFIH4+8Z7Kwgp$I(pDmO#m9~?e)#^&CNPClXQbB78@qf"
    "H^t7~SGnKx#%e9$bf1lWug!h$4Q-I^y(ijugBRkrE|Txwi5vGmPF0h?cemEOI~(ST)L(U1OWoZ-IHmMSOuVw"
    "V5-4_7dqXJ{-KLmk<%tw7Vg{)v8Q)FdFhhwwE~_2p83aaS>K6m2ktqD?H#4Y4!L{1YBHfPDLg$o3$7yOto;D"
    "7HFTM*|1f^?m1x*7vPyr#?V+wYlnqhkE+P7fve8!DxL?JzyG^a?^(j;2zdsc<w=5`xgf(<3eT3VW9^<G?LlP"
    "sTwWv6u#_dH7X$M*3E;H9|9GB+2uF_`a8`I?rjMM!e5RHlh13YQNyob~I24-i{+n~*B1KrNfW3w?8C>kAaE&"
    "&7MGL2vsi7TpH-(5^PDwgBmJezk@7@8{^6RNFJv@^lZMeh)^1V$L2p*kxfF6`xyTA<VDmVGeYVG7HOsyO@XN"
    "6e>9I4<}(cHQlcqp*-PbJ_QZY<t;@E{4@{h>)83u8uNgRoiM-tuy6CId<$G93kl~6Av!Ye-T_w@JHcN-EV>?"
    "GdAaua8I_TG>xEG?lmca)%dPG0Jj-yiLaD6*&<)abC*dL~@l)q=3u>DR7T;o|25wiyVnm;cP6SXJ7<M&7X(T"
    "X(-Xu4Pd%y?jDK~1X<#!_&nkHd>UD#3JKQN^NiOX%W91Jt=Lcp&@4`%BGAHi2VU@CniquF=iXF=4OxcLWzPY"
    "u}zS$^G%M%_D4NjDFB`)lyYPqb@%l+XB%mug2Xd#jvfRfN3uWR%OTdR|?79;uniCtx%Wikk(5MF}u2eaWBT2"
    "QyXt!5hYdD?3v=OFZaM(iY5i!y&DQa38ZG<a{|>7NwpYevHcw9;;$m^kDTb`ukU~S2yBpTyzLgN5Ks3a=25C"
    "s7E3B+aW5bm}YbG;4nP3nn2Hy)Gh))0kN)X=^R3=;Y0bEtZy0?z~7py&E%FN0_2SOuvo-WoTXgO(QrpU9Hh%"
    "Yo#|v^#xdtgIMU)3<m+)kVKHQ7`DQG&N8Tmw>1}pk>LG6Ou+*$FtlT&>&iI!XN!(tl25x02FUU2YY&jW_fa3"
    "J}BunD)@+;NcSy4yFmViEkP4EJ?NXMath&nFRyBZMPJJ))!6R1QF`a6hO*lj3+v8pkb^{A{~2R;ujw>ay+Yp"
    "Z>5rOqL}fY_y}`T_gI7vZ>OGXsMCkv_{rny|(^2w3yre7Y>+F@i6_F>ymz;7BeiRV<xyBQAMg>LiBKdu&fx6"
    "P+<$7ikAoOyoP>h~y0uf<o2EM5ou~bg=tb<is4wp>ymvASR6tVh>?_9sX9Zf*^Hk-syHXp)U6n5nV7LmvF#M"
    "ch40chFT-4^}L?wSXc5FfCg!zHCA41-=qR+P?Yfu)kvp`O;h`wuKVg!=GG@k*tpQfB|FsfY&`Xsb{>|c&eOr"
    "=*8jN(w<kN>dsm+x?%w?mD_amVzvoj9CfUB4Bw6T8q=boJO7$>KC&M&M2Uwd?q!0t1<ak~!%eb<z&a_T$vxc"
    "pICF#YmRj*-zyeI6sxw-LKI*G3rxdw5LSZt>XQ6F*P3CR%ld9kcJegJGb@7Inz-u<i$+aHukmLkS3gB*@x+{"
    "L2Mm0^CZ%K@1TX)Ha+dq87dcXduwy{u8o90J)Q1J3>#GpB-L*|6dGd;Hr=Y*-_<VK<#0x@%`^q&A9R*|}Pcx"
    "Q%u!!15>Xp=_?8M@(V>&4Ypj&82Imv+=m+l&e`8H6rmtLDY3)J%KRv0=((Y{lF3zg4$MN-LN%GMTX3Du|rNL"
    "8JMDJ1Bj3}c4S=sju#B$ktUhs4;TUj?$^AWjgf1Ws>Y^wzpAmpx9m+8NrJFioy#qkpTNH5hyCZU3NntMUIe+"
    "jdiAAtu?VR%U4|g#n5nEgvv$AxR2$HESZPbcc`?n(7P4P%iLCvTv}_@z{A`L@QwmR|;;l!V$|I80#c4nnSF0"
    "M^#fG->X~hGw4WK#w-LeSvZ4EUmV*nzg?P|&hVE)f+UM|}Y;eA3A>V>yIhv|^)TsyrLB+guYJ;vZz#~Ov4Iz"
    "b8L3<XFQ^_irc3r=8=kXy=ozse}?2WEDBs*Zkndwj|`<*TkEYw-S4(1;CKxtl5ssRg;L-NUx|P1&UJJLM8`K"
    "<oUEf$AgIL7a2A&2}jJZtb`?T(~Od=7gCD06I}9jSCX-sz!n_D}Exl&<-OxuPU*81f!#lM36(QCm-5mr5GY4"
    "#lC};TiGNx4f_7BZo4bQPK(8#wtNQFk%3bJpC?Kn)8U`}3nCizzyR6<Mlmc9Rut@ZPA*)x2Vus;QhzKvxR`#"
    "|o3k;42gx)sx3vw0jLk`msw-%-f*kosmXj^TL*7+Q@s466qz$xRxwQD_9|9-Ckpdst1d6)<e#I6Lh%bXE>#G"
    "g0Z-QE6<U#uXv-c+KZ5vs-@L#D`_dFsUh^CyS({!^limW8+D3V5!lg?x5qam^=p$!5I09sb7_22JvmReB&LD"
    "`wP-@SJhM<fe{T27rh>w7fToK9h6Uk1Hn0QEFMNh~`oscNecWI<T~6S1{^N>cGiKJtdmt<}N$-KA{RsGFRvh"
    "Gk`Y@>h-^O$m71u~Nv^FQaBFvt&Mp>%gR1Y8q|03DA-!>ThN7#ab;b$lxOmYX`Av{9sLFRO=7DL~Px|G8(XF"
    "3JcL>vS7l^L~S{~se>45w{ry>e;9S)(tT-N%H+1**#0VZpCp9x+SP@}hV>7Lk4g;nM(&^kfWf?3JnSHUZm^6"
    "kf}%Ubwx_ztyPme8CKRDl$-NfvO6T7J_YiC;oi<F3qrt>o<L)Kk)Qkm$CE0ES1&EO84QUlr!!wJJ@Ahk=C&x"
    "fR6I|s7#Yhu{2C3Yyj19@2oyS^F;rG6A`n&Sgx$Y`mxW9T=;dbaBmojT<t}znS9rSRywYR^2aEj<6F!!^Aza"
    "Q-5r^j`2cJSbWh@op@DPrIOIO4Q2)(;$8slBFVQ)o$(X=S?y6*Ykmu-CeB^_$RP0GyR<v2M>WsQIshG1kVs?"
    "RMUoruZ=H8xIOiF`bb<32tv`oJ}gZSf&UOGx7u0It_l432=~(F<UBtwpts@SvU&C(UE>I!sF-S&qV<+xj^Kl"
    "<#1f@*z`jMHGnQ5v^Ikb(`1YpwP`jUt>L_sN!^_o+tMk%w#HWU%xvx#0L`7_E6oW;r>8l*>n3E|d2vd1lSc!"
    "FXBVP+4{r4*iq=JZHvwWDiR=4HDk3I)JzLV=Tm=6gm#~!z%rk`h^2Jv#Cc9sJ0ql?cuY>Y7e(~j3rq$q9eqc"
    "OJyoGovT5nh7Z#E_%1FXrOI8n&=i3AW<cndAIiR6H<#*QPz<F^EXt_P_h5>Q=TtMfkVWV30=;f02t&I)4~_z"
    "dhL)z|u)x~s6be!8TKD$OliyR%vCB0-#-r)yGbzbA<6Kw1<DuN>VRSf4e?=7}FwhhM$nn}mpvSDz76$ymp0r"
    "fm-$EjF5Bm71s1mI0v3TEJBr^w~aGfi=kAAmM{V6D?P$xU|T<#MH!$diZG80T+(Nk$`}%1hgR_S70$7_@@We"
    "o$yiX{(}~ue>EJgk&8L5!;a`sShM6vQE!IKmb#d1G@Y=mgR2neiA2Qt#`tf<oNrA3@m2qE^D(PyMGy6ht}yq"
    "N8s{5ZbW!V0fT9S-^8m?s@Dvqvu&=s7QprWx>#x1x%x;1}k8?1$v@*-%3){Cw6;oEO(tCH4us|bzt;bKrdBY"
    "pEB%esRT39<%xZ>-Z%PpPXzI7#@1%9t*&AX*x-6bQmVm~Bwy?F2hFF{Rs4iS$#kf$G{X61sVIA^Eihb8QVG6"
    "pNw;}$`qlE4oneA*)9X7c0ZR{zlp#$X!MO-S>H_fdstPiL*~IYZH_&}MkD%T{;L#<W(Bh#@-)=AmI7gM^KR;"
    "1F+wSC30Ij5Mr8hjo0`=q-A5$(wGAIWjWt7hkTJgFj`B>8RHU%p#_jdh!)x<D*H^w;ZawDwLrbj7qY-&Pg^+"
    "S2IgGXmrjs+cWGrS-D|>$`0Ag&Le12mh+;5k~ig;uE|m^W0HW!JupP#bviP!y7vEQgTK|Imod<7ne;NKj$CW"
    "2DzTR4;Fg051A&#&E{=x8to@VY?+(v?LTFTn-34=qUL71A{(NxOv9+;;jGRw?&Jj``*Ot%_l!1h3%URR&06?"
    "3JG`1I~+oCI>0tL7jl%!HDK|{z_WqOY(PU)%yR4@hf>8$xvkF~8W`y;hjjbI7iG`ALyqJEQEaQOn~#qQUE>j"
    "~K1thPFUF4)*@$cno;6P7Ojv_laRIhr<0)i>pUMBg7xv7~Z_@RtCqvr1+I_~%P+vG9lEy`K)wPxtl@Ca=$qE"
    "Wvq4-I}AzE%DhJ1q^lACztm!<N_mxBr@{5N2Np+B7#z{%PQiOv53w=M7lzxR<53`dbwqL(Z#kuv+Zd6km7FT"
    "2MEry{QT+=Y)9Wj+pkWJ5AcukgX4?sP@V9)QO4ebaH?eIWez-U$xjVMY_U^mMP;cOy|jWFZ0vBVcf$Uzgpv<"
    "e%ZAiEDzPV41Uwd~k{b>`a4&Rymyc?v+G@l?P{8{C^sMMwfH$N^6IG43&klY*`SIY@c2vms`8zoq??&+wEVF"
    "VpS!Gp{j)xu6qQ@;s(UwfJCM&uo0oC0h4T6xqTgM_O3wYOQHqB=S9F=nsb=3S*#fIQchPHgeZ45IHB|<hm1&"
    "jrbBc$5txshojBJ2juVMrQmOMiFbbJYL#oA<mKNx<6QmPTv%iMNq0Ej%k^-Tf{;gK1x8$d9<c{(%1I^&M+v%"
    "WKpE*EC$~hq`Lp(qTKxGo_{dV1f;B8a5SA8np^Wh#!kyyZjh7<_F|G)Y;pfLtNsXBgrCcu{8&*Zxj%BR3SS{"
    "WbU_S0&fNw)5dGRiFmU3*GAlSEvN0Vln~I0AS~Bx-CA^xE{pg{_*r9kpp{}#<2%0}&k2gWE$zv|xG76$d%&{"
    "`J?$*R9g6vkB-w%Y$Fa^dT%ex-773}EMnC>0YLcWtmLL0&EZcul$xtYG{z<1wZAahq4Z2Aa?c-DjI)Rfun(A"
    "U}er^X~da`g|s*E8;!@JQp(Gccyax8%R93Ado^Z@PQ)rsJBh1IN!^=hb`Njr?t*vg;e2Q<6P2G*BRy$^&N=0"
    "_HgRb3nBz-kidUac<!ngBZ%8IgsMpl}@h?xoAKHGCA+I$L7h-#gwvI0BJ%3ehJia?u-^v1xv6{`_n=uoC%QC"
    "X4a$>!YKH45uZ2P*g^D+2OJ1ZJ!<Nz4~=K+CF=Ie0+HP{dTnd?a|3T2+Ms39rfepr|M(C6qRx}-hO@j<MGKa"
    "$J+sNpn+-)8i#s%aQupfc+P)qGQ`%)tNEr{8x~&gmr&gRR3;&tE4(2t7Lr1N?ULF60AtiS3Zp9H`%<GHQaJ#"
    "0<5K<{z@4bN#OPl_oo--vx(~&?2`UXLh@)PIeSv#BMt#wEmeK1;kpf~~$nqA4QLYLbB9E2!vTm9qQ@aOt+iq"
    "BD27cn#)C^8>lHLQn&&jdimSM&_&Er&-Q`yT_X`0M?*GZZt+08`R9JaUn%KlJGZNL-QCYA7me;C(o4*p}-7C"
    "+XUCEAx@2Wj-dKcZH7y~&^Gfp-n@q9Hk+P53-(-JL^nv-qBYmTiB{^XKC0ufEVJnEHz4{lHkW;Mi~oN|d%Ig"
    "u=91cRMc>i5PMpy*$tHu;2d}%%WyN3#dmAWftGt$U`V<2{Tam`F9bNiH3m@zyeJ0#u?Ls;fj3LBut<#Ma``h"
    "(+bvS-$az~az9aipz%;R6gwu+npu3`BwFB>4R9P0ZRx<<ok%!Kw3#GPUyXm}*TfYw`JtN3R&u<EKlIcGwJWx"
    "I*#F22s2;Wsujg=!dI-BG0$84JKf>=Ssw=2Z7YxEumK!P3L0t&Pj&RIJ<OE1WqJ47KCNxaTL6LIDF}tn@BC1"
    ")b0SpRd*sM+CA#7v?#0kKaYtT)CQIQ>uuwJChNG(~5wb%3>45w_03vFHXAaoj=!bWcVb{Ig<h{^b_rED0N?H"
    "bk-A+`kSb<HmG7=47ir6SGX<WZ!7W>twT8f_RIglY3p%L+9tNLQq*c=jHIrrBgB7r6}vbCO9cbsGZ+UOu0vN"
    "hU=(m04UQfDbQ=X;QM_CCcQ3P(6u-gxt2ITx<^fLjKgal5i$fo7Wfnk^Yj->`bOr+yMov29&a@@`VFi^$P^B"
    "zmhN);QwS#otSnJ6ea?`-OwX!yZlgjcqj{5mhp{@)Cx{8g0_%=wLjrU8avF3S&}(}M7TyR=!rt!Kp1B+xy{R"
    "_ZjzbCXdwu}rgmkXI`2AEU$lb8EaU=g2O5h9u2&e?LP90?C&PvXAFJ{_)swl%^EASL_bC*_it}A%6GenQz`P"
    "nEF$9adpQ+S5^xWHw;w99HoFQTlqR+xYf~Iy28mUYHjXAy-x5;cKGpF)F9ery6>CZh2eJi8vOD5&si{Xo}BJ"
    "r0<d>M)AUyyh|etog8BAZkxKyZlUc_p4hl|WM;&}%hVyqhHjA&yV-cXoOKsrX{-Z>^n1m87ync&Q1zs~q=q$"
    "BxG*lNCB7fbh!kyvYA0GZ6V7tjMN;{1AhzcDeOvKRPWW3J`IMSXbWDhsQr39ABKA{W`fg`SIY`dlq#&Ih!CQ"
    "xDR58tQFg&pHq>en!dpz4cv0@&pgQx(hKZp1Gp6A<Cs3YmjyVlA1}AalU2!u6@AS`F?xLskE6no#x_M->-xs"
    "MV4zX2K4yxok7tYtl*;!qEm25>OCq1XyWEQU{$<5|%_K<0esA<G_Xm(>ZN0fwV00of<Xv+iBlTU+9w&53qWi"
    "A*DEj4eKXiVz9&Ob@HZRa$_Dmlt^s_(6uxhz}oH?@x6%RicfCrzA_w3kfK$DByp0Lac@PKqeh`HtXtD!~(9R"
    "qn|?5deM78;r$&#A^)e&;M=mQUu8ls{ItNB#X4bvpww->327dKQZb24{`jo2sYf0@~wS8K>1P?!KFp^0jl2R"
    "F^O+n6$d6YI!hOeg~5>XG!I};T7a9J%4L5OVLD<;`N8oP;iE<6@W*O*$}b*DCLrh830hr4eWj<p{}Ss^4{gv"
    "sszX>cw!$boV<^-Bu!%lkMB3vq8UANsKSrkf>HJ7fL!U3pZNt4?=6#?+e%cc>#}WNIF$uNCl#t9-+NhH$5pa"
    "eJ3v*17XqoX63k$5)0E<UQYKU<sd1oZ^cZ}>0qIt2+fU$%Cp~o(%(h1E#_22yEyc7EveiwU<S?ck;<pL-U?g"
    "hl;l7UT<b8Unx45<gDsx43*9VW|3*{Mf45#jWpyai&CyfQxLQ_55c9S@w|6bdaj{+E==J?+zS>F=W^cz%n>W"
    "qo1GF)eOxc=E|{jB)Rh{HYA#S3k0LLP<qG?x4r-1Tl_;u<!cJM1)2HC+-Nb!U1-Nz}wuHb;`7H3cDjCM9Sfm"
    "><2kmNPJcjC;F-VI+nFy%_opEfgGhsh#Yyf16K1oUqK=Szfsr@pxR5G^y^#tj>=xnAT}z4RZ*?+#L)(I>iBi"
    "|8&gKb>IfO+<NaW2Z1S3DcSMmR=S$vjquwF%qyX#(fi}P8!zk$Dtt~vz?^svf~%=pdi-Gi#*6yVy5YwU=I=+"
    "x{tR6es|?}bO&9kA{pF(<=+DPAJSzTiB+7hMOy&5`e7K=+;1c|Mzcz+)wOGWk%;N`0dq5d*CJT%*umwQffj2"
    "7h{19XY1fY;?X>)16P@a}YN8&;M{a7;&nJuUn4K?z@NIbBlF(6~YBdV)tfu@k|B^2$Vl)tEE>(vWde}}Dw0!"
    "ZOBE?6afpbCrDSC2b?d!Qrjd>g8FO2=xX?ljsRMe6-WQOX4(vW_3rmv4qw1NH5MCmAZoXCRP0sNgC1i+Zl{g"
    "|@A$g#?vQ%QAAbn%-}NQdmLGSOz7YrX;(NrivjKY%oD<QzTZUs$E^kI2Ea!-N@ppu($++8%DTo#J2r5F6B=&"
    "n0<ske3gJ<2k{yk%_SQ`!;)zYSC}98N!da0)fcNGQ@VSawrKm4SyC=R+|@7U&k$00fHpYL+b<B*^fKGuyVyH"
    "A`QESvi0}UK_0G?G$A?EpdyX&%A6`V_>qxx#QuADHzyHI@{>jna#etGCncKt1-4~JgDiWU${fEvk&h~!!_Tc"
    "R7S9<EaD&o6qS&%BWi^~0rKYs~NIgp+fif2!6j*`{bcY+%w#@3GY<cF$=$Cq34ytqyfaNNZc@E~~2I|8Vo<-"
    "3&!0-IzC1a>zP79{Mhhe!IVi075*+kKVlUd7CKAg7fAIh`1kQ!w9@QqCrDWjUx;XuYv&1JIc{BwV@VUsB}g@"
    "-1Ys%Fqu?4nst!;K~FFks6aVWrZ;Jj^$>PBr+YG19x3ST|@0fa4zR5#-^00?=Vr0KB3!B6(#QB07CV;HYzRB"
    "slG>BEiP3}2#0gIg=+fNh*u_97Kn<)fm0ySB?h)e;>+Ri)0L<Cs3o@SD4~aXKF+*~Kyskon`)^ITkF4I*L8Z"
    "b_QF~5$Nnb?dGhApu|`~T;=(rcu*2m(w<GalI2;axlkT}Qvn#jp<7mzE{BoLG+~!MtP1LY^pXmZVH7q=Hm5d"
    "B{HNYZJWJ_-{vRN}ef|V;=m<9*am8ukYwmxW&*n*AWBVB5`I1wZXmgGrv99&{T;NVZ20(G!RuQjFwgkFr~!)"
    "s+)_40uZG~fP{uKKmzZve{_IlH*R-M4fNl@$78oDn+k8!9Rd>}amN)6=t)pCN2a$4Esw%qrT}@xa^tengeA^"
    "Na@C(7gw&lJI1D-;B4n;bh?O#t`qPAXo64hV!c`MyYaK&Bya9n5X8Io=^cums@~Rj>rXK43~0%M23-0>$0c-"
    "p4bt7y)^NeZmV!=oMx;^UNw$h3DICy&deFoFxpUHWaNn-%}6II37M-7f$)qKN`Nphvz3l<H_QgUA;BP!uPQE"
    "SmXzowL1k{cR6V+o4VxEPV)qb|kh++?I9JhAGqLOUJ+%;$_)$`gDcx%bbHcBF&YMDf6{M>jOj8XNN~fzu583"
    "}caGlrcFwvqw-eZ?+356{&y{F;5K6jH+A8qoQfyl|U<%*x}v_-<-7=v+oJul<i;`_Ed-tU_{Yffz+kM+v*BP"
    "v|v(l5To%7Vq#ih&{}Lks&fuaAVoIcqtZL|ggWDo)9Q&A=SXyi6)cgmr3hVQmA4$bVbql@0jliR~bOjP)GX("
    "$=$Q*k5f+O{DzDU4vRnE9SV?SUjWQb|WJ@lH%3b-gg&aG9vZdxv5=cvR99}R`d}#FlX8o2HvA-4&}QQDl@P#"
    "*FZqqZZ;FaIpXQ9VjlIJM2jN5U(B$gx}m$nv<z3MP3kDm--*>y#A;l~F%=9zHp9xETFH5R{z~Ak)JN|Um-nN"
    "_dU?|D@mQ#Psy2gt^!<sLI0h+qde!;#<Z3(61YwqQAdUerMc0fE%**9H-jDdoz@)Am^OF%ldq|Rza&TL&ZTJ"
    "E0sYod3*-0{@;JzgW(^8N<tH*Agl54H8@#RVdKyckPFAf6iqynrQWhKhhbSleozDjxVRG?(}lB2Y<xF~d@I-"
    "-gdNp-uBRWfx{YIO0t0o7o{HqaaVX}-#)jK=duX^@_D#Iu6ws|mUdpeyTFh80CS5=MQ(P5ee>;<KJf6Sk{_A"
    "oBMc)ZUV;x&m42;nDar1EX*S82%P6!5o4g5c5De=`edAmIMc9n(9F^S^z!c(0AQ{P$iiIkLslTSy!@o4SVxk"
    "JOVdSj8`px=fSY9eLbSOK#?}n*Bfw5q$_tTQN1<A(@C!sV|t$7PJo7y^PRa*ydRwFqrU;hDl6Izz^1zptPW;"
    "rxCP<wU4<X(;|Hseg^`1~rRF?;9gBv3XenBI;nn(UpbL9Rj5x)Vgc8+8csq3NZJJ8s>AO-Wls)SP%M0ZHG&V"
    "2w;|zRO(-pWkAViSa3C|FtGw^%HnNjR^fUGf%@3H2b-5<z1%%l`gGB@b?kFQ^vOA2KDc?vo?If1w`g0i_=j%"
    "n!bP5EvGXT9f>ra_*AAv3?I2}K601I+y-Tx`8^dFM!<18qF4zYWt6Fk`w$?^0QQ_MWBjwM^ZdckE7PkkT@V%"
    "W0B0g_iWd2ITpNu%{jIfc-p<#Df~fBl^`N@uvr-wCB%(?jHy0^MAm{(eTCW5%zqbcN-5OgNAWanB+tsa1gR-"
    "J_`P+Z>^U#tPALAB>wb3UFreQ$ENr5HaPA#?Ab9_pO81^{5D^rnccYp_dsq&SqEHFe9vLo*r|j&KB9Y$Yi8$"
    "jk;d803hn|xtt^14VDg=kG7ka!ghK9;QUx=BEHffUtP&aNNR+pEQNeA3&~^}B=9NywJ~5LDgaKpmLSZb;p|l"
    "Zb@ucU@lMmoQ!r$=7fLidw#l<O2YhcgYR<2Zbg^da>)43(dG{_cK3w8}R{rt>snk;0#Qf0&Jx5OVN`Kns3s!"
    "5f<lNl9AQ=V~<W3(e_WV448C{gJmfnmK$SOZ8YD+#|F{tHD&!55d)iUIl9xnNa{{#-D#^b+y`M*@AoGFvS$2"
    "2{&gs3k6}8y~aYDH^D6D$3f4N6r|>;*EV>yB9S{9k#sj1_w8~3ci)(jH_Bgmi@v}u(-8eQO;iKDF<B-g9voM"
    "8^|wXPbCtns%|-ccy#kvI6%jKKL+XS`YGw?;<coYkq=Ek`n&f;P{z=8_{5kixztT%dUj!lLoyrB`4-%HNFEz"
    "=)p<4AR1qC?W!9+Ko1*P6w$&kOYi}#sVdG!(({-z0bK8Kquv5p%j@U4X$Mt{~8c;?f>6tT#WYG`fQTPVWoq)"
    "mfv-&{AUpuX19dCh3KZbDdf(H!$?XNkDK{o&%g=M-bbuCNR<by#pB?(2^h4iw)_?_h<o>3BSoLT}JD#nABtW"
    "Ru;+$*<EA?{Qzy!#7k6QibKpYS@DqAu6*CkW!4=Bspu&nyc7^`TT?eEy}nZ1F2pnI%ZI<-ie6wV(ze;4@Xzc"
    "_iR(?eMsIL_<dz;n$X`0l(lM0%dEQ^$fi^IEAa4glG@{%-Jn`Vx)j;0wRq1_Bb~N39*Q0k|xwGdPQfXSrIH3"
    "Xgyq@c>Z1r2xt))eKErFhzigU$M1x$%w(|x+4m%Q{4RB!n~FJ4^E4&Pd^w1(bzBRuau}@19g0!rlcq#*2}=I"
    "jfSC=dJ9{P|#|RVop2={Siy54qKPEpw=|RuO7I}HxO{q=r?*s)x?)wOW^Gix#g71I-`p;dXf+!B@(+2{PO^n"
    "yA8SfufapbM)U?0T$T0sDRGY03b<x)ZW*V{Q*|7?2m3k7f>-<0YzLx>DU_46QZfz^SiwH#?OfAi)l!p5MpQf"
    "T!^Kh^9Ap@aHeg5fY&&;9OJW(*FkV+0Kb8zP+B7@M#U=w`_Mq$f0Ye+*JQcR{W*hC3U7V_dEZ;*6r5FfnP6%"
    "si}ktF#~9L+JE{y9hh?T#aU7Yalw;g^b_1AoE(43pnQL<}fJSpthAHT^K<f8;ji(*aYQtZpK(0b@vclqTEJV"
    "T&}>V#JNfOrL>-0UOE88ScI^Fks5sxX0VW!5TIHHv}>E9(V}IuLpu%NzYei!%ytsLCI(d0Yy)R<faF^c=@?S"
    "RhLHFS;l0}irk0-o2OEN!0x)PF0GQqPWH;>4AH`vYl_I8CzBazODdY|MIu!PaNM-z<%d0AEcwP1!ZmB`I=Q<"
    "rL=i-K5aSqpLJK_w0CVDP`G<e7z{1uS3d0brai6cTd^pb)?=mW=vTy8zq4^kTk2ylVsb5lDrYIz_$ACV8eV_"
    ";P|aOkmy7Tu97(9)V`(<Dt8fj5Tm=8C{hr?P;*pmYVALx8zV$=okaSsM~b^)l2i<W<F~#97FK>RbMkfuC=n`"
    "|Eo#rwav4DtCh*pqxcj947U@&qbjQ#uy>)J=|+1Fy&1a9cj&v=TX@U40E-~{Pi;-EFbSEh3D!XAW8^;yN5Bb"
    "D2qz1f&yez*j)6h9t3Ar0kW{xrYw{9nL>D`n>Sxo(|qBZzQb`&z9IModqAN6?PNx(8GfL>*}i&=_zSuRwMI7"
    "Z-9Y}9+86K0_o%yr*u%RoPIWbbk$BGcp*=MIo35_yt4C)9G7XG@$f_Hsat0MN2q>QiueVrbg)H;*J><@n#zp"
    "`bS*4Cv9W)pOPOgz{IC7u%swf~|3;`%5sf&W&wu8IgW;4)HUIt8NvCJz<cAQxif{py6e^4|SV1NF$oyknzQ7"
    "$Q1%XP6AlzBJYLQcXQhn)*9_s8=`+|XkEc;X`Ns0IFm+fjNye4&mr;IQDs7$1XwJ-T=Y5dGe)=%DH~VtvciJ"
    "rF~0WQ7HYv8|$Prq{U7cF>-J&tWooIbyvv-1WvfkyW8<LPQWywJ=0=!VU}y1aOthNDKx8bQAVl^(>kK41@m2"
    "3!Y8SLUn)($E6%}EqUdTaD$)&jfVe453!6fH*-uc7;q}3_)P2$BWI#x`-2OKFz0y4no1MKra63dbf9Y%xpL4"
    "G;wGOx;XZ2)n*>WwfxN@P2+S9Y7?uT@B3c*X%r1Z@;#s_`Knlw+-Y2G`vQx^GGsGeMCUBWRws9uG(=tGBObO"
    "vd5R4#Gc~is-z!XQug)^Ws0B+0Ybr(lX%M^53Qb1+0OvokFp%!50nt)SUAsxk|(==;X%1ao#7`qc%rb*RX#2"
    "@gF@ryd62iFfii9VmtaY^tqlrF-1Sn=Z8Rr~~9v{4VGSMFZ5{`{ipXVNz78`0ET2L)3pmrPA0v64TcMZ;7ab"
    "sUJBYf&yY$!mHN5Oi0(SB7DY-^U4Z><OwPEoBKDXiDxxA-Ar12$Bhs*;wU0u%571BW*^dCdfHONd4}*y(K{G"
    "G43iLn^ufXN|G{0_n~negfx_$=j55rGr4n{uL^OOWHZpOylJC&+2NdNP^R{S#`fodPLo%1%s}H#=|wOHM|sh"
    "(LPx#cjkgS^rVvdT!*U>n!bPdV>nMS{%UoZC<~q-*C|0MLQsb6+cAgAWx*-TzS;Qqs*gqg_Wimw#g3y6sxP)"
    "g*b;`IV#nUHCa`BqUl6-+Jspeg$x$WZv44kB@_q-HU5ohHhsdOd5Qj}Garsy=pcE%F%<Y|!x(Cu2Fw$8+&X1"
    "3jGa&cCm5%}KH3sFxQfb^#egVxr^TAp&<`UJQLDBYu{3{wd}BcYE!f#{;QHWGYLbgE`kKU!g;*(iuj-XjrK`"
    "vMbLrzt7Jf~PFC%cbg(u6mQXgCQax4L=)6uDcx#Qg8#&IR7x-MsvfqI6D04@Ivg?lOVOE8+FJK(j$jsqg8XO"
    "%wE`ot{h^PX6jiXBU`f?TYvYx_$~<bt7Ts5l(#YQQO<w!^1IfKR!59tATz#B#i$w4{*;_m6jkc@tFjy}QAlx"
    "-^SMeTV-)Eu2QL>3!=9xQRB6@G6Y`5ZO(ZnWfMpZ_tQY}JPZ}yb05=OCMmH);h2!p^Ro@EFas!gA*dt7@vB6"
    "LKon%N3(BJ{Tp$Lwv2Ej29Ppr2Hlb06g%e9l2JQNL#aH(Z3)vJIwIy7Gf^r{vefr(#dxOw?W3L?3Nc-(2Bh("
    "nG$XZ3Di>(*+qU56_$s36<$=#wa;DOBm9X`JoJftzQ4HD-a;e4LFBw&xBuC5YABv5n8D{EPLqXN>Q1)XAIq="
    "(9B|+8FKW7`rs*oRP@Q{vkL;yVNvZ?DfQvtLAq!+mu@~uexPmnO3J9p@rT^qw*-0@RE%$Ey-4$lCJ!6J?oX@"
    "egJGqjP1NEd$mk@7A-x0@~>uNLR{N>eSmMC*_*3E<s<RT>NiA6!I6o7i*<=)R<|vA>MHWs;kHVBQown2{mC*"
    "FtzWx<kz&*;JQ5?D@CDLy_$|HL7e1RI+#8KYHk7-XFuZtiod&S98sOZiBVRaeYG;bx=+@?|X3mCH@!$!sPF}"
    "My!^$2}8|+t1H0ARetQ~8L>OD5o)cRz)z@BqWElYys=bSWU0KR0%Rfu=<mSACdL7cPceW%yNcgBIio$c;XsP"
    "dj1GuEXBRqZ?rADa!NEk)sD<<i*v9QawcVHrR2FA5wbrX@DLTeUGYKEel!r^(I=tLz4(Jkp)tr3@>3ECx161"
    "mm2l%7tjPIQi-H=zt1c`*XtNkEZh5c4}%k1l^)oesMh)^~pJ@fpGGs)L5FRbNzZX3<3$sIBke=sK0Jqx~QKP"
    "3e4f-S16b+Yo)Q^1~+EO(kpmFTQ7E%JJMNYZC1ft^^5)OJ9tpLCC99DjR)R|1(g^z0@bd@s^F?sI3<P~_WV|"
    "qMq|)r{w9Xn>CS}HUtb&CG8>*x^CcPr?J##>F-x}fLb)%pc2iVEE#o|BOj>U}(h)5U2Bc0Zj%8Xh{eA4M?%S"
    "an8)AeuG%E=?3eQkKtUZ`15y~m3oi-4FcK`{^q?KP%e2;6a&>O2?LlAn<4gZ!G3|vj&AiUMEf{<WiztKst<2"
    "WQT;B_6GwJU2aOPk*iw{bb?9L)S=)1WZ}27TP^T6{9MMj1GWDhO!oMZh=T%XKrL7suKpGIX+tr~~)x8ILS#B"
    "=bkdqrY)b&8G;gSv$D3F`0i9dv466hN6VbhPxuEB>K86-4ei6@Hm*0Wf@#+A*bQB>8R2)6yk1Yc9&aws;rbC"
    "r1hCevr;y^n4Kb}j(ni#mGU-O)^&P2Las}gYuNaIezA8Hw*Eh^i)XPei>8NpV8)UeiIa1Ga9wtzwiAI}Xx;%"
    "xPQT6}&3tCXvd!ZpwYPyTRF1#I)hF&(756#{JiQ-;P?Mj$NIC|MeOhu&B-$s>sPV$tK7FzbqC4pjYW69V;#$"
    "u00+Y%iGd+GMl@)86XQ+R2C=ms`a%npkZMG!@4A@*f-b+`=!?-T1vd^ZGTxSRSCqEyY{W|$&@9?52r0Y}pjM"
    "g6R|2_Dd7eCLEX=#|7=6?X*p{v<KO}h*u75;z5-a2KCLj9l{a!Pq6;lR<ogBfH1(_erej&8e`R3)TPcN`uDH"
    "11)oUPO~G?uF87KMwqyKP<4;+y8LlsE@SnLaJ|&R%+qBbxj&2Zcq4@6S@P+{{!+v9n|>W89ZKL#2jXD@b5nb"
    "PC7hu1Lld+gFctgJeA`Y4)8ew9eM!935{l?hPCdtx{weTS`e<t5|B?;*KjRxtpZ|Va%K)KBgH6i8@jWUfzG+"
    "Fy2bczhKtJ6v!PMzdQT%HYQW9B%BVcFo*%zjyO9xb+{H(u>fZe~9vu0VemkOb;cqrL7u*BX8$f!s4=c&gQLp"
    "R7m6EDg<MReYX>|=+r(PAJ9r2m?OP_F%ywriL=9jhC)Bc%yvRlwMP9sG(lcg*s)#@6W?+%?NFhckmQKFn5T<"
    "<N;<QYxRY1i(o4Ip?<{_FJ#7GiK)LG5`1<Kc$+TVg5SUo8SD+*BE%gY!53k@F<XnT%(U{NaKYSw*hZS^q*M6"
    "$Gr^s*PIR>(%T=R_<@R7zXTBt#l29K#6eo4U!UMB;X!AIvZ+;OoMhB>tFROfOhK%uFk_q{nq@#0ChHMp4AvW"
    "eqHY_v?h)Hh8nvNc&b}_8P^ZfZp{FH!+NQ-z-d;hn30Ys;{sgU%A%<cH962`SdX5Lo(aC002k6JAsa+#Q+O<"
    "d4TWkFZ<1j_^dI#bW6eCVo8QdLxQ0SVKl|X9V0ZTox^`og*;TVx+l{>Zy|aVAAM9VaRzQBFZBeKsv%fN&BXh"
    "acm~!0jh&(RS<OcEMun-Wft|p3?4#}Wk3-{!zC}ka?8C5Q(nSL;8v3tnkZ160)V^_RCcQd3?%*5YM4v&43xe"
    "L^JI5`$~gJd?I4OC=35!P%#bDl7T`6b(#+%}1F9Ch)LsK!)je6ef@xv`ol7Pz*d#m*+^!Cd=9(80OxA)6i58"
    "zYcskC{oN`FCV9+bGB?BZtxT8?%Jma6KYi70^sLAhTqSN(?9<3zD9ZRBp0`^pHO-Ii;+)95GFk$;HhPnBFGo"
    "jFXF${xX=z?6;L%`9y7bw`+jVnv_TAd0=Yun}OgE7t`C8MrxtGFadK@Mj^K7B!)G$XvrXAJ~4@B%vC}t62Mu"
    "<W)ihCcAcPXsj4!L^^3h9fRj4>Rev3}4|cC5gB4_g45r7sA895LX-cjgv#8E>g7od)`li&5{@*~&86>z!?fp"
    "j_r|?=ip+;1q(Zms1Mirg-h2?r_G$78ZaREIgDN4Bb$79`IOAWNdONnJ2N@r1O2e+ST0j2~npp0OFvRrvE&?"
    "-p<a~Ft1tc;szf_`!lErx>VeviSG6O?N1{EN%I)lQCuzNp67Nu9PrH{TMTT+%(KNw%TX&E?jeQ>BKJ-L}tOA"
    "0JbR8kM=$$JBt{Y@o>QuLxfumIEaSK`HGT&Dz{2aDiHg7GiBdnT~GF^(lP{CSp4>V~@hP0w#)WZ2tUgHzFJ1"
    "c>9%+jTBXaicL3>>V2@?fLh?lwSlrcq~%9ziiUMPxI92UxXQXp?|2Mh4)3!;!p0r`JrH-j@>Kslz`G`nr+l-"
    "sdlkmMA8m|u$vCuY@*tW695F-(W$B&W*0fHJ+EGTM2|m;MgbC@XM@~WZnHxP7ZBW#1UAZL-MU-AJrpOCbBL)"
    "&*5Tqxb-M<7DuB#rZ_-b*7{CGhLYJ`^ZNw6eLfV^F8?RW6Z=}XR72%Dgsn%pWUr>CGhK03ZQNBnH{#MNXlYw"
    "~0B=chng(58it!A!ERLoXO*9dMAIf*wdGORQQOnmH$C8ErxLrqu>kb;Vd2-q=v0Qk=+GV#Sp6b3mKGVJ8Z+8"
    "}G?Qp5;}ZCDWeQOpF`VYts{>Y1I1?JKp;kMa4$f4l-g$=eTm9);Zxz_B1c)VC7fPg6a`qu*zWB%|JmMf>gPZ"
    "Ffoia$6Yz9oGR?mW|VNC>N%+dpR4mP3OZB^VPNbV5+l57ifi*yI&pE3etGpN$KKH8c4@gbEuI?ANpVfYp(IA"
    "xQvJ{Rsb2W!p+1vGuQogqAet7#B=0-)0a-v>j_~lgW)QJsYq&1ziGJ8~cC?Y&tR~R8#rB>MsBZ|y`{B(7nHV"
    "IT0HZDM%StX<!aMaPKC<(ejC+YFgs_p_?HaVCQ~#Q;jQNky_qZ6u_3B;)7nFb*s67~A`g;r}pjZ>6A_S2;6N"
    "`uX9bAR89T>c*{xVBl9$Q4!(PU=g;KyGCer7g2UL;xXD=2NLh%h8Lg>-d$NZoC6G<aLfxBFRRrt6f!;m8@A="
    "WhfvGn_s*%*$B6OzpF5%6oCf)H&;AeR_(|n`-gBAm2^S4*u))!Fedi_bGrlZF{yR@oXb3oegkwFt~Dj{QBr9"
    "Qg}k}L99`Q8}SAjbdWV7bbD;l5|(OK$Ea8n>5U*MaKpQGLuw^Mc6QMnq-3&&z(x%J7}u+!(%{J827d*LSmMP"
    "ys=w5&npT>)`C1=z22Gf;6FFaN-6{P4a!XHmYXlpk$Em)Lgu{g4ry+dDEY)tRq}|7QIZe|zAP2Mb5ir6}lQK"
    "~yyw3=O7Gl3t$QNTBQJEr)MbXd)s=VN`$V*gXSJ_PwQyq%hi;U1VSpgq`7viFYSlttn@n-|u7<^}^t?4SH?a"
    "s7j2_WodYj&#H+S*f2bYn6>-z|`8CuhL9bh0z}EH`Ii{88+a`vb(51VQCj5J(r-;2q)$<K7A6N?KLr>{5h*6"
    "clgI0$Vr-;og?cM&dz{`kQSCOaZV@3{U|&d~gF|9ywqu42CnukQJvtyCR6dT4kfbi}~X~94pI<hFe_^i^}|{"
    "hW&$hylPTfv7RNW01AXSN%3A5V04(V5y%Qn?Faed`i$#M<&-DdQb8rO+#rD@4PxQMJ>f6g58@0mm?p9+H^>u"
    "Nl?{m^wJ~%IZ}yE<Eh!=xK+a&=m^_Lun%K(}LBmvQen|V0c%i|T+1-h8HemAd!)?4OwXJZYux23Xb>^<U*B3"
    "vWoE={LIsu{LBLkKfxgDzf%H=x=*1C~cuDOr`T9+fSh_e*|w3&3p2xX1aRa#1{==T<sRQ>$h?Gh{BK*>b_3B"
    "~{H>_DCXCC7*|ft7m=MUuY+|6(~UlItY9xy<6}bOm^CSbQU{&>CA6D@i`LguZ|*p-O@r%Bnul%Ff?JlEG=pA"
    "%Gggs}ORXGd~zr1Bz9l{fMjgz$F={Y7ch)CT<{D7HapXL@}7@`N=Qvn`NM@ascemNTG<PX7vK8J_)?=^z7vH"
    "<ow`V#5zZ%ki`0S{w7ZEtJ^#Suk0a6wdW$KtWL~yJ`>ZpycM6J`>|s4w1}&G0pNM22q;X<*;CZ8O2KzJ{Epx"
    "3;8-*h1X%+2d}ehNX9A!WflE=8&ih2(5j=h(?@DL_D|=cb*)&<ksfedQ@fe1FoL9FP2%d?HANDS&0^?o$4#L"
    "JO@ZgBmav=7^?P>u?{HqMU2NHQpKHo{1-@T21uE0=FpKv4I(iXf$Y(SVg1ezs<oL1?51Yr9lIk7B7al6;|ms"
    "y<M(}<u$g2kepNAg8N{7b}@zTV4`K!RQZeOqMB^n}BOsh{utbU;(9Zi{?%gF*b4*^SIrU`l=K$6aH8Z^c5wF"
    "-S@%TUb_*K(n(${hj=RleD5nGMp#*!C`uDPpo6j4ssqR@PfBo!T^|JoZ^b_fPDnq7bJHoEZ^&Ua8uLKwio@m"
    "wdBwBYGa~4Mfl$)<z=?WXRA~W#J4$IlE5cKc?(0OS~+DUm%6rOT#25>8b6;F$&!oc-~ov)GyUV+c)3jPCwxB"
    "M(t(K!z?@P|@D{&+PBi$OC=OEYlB^Fq`7zlNl%^0@01u+tpJM7nmm~%C1j}4!G>20cY{!|O3T_3OnlOKgR17"
    "m*P7`2j%0DDgm*{uOmmqq;nkPIe{C)!DFd=dXCnaDVIQ7h#oUh6_ZOcc{ZTB?Ao>MaES)Mk%#U8PG-lpn?$8"
    "b`ry*ZuWj)NUK|M4{?*Z=r>=i*?GsDQk7jCiKjeFEK^9B}y01h##IyB+B~26Quc`&iF?1Ikpiyj^N^r2#D;+"
    "#bw$7I8Yyiv@@YnJHJR?}?-&hFd}G`weU@CmmRF$3PrHMDx9f%XcWA$GXh)KvmThvs_YSFCbS|IpJQ>C?VWj"
    "ER*Rw;x<3N25~6|s7f@YPLmm2v&r<9U2AlA+yOS{Kf$htIZG&Vh&?kOWkG-$0AT{r7{qW;7)r0Au7#Lqt^Xh"
    "?C-WqA<!myZGbz^MUYr%YD?qMCd_(AS5X=)TlN2NO;JRkF8!TByON4W)JqICr);R0thBLWStVoMDtUH)V>I$"
    "RF_#Ry2UcQC3BEX6#2p@$JcS&|LzT8?>^PR6RxB6}ZA@=b1^ONIO5@PdE*aWMc^rX(V1)X~<YBm;DiE=dH6("
    "jM0-9CobxJDTe2L*z8j-bDXN|e8rSp7s1iNU#e!WMj&d~l8;rHZfL7P{1MHj4Ec3>@AcMt5O0GShqnOJ5oXE"
    "fI85!le$1K~HxDKEbO0#7$7I6%V?<KNPy|CLULfPcQCn^Hhp#1yHu3&4Hd7_fuW2ROpUnHybT9!LTCY0#DeH"
    "AL94Y-4dRO;&V5$u-I4Nu&_Ghv0$j>K1N(h_3&6Pj2Eh*CAVIY=N0jHFJmMxz~SczRgfxz5A@B|+sliaf!0w"
    "Cgamxe!V`blv%`ogjDz@5$|V(Bm9h9|T#)6%14x_65?p{2H2#chobDvsxy5qjGDb#=`Bm|C&TH(bHzY3Uc$T"
    "CL1mo$9^jW0Kpu2!Hh^<cL(dF#M12DF?o34rR8~+trdzr+Loj$?QPQ0qYHL?LcXQD?}23k17B>Tw(BdFcRw("
    "p5t0hqx?y2=hJ9!`6ObaejXYuAqqdmK-z<UQDcc$eAev53n`7CX~i=O{4AplX4bFj|mf5{ls7jl_#ce4(=-n"
    "RPglprW`s*H~p>ZdP7oRm&b#VDblvX_1#@oi6MFVz#11y=8u9(VR6)Yk>f^2+Dr{gjxdx04R5}4`xK7Kd-_A"
    "D_UTtuooPogg{OloSYt<?OmLl*&ctmpeGoj<}Gq$+lekf+XJ-;EkyCOqJc4o9*9?Rn#O1sOe(REF_gSVPKTB"
    "oqFlQ_)K)HjI6P;K8AMbtBqmiPinzKZg(jnRsFd7-GJA{0yb{E%Yiaj4U!^JBbC6a8)?Bn?qqvdAWhL&Ud^f"
    "P|Ka>bFAr?J(mt?cCNAP+9Eb7!6Td&N)LT(jaVrvEMHGRR2fPR(v!Ypo(sDqsWgEe3FtQ$6&X*&C(2;OA_W%"
    "wYWQ^|+cec~Fkfg6kG;&N-CwSvnn`~q}w{L^|rEONPwrxIKQ?o&JYs!~sql!&_Dmb0Y#1{`-)9q>^113I2Q^"
    "cr<lI13dCeK@Q&`x%v7@!jFs`Nar@YBs%~O*+dhyG;fHjP?{PFt%}ZeJzWfvWgdwj6X1RLdvtgsa#&h_VL=r"
    "6M?}1EW#<5aRJ9F)R+#K*L#igx_3N0M>bCqcg@4zmF}5jnlE-!!W-=dWeTataVb1YCB&3eV5HmIzc~E)U<9k"
    "{*`S~|#}1Vj!ZnP<+5O$EEO0v<`_ll)pUpb8b7YlW^2{~DSZgfHl3>G<rcy}V1`dUGSYuX-2uB#CI=ThOqzP"
    "Q`{Nxu5;bQ9n^>mK*i)E4La|7(rE6H~u{@nLU8UH_l>Hi5#|JMmjI!eN|Ji$vN-@2x%nX=V(EZzVNIT~#64+"
    "Q4K%Ln1fVK(r`fEEERt?}603s=sRRj1(pqCejZuT-1}rW5F-DZfpA2VFNsCp)V&zuY2o8?VnT=yK}SD8rz0*"
    "a&t|eAJqfDG$kS9oI+Vv%^|5=O^7-GO~+#s8&LvCaQ$xlu0t{!)=@8zr`bQe)QdNxC@qeFlR?}Ymk{+%;QC("
    "d@K-NY;JIZrTfF!H-HXwv`)8$-cgHdja@mgRSwKZJos;ZtWoIy?ZVS15_2a117c1m4=zUoc#u<vmY=Si$}BF"
    "DoPGM>+!v+tx|jec3LzthB*WU6u3<G#VXd<p5@?wHVU;6dlwek#@m0o_>f$=73a}_;akVPqbSI6on-zo}W%D"
    "?xZs)5s1M8kRf||MLMFW=+y6^AF85;wsmoEtni&DcB1`nm}EM<)G?N4|1FaUfQDyuNqSe0BX!5r-&XMGB=tj"
    "PS-=5|_xsI>uRmWS;8sj;tWRS(qj=kJm<r2ty_q53q)Spgs{_=ePk|I`cHqg(-n14!A1`?0PR2K0+=9=@lbM"
    "5p!nk0f<0Z|_SiPsai1Dq2Z>u5Jri-sX@j4F*|8JW4WEYCF^T+NxqM{fmh!D0w>nTu}vGoyLV#R>My^O(|O4"
    "q?MUad{;!JczhyZ2JT}+3Eo-)!ndfTOgT^<v_FbHVkCH2HAI%iteZkj66NpV%vJJLR~qs7S^^ELSjB9A4+0|"
    "eB?Y_Uq-o_#3EB#oBWCiwO!Fn%K18o~x%D&r@gjnl&95Wz;>%$_f*7!K_<9$={xT9T{`_UXZ?*r*YV?J2o+z"
    "=LhOiY^6TPGDZ~xkPJ9ORm`ckA~fX-Wc``qnK{qlJ~@;jD`WpzJctSsuckT-?GCxD^Uhm+Y=co?U`>cjrs>C"
    "F59L;4B2{}LDZFOk@N@g=S4K72jI|NA@=pAU!qe(*u7_2;@t<2Y|~C<_SU16udHk1{|W{ZNyy-0#snsE{i&@"
    "FkY<J!JUaWT+o2>}Rg^hcy|s%BX?_Aqs^rOSvlY(#><Z+`0fck1kMnLkuyzRYBonLLFQ5$kY`~I=x8-$J97X"
    ")D+0nHcyJO5@qs%&ZS5)cEthkYE`U^S)!Dpy355K*e&82hyf~KL#hs!MLv}?xO*Y!iWn+M4lqhakCMeQr>L<"
    "ct8#P^C+1a(2QuycOBT6*#BOCeBhixR{=!hx&?2&9^Ys^z_{(3^;XbAV9W@ht3I)pw21Mi1F`*a#0}1obk@z"
    "#{n%1j-MQTBs()4Yf*aiABdtn)@l(f}@y%^!Z&`O0FkRCg~s@ts+5QYjgblhYjzKLO7+2dz;D_Xu~4A&vRQ$"
    "WQcbn3h4gd~4jCWjun(DYVd#5x3}eWvd?4=n-Pd)J_I0Z#DVi%U5mj_07yQ^1?KKmTf;q=r`(NC!u}_H6LDI"
    "eG-{z2s=T`mel1YovU2YwjaZB-f?Ki$_J&fNU}%5k>r)@bJfWjsiB&{jexw7r1AwOU4!|U1V3dAF%*c>%KXI"
    "RjK)Scx^Z~ei8inCHw#-VjrfaWYtuye)))CNboy;T!Yp&L+GKYdL^ciKr}=y<{+h;n8l^65{lfffw<9ZAxEA"
    "5$P1o$Pg?}JVEkA-ps9nTFZ`3U2=e89ZzkoE1UAQ5LL2d@7YL@Nden2JMr$UEj1>zXfO!g|$>0V-C{XNxqGc"
    "e&=2{9X^4iLz$7bR3oN7>Sun|rYh=o1{=c#kp=gG~g&_z9g;`9{XP(P~B9j-_Yxg};1`Wl0@<V<Q)VW*UuGD"
    "aT$iQZE*NkL?8kIAebN(oaM(cnUY;2jtDuQ2j5FYbHGLe7&9P}^lSz27ma<PKECiA#>r%xkgOlzsx~%`ElzK"
    "=<kC=6y6b5LfvvO)43zc=^;4?w$CWk?A#W8#O~;Y*ZD#jtix*V39qOG1LKVYA?WU7`s)y>;Z6JlgdiV!IUnT"
    ")nmWk(p&#`o29mN<6!e_0;c*%ufAjQ)~nD~(4JmV1GnnBT5a2Zx%B{Mw$0aRPFRb0vQav6&v^?C{ZZ`4*$h*"
    "Dgx07_k(4B84a5PTuk1#_>SJ6zh7dAqBc}7a?VwTliShQZ1}F2=h_k69LQjimNKe_p-3~k0{<F<r<qUapw9y"
    "l4P8&T`12OIJxL295tv6kFa9xGLb$B{0B<z4s)JuZ}p0Mg*+Ep(vx6E|_@-nhfzwV{!J5c;mWA{1OA_m%FDU"
    "11J3T6vgOtP}q_9!Y*FhoZ<Z+)4DJL!erRzp1jvE;efoeYOV_5XqN{Wb6uwVZn~c17V{u$Yh!8bL4-HZp`M-"
    "RM?p40I>ccv%L=Xfj+IfT=(Z3DX<p613LB`?%tOfx2!K8Dz;uO_c}<qdGkB&5I6>IzVK((Z9hL=(a8_Xn2we"
    "V8#IF@X8?wdf%$I$KRV`)csJkbB+QZHv@(!2i45V-)_iUUsn-3k(6mj@~tJrfG$ugzv^fSH2a3-0f*`n`b1W"
    "x?6CdjnzRLJNmM55zBg?r5xe}Mt-Gc5D#KPfH*v*@E^vek)sFDiJdPw2b&l(er8_DcjWElxQu2hdNY;<ehK7"
    "4eIZdamW~YLhHXvWPq~@k`R)GDJG`^MuBtTgW1Ij{fnLgAcUA-8_(|=#ChQs*te_vi+qsj1_%gbx{@cp-^{M"
    "Qo5OMkz_$eqh;;D0V!0=)OIVG7}|LEUc|Bf9m7b>;;jPME0`nC|^=)F{6B_bgA-xG2B*_vPiE8b;RdKY^rl*"
    ">l7%*Z`=&0V}}ky7#OfE<o70?@e&$9%~skPC&~A3p{G&MEOrF#0zFjo`kZNk*(!Tj{)%@ziX*#-+~HdVRaCu"
    "#u_<3p2f?Gre_Ky^aqm(FrdjKTK7EhsS)Hb#Z^@(DMk+=LeoosZ9j#K%N%@35K_ze>9s!Vjn;Sn7_}`F><yw"
    "4!yc_sIUvd)O)A|SKee_ub{kH&=TGfJqF85yN=Bc3TVsR#0xX;!7$7uV*=sc!5JJnuq^~>TK>I2fI2F83dn|"
    "mSaepmU8Xy;S3s~SJZBN_sq<+|Z+4`&l1#3O-eBatb2sFj&DD-oCf6NawHK{M!=W%UH#Spu1ZSJD$tT}<i%^"
    "XA>`fBajtpW6`d!PlqXZ^E~+(38>7T^w+BvnqqnTkYt2|0J^Y9>pJ;}G%mw^dT|fqXL0sERnFI31gcm8#bEW"
    "H<1nmTRFOh6I9qJ*8x7pekSp0OV|##FY=G-ES$xG`*FJ7>4@7@w}R(i~v#j)fs<2Jop9S%MQ;F4XcK|V~U^D"
    "Vqq0x67=;3&r2F-4DUgK59AwMWw7gb(-NWxe}4^NhO#UHxrgJK>M(bgTaapk5ey^EO+k>wc!@mwr-FLQ=Nw1"
    "H1%<tcs^#iBO{O%udPH(wm};jQeF2sba*rX9ja|rqFa@vRC=M>oiIV~YqM3-Bg!kxdkUmfd=~ISx$(+A(wO}"
    "eS%;M^e)uc`|4ai`P=YO*^TXsFP?zBr;ft9tiFvr+f;x;oQ*(38S;nT1UmwZO9mj_3wEV@(<M@8<HFTJ3F6b"
    "-r*?I4yq?yHQ%ZBimW2)gZ-ObG~hu3hxge3|$NVPFhMcwId!%|i$eR*7Tl4XGGlsqoD@Sy7?>5g0t-flqA$^"
    "?2PHi+nf~o~yToNdBF?H>Cbf-ZybV`~C0aJ^UKP=GCVoxJ=G@2)1x+W1$9~z!Gc|7D<|MdffvvRjL|H->sk5"
    "j3$UBuIQ}5?{4WV+vn>3g518wg%qtCA8d_rSR_iV!qnM9nYK%J6QXn%i&zQqjeRn+VF_bM|4HKlvC5QLEHcc"
    "Vy!6WSAjtIK^9;Ib$S^1DYYu{{QjtkHD;4|>-WCIKltZK=>^h@SScNX@#RK)0P3X$vknWKe*WiUZAey}s>Vn"
    "BW1RDdy^W5@EzfVdm7FOlz=7ch20Qtk|CD2Iav4`Sa9_E)@4{$iQq3i9d#}R!p@7TV2RHZ4JBpRI78%^3*UB"
    "R|66bDUWr5&q%ahF^5#)Zrvm}Mp)WvFDRWQ}$My7SET#Ka|s(lAoRlOmwY)ip+TLG-cu>b`6aTJ99A{5zKUj"
    "XK6JZyHEET~4NHV!iXH{nLH?qI2l+nnJr}oXxKD4@d+KbPZ;|Zc?4Mxgzfw(x+amU}au(!*(ss`_<(0ufOPb"
    "il;uO`Kj2Y1;QGenCfW(Qy4LYpjrWWe8mJpJ83vhWRzhpN|vGH6DqeMo<#z8kYZRz$XQ@^SRO<0%8%3#{TG="
    "yJa=UkZz37W_=~_~Y`yI=vl9{Ls&Z{_G7^FHcfwROkp)D6H|2fFW=$9nC>%Qr72?}=bYN_#!xv%D?j`_z3jc"
    "i!B8F%0lOo4loAJYAt==Al*Wffl74x|E7XCg+viJEr2>!G}mLDGb_7W143>qf8l{A2UCyVK_v3h+xQ=v7eag"
    "|P9ZHgD)Py458A77&6Oky|2#|QhhSyadiid{7ePF75rK1ME8ajsNc+>|5#ocpxJsd3y@Jt@w(o#FZjdi3>9W"
    "sTK@5RjqF;$?Z8qn(s%1KBPu%fmQ+c<ir3b%kN#lW)l5bJeW`BNqsOX)i!V;Gk>ZSG5qaDj0UgRBHHfgGn5g"
    "f54F}O-A!Ap8H5og$j9WT{RhfkXep-y9W5Fp(V3X-IhN$w6-{`e!}{0J)Sw^Nf{wwa6#p;4yZ^)<Mya#YJ>6"
    "&Ef0l~+e}quQ|^LtQ55h0H0*#jbD{ug!wg}S>Cje&qF@+qTR&9Vxt_|Pp}HFK7dlyhk43Y3o@C2a^&BV?6D2"
    "8QpnyPOj_uEj>SaTZSuQeJVXCcFo2m>mB9AphAzFtPYk3hL30(h<Qf{)qbO=nzw4~e<g)Cz5v`C9Hu|Sb6#m"
    "9(7%JH@U4U=BS(<e)~tvP<V5qqI()kZ0OpFU5|tcQLj2C3Kdt4OcU6SMGCN<KyGMYJCi&CEZEOpV|ICfm`1x"
    "PD4DAKZauj2H-%G_1isRFX|l+oXVf26L<mb)mr1gT=d9QgEGRiuj1A%J3wA#~)!x<6-`1jW)n3)tpD&^Xr|7"
    "&mc>hQlZs3iUD59jy2%}h={sbHGa{@r!FCj)mRNw`73e*<RGl)<p8qbdXy-gmxGhZ{@KCah4?ppIQYl@k^1|"
    "Wvy<baUn4QhztUqzkuxs`^BHzy`bHG#YmUw?t3t*LJCYs07y!<)>=p4{zxLBSU6r?X2pYVkyw9dR)c}wd^Iq"
    "R&Ei@0$ngTDd4JMY60pnVUVw;QZ>0D0!%;p^XtIkOzoS1HrnwwnGkno{6mZyi<FGo9#$<g%En~||M=Y<pq*w"
    "wkh9S76+;SOZ=<+2GVx_yJiR(Zj=+8x@=)OT$q@6bZhrP^ItL7X4ldWcM8na0^0!M6kgGwVyiPP=gM3}~tC`"
    "UixR&?P)XFVQY74s$G85(M1ZHZc5T=2MJwwjJ=o!1BR4pP;#CTJ;z-GTQzV8Xvd&yv9w9c%8A`(cw>r7ouK`"
    ")7z5P^6+a?VASdvq#uBMl<$*#RU!@~&sAt6Y2QkR5@Wheg2e~Ph-Srcxpf006`DTwt{xLjtWj2U3Yc?l`OX^"
    "K7g#x_UL$of$0~)z?h=A1vDt)QU7@Y{O8CWbbaU&>Qak$#x2ZPFQbV)G<eSth&fZCzz|92%A-cf*V0G*(&#)"
    "tP=k*<Q-f@#5wr}uEu~_g}808<#qB=GOjH0+?k18CIbA~8QTV)ExRXt0{RN7PJP>k!<a0#_e+PpeytIptxDd"
    "ECg^)DOTQO#~AMPaHc*Q6ydM&t6K33#kNhI-|23r3~Vq~EFp+sk)ybae1)^6lRKk743Ln6kYKx3Em7QWMr&&"
    "M)?$f@)($@=-&ncCXmrb+G1;%{lGbB^$UIyZ9E{9E!%(RHwb*1EzQh!Y((y-}r;*u!cL+r+=ikFWAs9f=R8&"
    "is#2{1jluwE1G-hyeN&kux9n`MFebM9~8KC@c{b_63HQ0q0+^%6ao!}NN8LeW_(GhHF#ai$Mb<9o!b{(Chsf"
    "-R0$tuA8Oh977PvLU}$vpg^bG;LTn(ZL&XzDyV;?ZzpdibK{oE{H!`+dTcHN%^6Ul;U8?>n7%b7VKo!FvSvP"
    "<8Qk2y!UscLzfspi7KFw2v6&QR}iG-TB=#!w+?C)Ld9i4n{q<*Ml*+W^_5-kB`Fp1OTUwD_Hc+JwkyveJCq1"
    "LSL@#2cnhnG~c=&4eWU}G<<S&|K|SAc|p@Tr&ygsu<B`M-uW>bpgSA;}mN74bJgE|Oryb{EW*?sFU5-BG0$v"
    "bxP@h<b-@IqsdlcC46KLY^9r(K-YLw5Loq*2>gUPTW;9OXbz!%(E)V>>zuJB=gm4--}`m&#z8vjn<687oS0J"
    "cjvy0i<n}q9v=G!cpVA&XDyLZzb}R_zT6ppwX^%RgS6<afI}YjOah%T6}*vZuO4~tjlvFzWqh5aI-TW#LNn;"
    "^$H(y1f+zkko9Fz5V#AzL?clw<w}-QPx%Jo__aak(7qqAV*4*A59NBX);Ma(!eQorV-aH|c+{`WnLK6rhWy6"
    "J4IgyKRR5G64a)DuVxESA!PBR{?=^ZV#=Q=iVZBQ%t<!Kl>VqmUKQl;tvvd|Rq7_ASXpC#l3J#`+n4GWt@nL"
    "e&fl=G>k*T_ohr?^g05Qo(LyyKC62-AgdGT`tCH6QnOnzhP2KZR<|z9XJlJ5f=Sk)4YF=7H1jDB@&z6<J3Cz"
    "S+Hcv|~^BX1wZeu%|)pl9Z?tG||ta!}ynZR!UsKEf{;ToHE}V3~4tK79;JAgzx%~fy%=H3z7XE2*RbIUC}GS"
    "6bjw%eE#CA;S0?9pde@(#A_=s7c%uSDodh4q8{D&u;#bY=;*clw<*j5UK-bYOV9Nqfrd{N27waNT&TPjqW{S"
    "R1enQ0^HY*%I*Na6O~`K81mFsRUUfGDaoZsnQsq(7<zcx759CH-0u*pba%6rIG0*};1oH!YR8POY*pKv=|NT"
    "gY9}OvJ;(AO(in}d6+Ya4qx3wxof#7uhUq=Z91heG}BDFa@n}SSr)b1=!u#UldnXNdQxV(j(ohL4`lQDSu_F"
    "-*+v23DAHo-iD*hfVPSqddXVzT;#$Ub)&Dj>qd04PXQ&LL5`;_#35XvFto3<mR_ZuR`R`1&gY=o~WJZhd?3{"
    "o%2GfnPdlMfnKVbvJOZGq(EjbDmvXd8l=1@Lh1rG$j9d#Z_V1>t&VO#=y#XQ%)jw`u*9->r?UVuR`hc;Ucw^"
    "zxc_Az}<En1e5MqbT__F^Ej2|RQ4cx8tja)^vB9Zb$Bd#+xsWS-yNR)bnt3B+J1F#boleZ*}<#re!v8O0gqD"
    "lhB!DnKM+H4aQvzt4f|2OafdhWot~bZ{ERO?KRAAMc>Mi#wEg<{$K#V<jyD`Ypz+!a)NnQGYbd~0ZM4W5fs*"
    "pZ$|YhxhU#*|PgoVxJ^<@RWYBW3XK;7|JYOGJ4v&_vellJXsulg>_!Bn_NOFUbh)eD0*rQDXl3WFQT_92@2+"
    "0Ofg(Ty2#=!K*C7lC<McAKyNUZ38q-^KMP|SV<{V$TN$5|s26(8P#OzA%2dh79hpwfD87T-tht|?^POldWE5"
    "aeJ*eiz7@U}?Bh>bG9KWEL@(<3a^NJ{yj{p?TUo;p|}V)vw#p_Sx&>V`2zgC3?Hx(8tNytAjIOo(%0CB{TN#"
    "y%@ICI=`|H*p$xI@4r%565Fg8@v=zct`p{IoKCLeGAW}%!m*q+Ogwy-bXz3x>fn5T^Oc%O(D#c(Zg3SguLt*"
    "b-*`O|&na7p2O<|TnTn&s9}mQK1y#h0crp5qZF8%t_q5E>W4Bn?elr%*B4f}%u?*L?>U|r1)kFs4?N=wq2ir"
    "|IffkD9Nd{H7EOu!PP<cRlPPt0VVsWCYNiZAvPA!VZ;z+<PECmS8Ozm=oD~4q{a#T_sl!S5Ytq20lm>^x%D)"
    "2t+x4se_rSRYzecZ1eN$LZLm|odOf&Dtg@-#02_!j(sV3$e?p-Rl_QLM75EI{7Jx)g9Mct9*>(165e?vE^a("
    "2sgLy05k4*DW6nys@f{Q1(S3K|MA71%^9OlnSRyAGL<D*BY!DzwW&IMD!z%FcB(cu7}uZY6!<c*H90o7X$u>"
    "mMD@{NU(Kl&ZsBXQ0lq5=cpDCTfSJ=qY^VLHqVkWzA2=1h!ur}H!l#IX4w8Jzw9R7?TA@VekhVV06s8N_x(#"
    "IO*u2a!rZ`|N)_u6J8&!DbmaJqKvMU_GD`0{zCES%O=7f%DYL);baD6tE0t;0FaqO2e~^?pAXmq3{+yL+Tbu"
    "bl3u6?`uN2gmcw8f;DZxHyX_u1i0QpOA4U0yDfJF3@wVe1Q=X3LGUtjElWHb=Rkl$6wf|IgQD~MOKq(T&JK#"
    "$`x&<*9Gz$m)Xsgf%{RT8EHKjE~TWoAuc#B+6*jprOp+@{oVr6`~W7tch!kd&=ZFyH`fT0J=(06dAr3H%0{t"
    "HaZrZ`~9|?|GVG024D|`XaCzlYEg>jEAv^KY)S)@-p*Rjc6iv$i0W%^u5iXg`22#=g*C8{8zdw!voNlQHfj="
    "acC(mcnc*>va_<VNgwp3rH9LJ?KhE;y3;4Wt@28i6CbAye{`N8C1a(mtu;bfRny5VzVCjDXBbb?aF~6fQknk"
    "5{DyT@1xw9Q1W0Xn>ur4<M<o_J+||)f(UPQoYEIfUEEu{ppNyAZAS8AsJq*Olt)tcSe!DcLL=nhQm|t$4@EY"
    "o(T5q;37K$0Gb{SO_&B2_}Jt}_R=Sl+5DFs5E{t=e{cpzo|`E#*596k>C<A1=X(eQ;=!gYSRwFjAHSWQ)9y8"
    "|^V7E33MyVfmq@A{kO7n~5|=KCF2#S)J!(-54bM>n5hZg?A&Ex3ylTzD;WxSd%xC+I9V6u}6hx-X?1`2#!K`"
    "|hG)P(~2+C9-lPEP>4*z%L+q>SH_-51NmVsLpR=dDQt`$*<u7O<@O9?~(nz<Nbr9ro}kZaVT0dAwJ^_Q2qAk"
    ";ZJxU_iaPUNdIU7nX_i=-wT#(gy867+ucS1spxRKiiFG{U}qAS(<HIc*Y$A?i&XQ=tsU_|fs%sK#~H91bb&k"
    "*h+QpIVmZ+>{ABE7zAT}$(({ZiF=(z%j8pSN^$Y3x)T>F~2R>et&}bqce>fAV)taoCZ{|wCA@o*#FspnP!!c"
    "1z`SGQlBBeO8ZU7EUs(lo)linhVHP@=wRe{0EmrP}-UVY%J1EWBaYXY50q<EoE!>MWM)TVP5hxQfJI2HG*P("
    "KY%cSV2m{iqY=zzdFZBWqi#uPPMl%c}J?)1ste5KIB=mSI&B{4zQUE<hHbU=w{c4t?vC2@uuW!&fDETQN%|!"
    "u*yzQ4!vW50Rh*5oeieOESOQI#6|sXCQ9_$F#(*M`iFL^@z3S8mlsQq9ftqVNBscmh}01%EhDiLHZ4103r4Z"
    "38w?Eexbfj$$d`&-I%9COMsJeV=(y^@8;fJCTZ^Ybg@7-e7S>w%BGC3=)H!LV!OMMRlv7crSP90@)Y-Kw~w{"
    "@prBp}6jCNLDc*Yi&9~d+xFgehWR%~xRLB%+mJXKp-0U)2rtuUL`bI>gP+}d%z5{0r;?Be@N5wkLZ<48?{0?"
    "v#i_eEU;OoQIml@6qLOZTkNm^lX+<Xa`S5vAt4Cp5~@ZMYHNPQcLw|2<!+dkJ8yv(Sy7yyx)MFk~SFIR2a;I"
    "V}2?1Xhv<>FnI-(6<$b-t=_*}h#R>8#|uOkLSf_c##8=s+hg1UNu-5gtn5jHzD8jHl`h4JwIZm3V+s5hWCMi"
    "Yplj$S5r${Y84eLy({pn>DR2p|**vmu(iR9`hGC-pj2$eoUpxzy((&g-q`?A;9ikW>EZrDF7=9!On8D>;5gm"
    "bc)><+(3Si#Wcb2f0{EA0F^<#%<#5XOAsS2NnR3AfqHNUrO=sxb6#QiYj&BPWKzsl1(tY>q5NT7<qI%TK((w"
    "S6DP+9;;kEk{#Lkt>dS0*Aoi%d)eeGBMB=xVEU0eeTU`J0Ei$CuBDrDX;O<v_*axac1s4{c+i9Mz78zr#tuj"
    "c-`ygkSxoaH`*nwG6PLm~t|C&y}6mg36z_B+_e6G`^yk*Ikig5$NXmY*cYML(wVlU%ZdXkxF!~@1qe@kgdxL"
    "AiXsIG$9TT~en_j?{E=?)_DfgNeSf(ol(di?g};s>!z;WWOb`V4PNQN&XeOfNI4`V~X28kZg82n2<i1Ixo<B"
    "8V)OCO5a0cuVHvw*pE@Wo+TN%wWebpBwyKWtuDj%89FIBVLWQUaE*Xu^6~hG=E{XgY}YCq4V;s;PMi@wt!1l"
    ";>Pc&?H%BMCdurr!qVH{J4Zk*SogzM2S1&jTpS!<h_i#!qrG1tMGr6$)MlRV{d533rg{<89Tawf#NPy9;qpw"
    "=(Ka52)3d{$_RfA4KOX!_h1?0BOVBbvS1{62V15R?;k(1L^9%l&4-^n2+x}JsT>Moe-X?mBC$z=T7X_!jwKs"
    "&{SLAn*|DB?j9r2bEn{eog)l;l^l{=FVy*Ry#?@O`U4=f;z;q2hM*XM`F-;2Ftad-^4?Ryu8-yR)^)5D{ai;"
    ";km*BHm5*D#+>SIYz$MO=w8g*2rYx_Wi^-FF9Pup6o^56vB~+I6)Hs<?WB@ZAc$oP4jPczZ^qkE=w5(7?Z7Z"
    "NV=;oE#n4yaT%F`OJlff?*%=wCV#2JU;F)66Aex1!a#aQ56ZX52$g6bcF?J362V1kBKx%1qQ$hImj|DAYc;O"
    ";pU}s4zA(bB%3|Gd@WT$`n(fb(<jR5Mm%@bn0Me)<$otL-5QKbctLjn9y*0>IztK33AEaCM5%bHE<Sbp;8DT"
    "=CfqVz@lZg9p%oN|o@%CB8_|f^^(3yGFQNK-Ufj5C>2ZwMN~QaglGHJ35Q}YFKh~H7r4*Y;v<Zx#j!(~pL~="
    "%qF=_R6+Axt=bs1T6kDMt-wV6hN33`b&=#Heb#%@T>QO3fJE$HbX=!&76<{~rA))o8XoZidI1|b1Z^^phl+C"
    "LPJA<cA8-OgBlv5;k+zDJ}wz|H%RP%g7YcA-A_EDKX&{kbhd4rbS4n`*p?XQao_l5Ge2f`}N}BO(ow(}*d>V"
    "h-i3^cuTkag`a1VyyL2PmUqAi^1~vD@gQ>>~J1MCRSVXWv#U#B&WUk+Izw$w%UlyPu`PtqO_i^|L8qkJC=kN"
    "p>;=%S7ElG8L&Ws2{;EfYG5Z(3qOfk8Hz#_XyTUVthQm-;35-cJeO$EW5JW<cra2div$|LBLl@E@nC@h2jZz"
    "9@w6}QZe=E7Ujjfyh-Etx0ifV*{$6HQ9xCWmMmZAtrZUV5Z6|LlSytt9_;;cdYLE$h4$2dW*!LM%04KSljyy"
    "627C11d8f0qJOrRQ0M%YhMKz@m3W`UPRVwtW$ue*`i3Z&iYF5j^takbDqy$503N=UoTxokGb%R32~rFBJS_+"
    "m;pKh`#Jnj121z9f6Wpx8~zAck?VI#<~6Di<Pa>UFs=n(A%Tk8tQJiBz!AB(C}`eVJNv!uh}El5#DTMD(=Ui"
    "lyAOIJ!IxHnSS*-f1mHbJM^oee6K{-J!zh8XFvDmAomE>K?-*f%0S^51sF8-s|#miwj9()-XQMZz!9EmwElC"
    "KZ<kEJ+%O#m<(UO5M$4&V$(Q->Sb{X7jmd0Qs59>7uW@vSG6r;XC~(f+GFMbp{bzMLmgpBZ&Eyg?E$M$!d%R"
    "{4r&N$%y+kWcxpDOJogFWc|gu0UJ~kxk}z(iEtQaU%5Z9xw2ImsNH)Qa46|2yX$fFI72X^uMU?qMD#z+JA+8"
    "TXXbCJQjXj~(h_d}4@iZJ%)!)EgH7F45$P>Xw^TIssFEjK|lP%Wv64;wRi~|QXaLU@~zd@Mw1Y5lE9T`?13z"
    "l)|u22_`Wv^>bYTGk3i)37K3?7*VQGQU;1BwA{B|lv`FQi8hDqO2B5lEa&Q?laKh#kR)()*VpuWn^=2dk+J-"
    "0`}h+7bj^Z34gt3Ov~ue+UC6&lWAfx~2Sd(!bm9YcM~05<d!eYF`o-LGmKGL7|6F{dT!#4G6qxQ<l`e4%4`j"
    "H+gYCHW~#{A|&O^)1)Q2meC_AeXt4MQ=LZ}U-Fa7KDJ4#SE?n~JWLmG-evK_R%(CeTh<=hr?_kPYI40|ES4!"
    "y+C|o3JqDq!&f)~|J#Hl+fZ2Auq^ukA$e|J(leXHiVJ^PT-^&K`9Gd#FJk(8IEN^270%Qe&u1DW=C$8`Dh|l"
    "rRD}AU28mpkLHlQfa<iauA*8Z`nq|}y@-Ozar=Uuo1qsM^?=(DHK1L^P#EFfWpLG;&E2A;I+Z=tt$xunbmrA"
    "GrdR9!)ADHwVJ7mhYXi$#17SvNfB7`QM5rM=V^^ER!+T!W7FkS)I;?vz6LUSha9TKcMP<IIRd2%#%E63Tp?<"
    "vaNjvSHh1xcU8H)5>;B&#y}oY<ab)J$ne5`{IQG5l$Q#0hTS9&6>ws<Kw)Q6M9MTIhTQZPyx-rQ;i^C{A%S@"
    "3%>KAUG=2Y_BGm`Z=f1=(da_<sZm!QFKnP1b=Uh`8{DXa4)_=bE#upW54dk?-Ncj_SY*q1*gQbbW*A)fnw79"
    "+odb5k)@7|*0&PeI%M@rI$4N4)XN;Qy2##4!M}z>EUP2@mG7Y?6@0B;xyI><^*_l@brOnp=wCkpRxhRb<Xc)"
    "FE^X@G|_(9zwX6zspCu|1eOqg+jH(A_ZD4H?Lm2oO}NX!*wa+9HP5yPjhWeO&MLSnQKxc!$9wy?_L#dUJCA~"
    "4J<x0jj1F(7?|BC$4(ya5mMa#bt=ouO9?@Bm-;R_sUGz>KI?7??!URHh=nF41dAA;^7_i%5&H<8nCA1@Lhw>"
    ")}?W?`5fU6ctWG;R;sFi$8|}uK(Lhyz_Az(YZ;Tpa1x}1cw@hY%TKy%e62Znc5x%?-WH|R<d|0l8OUKsJ~lq"
    "wB}TQJr-|&iQmcZ<4V4zc-?7b#frfC-C;;grTpnn_oEV$MHc`H)en-)gy-6ZB%wW;CqyXc6-NG?<q#-yj%3*"
    "IxD!C6I<dZMcno()wTBga#ZeV7!9P;y0+`6`jc|4dXmk;0Q0I9j7dbf0<st{8df$o17(jPKF*;M*Eu~@m3$W"
    "Y<%;YU39^PbcpTC7-3-ke^klsRfh^`%TDXN@d<Ha4sm_(}blNo)9@ew%<g2RvV6wc!5(cba#!P#VQ|Kj9qa`"
    "*}Z_ZBhCb!VEdiW1SsemOb&(Ql+)YBe+79JpREQUy+8FE4IJ&W=RPS4O<C$k**H2r>BYi~$X3nu=c$<4LM}#"
    "yjMC1P+hTj+Z@x7o5{|P$V}uSWU2j^}RJfSOlm&d&|RS*9%x5g-Vl7O0oa?3>3VhUy0}u`olToqF2Zr1Y(qs"
    "@qUL~MY$warH;GA&BHgT=#LU+n^gC2RX`HK#!zw;;Eh)@)1Dm4s|VTsHU_btN%30B%*0897M5gF*(V^Xc!{x"
    "o2mjdHhY4t$Iz&!AP|O<^nXb~fQZo-YU7$D%4u1g9&|ts=OOUsims=#<V>`mvp_Z@*4yYR6l?FVoe76EUj31"
    "?8ObR(SZ-2K^I=jPvfD#9@>wdU%G(l4_3jz_Nc`~6$3b5hV0vNkqS@vVk6rDj8KTKAIm$ix!>_JB%LWL@8aR"
    "Ja8>Y0~s>f!zHK4(hUsSBZqLI(X}K&Cb16nxEzr@WpFE(R+}kyv&U!hyyHHn$!SwNQqbXChHEP!Z15p@~F*k"
    "Y?4U4}`GWyh;C(MGnNHr4D9q+G@E<k=4@gF=SUPa&$`ovouvxc*TucuLkgDc%>q?U3tysQI>RWIvSflZQ&cj"
    "JIA2FBLmjAz|75C;K&-Lum>ja#LMo%)N{BN2PdZoXL}bXXSN8RLIQmOBk@^#<Rb7;BagF3E)plgH_Q1P&KIJ"
    "*BXWotupU=em@Z^Z6Mev(5`T8Qt4<b&%ulh*%cRm}rFI}PS;%El<}(?Qg-T}gyqL-vh*q@PPsQq5U~v)$tLy"
    "i|+wy<~_}Rm(ir>j22=hi;Wh5pt`)wtm$c9FaFY;AmMU(&j;v;q3uREA{b0i1JJM%;qP7DXt>UF+)x3WK~2o"
    "Q#eFl8YyBSceUA*aa_tE}k;rci_pj*H5lb&m1QC?QZAGP`=n$07;~%M1J!zZ#TPkt}=oCGai$5+QDfLo4edF"
    "Q_@DsG2E9YNCL}zv@<hKw&nUtI3$0REc8_$j54J>hS99c$ac|ynvBes(vG`d1EjHC-`h!JVJ{CA`yQ-IXqUQ"
    "F2z!u9E;_ke771W@UKEv!f&X|M)_`KTmNm%x&!i%UOo%n8m+Q=<;dWzM(D!{Z#-8EEy`nk%1%JnEtk9Cx~Ah"
    "@{FKc85P}SXC9sZA6ME44<*=<GjK!qZ2iYz!30j!35~8yls{X}B&??V5E=w}w{kG0ce;rB^<ZJPuMzxB00du"
    "Hoj3w4ff?)_Df!ghqbhM@&AF#FU^!cl|L@B6e^NL6W&TYrSe>KKo^nxVAbRKl?(&dSMPrPEb!lehT88FF)OA"
    "v10$9aAFYVYEpjWW&;F3bUmVt|7f@4EET5TRrfQG0++0R1#fJ8e>Z6Mm3CqZYVC^*?_byO&B+q(flKHN>-<z"
    "a$t57J~G}0?JnJ4u}5vFKZj$5e&-l?>0`n2cTI@*nFZQMcRh48<3(*gc|>EI5e&C#tojLy4hZdGN4CXr_u`="
    "_aqo!ve@t%eH!rAW+!IYjk6MWE%G}gV)uK{%33g_>m<l7PoKJwxU{io8fU#)k>V`B8;N<E$CZnok42Z0#cH8"
    "Ti4BLQDzR0!jsia|P*q}GcP@HGrXt!5PSNckDIs93lD#ayLm3YMtb6?4zOu?Em~)GN?ux&T)wtkK0A*+u=cs"
    ">1f2C821|1OLl&A;mQkUZ-=jR$bz=ovuyTH5#i#q5LDq+~VgR<X2WH7Rm#>Y!$0T?zktBktV_l>lKreElY0i"
    "Emr>c@AiZU3E$0V~36!F}i^D`-jGCe@?_kI(x!p&Fk75y5{i*;5q!LJgnmcas)^GJ@#YJBdb~4)H|cf*-CLq"
    "&vm;HcCW+KFF`cc+_@J<_B1}&usqN9FN5S%gS!hu!v^0qsD*}GPvjgjULR{wi7@P7(JK~J~oFr*hi;L(pGUU"
    "g~T}G*t3rzhzunfSp(t22v`PGl-7nt8MH*5pwSXRb_lw&Y!1`_YyJAUwMzYd;}+3hv!zzn)9?`A8IyR%b<OO"
    "YoAnWMzWu3%hQSuQA1`Cb{1CH*P<-LF6_VgfkxFntpxK=5b#x>LzoWcEXMPwNx*UJAPD$vQ#*wIThkkn@x~I"
    "k5SViC5Sq!wlVY5gaOS&4>=g=JBqR7d8Xf2hwtZ8I%u*L@(F!<Jl&S>%^$w&DQpp*K=1J*ch*azK735)f&T^"
    "BNb=h*i7xvrLMhM=-7*oWCn!r+kkXvi?C2Tv(W$}|fsEX~VmM;8u(8i+-LS3aXZr1v7t^CgC1{%~<|3iqzA;"
    "sL4k3E_KPOM=9JL=Du<8KaD#Hi!v6+2J1-H_I5@yTMu(h55b&@T{tyaP3&q>jlJbt7<u*Do$!3-^Qhc`JPdH"
    "^$&42OF@T&@6NI1Q_70t_p2g>0?Ifeoj_fr<uZl%3f1lXECz49Jst#kio`x*IK4^$jpIAWW@*e8hj~%a2<$_"
    "z7u8!q<%e^!87;b}U!!DG#xBQKim~<X9@fJhUvACw;u=mjny8a$W#e-x%Q;kcgOT<YNj6Dkc2nJscTtPdpL9"
    "Km-&Qf2hLmna58hPcVI)!s4Tl=!HmNMs^yUX9<{5P&++i&tV6@1_ms|fIR_sDc{-3l^sS*MkA9{WM>Q7pqq$"
    "O2oefuf;0BY%ts-T%F91fLeKp~q<EcOi;H~<baL(e@qAwV$&Le5u}QclO>^GM;9I^G5$m-+NoEeu<5d@}*n<"
    "S3{nha$wDdTGDSMnI1Z5Ji?9=RkA5u#*?6<!xS82=>N~@rTQ;-4}lz3<tx(E}>P8M*Iiv0kn4t1G(L;N&whe"
    "$fn<+O6DCnn;U*d7@DRZ=S6&j{>8@5t6OUxH0tNhXggRZ%4D%h5kFlsYs)R*@?Nzkj<y3dGMK2JBL2)20`-{"
    "rmq9@eBOGI2(@@*?_A~}ASLrU~06>Np<XGE!6YtE2JAb))`258q>WkF8->)qh8ZVF-ZPP`q8MY9A6~m2|P!*"
    "EW3#f(-Mr8!f^_Pnq@M6(h$3lOLU)V>Je?gp6|9IMmsJb=B?>Rr@2vE<gzu_KkuhM8ox|)L=YSLWANMp1+#A"
    "*%B8tU^oNE+UErd-ft_pMMnVAa>|2Md+)W#kbaZ3`2w_>8J-dzBR8Djq9TSoq-D)GR1T__05b8B`~$g-hSc7"
    "$E1y!IE?PRB~?TLf4Lqfx@VG=W`T7!c`>yv2(7j;+;LU(j%Al);)KCM93w1Ejmr+d`z7A(%EXMJ&cMv9r-c#"
    "#vdX-LI!2k$@P6D%kl1H_{G<gFaP{i)KqyzZRLO~Zv0^^UJQppLOD6tJkLL<06vuy1biWY2r*MPPEya?rz-^"
    "7K}Bi=*ZJ(eUKYmkq|?(8o|BJZE9#Bj&=^pV%F)pe|H-8z?+q(aRE=AzLdok<MlM%x>5sS3bcZ5AQ-_-AH}<"
    "DIb8E)OrA_gY{!av(438KEn-JSHtbxTERk5T?;o1PZbAvT40$6ir>0xX@5mpHcUagN~Z<kxo3HkCluB7h1jp"
    "eE3YGiw=!T&j|@6)q4EC$`k+iFWpF^De18;`{o!*D_AdZp!$<CZGJ|9?Bi2;i^DvRcm%fg7ZG{%*DGg_uC6W"
    "9{%W<Sgb0+wY-R+s$}QyBMVxIrMJRX@7%CssVjKql&YWt<&PN7ZVH<%xg#(JvD+8=>8}`dSPs{B417roO7Zm"
    "hG5H7Np>C27SHaf$Djy$o2N6~!-O(7OQWF<Rn%O*poz6~RY)-V)kYDBoC880g10mHc5;KZdqMSj8`~Ecv=ln"
    "nwG`K~x`U{74EQC)p^P(AO@sn0H<)xVHu2poDSGswB)d1|vQP4Nwspj*|EY4s`Ae`a+_qKNJU!d{{--_h&wK"
    "?C(-W}MjDOiXGB%5LjJW-?12AJ>>;XjN;dkQr<U$<$<M90AoI#ikWs8f0e_YsLl#vSvr`aU7gN}}0A00(Zx9"
    "+!n53>2`qvX4!m&_o}<KX*)GY2no|A&M9AHnCj+xLd6Lyz3yl1#0-+GM;c+~nj;9DaX%a(1AA1xQ*n00Q4cL"
    "qZS$)8hdC`J&e!+{zD8|D%_qP-atDklKcAJUKe0J<U+bro~p!Uf4Kl-yFn^D(3ScixV5;WYOynsAWGOBw0f$"
    "Wu;`@FZ*`NF<mM*6RtJQET6JZmB}nFZpsnx^QNJIr3gGp2>MRfECdS_5~&4>JtY8)ro)+l)dB=<n%@lk!GOE"
    ")o`(UF7jfO7zTY&QBEEC$A-GCxV5LfM|0o97gJzAbNbu|VLS7UU^1AnOgqB=31osdfKECVs`(fFapD+E%gq_"
    "Lw)Ey)M4<OM<V%I^>;kQNpPG+acQgYz>z9ykFS;|Itq9TAj9s4px%PpobsVGHEWh$#26Hd8^NNxKy<8)N0x*"
    "($yQczE@l$65g13g<^LFR!gVO<WMNPs=O*}a0xkvg|>cHMC*4ys4lDEOF!<hH;a9Sly~hI5jM2{wi81n2O8r"
    "C*fh07*l`D<Dp()B^Zw_v+D~5WCzb_QcR!p30kedhf~;IAi#t7K8)t__3p~_~X=92sdL==ES6+280+E{$Wyv"
    "$2Fa>Is{z|YkercA$3I=L^y{xdcm$Jy)NJuHWL{O^^jhA4B+X}+Gz{03Ea(Q!ha8j)bs;EjP@?g^2zrH7v8-"
    "9Coju?i$~)4>*0%fD32m~c!O=dY+@C}f?o`Wk++fkCbXfh@3icpVcZ@JdBVHPBgZDZ9&5l~4F3`V`4g-Lsoi"
    "XPM$_y>r!7AAo8m6&ld-Y^H;1%QVjN`$Qhg2`+Gb=#D*C!R9Jb-wSh<7&mpKfP^H+u+wN&y0=sJ@s&nIbIKs"
    "~fgSIc@&_(7no@GlR|cv;V$xwO~{1!fmNYY3fjtLi07Dy#O!OSl|f_5C!>rf`bKs8a-5Xw?iQFgr81;P1pdJ"
    "27fw&m=6t8jC~5sM`=quFKQ@Fm@}=Zbpj+60PmzhE}xs!p$~1taSD2?cnG`W4EY<T{qp%K4Y!ZnZ^u3nZsG2"
    "FZ>0{@@g^%NsCsbc1N6^oPU~2LOHbpTNsu#u?`Z-vSKWhrQRdcy39*=a|55@YXL*Fx$D1z-S978jhc^jO9r0"
    "O_UfV8wFskpcJG=6X<JZ~2BEe`<7pXruMLjYu_MCE*oxTv44g4pVOvZtvZXy(Mm4vZ+u!R1I&jbOAADxu{M_"
    "1EU?Okqp<Gb{=Gs2iI2-EyHgwtFj4qT#)^yy^FUILNf=avTVSpLFfy7W_MwS|&trLC_bYP)d7P%cB&#~OY;K"
    "Mf8PB~O6%9GU>OksiPjPc*kc3p+0{RNJcZ_}o_Vm2RN>jU*1D6{-l+ETx11K5|`{*soT1KDd$OM?3pOgRf(I"
    "Idn;JO7S~h<}8xS~e4y<I_dd9QwBxAGnFdtUkG*6uzB4;mwrHZBHiqxa@4APFGbMJK{*g&<W92<FSrD<fML`"
    "B@UNXuWff~mzWcS7*a#%4Pa6WaS$@E&0Ncy8YrNT7S-gPyf6I{q+RK%8%?PfcBiQ}t&{Y38u62**bd6)rxIc"
    "Ha{sQfEXk=ItOuTuyK98pH$${{D4Ov_-Z0$;);Iu~{P<)wOfA}21ikuJIM%2!%)s^PAkwAUlTbr)Wz$^F4*q"
    "_ye*u-w$@Ph#R17`4x4(aIiof}8eD~ve5#@<U&rCy3=5F63Ahn&5Q;V1uwZA?;3sW1Fe={_2{*8>-6R1<_c("
    "46vhC!K1x$Fg<oAp<RJh>sTWCWE{ZqHjx(^TUU81YY-VaxO$^NyEG85iZC;jGvmn%h-1%kQ!th2am*e|UW{I"
    "eYcXS%3Yr2&vZVNag+^RU4$#NDk#6ThaQxo)>8hCZc|%Tv~nm!GQ^${n8YvLie}rf{$nS_C8Xq3O-N&ncype"
    ")n4p&6DULhBjlEIz;#cXS{IYrN`+QYMFaq*WY2dpVf->IkIJhU>~W&!DgM%%$#PmGSnqf|nauL(WYV`E9mKQ"
    "QL?wszF1L1eR89%3EX4YbG4MdEr}l-})CCseRgaUyOwQv~T8+Q{^H*Q3d6;T?+Am)>;38k{P)Sk~&KC*7<{T"
    "@O;0nZutu<Z2>C~P4FYexjy=^1Q7X2&L%DG3R4bqa6xDzH!Mv<7bCbHy~<W6TST^b@l2`vy{1E6KKkN*4l_M"
    "@ICfS|1O%)Qg|#UdVsdhXh_A8QR%U;x(z;~#WecGeW8mf|LjAwj#Y{@?=zotp~5f1nuUfit5=<ns#0DR8dCX"
    "qmChb<s1X_v<$TXA#kZ_@{BpEA!}4@2$XG5Zs6KxcX^*7*iVg%Lx5oVE`wPjsckk(<xb#rc+o-)2Xr}(aKVJ"
    "T2AWC3z^JeW@ahNrr5hiTqxw5gAEMf#z2Z(<@qgIYyT@Q^1)=WO!NCVwqPqMGwTp@c@#~}GMwFp*D)aGfa4n"
    "?>;-7Y<$YPji$LaoA--!CSD2fNpgYX-yJ23|XOm4D+>u#)H&vNf?VzlC03#Ymw7)=L>MX2+llgCu*7sOs){A"
    "U9{SX(BrV8UNF1B`uUk!JtN2b#-P4V(O2{MWGP<qV8mQ|W@nd)g=k)5KixJ}S^j4BydxwqiLWy|EF$!bo%Jp"
    "fM<@82))s~ccssP)tP_bytU+mW+05elF#!G5E|-x<!b$;YJX(Xwb>7lZ_BRg92r<LjiX3dlALBMCvOSFXtN3"
    "`A=YSK|ES_0eO2^{RXU$#-qJdku}p)uf18SBvEo$_j+m05+MzfSd@QI+*;t_x9+VpbyXV9Q$NPERlA-ERt-N"
    "EW>nwh`W{|4JMp4cAj&d0HX8+0Fd!1Um4724)M$*@$O;^S+nA4t|l^_3F9T(YOjkB{cUiCfg~=r0Qgd;>`I@"
    "Kw?z)dZ$o?Nsn*7jzT#vn*V_ZYl-vqLGN!*1q-8-gQLH~%EvCgP#pIz*EvW$!H?&1Q_Y|#xEt9HDlMtjj(+k"
    "izWK+cXEoHk%#p=|aav2syl3lNBk9_WF+XKK}@!iJL3E+tVN@!iB<#Vl-j$&e#1P-Fvx|-Lz8gON&l8eBu{-"
    "puyL|P%|*Y=z!oE)>?Wgw0Yj=^-xP;L)Ug<V>hAX?$4Ema;2L`<9~{52y&SkR?W{o{6gZVh2dAAc}KOAE#xr"
    "wr#st&x4>d;-#bN)<Qiqp_4wT*@B6EU&SY;cS(`z*}UX!)oC9J;|=e7h9`pKKMpkJ{g1D!7dKbDUKv}WIvU?"
    "7>IY5rorqBp_bQlnWxF@9!#bT*$n{qC;=g)Tn29AZUCEAV3QwxK}pq;-kN$xAXXae5n*vZjglgsG4!Lrn7p_"
    "giZ?(O9wA?wRWBfGW3GunvfLobFmuK)={@EXevb_u4u|Sc(tmFjhj7z~;Vuy4wgO0FiV7s^8CiQ-bL4=k^4z"
    "cx5L*4%bsgepdc%;$MR?bO5~0wBd-NPF<;%Vo){qu4m#EjYuUxf`$1vNVa5)R~0>_Eq`;t2!_>1mk52j+xa^"
    "rgogEc~j>k@5g|NrU6zBl!TbwtM4z+W~1Hm}h}9TWaLh16-6b)-O7$soT2E)bl@FRDCGUzosB_#VUo6PWV62"
    "p2IyBWFMq@>tDtmM`!9_!S<oXV!kxl!=pv;gv#y05C6vyrQQ-bYXP1p`FT0%|H~LQI%g)P<0>}d;<TO5bKk-"
    "2<T4*UZ&8W4HRnl-+!no_U1+YuQ-ziB8oFd+@I1$pa|1cAv}D;HZ}N^bz}FHG-Wv1C$Dx?h&yZ(l>a5)`ts%"
    "U&#%5wXza@s#E#v@nJ#v^^YSbC%_>Z%-lx8r?tIJNkIO1qAkQ&W&xWViF`Gr~GWG)qwQ0gaECJ$zu_2nm>61"
    "C-*i(45=kqRPVB>?3e!o6VDFdV9bx1sOq(7(BCe?U<aEevYdM9|RS$|A?Z$a}nR8m2<J2hlZpQifNF&H`Oo$"
    "T)GzstyqAW0BRzByC}D;A?#;>!^==6>M0trtMx&z;b7-S>bsi~Z%-CVOMDBw(21z%mZGF^m=lxomN#JVPEe>"
    "_{54vT@X${#ztI-|?DWUEi}t4uorV?A-aC2B700c=U<y#~b2j0Dlv9af)(RCBDh!qmXUZiS0aJo?=SkNOHh{"
    "YY=mASO?F_JI3B&DM;H(IU;qRcbiMU<o8PbCb3-;#5PWlK7zp64O-tbK04y<j-{~Ly++1P7PPj&NwTD#9r4c"
    "QO?sTA`RsP2w)c1ME(6-WE+vHQBG2*)SkU<m(pqBk9rbxE*$*dz2h}OZ2{<2YehbbQ;H`q5@BC7E1Lp{U$Xh"
    "z^K;{KBEDaY|Q8==5JZlq)bgp0Hcqxboq?C-Va%{en-ezWWMp80%tpl+w@x6;sT*1>SXXl(5kKT|~^oG2Fya"
    "cM7cu^wr8m56#?B!7G6<0|G$23gtLT;bzx@1RT#_Xl}sv0n6ZHm}FSIPuWByRl0>IQUDIKEXeg4GbCS~iQB;"
    "3=tw=&(&;AEWInjKXCl)2=@-Yi|*Q=@kmN3)6IfvxyM%p)e0W5-;|#{39=TcJ`FZGR(+wgFzh2`(YHvOZcNF"
    "Cw*V4DPfc*IQaP8USB-tVx^q4(Zp7$=+H`5KVazsUeU*8UapE*t4d`YL*jN!(8?G$Wuzv;8iozGyy+S8CNO0"
    "U!(sX7mT|7o_qZroN?>$GV74MK8WC_C0@Vboj(nWquIrKr2=Q6G#GM#2y4v#&iW#P$Ydy3-uWEL8lccdVF5k"
    "<MGsUk}DPvXF2_C2?c|b0<&dyKXOiw3&eLFeB*bj%vf@jMQV>O4?ly`pkdNRdi6C2eDO=2*8w6GrXMTeXLQc"
    "aBDj(6tB@*9^r(-IO=xIgT(Lwhy%PmUo8*HcryPRFRDaN+j;E<fqn`^A8#nF})@i~YV1Yh3HZlV2ytp1Rc|4"
    "q=R}?ZOk(V&3<}S*i7`M1uF!@q<0>tb<V_L%eFy>yv}Y(RBah%-7`_i9zpRz9RMQUM;v!wqe}X2uA}o7S;1{"
    "4Kx|*Kc<I<GTFyV@nEiTJz0H58=k(NI}#d2N$P=Id!B~@meHDXV>P>)HBXLz)b@cuoYP177XzBCnVK|^Kkpq"
    "L{ejX#o<ZSZd*%6n^$}m9{wQ2k{j#PWhsS^0J32g=zCGLfWnwQmD(kuL3np(d8m?ic4U>p)>`}qV=rt75mo+"
    "h$L?xzJz~8WE!tXwvs^e@3$E%_w3lrX2(}>f_8-O1;nC|Z#9Ze5w!((jc&-Lej(6TJA?<RifbaLbq@lfa05+"
    "Z}KWW!!aTO0O-1Oe#RkMz2y{1xylK0}_reseUT=tVAw;;b<m)V!c*xzJRr3YffpbN>7Ebn@ot_nK_#I}8Q2k"
    "*y2mwk>$WZMgH|+F*e_#nxd?r<(LsP&~$V&lCvIV_tCV-Uc|<+E)nGXar=Z+q46a6Oxx`&;pQM#gxEcMVJx*"
    "6YA^JEX;_P+{^9mojVQtAuLoqI!)$sO|p?DR=m;zW+DV(V6ykBn1?e$jYfjf6#|zgNbU-LrU4vh{B%FVKLq$"
    "WFE!FLQ{u{>_xcu{u)syV4hDK7C}+R<TQ$cbE{Q+F*s4nuh-_YAG!gUCU2{N__HW3~(ZTI$s2J-$0EnVB&Hq"
    "i7|2PG6-D1iF1d5oEuLlSe4k4^ap|i|Qu0}d}_Xwv&#ibsHix|CPcsxp!5a~jN$*BwAibrHg<%a<;^a?mdWe"
    "jxpGR+}Ana@E;p55e2?wm4?qba~5b}vv4x5nn4<=HGr<7uA3d;sGV{METcMCFFmcO7zc7G}{^{xMK4YP#LI$"
    "Qr?cbTbhrRe&Tx0kLgr^5Gn$Gc2JV1zF~K)gLlPu@v=iaXnfV9k|Ck`JWYvcXTDjY99CSV=o{A0_PziX{r54"
    "jY==N;!83J2b3x@WP)~NC=9^)3UUtrNHUR^!#Mkp6nVyu&9jrgO-?6=zZ_5ZewiGfPbCTN8Dm$&B-rW?2}&E"
    "bk1bRbpgaq5i6aTdHR&)sTarEK4L)Jg{L0D)?2N*r2Zgd47Xt9g=8LdOW>lp4oW5n8Tl#eas2JC#c~7$?V^L"
    "_A%8T_~L|YR4mr0v}ZRmrYn7!bSDv9Il*=s*?bo6@q)85%+`u6mQh63=&M=xF|H1W~y?pI%a_2L8MZ3d4h9h"
    "*T#|MM=+zI<dcdv6Y>zfOKv*am^C--Klj2MZuR&WmbnLrIzg;66%SD;@R&j?!>ZIZ!+GzF+t7D5TOcy`dorI"
    "}H2T+c$5H4kxD$ii0Y{a(sD&GqPX#!RR&CtWxwCrG^_;#$hqLnG(^o{S+;O8O8r<Z=9<ULZIMnP5qNBBDKqC"
    "LGWA;moOf_I{;BWP#aPGev*~TcvcO3VB!k<12#DC6Aw10q=}&G7=hXJYdr6<Z;<ft-3dJYY<jwW-Ks!Io0Em"
    "^D`z&}n^NGSnGmC1;=OHCF>$tceDKrB-=~MK@jUKdgSdHhF6wOmbnkrsFB1Mt?MYl$h(p_0P#m1>126htPR`"
    "El>dA!}UB@tZ)oPi>J$rk{b+ec)r?apM)BKu8LthAFH;(z=zB)^wbsW@O%p<fi{0@%~&ky&G4*!de!f;Z`D+"
    "$$$oBV?cx1%KI+De1r<(uRB3UEx#QcQHlPYs2KQ9((tq``lz&XZ1ae>!=4%nhvoN{E^&S$8!3wUwE!?(;S7k"
    "vP>YClvnr;`Xg)6L$YI6kq<>RIwcJ119#$qk0?;gXkQ4RL?uJq1Gj%hJ(M;941MmuLcw$HkMN_$E`}LWdUnl"
    "WBR9^cNc>uwHiyo4Un^+^)9xGD~!)9Ve-e}!b@xj8PIf_GSy$cdO6+w>g#?(vv^Sa6kbLHi|{Vs46WokE~|~"
    "2>@^{n(-Vgxf^!>2#cwtN`8^}s<}ibSzsCb06`6ULUe=sh{MWaWx1QW#&F3YH?n-{9htqSDD-0y()T7AQdkt"
    "v;ysgpSJ_C)ko^k%0G`P!6O<y2Gv&L>WX=0y?FTDt38}7;;#<30?v}@$AD-^Z4B6Y0+565Rt7-taq46+|qG2"
    "|MUu^F!Y^a;Ku&)l%EWxMRJc_y!X&?&1n_2`5HM?sA?v>@|=I`pvZ*w^Zszee*XkG(UxX>p@kYwoqG9P2O&!"
    "#VUbc!E5_PXZxAAqksC$Y4+dIA(=T5@749dPmXRjph0_iM0Kq;h|>nr-ebapF-Gt(`F)vyg}^p09i6r3*)ht"
    "%A`(@=^rp-C}YeRb+vASx3-3y@u-s7tu|(m;}7YpeO=hlv_Gg>jZFgM(13}B8>gK)oA6nhO}^Q3M+jTBHks3"
    ";Y0fF`b<Pj25wXj>U`dU-ZjPF>@i#aH93KBd%ACcA=}ckGd8w<`BwtrqQx9rpc(aKeDtd5u1~}peo{E}Xb+L"
    "uUIIs`Mr3m7_3X=?>U>Y|Bj7_qts%la*cJb3q5{9e7`zP9v@25pPf27jWG)cjI9$~gw^V*ZotdsNBTwdfw=H"
    "|$$hvWqH+@hdaEJBhIQvwba=g<;g6-hj=a|cuIeY9eIMEz6>iE*VxBXPGK@Ev~$Q#&zVUUeS!uC^L(jd$q<>"
    "(*Y2%6PW_m&w7~Bd^v-&bm&6bg>0J@Gr1VV6kt{_dU-5%W)h{i&e&7pXFg1m$TSskMS(;E^|*m^|`D4ANkdj"
    "t#e8$-Av<S%$w|*Z97sEbFZ7K7GyOmv>f#W?0WAYb?#$jqbagSbf;;p%f4w9PwbZih_$r3@$ImXZB@mK649?"
    "^_O_PIvkG9NoWq`h#v5G?H-pDYqJTpaxw-_iZcP-8&paz)JdPhs&h{Ci`Q?sl3N~tY>1iH`=znAT_&`f+=(4"
    "qJL5-%XtV#eJ41E2O+j@q0@<C8vY<+I8)a^IgtV=n|`E6{QYh)gze}if?f~ayGeYLi4@`F!u<~5B8p1ZWsB6"
    "@)*ThLM=eV~g=AT%A@M_lR^Y=YiNd|_=hjkGoY0Zlg1)i7Bs;wS-=0NQAt3R0{_0FO%!5JG9Hj<-)Idk4R72"
    "jPb>!GQN^IIrSjyyKmhR44)4$pU^l0>kgYcFPNU#|Mz_+2dscb(khi>p_FA_8UW44Et8lZ^r@mn8`dE9R}mn"
    "R5)WAHJai4dl)T{;cAJTF|gwBQ{D7uc03Q{E5<>>Ht<O(v2G^5x_R%_*VC`R{Bu3x82C!W4+3w$T5nU8IDQa"
    "S8Aqjf^_BQF`N?mx_Vhh@*}JHyK55V_2~0G^@XnnkWFa40K0@aaiHN{9EB@tvzfGxkXnPt2Prq><RFG8fj{g"
    "SIf<Z95<61tkypXa2R<i&p;0g`z3U6E8{b7;ck+prT8i`dUbVu0{4Da~%c>gbZ$G;ecm|Ev^T1$(DOHXXB-R"
    "|!l?@x}}zFowNcyZ;d{pNE70FX#<jan?%P!`r;F7ntNZsYsFQMb`tHjvRAWa;SnCw+6gXj60NIXkO-T`ikI5"
    "A~X>+@a@xJ~{p6<otYc+}!oRU%py4b-ccfF2uU-P4}^ut{<k~Z;|(K&}irYSu{W8FMHGD=9|5@XWp|W1z7O<"
    "p)WFZj%Ip~KUTnGpQ-?`ge=s=IGV+GaimcehMmroF|^|m?i+s3u#c;fHYp}qplZ4>BIe5QL+m-&K@1ldx(U7"
    "TLu_qna;}!QNt#Y2_l|oK!{W!PqCJ^a$QDj~xg1mRVCP%Lp>%)n568$^D;<%>;q@ukq{6O+Ly@e%O#}}-2(W"
    "{KexTc};{tRyBiIbx?lkEL&INg}WYy%Q%X;JqqIHGDfMK>pU%q^&C-S9d_VNKAmnRaB)x%zH!k1saLNA=FH8"
    "jmm?8QM@h1d23#q0hV0MAnIZ?Ir(i_rPrFVX}8;EvcTC~yeB0z!t8VxMmCg+>=XwF9Sy0uZBXkogI+Yhwq#X"
    "XVgdOO5xspUek7Pw(RfJGblw<)VChr?hL}^Q<|y9O|%feV*vP^tscfX?PW<V~Qt4HyT|hJ!QS4?P4n@SDTn8"
    "A0aH=`b?v>m^P8FCM5p1*KsVMQ+@GB<$XESn4$e4sILfA3RJd=ISB({eNlzvq#XO=<2p9kS*}&Sga9NNV?Ug"
    "Ha8fzI7*r<Luv~f+r9V77{N?cYJP`Wl`Q-Gqo5sR-=^uDC;8`f1O-K0QA?5+Q$y8#zKCtS1ay0qnbno>PGa)"
    "?$Q&$+iEF;%@STLpf;3wMR4BbdO<jGkr;v$(@;l^OiBy1z>BU1Vx)>s86Q}5@+Fe#(tI;q^GCa5IxS3y<eC-"
    "MT%oBUIyf2mUY9RML@yS08IOK*vLHpe_n(p7<>X$BGnr5Ynq0SsPJPI3H3U05%4u&dSA4U`m55e2Y8{6Pm^l"
    "&7xInivJAMh(<X5ZvAvxv<ul_(w%FSTaDTxs8oUvsvGh<DmH?Mjn=>FF^lTs*w?G1ov3B(53!EAK*Gx<xK<k"
    "NZaJ_E*GmGpzXjn+uLm@1GJ<h^L==g&Ef*wIFK#Z(Hw7?<SD-HS-;h|NWCNuk410$?eVY2C%+wU2is?p<AcN"
    "FU$*-#q>#wY;pG5I=DPBu);Zj^Qml*`CrnVqJox8XJIR!tKGej7RJWk)gADzBdOVpNoEhQq4;#&yFT`FZRvD"
    "Z6jDSl-E+GV~TXGjm(Z3C%0(}$>CBj}p78sfISK=qa*$pTgLvd0>Nd~|t;93KdXm~Gf;xrmIa>vOnYMs<~Y<"
    "i^OU*%OGcb@j4*ra$<e0E^cYL$JK4D9gSIzP##o)6#u*}*y;zOj*vwY@q{98iaM7!wDhvBqn7O&g|v3%?l6x"
    "F?!hu*5sgsjDImZ~eE|#%|dF4sINmZ^K?cdDm~)L2Dm%nxvad&aRVk^U2tC68ibiHkD6C25XmT^o2OeuK_6+"
    "FgvbdbkmXPFAy1!z5veZVoB&rGHrHN#26?Rrasag(q{e$oyAlCX3d1*J{Z&^Xy-jMiUG(Q_47FZSAX5#w2`o"
    "ho}d0c{cR5t+cEi~0_<e8Qpv(w+ots?sL!jNK)ihROumWdJXJ^B!*@FwmrmZiF<cO}HpnFZB!l4{ztp5KKrL"
    "z~^6xqdwR{)3RoN;GS9m<7{SV<w5ITVBy<@=KA;Cgi^Z<$sDPT%W@d4#0EaGU?#0J#J9;6FGMjHCNXThYC86"
    "preyZ0S^>oNefVL86ox>_Y^gzX_MB^F1k+5L9ugB0M>?EG0DBaT^vnZ?OL4!k^>#pU>2J=HGp<=1VL0H|{&J"
    "cd*&M80048Q@vF>jZ%MvH&UGvTmGJk!7AjTW-p^-DnZsAj?t4#`rqL^viqhqrbLxUAhuIFGxRM?UwY3)pIb}"
    "1Ldn7d6E9jX`4EkG?~R4q?u}$PyH3s-rjK6uwwfk@S0&SgY_$Ni=wx;0+E3kXMC}R%#Bk18OFCJiYxX_+HYC"
    "j(RHE#XtInI=nR{eILoA?B6)>9FVIj|R)CHI2dol8bIF|-Mg!1n#f(iH!uiy?R^G1c7M}h3cJQ~o<HMt)y|r"
    "CLDM@|`wXfFkj>)tnm$!`D9bT1rx~gJk?tOc{FARH?e!25lu(IQfC%DAn5`UG@utbr=m(h^S;;O_g17#s)0>"
    "tHp8w1P+8Q$;AME!X_ID=+ZiNgc3(<-a5Fss+)^IL-fQa5cf?{DM#4c|~Mtc~7cmyCb*?5=q^WIn7-K1iOzO"
    "<&^R|3MMYH*O$CRBiZ{W^mc)H5t@cV*|9h^~+Z~>y{+~dJHIS-do1Yo$1cnQWV@rTS@B*k-_~99F4EOT3ZdG"
    "TZ~qYkXa`a^)9>7$WhuCisQ#JI}$$0<KA7&mIKbTe{2z)mp@Pn*P}{NiJ^lT0s}vO2B6_!K$)U#<?Oc41V;)"
    "p`pAILN32a(4|j&?7hWx1-oub8=8S?+zSV*OV3KR5L0^O)>5n=%%4L14k3RzGO;U-IGz}M_K8KWX1qle>3AY"
    "$z{vkBxc@n2l!>(V()s){TdyF_erhhwuv~fIy0n|2;Rb46<6YY?=p$gs+z)FmPFoD)s&@W{M7Cj24?ItX5kX"
    "3ITN?R3HeiSyFm#-W1$z$N!miJMZRf#I{r?89}y=xz_XAY8C6^Ng4__C(~REs|#z^&{-#3G#HMLL}mc>WoNZ"
    "m9b7ZT1r)(;6F<?4jXnjK`WQQY!#?a#or44hRg82-w&r2;hN8vvF~;_5Z#L2miW+|LhFDoenNP?F6sBeE45V"
    "pzyS5e}8ZAUps?uhw%Dt@aik$b<mKD6^)`@#v;k4X`EeGH{;y^(PsJAH@n}yRCW;ZJTt651Ao)|l2B}pdIym"
    "XS>^1$>YIiS4%UpWMrx>F36!dJf$pe)m@4@1GG7%l`r#nZ(MmaytbfEP6xY|`ckCaP6zU~-SissiqtioDBGB"
    "c~L}s_iEHKU>O?y2Ro=L$<97sx4G$R8z0s!w@IM!b&ej4L3eqa0`zTSOVy9=L!d&o)SzDf0g+b`v9({AbsEM"
    "N?Mqkm9Lq1hmVBYhnsWRv)8nZed2I9s_lp^2_*)0Cs^-=xNJ%r%7qj*&=Qjf+WUMXpT;Nt#muWGxYEb$4fn4"
    "A4*u0v;Bhiu~@~&ZU`0CEs~iAH2cUG5dHHF2h+;-3#>P#nE?SmKVS<2SZ6_5#%Dy(AeUq7H{gsT6nwJcM#>y"
    "eYNB5NND!>N2?q0nx7bKwkk@@r|M9z!{f8b>A5&OK0onSPJIxU6Rjp%3;`#I^=kq67h8YZJ9;}g6TKe;{$D+"
    "#rgfnPW>Rh6pWX__TnPHZ)(+FKtcv^&P)cn)TiQd`Mf(xeDYky7TR-Fp;BayhChnFpwv2>oi;kF-v65ztyj$"
    "0Yro>&eZr58Tg|!Mv?;xsO!!+wc1?<em<d@YXjVS&&OIJ}W@)UgLi$z!fI|%OEr0YnEFhvnUG9Hn&3_gYO1z"
    "8PC0a@9G1qg^g%kN0Cb3%0)&mgFfV(N+zf#ebA8Elv-nNae%2zjeu{#hbo3{xaVms}qaYL_J1>y#dw$jS~&="
    "wJen*PF`54VUy|SMnlW<cI>mqpLn9uk?jptGm9Xd~z)IPmX^+I@~|kd)dK>V2K8@OO`d3e>~r9UhYk+*<7sP"
    "Gi+FqF!l4QiqUe@X%RjcT8=5$Bt~lU8v5rdF78cglt4T~4Juiz7LteV<qp&Fpay2sl}Jifj=k6gFnoR;{@mT"
    "!>81#PeLn%SwP)jpg~VhpSl{6+EaQ2eMm-6vs+?x|U1?`0doq(Z%t|&hk{fr3wzW~8aBDP)?$gR7u%#A3512"
    "ik18HYDL+1t3EX7%mI`n^IhxX_RnoXDDxwzO8n1Mvs*<63w_w~8$sVwuNiX$km48(1GKjt9aBEK7n-oUO&Ao"
    "v0@l=l1YMm%XQO?^F#IFQ|81U89RNbH>gcM`x===pTLO2a|`Tl;t7&B@u}-^Dx)uOaklzA9r;-Y?Q5yQR(G="
    "fj^*1h|1qSN~oT{oj-y@AbP*XZk#yyD66es#2fbR4EKqKAD~#oE#thF8+_Wj;pY$ie6sAT0fZ{pZt7sbae6?"
    "EYO|4_km-~`=bXWZ<858OR)z_%0tgf%4?+r6|6(wNrlYI;XJ~J^qaWbw<f|eBT+=(n8QK8$7!+9{w%j*m#wI"
    "+uo|9C56?~~zw~4&!?K#fUWi;3=EXAkR}80-SJN-Q`N|8o_XI=JoZO4wmGKI81O)U<?1Vpf^LoPwOc<T2Luy"
    "G(WUKXD;#x2G4g*-QOx3)K@AE9ebwm+4s?EHZk#AToL_6We+)SKxiKYw5QjVSGtYL}u%DFxGJmb!N2-5^9J#"
    "1EQpb(B?KDzb|?pEBHN}~W|90!9_?-2(^&;mC(E6OT3VxP=vtPPR=gTkFHD5wp&(%ncqvNDY<ac@M(+O%LYX"
    "d;c^p!lb;tXYTSAr1n6jI`Hev7mI4?`X_LD6;rYC`Zzx)U}gSarn#e$?3#Z+^Dm9w5M{i5!XJD#t7$>)!(51"
    "0@)RTG)oRCJq?!)fYFiX$$R>edz#F%;)USm9_qD#^B>`NNjIWDae@b8I2<ZPiS0!hIf(_t50o<B@p_d+p~n_"
    "d<#;OB`Is`((Aur)$`8Q-KaKX&GEA+jK<YIbi-vTKCS67)OFDyQ#!uyUr9sSQ4y84E5)hTT!6$4;UVtX@^0M"
    "!$QHD}210t-vI!hEfX4ar?IdzCYWr&7Jhly3Z3=Bp<WXHS=EK?`|dzN4jRVhSh6_w?FT;g$_5>rXFcNsZxvN"
    "|6bsY%o!J>V(ipy;txgvA}(4QrAP2-{##2!&X3aXEd_zW0GFW1k0Mz6)>_EtY5qyDr#HR<05sS9RNFrRUF%g"
    "q)H~Q|?YysxruEV!|`VNHq;3_3V2l5O--NHe=bkhZ;6Q*bu08tv%$A9)doR3Ip=|gjs~GMvFcw+;dUIWN1y$"
    "vBBGi-Z5&coV7iB_=ZVYuCCxX!YxI84~V3hEt<?H;Lor*r1^15oGg3&HQKpuL6lIaFT_iNQC(HYoDf*IN7<j"
    "hK&Doncib7!3S)znot#}m_(+*Vl%s)$B~G=BauZSW3&mX~LIF&(l4adU)6Zb15YjC*xPwyf3ZfvN-41PhZ}0"
    "5<lTKszst|AD0ub(!JX1L9w0E(B$-E8r?1@$w0;OxZR+Xq*lUbyA49SL_2_;JZGws&(FYP`?lC@cX>hPAPEr"
    "=Gg+;bu}1G!dLBST$jJTS#^&W_~LcxX~>_`crb*xcE{Xhfmv{Novul|kUJvmxR6f&rzaEs4%5o$X04zkUT~B"
    "%>g-DouC3`eyp|pI@~v_as9(EnCKr*V-=$aDAXK)vYL-?th=J2=zVC2OW1Dxt`;Sm0{^J?Q6NpL*!;NU>6KG"
    "XT`fsvWSA+Ns36twS^zg1GCWdXcP8A7cM<|!E`yVduyseD;$-xUY1|!Z2SVv3gZK81jEa9j1AGq>PO=Q*f3c"
    "HDG<8<wV@Ghx6g|F?!lbNra2f>9;F4D7O|9-Hwgq1-EX4$B~d_Sv;k21ES(ak>fq#fVk6rTnw_LqRyKEupxT"
    "olck+(=N!JzN6z)`}AMEr!pXb&bo{8g=b8-Ck=*SPF8KB3?#E$&Rn*Q49A$<ygrQ#YsVe{Yt$v|Zs^>{?g72"
    "C2}Kl$*%j9C(vOeE-Apn68i8Ja}?*$H+l^hxPQX$}B*%;@XgmrXB$GYn91v2XRu^7rMt(aW8$zVW{e_O*g?d"
    "OUf8^$o6!Q`DeG-u=Z9_>$NS`_}Y|DOGVS-aV5=2Kd}%8;!Vn(WDNyw=fsqf(r)&lzf?<_(J^W*~zh(7vTc*"
    "C(s9RSB$fH4%^LADP!eO94G=*JmZkshel6G)Q$3?wf!4bf;#4z;yf+yOCYNAq+Mu<d$IL<zQR~JOxY2OFay&"
    "O)T6w~mqYOu0Hl$J5F@uSR-2f&lIms=SIKPX@sXU8LkP6TunH;68}~)r%E4MCHyxEhr+Q;34sz_MSsEq_u$^"
    "F9iRUYKVJViYVs;ajafB*<zN$n>dTg=4^8khm55+#Q)B0k2N=SjkWgC0Gk*DzOS)RelCKgl(-@<m_p+Z}FMi"
    "&<udS8qQC8vkQz7d$iXAIb?r1)5CglLJ%cP=}G7BYUM6<HY0s(}n%5;CqikZkFZD4C})z_HR`0}9bBp3mc%;"
    "bD2o*un+I{H>vNg^7eQcCn_el$V+U1_SY?$g6yor;Sm7&Xy-*6k+NG9Ks=~jsHDyg#V#h)gz5uMgnqxp+zV;"
    "a+ok(EuLLTnom-Dg(2XaIlg=hlk&kG89)K2jOlclm(>(f?50zg+jBR1!}JNo>3pbRaDZ<Z*xK>3Q6rbD-MtU"
    "ucW1ppTz?47uX&l!!IryA%;F>^w0%E&m9r)5s(O3JthD1oGuY;v-&c2yh8kO82YxhJDxKfZ(2ST7^Q0nE1Qa"
    "Y=jg2*eR3#~+TpjX*Nfa-ZIeH+_t|-U;KR|TEMRl*If+S7Tv!JYs5g5x`UV^oquPXfxW)4_y!DDHj16CwF6-"
    "Y37KEFD~Pf!ZjK+2`aGO)9~MFc8}bL5odDO}Y}#%I8rvG<q_@?#_=M|rzaK^h@;6fP({aTCC<j@y9zw*e<j@"
    "!1;sQXQpnF3Muu<)Y#?<O3YIu_kguEfB~Dxh}#~`QBAIpB2MtOU|BthWg1~dkCoj+aMUQ0EaMN)*0g(3|KN%"
    "H&O^lu-Awzmzn|Q9LvynchwUa2vN9@;$>d=(-_CGTSe6F8YS~NAnMSTbi;4~pn7hDs=;(IeUlO)k5;p)!G*b"
    "{j!mG3)1isqP?1}=D;cVFy9)Y^{zmzs&YXa&>^95qGWnuo_(MmJfhEmPfo<a*yqiFlLn-M(w`|25CpFx5Cp4"
    "DZP3B7zksMy4v6rocg4}+l5%NL6Ldqr>AXYVPSuO~;FaP%BU^}fH;rHtqxSAo;S^0G>9oNq6p7~174jMUO@u"
    "U*T!63y`Ha5V}p6P9TFIl-c$}um>F>zXh!Z`wFQ#0_&@5k$r?7~PRsqOOTTYY+T9s<8<Nj`>zJWp_@z!=LIG"
    "*mr8Nk*4JJG;&x;b<uK5VqqA@deCMDLGFa1T@-RnB^55WYqc#C<>o17hx8`3eEErbE?GQ!LWhSN)e%b-NQeR"
    "_g+uV-t6roK{ZJ<AfS0p9Vwo3|6xa1v%b5>c6Tia04{igjH>UCIH-}O;R(`cp7=tXV%jK5$_%3tl#PZCXK>Q"
    "TK;sf!MMkEPvAE(7NxmvK3dGPfC`1#|kqAT@gP1OXEmJW0r<hJNtMv5W{$IfQxZU?fZ}7O`ZtV%!sK@<-Q_B"
    "VTyfcAX7*3V}(`acijvD=dP}RfJ*Al+=U~+T_P>g^#;u!w0H`nQewQb02f#GxtDH!P}N9y7gtQP_w`wqiJ)H"
    "2GDu>dZS>8TOC+*9pQElAAmCz&I0v2}2|_cP<a?7ew&dh$2y3_rPt2OG~*HiDe$vYwbHM4c!ovY5`#%*1K`9"
    "Q>$P>uxh+2UVXI02uDp&~|jLzo+iJo@~~FznuWzfaKy)?lL7}Orv7Im9nM9OXl;GVQq%W_#$0piVwv6FaegN"
    ";S2$lZ5qCZ#!5XwoRj2Da^LGEQNZ#iP{PLMJYsuuKznAVG?a!gDU)(g2`%1t0~INwH1}tudsxJ8eCp`8W_0N"
    "97|}BpIgux9M!#f6%21_zu``(NXWk%)vXNuj&=E0|KuAayj>UqGsgP0#Xi0om>R2On(5=PLP)6Pzm+B9(F;w"
    "IX<7!OWBeZec*})%KerKQAR5g{33$<-Om`U3lIP#NH32AA!)c(&o!$V?(^BhbvagRHzFAs)L#qVVof>I3^o("
    "s(K?^q420F{6TK!7&RViAS+W>pCGn1woxr#Ai6zEnRvDU1&8Cp2n+1K^RzPqG+qqKZ6wDt)WXszlY|NmH{M;"
    "oWx^TM|<P<2m?8)3cp-)Ic<;B@c~-^Oj&8-TGYvOXLJnevbwCH!u`3O|BC-1lUTybyl0o89|X$Hku+T^cw~I"
    "Vvbfl(Zld^su7lrXW{PbdW|CukH3?MR@i49J4)3JE5m8U*I;`G-Ai_(Yq+S7f1Bp{Kmg;;ZY`x#ZTuAqZPQF"
    "e^DaL$)JDU`f`-ys2s2t%oi?P$OfHYAqdgXgHOAVj^7g3N2>pN)5gi4p<{AaKS<eMAr>(9Kw#^Mus+;&h%09"
    "nIz1xIvj>6OuB_O>M;wj>}dGiIm29pY;p<C)GyI#$N+iRGyn-b_a0I1!_Rg=O>+cq_6|03jkTaFXbmm^|C7G"
    "uwVc~=siA6SSg07e6`Sx%OSM_Vr15>jNU^qxl+On;0o2DCo}5GO72b~GQ=87i@am=<X;6VLS>`)ENoRnvYzK"
    "u&vT)SJd=q9$D*vdyLYY_(XWVUvdiv(rC21HJn}>$!8#5<S^7!K2evJDi>N_s%Bp;bRLnU4o?t3P9bb`JWrF"
    ")$dM@&L(18Zr<Bsa(tkvQ?KsFKRQIJ;~#^1A&Qa{HOcXh_D1KdW9D*XRS{K(x92CX_s$RZiGM|lJ|<g{oDtC"
    "9S}W?3J-b1wI~@^aV|_{P>a=LnvP&IGaTyVcXkIV}C1u0uEfB8CA5PZ0&S+#VOw;M(9)2*QQXg;{&NlA(;(w"
    "q9O{oO`-TJeWJ0RnZq+05=ZJM(BFdNL%<oc$1Azc|HIVe5>KxIH>0PuT(3yFXrrTW8`F~ove4Bd)tU~_t`z2"
    "+%xO>6B;f4q&U+hllJS*^@kL4(vDZ{t?K-6W)S?NKVT%$^({w8~SnX^#LDAi1X|de9>2Y^i2~3DnO}PR5%df"
    "f&%gIfS%voy#}d6*TgFz=Ozhw<4Z9Skuwf5Yx1g0;^oj=gG(3#n#Yt*<#x66yw|bGE0(5LL5_D5H$K{@uMh3"
    "$dFiLZ<Hz;WBT5!s0r*$z>7;=@sH{87-+<iW?e!GRF88fyPfWu)x_8|h5EpyWSASlh^uf1NIw44*s|&whcm-"
    "j=!#LZG~;-_fiIEs3Z_|@jnOveT?2m_>o`>Y$Ym!4DnmpQ!?I&#05MMIsDuOQjC?e!c~XS;4Fk&F{{H06xwM"
    "*`PX2SUf9`rE+A5W=s#(5JnNPr6ldeicp|sNCJm>dj|4?&PbepUK|71Dm<<`mYspz6t&vwW+b~_x`U<fmtpb"
    "}-2uxz|Ta?L57GltS{s8e(A=)q~QaCA|T(E4^4vVs&Xk_zHR&<72PTl~1HIk9V^W8K%kwB@U*;;w3M^nHij^"
    "2dNb)~vDjta(D(BUhD`b3IW4VR07_ExJ!*??0{qo2V#(9j|X3cE`rw&l^>lnhGPF3-zwpahlg~5NX<fnRIH+"
    "7OeRrJoDti`o7BFqB*a(7g|5PmXNaAQDfsR=Z^{Cdis+By7QHD-&GZ*2W;*6fek{1bjuk1uXl5tL#i*W;~py"
    "ipYQi-7$(OnW&{PQ9CHG#Hm`N}oi)z_2lEn;K;e{VY*WIO_A*F6%tUxqg1wYNG*#55lsDuqN(#_iXW<n<=%+"
    "c_!je*lVT!P_4q8B_0(fo^smcMIX?TwzeVDIO4#hb_Y!<7Mw;O{EP4*K<_fTT(c1h_29(c8X(_xz(bh36sfO"
    "TE=`Vw|iUPs8zogHaax5#3+?~yI08*-5vI9WX9YGkYo$;}%QFg`21v(-KGKiG8Oajof_${_(ac|CSBhoY=lQ"
    "QEf~{YgEqT`IOdic25L%P=nSGnGOQy-|y#bK>=7HAb5Tb@<Ez^{teGjJ&3a#!OY;e%th{W8G_dOj7}jwB%@d"
    "Sk8zMPlv__Yvx8zZs;FlZb7f1ZEV>)_<cJFKZFSalT5>T6&K?j;Un*qk#uU;H7(f)Nd}=o+xvUR`;()|!FIp"
    "Rv}Afrn;UCou<PyCZuBN<P(9oOt+4)`as9@)C(r+}nFq!3s!1uLq?EU>;(1<xh`on|lQJPhF!L;-Hb@nTqI{"
    "0CdngS>vvg8M;j*G+GYn#s8=B0c5J@(V3lQ0q00|NXY0{C;gdoTa1d5^fnRbAPEAyRnR24DCyM_X{a^7S;JL"
    "QIOGEc17tpk(keB!%`0dS)vASNdySReLmeb;KVrShu7skL3yJl!2?r}l&hVOT={{vn+o^*D(qH6nce5xwj2y"
    "iN@+)e@r{IR(f#rq;tg`7PGLJQ)q9!mrL(G?;;Pgsbm~orw{-e1}EQiupC{aqh_xo%>3el1b7J^<27ou*JWx"
    "9)HZSkstZ*))X~LcVR`a{NdpW=yci)wUB`n%GIJrCJzKb*eqt=+~AdiIq-R{E&Qh@lJ9zq8+Tk&=uk^Q!}r)"
    "H&2MZF9AOti;Sn1IMD%Nw;Q8@R(IbK%-Y`tMhsd=BNWZ+YB&wxad)1=|t}V^*-q+tohHza%$MB^_jjnsp2(5"
    "UvNR*F<yfVr$i(p$B`hsSOds>pwteY$y`A&QIA8IEu9B;gXqC`epcR6v&ue7Dtb&UVxp81yq@hsykYUl<2Yt"
    "5>x#|o=0g_Cmo&Cz@O1ZL&co{}{gSn<){k87`QK4FxC1mMo%4mhQ8V~*AI6~Ne1lsCz8O+L4t?mnNw<HBThn"
    "Z7_-xp+O2149ki{kemEWpv%+*A-eCh=WxJJbLrpyMeZOJo$2)Xxnig2b)G_0%;v-v$kKynW4wv0@au5fHJna"
    "=}|Q)Q{4PUP5(9All}Tpb`#yQlU})r!nK*w^~8Z4QdbaOD}tUbeyc%4jD+7_@Ce2B!8GP0?}pd9;PI<4GR!6"
    "+UF!`Cr<A~r_DDrIB>~sxt2C9g^!MLsb0K-N5Z*!3kW>f;`QK*H)>IdcMO)oqDycOICJJyswpy1AwfHRabVH"
    "`CR0lW|WWzuf8;5m`IT(l?Bavhv`SRjk2boBpV!~L^WJ03(DUmupE2p-0CgJ`~6wwRmbdR;GH|KcDJGdb#Ts"
    "t(6=q1$&?LTBQ2<&QVdVsY|)96sJfyy7}y`tacQ0&-4nj%P7V3tE3?b@mQf3SzcivO@oL0uDf4EQXSucQEW*"
    "UVQ1u+X}N>Ni%)C}i-o&S8FfQNdP}XHy{E9P_`=eUZ>KUB;LoPB_Kz8&bpht?fX)-ghuH+<XUa#yb-i;!;D("
    "v|LAsiOYs)z+^7_Vp!yN6o>YwvDjT7nWNT+Oz&p+6I1p;b03*7;-rZf!lVm_nym=23<zWZc2lRV=T>`~0RC+"
    "y2|9+RK3PP&<N@<Uv}ccXgIQ8{WOTRJKex_XX5*eTjFbcB*vQ22$qM^v5XI&*?JeH0ZjLNBQ-vQo!07>7{US"
    "8s^j;^|9LTWy?MS3?SY)vz4~6yuHLs+&57beNkjaX8kyo(?PVa30FO%1MOymWZ_D?4;fzS7TI+}RYl%9-2fh"
    "9WmWpXOsoF2a3JN;e!I{97f|7EiOYcB!Gy4w!~#>tBF$=}ZvE~*S5Azet8p-@u~CO_}JJvxU2fB(CS%r6bNE"
    "|4>ZUof-91Yi~yy_@)BkC0Zb()_Bo`>NmXLxBygLnDZ88rIloS0M5txlS^>DdO$%;a}fQ6jrOhp@^gC+>+){"
    "yExf)=}V;o^dYmQw*s<AV1X)~Mx0LeWHlrREnG6{!Tx%B4UpHfIBlvGdiu6?bWySdBU#ha+ksZ08XkEgk7=1"
    "PGKa?plfOG7GbQ!w<k-31V-y$Jx|RXf+>3IU%lMz(DGVme!{hza$?M7Sc_7_}P8&`qKTl34$NQ5rw@Qh^+-w"
    "=RKrG_21jJLTSgRx47W^Z>DqE+he-`>Dpd?wY!1kF=3CpOgPcUB3?sGrr0LKham!nne&a60tD1ep~zAI;Dr&"
    "-rw!nO9EiM!e^!kqDMTiZyx>vT8KOt{OPkluY|G%QReNNDaotF<JEYHLR~T4RY?i^xZ4OLUtS{ya$Y=B{PH`"
    "d&+)FqwsOS@OX6y2aaLYkOsf-!{gxMzgTK+X$Xsnm4TuGw7!_=_1wv+g9)6<%~N1IGx3DW^NY};#I>+=ex%G"
    "Zr`aWhba(-cZxR?+v;p?MQpWh-OZ%Els6{d8fjhb%Ga}zx?*_f>LOdQ>KHD+);1V^!?k)&lhQ@9amv)yJxf;"
    "F`pJA=#`R%^vbC-!wY}-fo%IS|6ej7t-YmU)OBCKWbrkDp<@&{{G9ubKlL##!eyBr<5ZAJR8Dl^rww7+n3n>"
    "Sq@V?(^aMBp)Q)ExI43W~%%x%(8)tsrlcXU2EZ4xDW2M1#R<mm0|V~!DQ+&>l8<91`2sQnrxZrFdF%XNYS?M"
    "6SMJmtfDg1X7`O>0qdCY&Dr0?#<3GL=DCQ$^Wyk^MCJ8DKBiEO7F(SP$UE18QwNj0cweH2LN5Sg?KXbnozN("
    "%bv#<n%lc+iW$HkS8W{)OP<nNF8ku2D^qZ=+;>hskTWKYwId(JQtulO4RyEexs)Kw{`;7RKDu(f~GA492{`8"
    "BnXE`>DvB&F*b3DpxZN`A49Zxw&=+nkAN)u*6ZC*e9-HpQrVq5C8bGD14k=$N214%NsWt}IBrq640|CDqg<i"
    "2!Sw4`wBAt5G8{xcO;y^|r0Qc#2ZfB+plzx9hQ6hW3oGBI;9I{hH(tL8dfhOd0E69qtGG&1Kwp`97sMtK0jk"
    "s{4&Z*^e9f6RShsbu0h<mMF)qvIu&ql8e{GYj3U6EMO9rDrY7@+O^N0%5r#6#ij81NfG3$J>cKW+Vj1Z1O_2"
    "UVz_AW(4aTp=RrE}0V6uT+v-%Sdop_GJa+N7m(a9+C--%XjcmA~3N(0$yx-K)J`c3~iUBB(Vz@U<~8TjWe2b"
    "DUe|k%qHYLH--42VbrgFLYF}6tgsK7)`wo?KCTA^#VZ;U0>h^B{Egkbe1ocX<W6LT3U1lj+dAMBXa`5{yj6T"
    "(xW&?um|@TfQE9n3a!Tv2^TRfR(h2i26yYT%QVcU4F&;kjg)Bta+QjyzkXoCq2AkMyZUPV*|WyU<~xR34OGb"
    "C>(+Dk(|}1Q_sC-bE`GPHwq3)0_AwiVWUf?o@4nQ(Uc1!m$4@&FxJdz!AzK$28=CC5FfLp^6KqS6ukB!4MHF"
    "xMn<mt&^%JUyf%a%K)30aW+{iBHvz!5Mc69GI1pAcoM!UD#RlB)ZY>Qf}!-czQDlKT%1mv625T$)Q5YN`^u#"
    ">OJ&5->z&=-L$N;Nc)W%e(!8eqT5Mj+2su>BiM`4}U*G!q+rZ@ArWDg7dA;hr?I6_9Z<@MR?i+cKwVyH!)5+"
    "H^GvwDyg6G+GbD^o%wOU=ThV-Zxc!WkWK|UD^8?1cU_mte?^m?CN;(+i>&V<QCB!f8r?xMM`%u)GmnvT}eH^"
    ";9AEyW``rdi?RY#h^z$sQWg2`BJ0-M5I0}maBmyt5vngcaG$pOF_;#F$>i^cXXj^9YBo`XNa3ntS&YCGerFM"
    "W1j*R`rp(Z!QQ>S%GqBfa4P1q>9zSBhF0v<|VE>)-3Z@b~(43<!C*4@>2YRfis-0(}+Hl<YsBZWrl$wn&Jsv"
    "wL_-{FSU0Q6eIlIwZm=t<`bqAg9hO!gIlnULh4@zZZeT+|Bd`M=o&6&w***AH~)Hsni`|A-9vjS>lMWmnK2^"
    "H<D)0xUDgpQ-wn#WRR=B4_x(2SVnKhva&ztq21SIZ)w#if3_yw`tM@nSho(s(VsCSdTT$(5|q8>Hm)vuZYSX"
    "*M%q$~gR+I83XXIufBbY#yOSN}H{VxYLOnWcSibLbEPk-HOarQel-<y=Q%IhS)EsCvV>XfwYWVtA)97XfSZa"
    "B5PLXlzGTYlrb}+n;OJ1N!zDykB=#1Q*I(EW7C_h=?le^K#WAAQ43=n{vkp1M(<fcdPFlc;$gSL0kLD*V?m)"
    "W@Z=8ScT=ZZOdKH4cC0v3G2jR7Zz_NX#q%=SZ2mM&94o-v96o)vlF2b`iM9=u)mHT<tFSHX<r(Pc@YjjhuE4"
    "rggp1Mt<3mXW4M0iu$G6eQ*0~C2ko<3_=@=jhBNR_4IN%QrqHH`fMB1y8(o@XqNjWX=7ip5+_UfQYKPPEC!5"
    "0;>64u3nXpmV5^aT}HZGMD|jKjrkloUPsAt5vIt!aKMX?+Q(c=AzVZ=pDV&n&{*xD-*oLRb;7ErCulUm>X<W"
    "#TA8B$3ZG(v+$WhS}wSADWkt4wUtbItFm@WQux*_&@Yx^7s8C`S)+9C&x#>2Vy6GwX?(Z%*!F-dG&PR-Yh%J"
    ")#05=&Q0^%n7LDC;j+BRT_6K{HuF@F&(={O<Q+pmy*b_c<@Fvjn}Qd3b`#HTYjTJZB=TD?LzoX^i)Ml<R$#G"
    "W9uh!c-_74MrUUl<#a3|zIHS<?s))md^Vz%{&dd93)}y;a*tWC0*S9kVe+tWZf`8g6Pv+-V8BT9Z?8vU2xu#"
    "lYZ41Wh76cw>HqJt%h%jRf8p+P2>b!xifxt^BCM(j(c0}0;xVykJc!TunWdILnVwNxOfxaF36~N5o&`J=UU*"
    "mX5z;O`UrnQYMkKz$82T)bu5zuEiWw?Zx(+Iz(XBMZk!*T|6pC|@13(+^+0hK5$I{Ag6)cbe}@WLZ9L_WiFd"
    "63dp@Z;Mhfb!gyKwZf?nvxwV)~3pB%<)AC>SL0oGJ1!t;uJTY-b;NG+DV{%h;xVNHjL|&RDRSu8aN5X)$0RN"
    "fJ9%`7bU$5J)D$Nn1{Xox@<m10*-~E8<gv%DVNWc5A%zyPi_^5VL4qDNw5F#BRFZsMGn~+t0JKp-5^H(7_s1"
    "KHZKqvB$>c$D2mn6Ml9fvN7Q6Ve>K%rPfdBNmeHEHQ&SegbRkm@VBfKEV$^5;gu@yVN6A!P?)AI3M`bIj#YU"
    "f(-}p{YJpi5n7wtSL9J1Xn#G3-9M9k>GG9z*U!PcBb7A0l1Dz5NwMS=o#zo8>Z<r!vp4%jzESlk~ZKt`Gu_d"
    "N)lO+Jn<w$xc<z~36MtgYlrZo(qgWQ=00rylBy7eXg+4rz|MIuAKv3|iQ)dJffV<0QwgV!f8sJ&|YgQB#6<K"
    "ly0C{;0gQIgh2()Zvtl4I2L#kWOz>KoXq_-Vgd~c_}}yM#^ZQ!)(U-&*!@V2JVf~>dxVM^uvd7V&)ooF#qN0"
    "Nl1);I8?a^|GsiPE&f8Bq>+*4GYd1~Rh8($;0&+Ypgo3Ny0Ih8fx`^5Q%j1qN}uJ6MPku3KNf^wGi(WS_IYQ"
    "Wr$&pgxJ5Ml4wL~tATX^hkGU)=0fNYTd;z28fya4B+}4=Y)X+IbiBiM+*rwpA<xTn>s2Opu=?lh#^K@0-^ct"
    "R7qcu8@a+P~V)dTK}wppSMWQJ<4jxGm^l{>wM#8b@_1m@L6JhA|L2_aam4NLjiE@heQ3pap)WyTz=nHZA3L#"
    "7Cq!ZP6WI=#LS8boWeNYRcY5=EX@q`wA?I|=HZyz&gQvEs0p-PrV^sO*-NMCyQ2%=Zz#HLtYtV&2qOWfRN$v"
    "WgcGrt8KZmtlr`eE~UxxG^T#tRfV^@+MAG@y}HbYSmC{!PN@%z%bR_#&lzJenbs8vIZwCK<Ow##6ZL@q%2{C"
    "p)6-18u|{-j)odoBD?R=S#t;w709m8D=xNPyxSZ6S2+0B&fweW@WtTrIqVZ<1&7`0rr$>e2P=?dqUZKhzg?S"
    "j><et0#o{y}YIuc^yRq`4A!A}L$|Y*%1Wd%8I=cxW)W+fQ?Q6esTukR_c#ZCBF&N0<AJ9F5*q0Y2#J(&l&OV"
    "@uj4#WFhH>`6{~B3J@wRmUo0GEs?roM-mlxRqAFyE$JX+1}x8dk;7v7hHJQEc#tfAI)jj2R?i13=l71Vc-!|"
    "5*`GCXB=k-a;k9=yEB&hMA;xJ=+oagl-7Bgw9nGBy4L1NCr`O+Lo6GcaL~Kj}expsxu`yMRv0^GeVgpNu|)*"
    "Ru~vkz<$>*i*RJ7oTKLJzQj`F`CH7c{YIa^9l^yeC;frjdyo0vU5U>I;-*}T>5tBB76HfEN{m<JAeN3MRuGY"
    "$9Hdv<U^9i*Ks+n3Sb6@3t<(XFP7>;Ud6NO3>VuN5KAD+^)wEn-$4AriZ4~WFS2)sw1O@#vfsk2ild+I$5A{"
    "FS81g?)QEflxx(|aN2se3L*_F?jMo?rXPWzRxJ<)po)-%`rKCx=`e+?geAFGhek|djH2fndf>6T6s*DRImlk"
    "0bUK^Migfau~YTPU?l5jMnbv!^*rMk6-G=&WUvKRRq>Z`aPLsPYWkJC`C(ILO+Uv17?`e7MPS>5mvV0e?Gof"
    "y>rAtGR0zu>NT0sV5ZC0<}{+}=RAJ=I#Hv2pgHwk3!cqPL-xr~`G!*#~_tM@g4`K=)>^KP;;$9F)cX7CZUQp"
    "Z~o6$Vc7>+)ro~LuqpP3KVRp>|m)ouXc7kXFVClFn0D>bL#6EKbC)($RRKS8yZa|>I}T9jJ+_C*`QIQjPih{"
    "HW+<1`iz>EY`0N_o;L6aYNH@f=#%_4;|cAi8y2gq2XG)bxn>owp22`-8A8v5@gj$VEC8q=A_kf&xGxw^Z~Su"
    "SvE@_TzPt{!<y(|(y6#JH>dbp%osS+dDw|E0F7}H#S{`#Wmwkv8-m@wDtdZMnT$1vU9j7MATGQ+wIZ+fTfb$"
    "Xk_4dz|H@d}iPePNA_t1WG2IiaKY;}cy*eD#caEV^;d{r%16;;vXK=@d{rs^N~ICLyerrb70JUt!9SyGW_k@"
    "eHSk4HlNGc2npUsV#`%z8Hd2zjY_6dR9Ib>QL#pdxrSF`^9)Rt26RILQDO%slxRM+2GiFA+sR?L&bK&Q5T;a"
    "XRA?w(gig=N}hv0QgAi#{vTY*2f<-#S?(-e3k+-VJW|=F8y0V4!n;eT#u@diQz-~+<&M#_U1+YuQ-#-6R!~k"
    "fmFyMoKRLgv~h*<OpFWKk~M-+X7>Y(o$(M_1I%(M;R>(F;95R>rm?;XsX|gErJ5e%JUW6veG-mZPWBzT1C`%"
    "}z6T7f(VPJ)9%`0n<!XVW0+B5olNv9RGLPb+s}oP-?4PT6C5?ZE(MVZD_ccnTObmlW9h2P1?A4&wtrGhIC>Z"
    "By9#(ezuI?R7qf-DwDqPF#)S$7u9hxTAo2ROSZ2diG9v!~rWKKDsa!d?1jxFeZ)JYbPBYJCO9+q>Rx=V^O>o"
    "!0(^6eO<SFaDijk~X=Uw-pd?MeDBB0GM)Q@hKbXW++LEo|W7EWhj59x-^2co5K>P_ME|LHq&oOsw0OQKk<k-"
    "0jAx+ye3C@|^Lony|*{mE~EySXTE6$&!V4&E2J3Vq@`0CJiGNh(>~8$~d{LjWI)heAEVyEFXgXpQlm{-phD~"
    "ljzzW;Z@lBj9QAVPs!ZW){^oxZhgQ_ShSu+T~y<)Q0r87t`YjL5{m;$gLGGO2NhLj5y~*|BE~Q(u$Spm2cI9"
    "@!(UDKpq^lMl}%2mnLfPrs_hY|-_T6uY64ki5krfe6A$ZPpivp$NAjP(%P7qKIHagZ*id>GTPmgqE#_IIMAb"
    "4)s@@{}h<}WC(IuNW+TIIxayY1QFUSkxUL|w_b(LiRKxT}gogk3Jix=XXS6`_>mo>{&#@e5X!(Waefxtycgd"
    "{f$z`{gJ?Bmi^(~a;gOpZz!8_-o;NQjR>z3D>0V$vN4qtSO6sW}SoS(NzdV|vD^rz-ieKF9G%O`v;STOuvp8"
    "rpoJE<aUEBqTxkQ%zm+RMpUgu(Gi=iYC+_I!P-)jP=<3Fc!U+J39ukHQ8>y-a(U$?3N$K;^mG(J<J7>tHM<!"
    "+}*~rKzUux!euxE@ezN#30DBq{GFl45v{$_(zFTBq?GnMuT`#E*YQ*@IOAE%9CXIknsbf&SGRTOtUd}PHJ<~"
    "3LT?^;z`?hkOcg7yBj7O(r2I$mZ-2zu3j#NVuwO?^t^cp@5dGRA3r+B!+&257tcQJ-+wHPL+&Ncdf7FPYxOZ"
    "G9joxakd9LV)_YQuqVVwdbI#Q1j-V^xl;Eem7q+V5v2xS9Ogy5!A<%=8~Swr8)AQ;X0j=9d=KH}kYnu&vJv)"
    "VX=>-Gx`G454!y(Wl)IH*TFyO8SCg3;#z+>(?BkGhmnWBJGPALR@#$MTQoKQ@yDdG`nx+t=9;Y=NqvS!hFPg"
    "oZWA=@l2Cb$Ikp?>_RV_^fth6#s{!#>eKodGZ6KbE?}UvO-c!*j-YeB~L66%*c71q<40J{pSjd`nofimt}<L"
    "w8&xJo5K+}z3;}K)(_0a!BXk-wS#4yc~|NCzB4En<U8&X5rRwzYd8-(qo~m2#>Qkr6@kebt<<)5Vkp;7uheY"
    "AU(eop*$O{Dl=|#2DZ_G>B)tv~M#;55m<S#O>0ouuPBWuar}H;4UXnc-IWP?_&1GID7*Tl69bn{dTjz7-#bv"
    "<eP5Hhz5<KWS1CUnRV&%fRqJi-9gpofmdxC+BZ_oGDCrb$BEa3!-83r;t5?m-C5Up5|5JP&?{2Dk&iiOKTQr"
    "$1(8bWPfGKwS_j!_N4_It7GHxsG8S~CmiaRJD;Ah75xDMz1zeZy3Jw{D!VMFzyX)S0sk*?_RJBo}18@si7if"
    "){I6Dw(QA;It4MV<O~BD}onrzAmPuox&^5KF{*kk#77S`IWL{Xe>5;ny>e^U$zG|DffhIFSRsj%q49juriJz"
    "YYjRc;Cv}cj9h;!W9T~wwBYDZ^~fAae2us~S2ueK_6BHqUFU<-k#j6^muVo;r9Bzf9xc*)&{;zdQ8T~Q>Y&A"
    "^&o<ZhD-)|sN%mISC%?di=p#YiGf^e8+fw;M0P(3(hm-0iB&OyB2x*epZ7tEml9n6J?%vZ7c;URqaBs%#+v&"
    "!7K}T~*OJZLKj*@kZb(ZK@Ww%*=m#O{M+JvF!oMCzvG^^{7=b0>43*0XOI(TOXhY^UYAb?87PGv~f(?O@TPG"
    "xnk1qe^v<@meq<_3T60-Yb<hd%)ft78Nd_Xq?|mO8CaVlI!xyUYK%OjvYGd3R}LYJVnil_94U(j*>8r7-$_>"
    "^s7qoE}V0AzfR^tRxaghp!LMh36CtCmA_jmI$HthG}Z<o<`mdw9n~<I<s}n>eg_OTYI2W6}Z^?M}9TJJfiI+"
    "+P-|?zhz^$FaK0Lc=`5Z`h1Iv$9}BI%hPN%vfpoX0ojkWc!Biva!jRes7OP02Iow=c2wUWu_Xp;B*7&vwk{h"
    "&_$~Gym&)=V`PEPtfmD|N$gePB1dB&6$wR+;`h@qvbwXa&z|5;4DR0ywXsCtfG6;nvTRDu*ES-kbA<ca4<!!"
    "QDq7*~1I#2Vw=%MFjz`Y_TFnEx}3)xkU4q&r~S+p19dO6~`=*bU17rTA&thv}H-eP4f#y<vXAZpvK5>V#E=r"
    "^i;YiOsu)Kf0~6wFN)L6KKVcqeOMz%$q^dGf%BAQIfy4}LOF-XnO%=uIE<S6gg-Dnw>8D_&Kl^7dGKyQ9y`1"
    "lE{m>Ag30gEznC2U>zV9G%iRj$`x?*Y&eB6N>n146g9ssj5`n*%9AE|A2n-<*Rl2i9GD#IEgfw$9T^EEP_tM"
    "IozGZ8<P;bYb+fY6#2Dx8VRLAv9YP_H>H2P8~Ot}?j*F*5atwH+j~Ebx3|Hyd<PIsKlU3CH|~ZGi2wKezUO@"
    "R<9LS(lpfLDe$%7)ES0ZPCDF~=hZ1_mRhd9Z4|6`|_fX-JQ-39Pt%7YIt_KLM-|w$!kVHjmI_PtLWG*9TB-G"
    "Rxy0)QTbE|q1za&tYCbCurDx0T*O*JA<7Mj$!PVo0o`djYgUrm*xr+q^}w|E$S@J<`07^{gO4kl;&!QD`LL1"
    "7}^DQ$VTt}S~E&UIf|tE+PDO}|!YO69<3_BO<N=R()LwRx$Y6GSU-YZqx_kPb*4$D}|9Eob-h9EU3L+MQW}k"
    "%|2yr(gf5a<6~1^%-ZX=vSF>1rPG#Icz+(!V|0~*mPBF44=jgNvYUUKXKF+n=RfTimwMV?Hh`SeMF(se0Lg;"
    "Qtd}5qjMau4)#<{8RcCHx4_1#G{wpBF{>6LXBb`@8RWh3x@hwnQGYd?XIzWghPqC>itt#apuQ-<pHekPyhzv"
    "(8E8HQ<C0?zd3osrLE_GwO0)<Bnvh@Xw<WKeX)+f0&&ip4er;rNR#=3bHJ_AXl_`}0g6=RtUW=N#>y-P?@*Q"
    "|ICi}kCXimJcC$IH9E@b_%`F4{h!o#-?PzQ@xTC*@x1nuaNxVo3vBT7&0puIKw4va805PP)sl3c{UO@Y}MzU"
    "FM*YSSi&Ra(Yeo}n1pK^Y`j+}Jh;fdo1?i2~16pPa!PjgqOp+2Qm42pYWQQ4Q!7ZyW+;#JIr-zPLOcPUqc7n"
    "`Q~8H_|bN`W)v<xmw0W8ArBHP_=-(@;ZdvJ;eZr!zPD;XsCE6mP19H<pyjTd=frQ^*>JzYjE^SKzv=op&=S-"
    "o)DU=ky3p<JGGo9os<`ow?Z<{%WG&lqqc!w^8FaayZwM{vFE7OFmAdp%u%Bj3SFGBo8?!Sr0#eUtWAq)Uq?="
    "|wYz6+a;9zEL+R5`;lI=5xe5*0Sg+hBC(f|nJm9qzk11foXth|DJuWQ}VVdT5(=5!!l%n3?Xd`%RpB=~Uc@l"
    "vtng^mQ`A3VZrykG%`S5g5(-|KCs!bUme4L~o@<iD=7rlZxn1w0%_DafS>_7R8z~=U!R(1c816B<+9U+n&W5"
    "F8T-2+KQ(XhTV62`~YZ}d87dLA?LbI}7(+*dn+cqyJeLxkMgG=0_Y8xsHpv>I^^y6k(Z<lhgwoIhNM%336=b"
    "2OpF;4!fp2Nksn%@kBgr0pJDoGy6quR)TAk8yOLNO&GfpZ$)DVzdZ`u3~(_P(z@iK;npaw6;9j&CP_jo9v_t"
    "%Dr4Im@u~=>@-2~HpRNCjnM?S-`;!k=Je!mlY{NnpoOL#(?U6Z%(7OrxrFUyQr(X2pMxJ=ur?;I^hXgrrZQf"
    "bu?A!{GMlO13z(SLXO5D{YkxZd!=H=2=Ah31*nJ#~#@>Y4_4+)Tu2vCE(IB;s@ovzBRab8>3+C;8b7g^vsfZ"
    "xxy|$D}!U2_rc7>Uh5XkFx<BWDDa|hV)&QrV|1o8!do02XD>BBeShgeAIhpECyA}Ok{ycKsh04IW=sW%A%vq"
    "JlzpUa_HB1YX{NDVzFV{VEVFg7&8D|~gk)ZAC%YE>o~z^rm=etjxD|IE3te&o9j%a}MoC{sz6O1AgS?aKCf!"
    "FM{Kp7a=nkLebj8zjk+JT<6u#Mpr@XtbYGVI61Q=S@90jnJ@;TAZYQ)1f*xShhp`@rS+zgA4+9s%Zk(Yq^p@"
    "kc2b^^lql$(nfk3Be0p`BZMPRUP3q;v<|~=VJR>@+;)kU59Az9^RgO><Gdn(;}1E&*v!JEH*D9gt{c~6-xc4"
    "BowbS48BRwt0vCHv6;*d1K2?xbr)c3B05jL?(3%ta2WtLl99?60P<Q3WO-S-mg|{28KF1OuNn6KUcW;WdGVs"
    ";l2|5~BHZ`)e$CH^tvaVeit#?BoeI4Kr%`_|g?qbUz7KQHkVcc+&7&U8AC~PBjmfxSOQhZFMO$Di;@=X@<Bs"
    "t&u!TglPf9igs-w2}LYSAI7fF=eI$vU64hT;0ykQ{T=VFx$5eQf;we0QzJ*|<a=9iIJJ3FN%q%!F;HBAdo3J"
    "bO&ucvCf+42WI_TYaH$2%+MytxY(J#N2E$bD@gE)+)5$)gFnR(rw~?UK(fPRP~%Le7!LCU3~td*dWQ%?m`84"
    "(UjEr*=DTGo8jNd+<Nn}ZLnQ(S4`2==h9sm30`%0p3fmkY744nm-NNf+1oSViklp0<R^@)?o2Vos(1&ug_e|"
    "jBMh2BftkpFa7IY8zu0o)qx-%VoxeRjo~l=5LN6Y8H)xiM(`kzN@g0Khz(s;_RVo(9ani|vEVENQ*b$Cfs<C"
    "^>F3nLyWR2!PqKmBuv$v?)-u~MAZSU~>@c5VM<nQN`)8oA(LwT=#3e7F_A!&qp#Fyw*{U!Cp;la_QjZlxFzk"
    "va%j%~Nf!m47HMO1<nX9&8YZ{o11u3|`}$sz$n>0vsEiUiI%Z&qm;e;%$KzB0?xl=)wV;nhsOw-4BmNC?{%y"
    "AE>L(C4i0<8MM;jN-Hk|7a|`BvQ4@;ydsKiB&b5;tA;@d%gGf>Dl?-`DFUj@8^><$d2LszI}xgGFA<L_s;f%"
    "Vd%_nkVJeC&k|PWFS0k`Js?e5FXB6pY!l!y?XnY)o>lZnCCA1Ta7(|Hi%z5Afp~UZcAdVrrzYkpd7MmUXyXg"
    "TDw{H}pL;$hlBFt4QX7p=+1@dRZmO3+b4as(Mt;l*ZEi~lUeXwtuTfqc)IF?5dTRQjq8H)EDI;!|GHvS1!0T"
    "RXO4T)sfNSe~YMgiri`sfsBDu?C-I|LHEu!R)uo5x5@!eS-RMp`P>5radLk!NRJ`H522UZK(GarTi|9%$_n>"
    "8tXL6dF`J%<x>BEV3wJm~fN!=%gsHyl8sJ#WhP`A9c(gWDZv%3QX}AgSRI(_bMNjf4_X?yP@f)zr!bXq@;sx"
    "f?_P#|lJK%oE^v1=Vk6#879|L>{B`q*JP@7z<Lf?9uK&GRWoE_@368Rh$DZLx&_vptGYccw2XSrgyP*aCim;"
    "ya(Dcwu-5TWG4?{Ptudh$i1QIr!!fSxz)+vb6K70aeQ(<J$O4&)mQ?wP7Xd_3RQrqQ1;}pj;G9&J@%K1m~m!"
    "cTG2B-S&|H}!$kN<mP;WJ@<miNk@>Eurnm9EL?vhaB`?Zx50*HpT>xmOTMoJQT(*^L?`+TQGE^gbVMQ<+Fpf"
    "&G`|8VYz7kjWRjkuRaR_LnXvm&xjikq-V~kK7jR{uv{862N`?E`O*qt9}181ClNHmradSL8fh81NAYq%hM%u"
    "U)4gO8|0QvI6W;={ui$gZ_SsUx<9cO3-lbE(^w!1U+C(+Oc+$xf8GT6_D(8YSfgq!_A$6d0-QdKfx48NF^}="
    "HGOoD@y;6;O6V}Ax&HrqZO#22u@R&h91y?=IvKIo_srT!)~nmvmqsQ-FB^QTlQpVcvP)2+2{tgCoM?V?C34_"
    "$wydIoi*q#x5q84CtFWejI<(0h4~yi;nK+*PvD1teLF!!>#6`Icg5j~e2f5TYT{(CU!tdzH%EKFQ@9x&5ylr"
    "=kds=b;C|y+8I`W!Ct|hNC+j;so`bddu|b>HsncCm>V_GCi`$5k5M#0Z^WGuGNl;yCQ^Y?VbjJt0^w!Mb&!{"
    "w2nDyG}j6;v?-_ONv&>&`5n}Tz&ihrrS3|4IGz~e6q!(etO{6&_l<6qi3%+Vp6JF9u-(m=tyl@m>m!jqoHI!"
    "&9ixBL5($pJ(I)f8r~pWo5($U3wr6z<g6X&(=*B-omGBkZAPEL7|DaP%83ZOHRcnpU>+o0u7?i}*SzL7adv!"
    "cbNKXr2kg+Qeq#nBv1|m<zM=E-p&s%G={Y5}+F|7ZC6F=DxbgGcjLf#QcJhIUv(YNoHAvn!S~{q^QITfJj&0"
    "72y>`@i5g;8V$uhwn6|)Y??!0nOH3$Wg~>5t`ve20|ezF%r3HgRn2mse~xEq2+Qi(-AyvPd1f?Eal~OQa8w9"
    "P4@uKdce;)%piG7n>nw?4YEv3k5W$#TWaoD|Q}F={I}r^c>l)U|1Vvkf*U2nQNAT7zFg>g81C|s5_J$VEF=o"
    "CJi#UYvjdCd76ZamV<c46ko`zTXhj?1VXrY}h^CYX@<D{X!R*w(G;am`2Y9wZ1d4u%XM3qI1*U)y6Cb76!z1"
    "-RT3Y$+YOpv9xx}i-c{s^@KfPz^?ja=l2Tm`t_5NlP%nP$@&USuzI3uz*ks&T-Et@IP|EX<;3A{0?_4FIn>#"
    "i`H);i?bCIah-pN?}aG8Ej?fQp@rRVpg!?YSIX(NhC0e8Lx6}!i)y)CN8jvk;6&}qbcI-I>}-I&p<=uLV@b>"
    "Xz?5`MN(d5Wxj}M&V(Y3!veDIf!q~4KF*N5J4R$f5kjEy3b~8_Cs?TgD=SW;Kb}qn7U)b{!Eqp`th{uiRDt2"
    "MEHYZ0(fVa9!Uka$kTAnY!_^XM0ZqjTY4{8102<G3r#5ZNY6c`orqx-_VAsqH>H&0VkLduX@ChJIgva`ZeR@"
    "irP^WBZG9F_O{A^p}4fi8j){S1eG+0)&1S#@6*Kc{d(mqSwmL#h#Z48PHty_6Ey$Q1@P+`gNmm#vq_@!zlos"
    "73DwUw8*tH)Tp+}vm%4HQ=nWBHVgLtVVI$(&>Z?5FkB+T%tm-FK{8>ct0RtUyqcH`*E~9G*lm(``5y`zL!xl"
    "e7Iv@9gbs4*cd2zCi5s#on2y@+wSeOt1!VRuxICPUH0fkhsjMaaaOb(^O-4q0?d^Z-=$Kg2r{^xX{@{NE^W9"
    "?|b{_N56|ZlK4r<!d;d%luLO}ls7<3thSdA$nnNgZm>*O<p5ecpw_c)0jf&Ay1v2f5O%mMm)-sz`$n-6m+$e"
    "J_Yh1OLoD+Gvr<3ce-Xp+J#7L>79}5&XceY{sik0|k+=!V8+3w`=CYI#{h^3f08&dqkVg4bf->=NC*<;cNE)"
    "9g!#j?n#tqf#hO!FpA?g~HCcwrW49J&Sg|PXtu$BgOM12iQF)!lsW++Z$5S|cxo{2Z7;)|DGsqUuXk#W~pk?"
    "j)FUWG9MP*WmxfUASx39=pho}6Q01V9w~C&y>!r*HSq4^NIqpcmlA4KCk^e8Lpm8b{b&@|y_wi@lmuvac~eB"
    "`T761@~b?FROS-dxKSoZj`AYP$W8FdH{$pjRPzu$e9VnGKCo2_k4a(^Xxs371J+k6-CCY8-vcJ4$WC_tv7q?"
    "?0o|GG+Iy~YUJN=AeE~DuFJp8aODnaW!qNKv?2d?a(Z+ie%{+ZAF&EqAY1DyOw)T2XSlK`*k2`^i6Xhaslco"
    "9qwMzAtpAGlES3QPj79GI04>h)&ikP_TZTme9f;M2$%zLDno-8O5YTBcu2@wR^(bZ`@Btu(`G`bjDP14oQ1Q"
    "l3A02<gsMb`@_#0klaXN=)6X6r-(^B^YQF2|xafad6pfB>t83;j3qfVmAqwt<*F1`jQj;xaF6fUCvj#(5}F`"
    "jc!k=HW}LX@dMKEkG9h6hhnLV-ev`iB}@<tspyBw-3<uAT`f=awKM+qIL&2t)!vLstjE>9lS$Z_%=Vx$AaCa"
    "AK>6I!K3}o-Qn^TqE{n6S~X&ymPs2Thlw5YiOBYY#rx@!UHXaQ{N}3|IjiONP-`=K4hGuR#6llo}q`>nyK4I"
    "fq3?;02IrMEd$?Sq`&fD2^xO~dOA<nQiWr>dQMOuQRC9)-8na!F#e<)AbN6j64)jG+P-`+qzB)VKesO*Y%jt"
    "5V(UX`>#FJ%<J$J6hR?K8D=h`#b0uh$^Io}J6|);SaiP8d+8IE0R5RMPJPLx6r01}v(M)4!hLI2r1#+llvwK"
    "m+kO}f0RK($M*oWcVd9NhRa+##}S|@<*PF9HxJ(Qs)2H$@B?YB6y%j6?c_m+6BkHew(3v32dVS3uof#AT7)k"
    "m{T{7+Pt!0?dXi$U+7)xCH|M8FW-*;fXWS-zlbI5~Tbo(Pl#<_=?nPnE=YD94JIU=M)+z-#2P_DDWf*~a3Xo"
    "+UN6vOLUM$c#ojT{qjKiU*|tD%8jM^K?MEO)^;7+g3Y=suKVaj?ufy_@5qdJ0k?~(rq0nG*D;yE6L<d;0Hv#"
    "<T?fuBnp*21a+3Tt4qrbOfQ)9=CBjuDY*1Xai;?ec~uKtmPeA2cz0R5tItjFfPIWIWydC41(&LqbIqXZ+ck_"
    "9ci1?5-!O~l*IIg<JL?@SZ~VFluYuF6@@^>0BEL%GMLE*Y*9Km+-fZ!%6X|C>=$RoSH=AIi>297#1C|Z-vuN"
    "GekKcaBJy-*(eFU(Y>>!C#JEyW)e0-wsZS&_~TLDujadi6<Q5+!L_~TOY!wrxYN!I_L0OxcGxU=R}CG1qLLf"
    "^z4K$V))&`1QC-Dt}p?ov-Q4BfGxR_AE$g-=EgkA$bwk@0%T?2jYW&VUHFS6~(xafKi3q2N~hX{=s)f)V+|p"
    "9*x3f0BPa)X6fT16Z)>Z2((_grBD1bMHSvWkG1_{m9wMRb@NK&j#6P$xV;;)lG7C(GA<BC6kR{|Cp+rBPKhG"
    "$eM7x*!oUfYz_aBCrDd@AHw8N`GY(<^NX!d$W$!#$_K5;Ku`7VUnCH-slMjZ@|q1kpRMpE5%YU!`L0gS*Kwu"
    "w9XVkrTkl89RzSHQ{~u6SI%z1V=Ta8wJ9%m{vPW2?x9BEch(qE}N#c^!-hY557?Sa3B&xeyR$EzuP%R|4?iK"
    "KONYY(dB6IJOR}vSoe94%)L(h?H{eq>KudNYA1<J<TQ=MvyFQYZir3wT0yzUcMZ1u^1uOCpdywCJ}`WEg@U?"
    "uMN>-<DK3bKxMJhpCfqj@=5A_TlI-u~!#s9f`*rlaO3YZxAUhpk@O*JpWGV4O~VIXpW*{oNGb?94Glct@Mcs"
    "Ege$kVV$y=Eyyusz=+pX4E3^f{P5@7hfnZz3sghA4+MH)_0ag4i2@U_$jY$VER<xxrcQI5V~Zv`-1WnNQ8hp"
    "wTjbJ+~+GSK|y#(KR?(Sixn8IP@GOdoKE-uGTHwXl3K{6SjJUPj)oJ7j#YGox8tfXNgG?``Glk;dm}OYAfm#"
    "C;w|&qfC>%aGa%NudmzY0>>r(+O%AjNBNLJdM+Xexv7HX#O#T-lrv3=*$Fe{0Y^#DFLMl#$SY-KC4zXr{(L~"
    "<U8*+4rt2oXOC<6T`?ooIS{zut`C?RAgX;a=jYj(>$iCONG0b)Cn!}3r&C)EyX^NX#wR%dR1l3#t;)<eU?0i"
    "itr08t_-Q~fYy2jX%2<{6nbsZML7?_eXz@_&XS@zc>{XJ=Oo&;?*k46<5~ofa8?{-^cnK1Wt0;3)b;I3XArH"
    "k?i&Tw*$X7>Q4K>A~&Drm&?nabuRsDoT^90m=u^AHpme0D>ed*C_WMoB0<O*UPW~4;o*ff70Y?xQMF|A`{fD"
    "Wg1oxhN>=<_hoxL#u=~|qeBg!T87n4nq0}d0QcJzXCa@NPCvv2W*$$QA}pYHReZ;3JYSVz+VCu<h^uE$NL~c"
    "uG*6r2H}c{-%#weR|4zPVTx)nu!p*2}LAL1O!Q}L;;RS?P{g9ME07-8YD_kugq4iBGvH_fyX&lKO>O~N$lU%"
    "K;hGcB-vaOHz&{v+lo;@$U31_$Ab$pyxKLZ;^4TV|e#}a4`SKZ7GoECXr^<dSM%WxKt)YKdv%WK#km^C9yw?"
    "f!x_2Ll90f1<<?)EI7d=3FAy=i0FI?7;)5q;DAR*Yeq%RZSV**r&P`!55r`;{~<-A5t#ljTT|)5jg2i!J$_P"
    "S%kx;&*Dr{Auj#Fgk97s(P_?Ca00u!&O5qG?IbfvH8W8;1I<T$UC`?hulb6!Ty8zw0-jHHl7?+`|;?weLR^Q"
    "oQW@oyI(!ue(;x1l^=;uZab`Eb%Uwk*N10kkb2-23S}P06+uQZ&98$+q#r!bZ{w^SiM0>Dk?V4#?F+#85`q|"
    "zix%{HwO)rw6)#FFo_7{5;fyIChsAzX0|5?~rRE=Tj3qLP)^z5GPxy-LvWu-@QDY`G%Z>muDvG0N5r<_h8AX"
    "5PIez9lA)g=dhn)8x%;RXBcKeFaj?IcMn|kD8i*1U6&fVmxGwp~~5gPVbvq#FRh{HvaU5{Lfv#JQCT?&vy`7"
    "zfVhJ^qu-Xd-<fu8Zcj!1nVPNrp?deelQveXL$^}?Eh(P!SZP<+^1U>Ro7RsL~JU(rKW0R+fTqb_=%wga(Em"
    "M46-|InvAEE*}!!ZL@@kQqXJf|WP==0SYI8SwClN`EkM#<eIy+FUI`MsA@11_ya0(m|Hb=cp$E$XGI4rK`nI"
    "96=bi-k!mPk)vuFrb9dNgf^EQ%g3YY;`#P*{IMEIQ7uElL1JR?F~r2emi}Jt{DaUC+3)qoo#kN{U&QhQZaEk"
    "-QG!o178341ieVtp#hH{4;ZvfUZ_5s3+qJZcq{}9(H_E&K<CLiq4Y3SoJMX(Y#$lFJNth=80?j!uu5sIbaI3"
    "asR1Kp^kzC?4E7(-GIf45k2RFPn#2@7lsi^HOKsDnNJ~K?Bhl`7fe*k2g@l#H=*2eMKb(+O@e^Be>>3EDyV="
    "M-~Xwq6JJ*rBoG!BdlQCudNE1)mYfziAoiEPvbdA7<dZya<Ve|)G;(4Kx-reXF+bqI}c@&R%?jRx>YBvwOlC"
    "}j{cy7`P=Eb}{W5BiHLXZey39hTOb#{JqCtzN@b0s;;{Smt-S=cq%iOP<awR)17qYBMYM3|%nVSLS!VE;TQ}"
    "SZ9hCvsNT#H>!(0BW}Uz23(fr-3MX*rk$p3F;c$OMqNc;Ztp6_V<Orb-z@1jS1HL#@ZxP$Cno5=$?s@4TrvW"
    "0m~uKwh+(F2Au>#suW}Ktq6Fbk_xATEZ_XzNl#{8Aa&Vu+HKjl)Z!k{A(7f?;@0k|qy|_xU5K>4A0Amtoe?8"
    "K9E#jXm;0&+A0(|*tWTcT)H}O4IECvKB-cB)dO0f@IY{@7WI4a-DKo}y5SBV<4G{H~-C=URbp`s8z$jMd6P<"
    "|70z7nu?n5TO@H$%nXR#mJZk%GdKz)uLMt&kst$va62X8POjAV?9Dr$RF04j2`QJmYA-@t7C8uQlzQ;)P*3t"
    "n3qyvY1=P9rj&xL`Q@8xlL0VbSMt^dFnhx91J@aHNV=)ueB^tLfWaPDu_WtQUV;%TT%}km6uf1qm$n(k6cXw"
    "b8UY@e4<u72#S#_Z&yGzEgA#y+;+`%t?Q9_9mDWEfi5)i;Ox-I1{mZ&mdfm!IUq+Kfd`Dh3Qt^$8fRohw8H}"
    "EqtCmAX&9%1PxAR~^YQJ={^*Bq9)vK6y?u6wPxx7SbS$pI6cXiXmH_EQeC(nC&rBfpPu?D%_nu)ENoKi#e>p"
    "vQ`v&?2T{x<j;FoVXcw3)glfP6=hL`J&zSv?p+_^Qf4Q8;BQHB|VtW-uem68E<iX1SY_<(P{C7Q-;mQ?m=_t"
    "n2)ch<j9=WN}Zlk6)0xY0HU-NBdu&?Rg3Xs3MAQebcPYX4j==@{~dY@+(jE`_#c6)QxBrd->Nnn!Q0-8+Vm-"
    "s#m+b)KN9>JykktG*U4muV80Q%Dl7Ng%3Lt2+82E|U5EAEc2Izd^}cBz;O9cQ(=XaW!FK7<EC#dM5)@Dk#UA"
    "__Y0I@9b=wVy*BVr&F@+6WpHLuzk=?N`I_R6O0y1)1Fg?w_<%Nde=HPgU$XA>gF%RaV`b}mF7xrSuq$a;{t5"
    "VaWNEoI$@bomq1afZgl<<#qo?}iJ<@xUrHaZFlQB}8Co~MNx3ZYd@jAgSF!RIDzB}PAkBvnrw2NWx*Aob2!G"
    "PW8t*^YQa8p(nHdTi<9XRV(;}Xiez~uD@RKT}{fwe~LXS8ToZq-)DUboUgs_{igg_xlBew{X^eX>Ir+_jLH("
    "?PCZi#b+n19vL;Qe5bXS4XdK-h{<fE71&bCRoMMrOSDBMj$YAanGVkj6vAZE^~tqF+{GRRR{kZ=5j(sY<A8L"
    "xmybpVI5|BEBxb@f3(MjpL;X$+?QFJMinnm&-taVGQ}1@*<*R3F3PdD{@cw_NyWt?28xT?C_Vv<8y#~x-C@}"
    "U6n)r))MMH3s;c1HA~|VsE=_nrtus@l4k{kK4Lso2&^@;_$q`5h~r$`BvBM+l1m|8<Z^+Hs8!_bo`V55;2$6"
    "sW;bz3>jk<^o__+JOG%HiZ4lQAq+^8;Cm~MG1gMqCEFOvj7>8_D>BX1j)!;s^23M;j<z~L;$KQ({d6C7**&v"
    "NS#Hswk3p&caSY><xdNu))88~9cQD4^#KlFYH@U-tm?`@V;;#<yBROq(LB1cP1Jh%feiA4-)gGspvgi)p(&o"
    "sZoco|MsL%*2B$1P$Qn=+PYr1`9TK|4|T;^ofESA(5z1~0!IKz#AH1FX{^DF?D111M#nSJrS5^~GQyQ_GT4p"
    "5Npd(Dd43j|VbKjnXBliVOro5WQ?dLhN3c<bW9mH?iT@lz#d6vmpx#+iBd(i)%Q-&El~k0<&3c`bM!c9wAC-"
    "Ku4m41X^%6Q_CQu-dg#peQ-;SH=&N}q~%CXPc;R2{^-`c4?H8ER62iACe{%BMp6MLHJK!?L1JZamF7f$sHGN"
    "<C<a-OJc0qIvS7LhSRa5hTL)UecqGi^jTpz7I-z94Kgc>Yjj|ji7b3)GxhNs|aFxoR#!~e}AVEvj?l3hC72y"
    "5Ki+G5mIK^(3F&c1v3cZh)jgo3H3-GQHeLz!Unq&ON<`WT%89N2hRCMZOY@pz5C2a;?C*SL6?NR&_JU66HC&"
    "h~{pp1HOkKNA&?zm-sH|FQiH2Gjy9QlS7Jo-R>E#nRC-RgV4(4$gVt?L`_(<SfwuKt02Hxd|$$-TlLfBgpu_"
    "ZL{VvD5kwJP9E_GX@VvXfiK&6bc0{{s<v(N=M0@WSv?DDIk-BstU!HL(CqBMfQf;M70M!&K3g)gdZ~E>~LIh"
    "3iTX226w?E!sSb&{C-5A#vLubi_1!G7uI`l8`5IRN_>b52uhUzKzm1rf1BW*4T<bnY#I*`MCZM*0fo=b_l_p"
    "wD!$K2*7p!zi6RF}Q6?!Y&w!yp!|J9Vh~ty#Pm{g#Gq5y52%<#jz(7*78LSxXkRZx4suuwyh*^<i_~ac#<VN"
    "{51~INmBlj6a4d85vC@Eq{AZ(H&?&{ar@+<+=g{cIWq$9qbD4R=#WN!LF2wL3mEK8U(_tR2pgp!<9VzG{`(&"
    "XKm1%gT$f-kt&_qcd9Q|eI83S@1-zL$KdyLv)VD!AJnT+4h_SQ{-}aF;dqv`hsZiBDW?tIDtT#PTsqbVZsnW"
    "TuAE2RN2QdY3IVB7+WtfxsAwz!`414apBoqj-^<TZ+7Kk=eEaeWyNknmYta|G(_Ld2`!Hk~jQSYU%e0ksgw^"
    "+&8_PdDIf!tsPtPO7eInOa}>(MF}eqU<06Jj@RG)jlAl>MM{<@b|aqoW7;ALg*vh_v-0@m{Y*jIhh7|U7B1H"
    "j^3J$|hVvkW%oGgN!uQ85#qmVc01RgQ*mTsY@a~_&ai8MHu^-jz;j88?)KDC4m~K8{hRR80<N^~cc$-)bo6e"
    "fO^YDf}u=WZ}=%4VkVgh0-w2f<HDav67s9Q~N@6tBznc2#N^lA+=ev1#KDglAI4C|V;gE<_`XvGD&-M9MiEa"
    "2wtajVyZwIE<plsm(53!<JCzCUhx5un6XU4IG<x?w2pQYBZzajOux;l%~dHQj-su*$gg8eCxt{C8rX703(HW"
    "I^(a-Jl0oOs*T^2W?Y@%53<7=KrzlTkCL?<wfB@G*}3Nv>c9G&TstB;bPqSKag{F)ydRnCs@}qvTS`x-@uqT"
    "`!)M}d#*EBb-r97xFd(~2@;yQ&gBS}YTw1KR>hSbw<Hj<Tb{mbEQOf6aYE>N#Ne~|HcXS4@({h{qJI0ChO%i"
    "$E?FD=JCRO@9A9dN3vn<z4Hg<U4;3{fE3^A~vW!fIf0O`{5gE}6!-bw}1`ChOP=n@$Z3SCblO5F5ll4WA`8-"
    "=+!U@5ptKNHR*FXVS1TxGhhuHM$j}o3&4bpqPW;twhky#Y${-|3PZqc|fbG-oCDtT$dhDd884v^~Ak+xgbhp"
    "z*+yYdpn`mSMxxPu}$K(CA$cuk{TkK==URtly@9?hPIUCFs7aCz=;@0(gf*Kv?uWy5jnsr#J|JLZsm-1Z#|U"
    "-D-F5^ff*f~a@BUIy{@v6^fWx0mz3zUy6#4oYuVQ==0OX2=o3V;X1?9yb!A#5%dL8yHV9hnR|U@PBvgmx|M}"
    "Chx{(y&i@)`Ex%Y-Hv!IfBAAl_d2lIBFdUvwYPK+aZ-No6xZTLy;r_ySeSit8DzOmd$VK>88F`K)!PD(%zi*"
    "LXZ?0yVS%RIzzLJNFou%xxEz5OzMrm3^B~JZQ;W)(${H2tTKPW5VePV3@P74DrE9xV=3-a3Sk$0#z4Rx-m~$"
    "!j{PexzN+uomOFPgmH>@5zJ#^$ainfVS^>&fy_)6o9vTO9wGuP@ePO$|^bVNo$rn|$bmG0QcnFF|S>URL2Hx"
    "3s>Bd`vnuD#3{dDul{BetQZ;tq%xz0j&|?3c+W=a{j7yZ)p_7|q`>b`Et2OpySGfT<!g3SstHSIb~-xkUssS"
    "oYBA*dHx5Xpq6wptykIq_^dUDw}~2mKin;a3`iwC-`c+AV+O4PpBVK>}-&L&yn0PyF$-(9K{KdFyuj={hbaa"
    "HEkx8bzCs+<4b^xiR1JRBK5Zev#D-ax#sk_GNa`#s#k^{I?6mI>Fu&4)O3s5fW0|N14!)D0NZa0t3vH}Se5F"
    "A_o77ZqE4fShxYOawv~D=0*|+nhPbE!f&{ScfiYz>9S3QcI7GZmN*XlUiJ<@4brQ~W1{y0Hj6OEJM^+$zF&s"
    "mZ-<|#4>$2vqs+Wls(AJN+3)MLJs5=-M;D5M=rC3i3-BpY`kaz<SgC^#{Qxmg_(t>S75#rnz@Up!`Bm5q_)b"
    "`W|<IYys3F7s3E4uMK*&o54w})PsJ@o-Cs5M?|@eP+dM4xrH#$o-j<3(}$))~ha?Nf%c8|1<CcEVU}$kPVs5"
    "Z-g2q^klo3eh26+(s9p0KnUlEX0}VB7YPHUZZ1~28If)3t|@{v~Gy6ropYQE7ca)6}#iu*8{(lw_7L>;A;cu"
    "NP!PZEfWG}WZYSZZNiev*XZWb^CmhlldZUK8Qr;h*R2lZxyP($j9ngvf7N}<BRR)C=B+<qAPg-I3o#+$T)#Y"
    "jUM4}QoO-Dtgon&ofs78(fHH_;A(vYlLuzEoog&ROEfP9md_FphLT7pIt`o4Z!|TA++XmG0R;QOi9xgi-3sg"
    "<o=DEGPqicN^s>YUIN6`<A#1NBm1wm6SYOyX^PZ0`Z=wi9bReSD*7VX&szW1`C&pl#Wdo_U<hMQgoS<}J5^?"
    "Fp^=qb(j;+eF|2Au|dv28mys=WbTw*>UpR@Fpm%noaEVo23z<Fwm_qHObfkrmApMjvKL#IU(;v<jkteagFPq"
    "2n3;k|zm;q^YG|LKaA*vkY${-c0R)YNl%&bQ&Diws8F{3XR7O>(%JgTps!Fwq2(#`+<D7B0aPosbvtx?I69n"
    "9T*T30uA*l=N8<H#-9e62cAo-kN*G~&9K)dARX|hSqCDoO4(P(P#g?+QdQQDTyq3SQxS7Y3B`m?)5rROeCEE"
    "?coPveW}yVyhJ56c2`trQV!WDEo%MW^<$5{#6y|MQ%1#5~4H1%ydUHm>r6`H{F))Nx;FQ;r*gD3LCl0T$1{>"
    "<=RQ2_txHz1=_GsmX{xv1-&)3UkkcR(6D^;NYeZF2r`op+&Nb7l=>m`W4e!ma3?18p-t43sB7iX;g11r4<0A"
    "u79q-U%^=NGPCK^2A6kRX}t!K0X;3Fl4}F(q}Dxka(<*Sg-93g;405+lMY3}Xa*Lh__?#OsV~on(_mmO_65K"
    "O48cphG{&%?BC_5#x6RY@0x}fFbg`5c$(XgyT{v_g%`&X8I3Cy?s!r(qcuNHwS-jXZjB+gahRmF%`(3AlU>#"
    "w6v?=P@>YbDVJMxK#4y*|5%x8ch9Qz6=SWq6*^;rB_ygb1mhRY-L*RXwmwaLbhmbEeX&D8EVn@V#OWrmQne-"
    "Gi$_7OUBUnc$})pODfm^I<X}L))=(}!#02XXg=bWV%Ra*s1t)gk;jHdlhN-+p_oKD~xVy8ERU}u&Q(dJY8aH"
    "+xaDI)IkR~{ATGkCGKoSR>Sb^l*BBY9T-2|MT)zRquP`y5Wb9|xpcb%1k>OEVhS(4iE4HbfG<7A;OmHiTy+("
    "PJ6dC2b|wl-k}!wO-jTl;y}4NKKgcomvj9N7n#e$)UexVGJ&x(MD1Eg?hKC8*9v<9xx`U{E@owJ#j?unf8yb"
    "|tY<$Ll4QCTh#*$~Z}sm`IaXf9A!a+BKB2h_1wsEE1}jfBE78{{!96S86GOFwN`nA6d?^>V{>JpdO4tq$h=="
    "f0o=80q14PJkbyMp}|MLXtUR6#U!oltDa~92<S4^!A%i5e#bQ{VG21T6KtV0x$F3gp@yWQI8@;18f}|I>`}8"
    "fD(^RDGeqC+GP=X@_HezLbNGa1qisGWUGpBmI5@am3=G8YHf}o~M>RBkedz6f>?7~5uRs4dMX6n>S;4>>O3%"
    "BJu8Ju>NPcp4b5LO082AMP8Miu>pX<u%?jTmn)z0N$9ml!80)GXnYPF@$D;B5kj;ZRo$7gq`knRvgp~g8k0F"
    "~RtTnCTKgg;%S6O=XMr`;MkUoGNU!qX+eId^nH_<utGe~{2Ovbt8(6#3{k<E55`I_Hk%EL~J;R1srHMPT^Sc"
    "#7ZnFmP%4BTHq>q1oy6o=5ZY{tbCaU2<=WMpck)HHuRg4{E0@t5L9f7dJWxUeS_<#rK#J&`cQ9YC&x&`V(i5"
    "UyO9^t@ew&AnH$nAKjn_mPE3<49>}|rUL)^LIlTn{Lp7t-RR9Q3YM4i08^d@s!f?w&@t*Flm~^jfKH=zh%$B"
    "Y=sG5%H|r=5d*H>BGbe^fsZI!5QcwIjQ0yAgl@$Q4FIV~f4MRJac%+F&_m6#!e|Hu9|6E6as9aHW`Xo8S#eB"
    "5N)AztdcC1Q{HEF9Y=$sb*5S+Dx%4;c~bExFw{SwBrAdNZk+1QTA`<qRId|;N%m4al;oj`Y$LeTvVO29a|vp"
    "xBy+pfy<Nb-n84F33><)Fh?c~Ee-iXjp|wMqzBP0XgYB`3-0w_xL`Q{{H6IO3{sG*d{uj%PYUJq=yx6T5`IF"
    "4n1V!JM8zdD&O5j}I@};k=`cP8An$Js(|^`XjY9TF>>oUwl@!kMLHAM?Y&YgBY_1kkNz*=)Q&zZ|i+?M-vgS"
    "AgQ7RF?v)GIW0WiU<(vcBpGRWKRVgP7Kog0sCKVZQ`%3|YkWc~*BO(2+@-h}4vCc#SQT0GKJXm7|IqBixK*a"
    "1j~(^zX2}YI3Xn6_8K4*8SMX&l?RW{0rdU&gLPtE`1pvPifZ*=y^pqAx$c!RU>MWqX%C^HA6)m?kK>tTH7}J"
    "9fG+!z<26<%iSLubC#kT!auJ@m1xt*ccto#B(KyNi2Q-ooNuMjivvwUNZ2j_xR6_~vms(pkY=8X|*L9-KRMs"
    "~c;TP2Oic^z~K>;e{{dUbbO#Cn3=vWAvs>le{of<Fx<HSYlyZ=Wo;7`qJ<c0RvNQ1Fd?i@2~~DT)pL=FrFrG"
    "l+7J&aiCz#<33XA%eEA#CF-_=(w&N*u8H@zAYe5;JQh`(b*b!pXxHQ(@siFS$o+gs_r9l{sr8W&_?gWQ~yPB"
    "kOp=iX;{sRgyWTXk1IS&59=72H>%QGjr`QU?mqZ-9I?5>-E@NOLW*pOkw!*4gOYT3-w6mjz)=&-rO{iK&e!v"
    "0^ZAD{XFBY@O|{(7u3pO0aq=mqTc!Xzf?YhA_G*Su2y)4uKmn<BTIszsH*tGW<APB5Z!V8UKo0ILu;9{v{41"
    "3imskiY_Rkr6S1bt=j8g0>LCNrD(8XxR3E6_1H*&wL_RdBZXaCp}5$p<F=gUQjTkh2x+0gszPx~^`q4`?518"
    ")vi=GIQl5zFScGBtXu=k?{17}G<QAl|9GvMsIb&YQS=X^)xFJ0mk3-yLnI&KB#0Y%#*E`%%(cEygGDC@!U9;"
    "#9=&<Hb+0y~dvxvn<lUskcIbHtO4}mwC*I*#!v)>T*FUQ`G}o$6<cycMfq`vA$yv9dipqK$<YOW$9Vc(J=Fb"
    "S+3vLEXeg$l5Q{r*ck;a5<q)f|7wk8Q0!r@Rq*^jUx&F5b<W_D>ixm%F5(12+U1SjWcb$|Bo|<O<}$oW842W"
    "pF=K#+d%pm(<D64hr4?FWof2vM%QA=qNbYYaqgG*2Jcu{7#zY%E2{E0^z*KBI{C)K1093#?$G@E&TpXXCO!j"
    "|yP$!cY&;5R0Jbz$6`-Oh?AK1@}LO(Aa*w1%`e!hEPKi?Pn`Tl|Z{7~rUhX?laW1*iPfqWb~zyeKlG~S*a{P"
    "yNR&eH_W6JS7{yd1}eXCn|z7YDz-9;xG3i2619`|<h3xuVZb>`@afSt_0#pInT78=a}QXUA_2&i<kPI{F9m0"
    "m>r;g-BhD{(gbIp1gbgdK`C}zsU);Z4<0m^$~UO?&9?L<nV0tW^{7FpA6^zr_{TX<Ntg&;-`r0?LC(eDlQ3K"
    "^~|bu_nd_RfNA?<D=!o$wA`=z3_QDAyS7Pum6NObGCz8CdNw-#?F9O2JM}Kq+33~iY;<xsvNLQWWb~HtGN$J"
    "@S`ET5`Bt4fDQsK6z(9jlODZYVU$Yg|y|)cO39tnftql~q+n&~Hr%f}y{#_O}Tw&FYT_?AF*;kOW6-3`^TMd"
    "5m*6ux2$;X>us%JV}<=^g18040AH8*&q53SN<zMeT8`)2*9{o5iX+dsg?JB-9A?GqQbFFsdCqgMy-USFu^rl"
    "ngfrM9~VOvfw7v7PqZavqO43`V@&K7zTPuU8rrh9==!KL)FtJBo^<nDlz;0OJ`*tJm=&NoSbLi#eSc<YK}t#"
    "dy;Zfd=E)I7E_54P6J90IUZc3ifz@t-*pZ<;yo|7E`=RO^-&eM;9Zv2*{LB?h&pZvFT;`&W?YBWv^0GCUG*M"
    "V7GDn>j;>y-1JXRxCuP??f8VDNa?D3c5r+?Y9IW1dUnxOdjcVnYR;y>r5JD&om?FEU^Pc4M=y6C*jz{285qy"
    "_5e7ybR}2hSe<Yk$(kgJr;9_|Pc|c*#*|>dmm_+MkOdkm2x&EZ*Dh;@@Q$W34uG3J^6Sew}^V1U{Hh@@l;Hi"
    "QbYR0D<C|@;4Wy66~?e#p>1z=h)MrZ2a=tv!&zJB-SL>-?C?acw}h%TlBs0kHZrhq2_h?2dY0aXYQRGgOp-f"
    ")SOXmhDk9c|zlk!XJ$(`V^G8VTf#qR69b^sSfMdy=Xag;OhxZf|dgKBRI*J-d8~YjPA`L54C`)dEagD6tjBt"
    "98zmx2Wp64(5Gz5EJ!Lt}v$8#{|5aBN`p{hmgFvoTNN)QFpS0Y{QuB9cF5&gKP~3;^dYKf??t`6{sbU_Z`+9"
    "n@{F6kA@ZFlXrY_H2QmuLNQ_bqjQC)jUKe!Di*Rmbu98bpG8hn4e?;%EhNB(S64dC5MVS=Z%1dn!_$+K(ILF"
    "2R%vjx42b>^8g`}Ad?Q;!_c=Nb0QBZ_hO2duV#;|+$=#F&n<uN4o~tR_UQf@|+34--gTv8OC3Hlsd5`uRuYN"
    "Nw!A^w@63m!~?cad+YmFs?14R%~rt%zZ{<V%Gs0t9qV56jlQe<wCL{V}Fza3r&E1;i!Ebf36?TBmuhWS>fR7"
    "n>)420x>??*bgrL=XhPLio{9aD7o92bTcC(L*~sMcPb@X`)|fzh$YVhy}3ch^aVOd*J{<Rc+PL-J}ugK%Wrb"
    "rQ|FI>f=rKe*mqUyU5m2hP7Gphh!jfmHR|k@Kf|6$LZP`epAd>wo|I=tSOYGxOi~N@6)KwUg0%Dh%{cfx+Xc"
    "k{E{eUF!e7pddgg+(i^Z0gntAy>J@b)+o7>z=9xF;ao3QNscviZ}euM^1FoV(`3MGNT`6I$HZ?s6+tY320BF"
    "XLrJQX9F2{!FaYX=a3Y{M2$n?b^{irY9bkCHCFADBT4lioNEM=K;jA)c2z~^>);8Wy77GZ0MN>^%B49UldfG"
    "QIZ&dHi5B}bu)7u=<41oe+p7ayS>&qy)>=ITKT02E$<8lkI*6@4I3kuB@pyLvQgU));kcwlg6rrPGR{`b#b`"
    ">`Wx&F!Mz1_0AFzz$?U=(<&dqF(!WjdGonXvaHCjcX0hS~`J%901@0!rEx1c<p87X?V2<Tx3<FN)i$KFFYsh"
    "?dyX&aXJmaej{{U|u;NU057Fh(0h$rC<b~VDax+&LA3HwfFx~QMA2Imqst<1$T%;Rrjvbs_>>4p4j}lZMs)a"
    "j6W6`MXGN@ba@0h@p@$4tT7@zk((n<E|d8NcbL=@Kw|Q0U$JPDQmL~f#?oE%0R(mq)|n=8iO?}|k|V&BgfK&"
    "4A$qC5#SlO}evNTytECS<q0>pD5lW!w{&g0r`<fQ3^e$e$$)zb5r|^z|zq(%_UG+@NE_gcuWCT`P!7oW*1$x"
    "7{-j}+o60U5^dMa>txQ~ERTI37HqT?LUvhnEU_Lmrcvq2B7J`F$lr@<8<*E*W3(~}X9BwS`<%$$SFI4q|Nm|"
    "m!70ERsyUO|l>lsK&BITf6{V1()P)p?$-XF00hGrr4l6)oVnNe28H&On|9U`JU3l8vx=c4=h6hVF%Ue@bRNB"
    "s`_wExcAjW{+XQs}-LN4*Y1Ml^>J!r0or?%VkRWla(f2x4m!IwPtF2Y?vv!n25v#e?7Q3{2kONa~UEt9$jTY"
    "cL6jRR<j7p**C2Ksu=THxbL9m1E5SnjxZmlTU3KV*CFU)dtV3r8GEdK3pgr*UsH`jVy>tngXjiz+iF`Goa0P"
    "y;k52Le7*Z*J5dE!SE;_LyIC3{$}&B_k8y7|3VUom>n&b^+j`FmJa_x$+kLyr^5OShHh+i!{mL$@dFszTPd#"
    "4+DKMx#DC9v%S^y3h#`A%Q7QYOY0GN@?fPT8z(F7EZET8y|DL#cjy_oVDT?=(nR5R8D6Ql6#2sbt5JK-kESU"
    "ad6&nd$-0Qea-OR{{u)O~fF8`G7>G^@K{W9CL3uCr?t0$nWLhZon|FrZ5@9Kciq^xFm$a8sL*=h|oU84FFi0"
    "3&L-k!Frm2xEn}L$vMSY*S6e+o*^;OtyHgMQToNUp&lzvhx|PlEwV!@b}T-UzN|3)xZB)?d|bta>Lb}5o7v~"
    "=(`08V4$p>JGR^0=<P6$<5xuivWFUbkX=pIaggU}c)11^5$UU7w%IErUYouo6b=WDyaVh2GUAux_~`8PZDHk"
    "W)EL;1v(d@Hn~`!Ry-kDTcP}^2aUd;@o)qL>?P2}JKdfNlA{{|NP>~2DBxus?k*qH@+m+yqa?ZaoHnWVUL+h"
    "KIrS>Pk8s-Bse-Y3wSD?8{gVnVu{So-r3wpTJxp+Cw-~IX@u;uL_-rybA#1RCS2_8eO&j0#O3hJ-|)WOai%e"
    "i!CwA#Kp0A){<wO4;dU5pMGE)AN_`J{RLnpLa^*{H?u!hJySqw{j%0w@a&H$dA;D3YvE&fCZ5coetTWa-NVX"
    "~n;K_ShW>Z2x&Pg@qX_vmsdroohwHx)fq(grE%1bzi;K!7Y`&#QqF(r_!|WLb{%XK}6jrD+-b;Fj!=+hP4`g"
    "DkN*Ts4~2@q$tTs?ZQw0=V{YhG_`N7Y(Grn@}-C~1y_Xhdg|m<O^N*b{XWt?&u*#~VMJDoSm_{*LY<n(tm^;"
    "^F|tr4e!$%^ov@TucgP8Y=q}h~;{8-;H^)2RJS7JJ=?NrBC~ZD3m`IGk5o0QdjTM{rR1(dJ3JGdHO*WMK?FN"
    "4mA0u&1i2$)@Ij7iAzOPdGOoa3%Kb1A#zk*!*S(IexIYjK^0rX{sA|8hnt?a}H!%VE?w;|wXYL|$(?)BX@qQ"
    "yxx93<RF8JjBO4C)9#El4zytK&CsMn}g77b8=9OYJ{rJ2f55JXo#(BSmCGl7?4MCEA`<DoRWiCc5jSdKF+g#"
    "3k5~WALb8XLd}u*UT1H9Pr>2Z6~t_>3+~>x{k?N^v62Xd9)#uHWfvnGMU$5hS3+Gd)_5!eywh(UU%PDa4ZLD"
    "hh@FSJ8WGeRh@&~bxv_v|G9s6)v6AwO)l$4ooOdg<Z4GeU0*q}KUT<4Up2B#|E2P4zi2S^Yy-Yc49c~HIc4o"
    "8>)fyD?$ud9de#szt+_T4mOcz;VNMS46qDJo3;|O+xnvyfL584kc8$Fu(l(IXrG^vEWfqVxWeFLX(6K}5I~r"
    "@tWL$X`^N3bDT?R4w1F^$(OxHx>Krjj-Z+HcPzthVwPlI$*P=I`!for65tg#i=&U;>_v*fOPk!E9RVK<+s3z"
    "$U4K1T%kp^Za*6uAVrN^<a6fu72L^vSCkX6ntm3nT~V>-^x==;9yjFkXY7CFk#xHp}-ta04qs_DDB~984(*T"
    "%30aamqJf(Rsp*8w3;V9J3Gfmu=KMp}mIQfVaz{TQ3?qdD1RkW6@prgxMg;f&Mst0>2#-T_Wyp9)xgZetmp?"
    "A&&2Pm?0wV#-bPZ)xn&-QmZsc7QIUy&~5h#bKIp4AQ**Og*la{KR7v3*>$*D(W`Kaw#*ictaI{S(NNjP*JI}"
    "HG$koZF6Bk4vujF{i+CN4?>f5)S0rnpzxC=W4FJ{LBv0`Cr*bvIFaXGXxkPxqXu~P?12tW(^GT-DFo?o`LP1"
    "i4y&K5IN`*LTa?X-eg<Y|Cuyh?~u2V!rQi>@1U?I+EHStJ6U&cNFp*~!809}HB(k;5cEQpghger(q>4&+3L;"
    "<li8TTep!4()A+Ece=aUYPt;WjtnxlB7|Z3rQKvI?Ulr!a{qOR5%0EEi#hpnd~kcSv;z8J@$|0k?qw0G;-Lk"
    "xj$-R4g)Wkoaz3I(`jRDO9+gC<tvxgGlXnP!|J@D$(SwUWB>*_+F*}eHINXJr44~d=d~z&{2E(8R_(k<KdrT"
    "W(*X>R8#WOPrJ<6uwx?ZYxGV*AFvCz0Y=v(iw@d>W?=LIc$}ODKv8GUIL3>&Q$nejU@Teh|7dm@RW}?}FAEk"
    "r-zd7d;u=HmQ{0dVtR89P(3Ld=(b?tv&#5;_UJhW*Z{LnijwYukum3SY27NwY5Hw6j1n@IJevwMb9PS{=N#h"
    "pFPUjbo<vPE!RndKyZI{kZi1a=Ig#L+m&$7~EQtZ6nVIk-TH{p!2ltI-sVLol#LYsY-<*8RE`a-PVWRau&!c"
    "-{~Z62!=TbffGniJv{@>^qoi9>(2L66*Kd#ngIe5|lXXKb}`k6%Bw+Q`RU$5tPiV+Pj{nj_<SNNi5+oOTP&d"
    "7|FfLyw7BVXn@9KYsgmbks%ccBDD@M_-*Jxw_IZGFv0Efl?P?n&pz|sSu;eF;9R5=O`_<exQ?n!69FcZ4MbU"
    "*}$gN$EGvIx(GUDl~Lq03MPJn!VJ0s$(ZCc6ojD%*B3}a7U!JIWci*jSuTb_A&;CzjY13Nnri(|QZCWt0+MW"
    "%B1h`0W1jp{FE4cp3I@T*7}qO<x5476w-7HIgBV?|bJooQsI#<$iVawLLz5vLQ$1;N9Mk6J1-{rriOS9>>-8"
    "PC2rX>f!f+RMNy)w)#HuuZn7GmG>USW^1J2u|`U<RWy?S@??rh{_aH1XdrM)P>0C5X|H9A#u4ShosymydQB9"
    "lpkIHOe<$8;&4qEp56JlTIS;gy_(*@UJ!?R!_E7mb3%kr)-i!jlu8i0Bk9UN0u+zaPB#fdW=$mp^>}^LLlu{"
    "h+_si=Td;|McVjcR%mXXTJpd!R(jspU-~y<?@H0zF++G)BcaM{hww(e}B37<?_eb&%b>4%ZuO#U5pmpcO&YX"
    "V+8p|*%-IZGu|HV$A;a<id?s2s}EiO>ajhHg*#(w2(FaDHAPqQxE>T<SvRK!7T!E9{~p>z6h7r|9jbqAek1}"
    "ZjkcCd9(S}4Bbm}@>j|eY+y?UTNBR&V+A`M%laj<?H4%~U%B_PGm;NIWjx)AelKJ|vJ&a78vDFiZ8C(Nt$ha"
    "Ot6t#1DFgciXh~^9_7MD9jmUG<5qh(3i;Wgw*ef{trC~lnLH3jp__!`6cd1wy{=&zgCeM9>93mGl%;xJb~4+"
    "K}H7wKCa2_!I>oS^zS{pg1DxADF2_cDqQ#5z^%t0IF<k2{3oCpYY=;J1F-%T%_BXV+;GCu?wpVjmp}7Z0**y"
    "_6&`x;bM%pi5)eEc(!G;Z<?8pa}~MuZWC8dih-4;LQR&x)^F6NK|mwdfE;Cah}|0v?YT513rBn&gVK7pAtpU"
    "+tBzx9Uh!hAUCT0?CjwfcNJU(V3DDqAKLR-1fj}(jt4}G0oB7>CUKI3B~Hq5k(19K7+x#D$Y}74Gc`|xJL(P"
    "WHGLnB0gVtroNJvasJaU#>b(s9pOWSv#1oB4SLm&AlKtrOgqDskQGIn4%+e&w%$6`%bXK^9@vdRU6H!HBCTJ"
    "zU4sJC%1JNOAe`l+mEF<2mD3*ff2{v2wsDf|v7H!mQ7`#h^RghwqBgHDJREJr%*3IFFgr0^&FwuxKpj{C!YW"
    "?h65<6f*r!hG|4SkcnU)GpWfxuMC@hlX<M!0u5oKUvt(usl6Zn4IDI)&L6QZ9rnty!{bBR9cZAFiS>){^ZO1"
    ")Dyf@Ttk1!JI2To-9P#Nh%BxZkYjD7-;ulfLRE7PKQn_*L6e>H@Hu9412YX1!Im9SZwANxR^M!hJea($`J9!"
    "(Xw>$k0@gH4lbc-e>+megsLaYmisaJ<!}o@-U3dXhxw**!<o=0KK+y+D5z9sT^L0nkE9%n3Z-aw3OL;fdZJ8"
    "*Y!@QGwC{@OtEm%p(P#3Um;jpz?K$-@Rc)-{+@TVW%qCJO>KuY$XVV2DMfZ4_#L54NxkgAe_OpALqWYmh#EC"
    "Q<7~r!RG-z=<iXE_^cfd`j={j}--KG%jMeYx$25TmAWrO%p9gelYLzS7+6y(Vg{3sa3hA|gl3<~p{{PRt*^J"
    "d`;w%Q{E{xs&7{Es2zhM3N;2)1wX<ChlOk9D-U!li~4p^q5F0}(t>w>rXO!UqR9FE7nYHG`yCu!=!^N}kXw="
    "n!rsP$HlVIjc{TLXn*1YsS^-t24^@<474BuuQcSfoR#M3f??tycowPCrC2IQ4=DmLJ^IqEpg5shQV-N6-gwh"
    "+6c@X+Xm_?M2&&JunaaFZ3BKwJk<pUzl-nVX4aJ$jC9FKJlO4<AdD`PPs+lGK1+3;Zl28{YcbL}s5dY^ymb>"
    "u08uu9A_NS(M+=8lCm?JI7VA~--S@ryff15DJX?Z?+T&!)G>B){qA9Y#Bvnz9_bmkt3>UDC&dqF%4q#Gea3~"
    "$OJbZSYq<L=^rn5EL%c(D>Acke_VlV~D0+X1$Em5)ZSl3?`zOTA0bk*B5jA!8r{Kg0<1xDewXX?p!`vZG)dd"
    "9sV^+lqD6^pjxxcsi8f;hX=DMs5bgN?fQ{rKGIKO_L~CrF!eg#c}qCkVzZpxPU~AuDttS}By+j3B##g4|52C"
    "U##wiS_3>4jJhjlr8a@)AoZrilV-{xJ%U0@hjv=7bN-|Iu>Zha|kw~7b(yN)p2GK&>~FH=WEu^6{1v^Werht"
    "5V!;3Hkn=p;M0*ILyGh|4$v&gGmkfzs8fha+y9PyARL3j+~L>blcRG#Ox95W%%rg|qIVR?uuceV&r#0Hgxnw"
    "WL>OS#;1cixiZTbcC#NR2L;Mj3<L1|TiJ=H^--WJo3b?_q6RHRHM48-_=!sgRjX*r%Kwffci;x-kOCowL#+T"
    "?h6d#9@_IRc#9Dr^`9NH<IISpbp#aue#DpR8lq#6J;69}x3+giqv8L+QQ9nU%FvevU@!iUvmn%u~Z5GzDrHW"
    ";xg&ho?W34BBU)CO{5&9<qiN1l5O<pW)*TsTjA1(577>n`i6S+IuWf86yVLQH<GNq6hO4ZRfJ>YVwXVNp|e5"
    "1u>AyaL)wfgMtG#$Ax&Eq0E9K!Ylc32!<mJVQe!*cV<tL9OMv;1q3ujTP5dI$nn{M5Msny1vj()^wA>_!+_4"
    "E{Y=(*@B@B<)cahdqX}@wv@mHTla8aKnyU-AeTrZMWv83aOrYv#(G8`PeZLB1LV?2X$caXieH1H$LvnRdni&"
    "3V{T?lNU+HIPAxFF1z59~e)#-W%oOD4Dp{>Da|(|8Rl+HOKt>&o+yPzz#sZ2}Me~$!0K4%p&SIuRd3)VFUVC"
    "wy+*6kQP$A2H8K}1q8^S@qCUhE8HO@%3y?FlO$KLawdoO<Jt>ZBNrH9cZy)f%>ye0AC9#~*{cGJ6vFM=Hmz|"
    "}~W06ej<Re`!*FN0W7?i!CtkeU#3?~>}1IWfaz*+MlffN)#d-7sG06lhh_zlgr<CPQ>yybhusPm+Q_iLbk@4"
    "qqPw!cRgziGIb!kgZb$TE;GXC<saMjv4@7f@mGELKxWiGU787cqRcleaQ!)UMJKms`#voj$6R*Y>*nn_n5$F"
    "^iY=uHEwy(Whg~S1XH4hk6IS6+!s(2vb{1^NfHYpGtUaJE!hHLb6sgEz;fxyY$uR?AXbU2%T`6kW1{K1DR@+"
    "YEtqFE)E$i(WVJ(P)O?$)ii6q((7@k-N`yHtxRy}soa_ni0>wma3SUFg35B}vY$#)PP4n$PRV<!h{owKW>HB"
    "9+^dy9F(DCbYE)IqaV<xnBl?H<&%@zVXU>Pl`Q4g6qxKM9S&o7kdvSA$NkQfBP*3cttvHa#{>f<4i*u0Iz|7"
    "S@Ac>#&paHJ}_nJ+?}W^&4xNFRy{Un$5w*o6`}<ZXu>Yiu|v8!|240Wpz1e!-(Lv*Tq_siH`SZs@U3HXfp37"
    "1%}<Mf%_t6&8~6JJs1kTT+}jO~@%e>Mq^TER&Dn57En|730)7d{|fqe&p2EI6bTXB)+`s7iA4=zRHg~21S00"
    "NyVcxCxYJl-Ho6JLVxQ<Lk`FoQao>1a$GqgOYFfRMH_+MFOg4#TuD-=9GU7y^hzKtnkBo0?PU!n+Eaf?<3=U"
    "7B7C##k731((j3sR>}a?zI@@5ws!mFy*S0d~b*n3}QbatGH@9vIwVvH?IfaLLeCwuA@BOWwLxU%{a`+GN3h%"
    "ZK4-OshkM?#-l=$i$$1SqyQEAT^?G~bYtkHfAnK`4~Mqg&QTM0`>`f!S>oof>*A>u4>Oi+a6w!_X4<dblDg1"
    "e#qL>Xy?TM0Asu1U;<3p!m-`fA)dN$955D@lZLtB`iZ2pFh@fgoAX-3q@3wjHgm;NBVItsC!nEFtcabe?r2P"
    "?>$;&K3Rq^gY1^=-lRMnCXb*$Dx6#0!kgZ8KG!aE?P~**2Rl)oAq482#D{7%Lv-?I6amDn>gGc*BsyguJ62G"
    "(0oEli(P~m&WxT`*H!B1$ZQ~4u$hnZ4qtK>A=;V#snZ1QHngQ6o&7#=T~N8eTv@Uwki}nCDcaa|N@7>-ECdn"
    "24cD5e>?+b$Cjd8@>5d1saJ7btIE14$yLJZ#c99U1*4#{4Uqt}}u{RvjSxPCdPKB{?OoxaL)ZQvduO>9P$)5"
    "V}|3;;>AI@2GEGW?aLpjMoKb+HHcJ0U#Koay>9UjW-tn~eMcKYtE`t={S#qS4yJ3jeM9iF~BxoAJ_Kmk#c>2"
    "B~wpbs_;_5S>4#QZa}+v}PtV(q;+xx=(uDoU8#W%Xv3A1ll(`|Ll#e48EyjabvKY8Q{A`kgH&`yENF#&Cijm"
    "d|!WcVzHA7gd9k*651AjwP23VMN$0EP7v2=W`UcA1s7y1`1z@I6Pp{M?NICeQPd~tSX^kux}&sso?7}?eSwL"
    "nu>hN=yPlH<;MUA`=*IbeXs`Y+=O+4B)ws#R;eWLP~3oyE!RoRl#+XonQEOu0d~0l4h4)J4&GI*hZ_4|rO6_"
    "UYSAQYY#ziGtUxY3Fa+noNixi$1XHgKTtdub>xL;{UZcb?kWB%s>OF=!NMb@HK0JN%_Tb_eR@1R2;o`7AF-j"
    "!gkQWjAf-s{t$3g9P7#czD{$1;DGW)P+b`%O`feX^`Ut#xAc%v(6p<lB82vuNyqzD8lRDc!?Lab`Ctbq%vM7"
    "?Zp1LRnNijKk+q`5#G6X99^L(uNfXJ!bEggZJ>876|n6mkJ$7D1zVJJXsLnKDnA^>{}x&J#|9%n0q8@v#kV4"
    "C0rgp@vt~#(E$t>THcKGMTTYUItI4DyWRD3ev=3*8Wjzuz=8P+S1;{_B5-Z_)@cOPdSp?RY&3wkEP9<fz_+="
    "UpcS`D(+@r4H`R-tVva;L6KNDdrX?TvxN_x+ZILLBTs@schy1+(4x>v@nA>cg1rP{?ig4EQNui4lJ<z%9IU|"
    "Mgk6tC3K$Dk@AHpbU;PGF8$Gwe&qbpF%sjw%Oy?LZ$|8QQ7dpo3=#aXWUw2)hk5Oj{SfiQ!LU}h-IUHqxqRu"
    "2Vz_I@z>P}{OQWeNU*IAVWLQnE-<{x)Y8V{t~a65%~TMeGMuRoi`k=CBjM;vK=sgr6gv;e{TyAIO)QV03Kz("
    "RwcH>N0N&>=ID8`sjXzuI*AFqh+BM`s5Yqu2kCq8Ji;!NIdk1%UnmF@;>iX5jkm<O`(d>bZJ<diK|@378M;S"
    "n_aoBP1k{)+i_=ofFEWiBbU!v2_euvp5ff3>0H<9tW8kx3XlZ39QvYx>s*ei<xQhu3zfw;5JOQcoWRB7!?ry"
    "m~3+?Jy;=gv8F;3{ZM6vrI-M-fP#`_e9!{eNc!}W5vRs2D7dRKjmfr)5Kx&R{w27Qq5~4C!(biTPOn{{#<Od"
    "iZ9fmrnm>|I^#@sdY#(co4->HdAnS#!JIV$@>kN>Dtq?BBTK=o^bu+zp5>5fl_F#Sss4jDrgMy3ZAQKpch&`"
    "E>*e6J(F+9vvO%S=r9!acRj4fC>Utq*b#(J%-5#-h_ehESN^FS{l$OMCqQsPm!8Y9<iQ6&g&nO*Do8jzy^*("
    "@Y}!uc|n`a0x5Y7od6O5pL9V4Ww+?dZ>-rt9=Jh$f`oR}?@(y}4a#_h(0ej`gQ}f|1_E%P6QKbF>jo8p}p}{"
    "<7Byji_`CYu!-JZeN0`DfEg<Yd2KCQ_J9!RD*A*`}AE!eL1_hFze<8cG{M17_Ot#?w8{zov(GpnsU%-@#lt6"
    "Z!BAmPrCKaSA!ktj=0NXs4Ag}(O|PQT;-Eu{Y)vGU9u~9Rc`ICDWDBjN-{z;D3Mj|B%H->+fm?mtLDH}tS1<"
    "`J==a>>L<pK9h1rlN_56@2?$_2Pe!75oOw8(shUrrlzP)tB+9Njf1sl8LQVFXt@ge_!)c&<f4mOqhw&^(V=@"
    "$$>E+>LH}QjCF@bLL=u8DpD5A3CURMkmK)gUFD`l;i2h|LRTGueZrnC@~wL(E~#4bRIn^^u)G0vr4CfiDU)}"
    "3R3gdq?*{2Eu>4zutwjLK;<TofG9+0I4>uOFcHo|DpSSyv{2Y$4#Uq#@KyrE`k7#;K;1SsModv79zTi4sXjQ"
    "z5iDR8!C;0(s^yZA84mqyx7WDeq;Kw}PiyZC90)-HPU{W>srgRV;hE=Bp}o%9^3FN?KBRzsXQb`fv46E4X~q"
    "s#Y{_9BP#idFZOv47F}mzsXSRR<%;oHef>Z|A=mv<>|*FLS1{m>jE?hpKrf#e|}+p{?7gRJM;7R?$6(wpMP+"
    "F{=xkGqx<uZ=I5W>pMNqx|Lp$!v-$ZK_vc^C&->54`8>A|{E6C`?t4?+w{zY1CcAHEyYEeR-_Cd6oAAD!@xC"
    "|ZeLLrUZ_@jA*8ASH_wBs*y@~JJnZNL6{=&}ug*WpT&K~=lY-j$$oB0bn^B3OCU)Y(y@MiwP&isWp^A|XC7j"
    "ZCKMPWWMSOQs_n2UuQ1Y}A;#QNxg4xYnnVAgQ5M)HJ1=<N-WQgz{D@Tw{EY078%4t<Mg0HL%~>P)vgQ;M&(X"
    "kk_f2G>e~Ra_=>kWm;ofcG=Z(bCIsmPFunvgs;<9c-sR<*fP`jB0mi1q+;E*<1~fYh9;6Dd*o2ivg7O(aBNY"
    "T`;Hk9xlqjF02}=4<FgKwO%Z+u%3Md?BHk=4r4`o)5kr7-?BDb%3&<Si*&6!0~?7ME+|$ZjCCL27FO+!;-2~"
    "u|IjB81z4%}xYg^8TW$}s#9_SF_E#R^pN5M3zs~pSm$glA6?=C`gcvQilL2@9<`s(*E|mX@^MxIIXZ%asnhp"
    "m#{MC`T97M%jsL2GvdnS`Mxb3=_nmtf&Awe5X%SjSzXAJPP4+-0G-e2C#!?aC*(0+9(_GOaX(DpfFhoZ&svK"
    "nF^g(tS&F2rM)Wyc@eoGf1RLO)IJCJRhv**GtsC3p6fJlzy`3OdQwE6_9be8Mn&S=;%J{sa)gYJ~p*fWnF%o"
    "ME8Dx?R}eGzc?YKXhRT<JQ}=gWuj9APz@(6;pw`p*lT5)V-Zw|F2{X%we*Gns-C>{@}Iu{u2XtZfY+J4sF)0"
    "EQd@QLT0KRr0%LdQSXm0em{M8A?YZ2b^Yb8g@Oi8Qb6F+RuSpnV}|9T(dA`AuQH~h!*$bvP6YqiFUUoJ^}we"
    "%SFm#bu>rm7?Q6KG|3ZOA^eUMF#n(>DYfZeB)j4|AgfG!|5IZ@EOg!RnU-*XnuhU=Xd<GEa?aqhiAN_~R0IJ"
    "K%c<?9#tDI!(#UlLF9=H088j)|j1+6|pn7RNmiMs7WLG1Rp^=aJdfPK_-Rq-0;70B~GUYFW9;lfAx2)+s<J;"
    "I<mOv0?_FkHweAl_mEs&Bia{%uIhSTh1Voa%2uwAR>fd)zXElC_$L8A5fl_!Yw~%n@_MPB3@HWu|i-F)yv+p"
    "HB>fI<zbg!VN=60>@h;M!Xm~pX!B92}xixi*#R|B7BeSFG}vXw&yBGdAI~j(Yt`KVF^J7E5WMY4yi!&4AYaZ"
    "gEdBAL^^Z3LvSKs5?nDVxx;8O7pGh~m7gJPp0n|tW9+f>E1g4d7mJs1%RU5vQCJ`Ua5*otW38};_2jsX9Wy="
    "27SvG_tdhH_B`#rhH`6)Xgg!c*kaqMpZfX{aoM%ulezaWW8wwzG9Gj$L8XR?$qiO@?A02IrSE$`AiFV*F?9r"
    "ixm@I<~=!3Wlgn$kAC^FVEwrmfxT=LCI5BG4adrhpze-dVVPE6CnqKnF+bN5wr1UE2GnT%lU(v*q=AGdg5K@"
    "@qt(>Fhdao$doyAOzjn;}(x?BKb<*aQrZ&c3+hP@y(#Cx|nuPqU6e6^|wIn1i~>VDyrcU<{T%9((poqxgkdw"
    "k3j)eD~pYktSasnwNU&vn(G>ng88Y6PUTZ`+xE1kGmOE=sGStU5r~Mp@)hk=*$ekZqPt|&Tu)~{H*iElRmsr"
    "JYshvL^#(mTUx1u)hgPk6$1r02+d(k1zh^-m6NvrlFfMY<SxeiaeLi^|BV1{!gR8isiULU>gm%Uo<Dtx7C%`"
    "|te2s#61hnw2m?T&+qS!Dp|sKzLAnBiEC8YbqjCf(oGXB1u&oFqM`fkRBZ4x&3o=txjp{nVSTPMBBLN_9o^r"
    "CxUBR0w;#znmCJbo;Da&=1L{Gaj1^22&B0oHR^XB-%kuCT+1eb82dH4W?RF^{n*9-b0MJu_CA&LmVh0cpqSc"
    "U}fFw@z<yT=o%A!ktp66YAto$CSc;%i8$h;M@^oWr77z;=3pJh6ZcVyawa0E!1Q!mKw-u024r2q^^__K{+pH"
    "Rb2~v{`$Ko3AF`QKj|w&uigmh6_WMp*cy_^0RO|CXg$A8v?pynHPr{6cz)o3gTKe%cVkN2pjOELcLtC7XgLi"
    "Q&L-HwXDe_jB9N~of<{4?(?RD&OL@)`Gs-I&88^Hu(sK3;qF1+QAGi4JNw(g>vyAb)&5IG;z6rh{g(Mgr=r1"
    "qCfK5rU)qW-_vz+7&rV;z{`KJSuaz5IxMkBFFa`*@9llYfN}+e*h5wh0>B}I7pe8GS!sSxWLr(%7LuH&iNk5"
    "d;{IX7PAeej(L!`on7^V)=BuZvC>fKw=ZB8yc?p%@_0uy9HQ=m@XoO)n33oO{41mw8ReK!%X1|D!GiBaeo12"
    "vWqV7l}Jm(&AviWvh25O#yP9AdPWI?@cP;Y<^Q-$^v>Wm01qAtnGU%E&QTj0T<wCc5Ho#uMi=l^V(!V9YJ$%"
    "2%Nyn{BmP3zkYE1;Xk6CF2(%RH(=#o1RrtkR3DH16b;P*=dc@>p^UEgJFGz_jm2Qf*&w1_%+CM9Y475=01>r"
    "6nIB9-*`lx;Y{G(T%)|B57-wX*{?8k3Z>WBI_*@ibk3K4Tkxq%d0hA}fP!(#K+VHhPL)ay;>}0(-$iX6u4B8"
    "J(iY}&$Z2gAwT3?5=*<^3RDS_8L>mXwb*5B(o#o32)Yj}n{949$NBRt29m54>)B+Y#(LA(T=WYK)I}F%aCHI"
    "5S@g$sAKNu92t?O=Uo(6;JWo2f;t*Z_F6rV>Wt^4!!a+S3u0+<>05Vp=^$D}I{!J{E{fyb@E!_5^XlQbzvPR"
    "(JdBi~Vu(({*UHK%PB)6`cjDm(76i2MmbuI9Ix4CAf1#bT=`wbQ(C*FTu$pM*e^ur>IAsFIo&8$D!A2><w7t"
    "cpQa{?~1=lO|l6(Gv_ZlJwIGSIK}hH>+JO)&OD5aY9+KH&oFOyxDkMIoH-Jr-+4X0h|!8#hAZP?8)4fLF3ES"
    "v=-%c$~6N3GLZ4!ihZpTv^#ekY+b=_kvgdq-kSxth?zQ1q)@2ekzjXCw?g1ev8?xhAzB;Z#}@lyEKGrnbV9V"
    "Twpd?HPfyh0>B+0t$A=f_sy;g9AfIWmFAC0K@Ioxx$%{@1zX23wo#ksppHA-Lj7ztH$)(s;(?XZ*Fa#SJ<N^"
    "~Kmb#Nn6#(18s$a5ltId=XtLAzRMQ$|NCs5zs_=XXtEyi)}x%UbejTk7{4Oj0M(7XFzeVXYAd!SGb3IHQ&*K"
    "#>H-<7x9am(3i2qLvxt<Fam%B1w`PjIUAyf5FZ?#*C(J8u+6=ROlDzlHS}{z_j*-fDi6{vXJ3O-$W4;JMPba"
    "NTefVs4>`eLVYJ0!W@DcQIr&Rxd0yv2nXyA1%3R$L-cvuWyYH&#ZdWb5hV~I<W55ZKXzfQ%`+RYR4@`NQUX3"
    "jNXr1on2mX*6m(z*k#=DHE*q}-)W?{Oh1^m7PZ91%J(;s=$?Y@6o{_4eiZa+>C|3BDt!oV!8V2!RS09)0%;-"
    "?h9Hq*fl7jf&#3BV$y?*x9ArKf8uY&nu?!>C!!8~53dglSZ_H@aRFsCG+`RJC+-s+JnGj{aZg{(qK`jNymmL"
    "x)LBdeLdYSusHqBZ_Q9&(T|7duhgQHPC*K00(Ic{}aQuD}+yvxmgr!&Gg77vW|(PxZQi?yc>Qn%}|2UmYMQ@"
    "<US$fq{ysMT#{>FimCaK#Wp6kO>ze})nLS%0-@4O(1Xc>}oI7_SH*6sVQNRBc(K10)Rt#!rP`Am<CsTmgXon"
    "808Rewa)aYru(_OgLg+p}kN-8g10s8GWs7Obnd?C@OLCM=($)KkxtS1{NN803Zg$kdVp+vIwxL#HEu_0h}yz"
    "##ah%tG#tgMsM+ceRsVn{>0gU#h+siw1y=rK6~o?8+5m;B&BW@op4+iL&sIbggY*SR70DA?6<hPhUmAu>rLT"
    "z9<v(C=nK!?!K<M+mm<~i@YT@F#AyLw&FxO#toLHv`Yc2JqCVeUZ@#EwS-;ObmM?vEqCde8?sv)kTSw5C6z&"
    "qVo8f#3$k>p)OjSBCTB^5-0>Yrg(A|5IF#igCkzWyyio76y^W%Jj_1XC0LA){FDomwstB+CJBjeAlY#z=tX&"
    "GzMYS;N}5<^H{<qHr67@2oQF#j`BU`@0%S2@PDY0obQW(-{{#8s4$zAXu<)#=uYr#_A|ol@;Lt`B_(g`m?+N"
    "|RAo%%!$9;k@e#)>r1uloHWdq0T$YEN2Mqa6Y8PfilOps4(G7h6F^$GZ`A;U{i^PyqG4Ac7E9`>-@_Okr#?K"
    "?^7Pinc61ASb+7dgzQsqjuFmu>{{VWbNe;0QZJHp8RTsv$U2e=_lFwRO;9YS(1^<|B66Aof=DJ)JJyaDcIYl"
    "qrvH%r`sjrx^UFx1lUcIbwCQd>{LC*6AO@}t>5pHW@P0xP@<`PN^*e+hW^;}q+P;@|(p}ZKiQbx=o*j+O04&"
    "a#XF1%6r|c|@OR_~(`;VawNy2VF7B${*KBUKVi}W8NwbNtxXSso#fszfIA5?tMIVHmF`U?!U5DKFnY`a1yh1"
    "dlrvdYj#Ex#vdGMD2YX)<vDmgJ0V8d;air&DndTOyPJLmtZauY>G53NNeV06^=oqqHAf&ZHk;b5t|~TS1g<6"
    "kf{Hw;22WtvI~YP94XGXy_T5(R}#uv8xK1TOaMQqP3ea1+)#rPY`CFBHMBam1J_}rRita!HXY$Y;WOEVE##S"
    "3~Z;|fWD3)oUPsI^sn_N8bI4S>8Q%%?zStS$i<yx#gtLOOd<;ABY<8p*T_z~3L<SGSs<en2j3ksj#;G9I-2P"
    "&YLZ(3v`+dX_!;@DPZ*$H`UdnPi5pp-woPLZIEH42W(W>*@Fm;azt(#zGxYCNwv`oxZm2FU+K~V2`Hy(%hUQ"
    "=H7kTs(nAQ;f&v{6@IV@1sXW>W#=C0$G=^1{spTQ_$(m`7T8R8dr!c}tHmvKy`wxDnwgAwzs%VB;6Q#57>Um"
    "B1gTh2qBV1ythd}Y@<ijblom&NcONoM|;U9a;nvVWSP*`GGqZaV6Hm=fI}r77R!z6}Gz3dvpi&zXjwy6TLK2"
    "3>V7o|V(Z`y{=o=@81jgmY~Ne5Dcn5`WzGPT$qWkIjCc=|9#w1B^BE)A>rz99FfCCvcx>!~YEm&AJGsn5^@G"
    "T0}_zH?bd|_n#9_HG9}U<Q%j4Tl&GxPz=(mTQx)$grDfc4;_aGUEMPKxw`%E{398PSDXAAf|&>IndRp|DSey"
    "T$m}PEk9<b|e3JEbd>f`o{NYA#KBCaR(O|Dokc)G0@jKZ_em{LPqJO<UIQfnLbvQYA4T(oEHh#(W_r-jyPYN"
    "H+NXIfEKgdBM!|g()>$spY!L*7B%iVm4{m$-+_FNs`4s~350mRJkRS;#m_#{~2DNfe;YMoP>TOnpqf2#hS_I"
    "=3zEj%U0QT(Ea0!~!0)4rfiMqJkyN(B|N7dX0!O)dV%<R(5Gx85?f<M8I-bnchzXm{6}AsAw=*A!JtFVq-cE"
    "YwE1BfjT@0??NIC@Pok)qe<Pvpn5MHMVStmqr(WI!UEhGqPS9+^HzJ%E|)}OT}>9;%)Gna25|PrML2@=V)1d"
    "jCD5Urx64V)(^8(t5+Qs3P0F+gq2*!me=)=*Fmr})?AF|W)-QNo$!LjCA2b-ph6nfQg|a)taF@;6CK~`C|PN"
    "@RP=yZzk%}}6xqF$%1P22#3v^8O)Bo-(KJ65)K!+~SUHbT|A=a0z0T7hvRHx+l|c#+S<cD-efav=Gd{>HfMd"
    "GXNKtUW_vfvA->Lh2yYDR=`vl>&&qb?uh`r(Se#yx-ZULT>Z2*6jnm8$a($4(kH2&fZ$b>OWk_94Q!^>H6$M"
    "<zm^V{&x;kbqBil>Z@TV|Uo=aeQ%j{W6n+qCSI1}-Sbw46r?9vzHAO&C}ZHXdz9lm`%It0Y7BiRsm+W`B-EH"
    "I*ZrhbaQJZDg3#wmilZwG_XVdS)K<%ZtOCBD#OYtM(F6_qO8$F1~CZb!sP0W20nJs8v|C0u+J{j~s1`cOQl|"
    "p$6@$C6PkboZ~l8l|JZSDI^(`{xH7k@{T&|hK7A~VUd%~$DF0__iOym?=mgeN0Pih@2X3Ph1OV{fD<$Nis=e"
    "Htu`msDNh>(vl}!Y#C85tS=}$*#lyJ|3$rNQfi0Uf{<#Tz$#9k1{Kr7l;Od)ryWn1=uNG76u!EfrFfn0wq{W"
    "8>+?u4>uni{+^}d1MiDUBl6YLVg%EZQ#M$^VkmnpZ4++A&*zFjKxN;Iuu$A5>h2+Sw>)Y2l+l9XSdRAovpcE"
    "16*g|h^rxQ64_I$!jDHr`db1Eb%@Gncrp*O@}lB3wU0?g|ndopFKVg}V5HYu+$6Yl<~PDQ*oN6SG6LGP_zZd"
    "mb<TjX9M2ldW+uus7GLK-IDVh`}>N>NB1A^r43GdyP3Io#hWPr${8vutGeGlEKg9&x#fk!KF*}V(84(YqN$W"
    "8-8~AxLvfNQzIftRnZ<o>dL&@SYkoqhT|4(wkI+T%%&)T@{fu(yt708ulsqaFM-CQ4<CI?O$`$;$G?(VEun%"
    "^dZQ<}(BDp}tlTmo!UxeRS!0cT!;!PxpIraV_$DMHp&ME=N6*5<K3}^t1_G<62>3;zCF80WzqI3}XWTlVdhC"
    "=f#5qk+oH&T*0phZ-STEfz#;uXSQPX9~QNg8xFOTw{q>fYX>@&pD;p@0g-!r68r8*zrm#{PzjvjBEG)}x86;"
    "{~6t7y6*nO2jcZ{Hz!GuxqOo@)ENq9Z(X6(C9?U_cGcyU^^?q$V&plH0Z)+y)`q{&(I+b7ft}=H;qAdliI{v"
    "ISa}A=Qeqm&mN@t?H|2MZ;rAv5g|C{PX%4c^c?*7(N4qXB*bva!fmgsx5HtZ}m6Sfb5YO#A!Qhi7x}pS5|uF"
    "ZVUBhFUrm$mn@2f4KSY=+K>4y*ak)aDZ2GArf6t%&7%58@z;P>y41r8PUeDb)V!Pp@%%FRRFG+8ZUhD9)3`-"
    "P$7droy%kywo8PJ!an&r!JKgbg3^la!(Zg>Y`c_;+CiBX7g1eIL=o+I$cYLDGFs6Nxtg-O42B=WgKHLX@i8V"
    "%8LHP_&MY)uf8<xG7snd(!M`zL(u|8~#%*vLp(hD%;P^?vuscGMywPc-7RTxVje%$wJGZ++NzXxjS&Sa|AYz"
    "Mj2;&d_NAsgK#;7U%ngMm#YyC3ltq?37=<?Hk^^UDJYj4Wh?;uDDBf$hsS%P5R*s9;jawN|7rT$;A-U-kV-f"
    "VBO0{fQBE=3)BhX~!uBfl2;Q(bik&jFS?ex~+=HV1|WP(Dy@yT;dx_naC7|I44+EDe9Ze*UJP_aH$r#KoBfa"
    "on3QPLz8L>u}daLmr6}Yj)2V#Ktw_Y7O(~E9Um1`kT-xc>sYnpB)>L!9#1^;7bTg@*Qvy-QUykKhN(PRV|}O"
    "sH9$oe>Wfn-!b~mJ@r>ye`nXJz8$j_xM22LwjtJH$ry<`FAXp;Z@Va2+BQA{WEG{fb89Ex*fE|uB7bhkg1h-"
    "21LdXXFX$69*1F(qbG3%$xJk@%t)-miDmj7ZZ(lOYYdp)d>jC*Ivk}W_?<?)6fnL^NWEOAQU?;?S`Wz2<<&h"
    "0swU4!WnnsKQB7>>HsSpSAdgit?V{h3JfAidJLDUD+`5O+lqKj`fs;*wi^H7y)whB6@yS!#VzPqJySDnL3@!"
    "_B5r!AmHKZJ|sf(t(*~4<Wt9s%(EQaUluRxLlhPTc9ogmV!|Ni9oGEEJcmhSY&q&tzn`#$o<UHqoHDoT}?91"
    "caic!QIgzHtu-o3iipj6o)u??pmkn3%v47k#NeUaqZv9S_OiZFzS7{XA7&Gtyzll8?)pS|B0Vsr1Adku0UR("
    "|H>n%oq#X}5jKxTJs~s^i+V2=Z2;?UeJDkK2st&~vjhJw|f}2>5k>HE)Q#lo=QY3gh*zN4}?CAIez`tJ|L%o"
    "E&VneA~jGQ>E8GD%q@$A~|C%8;*9rUY3m}*)T(}6vZI>e3P%Bo4^4QL&kN)$m<-~jD3xa&-56GiNj6euKtKT"
    "*?@BtJNkPogVm3Z6+cKPGCL&7j6r3P*)eZwSHQF7UwA<OI0^&A3r*n{P%gql~XK-XIG5u$RFQ$cl*!V_MAIp"
    "4!6cwwqb7hPr`>3YR81L|RS-UXI*2Q@K=kIp2+tZ@ih<d)cdl<JY4j1;Kt<9&ND5@8ufy2$ylLrhLEP^9+!<"
    "9su4-&%#AG>k?S*JPfX4w%ZHp5gxh#&}nrF5neb052a3K+~va#tC`F-IPmu}TY_u_DS0UZ$>j-!xqIMw%OHX"
    "PHc%|CgOzq`>$>M3v;;dG^KrK)3h%!}C?lxYIGf=?>#H+d6snqJay7F%1iFBzbFAd-tq)fD<y~V&k7}N5w^`"
    "onAf89a-<!w=Nd@iMb`R+ru-uv1NR)$d<k1cZ8KjN9%-pjeL?TUYb=D=9Ux}RZ#P0;`^{~FIA;3QLce|7>oL"
    "ecoIq=+m2EPZK=YLDTNWtZ5eo=K5EpRDgX+RagVKr?KYEU>k?{N9PFsn#S5#0`Gj08HXe4_rqCTGDyM_5>Za"
    ")_~xP`NMoB!P0pN!+tTiIX|7#Jt42prmO<KSkw-7P*qIIY}r%MNhLx2Wi(kr)=A>auoB8sFbO5gvxy(^W*Q`"
    "!&XCzG!?VP@=H0TgWxi--kfUT&#*v_x3$S!5GV1(tTBgppryakW>A#dZk9$#(3qQIGfpxFfQYRh?kdmF{W^!"
    "ZjQmqxFz(Z3wNKA0-qLB>_?Avn6Kse9%r=<OvcjRlL2)<^Cle1quaAoHl5Sm<R4*Juh;t{WW^1zg+Y|zTNd)"
    ")#mohB4H%%8E++A{<foBm5QYaRKxeU<UPpqTn^@yUJAHo5eWv61TXHEY1x%y{6E`V|&3Ne`F<|Y(d-e}HEWw"
    ")^Gpnc8!)cB(8v3*Z|sd?>7&F_s)!wMX+P7$BjlA!%}k`^aZ0t)g6T>%Uvk06;F+hWF;p4ax}SokZ^hFzj}X"
    "o{{t&a;0T?$Tj#^00bU=tk~AD|YL*$I~VecGMJqD)J1bdLZtoz%|%s+yAd5jH!#PqyD7ice7R*fr?PVq}bFj"
    "!hn6Obzu>1+d+l@!{;*O%?DGXxTy+-;Joa*iXIAXs@is|G|7`$5={~apog8cTLatc9RE16L1=ve3!4bB5i0`"
    "vZ5#RvGu+$~oj5e)+bu!l5H}?M*9@WEhw3?Y;f_^Q4qkY;NP@8RcxD+viaj2~iZ~C%!VGdPZgktV1=+6fEqB"
    "T_5pd&p?>WV`g!RSE+zqZcDaGAZ28Cuq{p-0D`>To!DbRo&Rh;B5#7FUBRTT$VI~+z4oru$|`V~6_8}sMt8K"
    "q7Cx1oAIsQ1j`r{g64r%sazMAf(-E`QWjtf94@leK{N;dANxUrY=Pj?bKlZ9_5*Cm95dE}Mttj%o>3&$)X?%"
    "hUA?f^p}}Q~V)qJ&AZ#bWS3iI5>^lreJg$4tm5qeWxE(mxK$lH*8{s>W=fBhhzkTd4t7-a+(+G$bO3$`TZwd"
    ";4}s8r+V9X8vN>mXF;!97<+&86w=qsvl@q2K4UwLuF*&RzcqNo`FDK`$+{4lQ4S-i3LC+HltV`7KTY8x#UC2"
    "NMhbs_+whStZ+C54kcD3|NkMNV69_G+`KBuof&lX%d57Xa=?MgubOhllh`;qcZV9^1F>)0{>;dIKLrpe-V0}"
    "7AH`pwH$}ni`a+nJ~1nMfwAdD`PPo_-lDoy6=nKs~4(EVK)#{h7C3>nv1()HlHyJYFZ&S)I#7>5=Q@Z^FeRX"
    "t_j(o-yvDOFycf<cN$2Pwe-aTP;ldT>Re&osCS;GW-AmwFa}`<|Yli-aD5-C(H$G<^c6Vgzr>Xo|vx#)21$E"
    "4%a30a#d^5Zu6Rn1M|uL7NiT1IBTC3TWWZ(sdNZSJ|@{&tLr5d;W9p#gDxt);;dMx6)~kj@EKM?T+Kxm*xmv"
    "8$`Wz9OmEmWGk2RX$SWc5z?0lRBt8+ZjhUBe=`GG{W!aWtR(a^D)jzC8dz5a(Os|su$Xa-wfa)s3*vbXpxk7"
    "g7(j##-sJHOvst*p>+&?!QLyP_0BgJBSb*0^l;rwA%htRHK<7OUv<)!9aUAdW)zhc%(J?DSUj`f8ZQkF%t26"
    "{#WSrdn<>^y36)ZPmVMzT`^$bthG}XWu{U;VG&HcSUj$hbM%V~j6lW~%U>DVm)0rqc^gczXym#0tZs}1OK&("
    "!o*r{Q8#@vuLRzoQ{A1`6ZA@j7(5I5-~kG|7yk2(fjb(3lwjOj@d{3VWE6_nN$#X%NqoC1#e(j=ViS>cjt0V"
    "a8O0^$dGsn{Tp$a|)*-+hn<3zNE@+kv98;^hOaK89Rg#Gg?bg0L|G#Zsrae@bZL8!GOIF3F__f(YVF%a6)&1"
    "dK&9HnxXxqR;TA&P=+=K(fI-T5c-MN%S*($AIB8gg<F$_q(tKlCDwQ7-~^9kpr$i$XAJyAWeEcPI334rL5H4"
    "JWE=K5Fk{*QLnKbThm+o@9FoDMmIHbc6<F-q#+yY@{2eR2LxwX}T)5F{M!uDRipZnS1e~8K1qmD;oI@P|Km)"
    "`XK7IO%ID#=d$FMjYCewfVR2^WD2)Pk3(^EwWaz5TA%w~cLZPxKFCjqMTxF-QC^}xo>!|Vn|&c1zeC}nxF%D"
    "}hbL<6909$to4ghr*1)R~W=qZ3?u;GWM^9HeP-r&FqpFb@}~vE>7HrD6xXo|hV6Lv6q{Sha~XdGvdks94E3o"
    ";pB){d&~IE<%Z`f>?77{WxX-I^z0)6M&Oy3gCC_(5~mC$XQ8Pu<wrwvlxMgJPMp0&|zPlYpqB=-0q_o__=3="
    "k4S{xIEHj}Ab1cC>?_cS0jea=qVE<~y;p`Xqe7d%*K08M+yjIFkY+GCNM}w)F#WV8W0=xc=Q_%R#{#l>tc>D"
    "ue2C!o!y4uREe;MZPR}N<M+fJl$;HKMHMG-dFT)thmkfXCjN{SYM~CmQqUgcJ#puo3i`v%vKREddReZl)nSX"
    "Xbf<R=7Hd>^0EZIzRDwA&D{!gsL7!)N&owPu&pU2$A%ZRr#ff`5_(x~|LaZ4r8LdR4K3}^$KQ}nknl~HQr^*"
    "W6^$loYfq_3txq~A|D!Xk#c0)!!JB!LsylIoQXa;(U+f};}7bVaRE*5DU-A)ie?!x#GanKmX)e36>P|JM+Ad"
    "yyvp)UjmSP=p*>x?XEr1ygoNwuX2KvTRVIiz{RGBeeLUB55kZdf1ET0iUPAUCfSBs%MYSGAIJ8;1U~YwM^n9"
    "2VSbMoV%<`b{(X&#gv6eGK26<;P0~32syYxjwCPR8B1M=%gQWY#DS<bLQ-be;YuMDa^;B5ORSc2qZ6dQg&7q"
    "KSk_#Z6#pjVO&MF!MqQyx6z?ir(}*;cIZ28F0R=IO7W_PzEqHK@s*Emi;^G+M$YP{GB~J9e1l@bzF=y0y<ZT"
    "$Dgh(%v$L3_EI17M)oW&=o-IU(hfhI??lLxesUqP7=_=kk)xIlSjex;yO3I~{;y*oKMKKX4*z+0%I4Vr<0O^"
    "Fdsk%+Y^-~=IN6#r(-6$V4lXzis!Clog)<&P;?t6pE1uzy3pd^0L`>(~XdLgYi~4#BkPX)h*|fv~2X?x;Lsx"
    "IhigL{hpaAsG{^lx-uOH^5j*J2tFZY<PxqEt(Yg$7LAL`Box><Lz^75ALKv#e}-90jaDEp3NxKLero=He8y7"
    "mbv&zzyWWlba6z}HI$<Z<QM`JxZE||aTyRnMq$EbqjOF1uL8=&W(6>QEH$90hS@T6LvVQ6ba2aUmk6%y#0Y|"
    "NPRR!h6{#U=kHf2Q=upSAl3{qPqMWz$xLl=XtG|}>kSUHwV2YswbEf|5(f#BC2LD0TC5-brC_e)3VhUvGRFY"
    "j?k<@0l4XFzTAj^(<LZn?h*VFlB-xgQuOuZ8>A$D>jqwo`6OcU$`GdB!~lZJAeOO)BtJpvRLMN}IvPp&l3U4"
    ";e$wTF{*Cp<Y98_<>dZGt|k0|hL=0diy5ns%oF(2vg4kO$b0f-FyyJ8=rj6&4#go<>it0?;`E=XSD^%XXUr9"
    "eJk$smE$pehC*U-?B3h-&3xAY`-iBgbrvs;Sd_c_KpX@=TvUtk}PG{2hY8<7<Llv3X?s7-MK*Sp1IxQ3Ea5&"
    "v;UIE2^{#p0+6={spoKsW>7w-VWA7lO`3af0m!`$th&?Ofo<n>;LurxMX@a)oDQ{BFyd|lC+24Y!JFsNz~1M"
    ";<L%!KC1=@Thc@#^k~VAZ4i^E|)1*oPSXZ6RJFzjFPi5RwG+>a~!7@hQLeVu~7mm$#U>PJus|qJ9n1K80oT6"
    "JYsxllWRFH_GBsb7I+b2xEO`6>8Wk$wk8n1F9(a}mKVb?R1>ul8&Lz3Yvz;(5g-w`y~Dx)M5!A~Exw*b7``t"
    "jW)NYuq$0=8>Zt%z#UOo4u7^&>~36a~pk5u=`qB(q)$Q<KiGpJ$-eQJ-8^h=v&18*^&b#js+yHvqy*68`4~W"
    "-tTyC3e?h!>|<SRGnT7Q@K-r0upy1@<dqefY{D8G(2t#1vQH23V3dzmIM7%4)t`{4mI({`<Jf6JmBw&9?I+<"
    "YfwA={5y$&0tkFXLGQveDk9TxE)~DW$(<{AD_Yp`Y-zZ?D<jG7`*IENMz|1-AWiO2JHybS`_@vwPwi5dgsfy"
    "P?|@H4-G*YO4k5Wj<PuPQJ&}fbr}Yt$4cl-JirPk5yOYJeDeZn;?@;h{*LbLH;rvcmKVb%ZmbHdB{xEaymsw"
    "7huOOyyc)ZmL8eZ+Lj$5ZOGj@9=CFx?lnv(J#pqUAjmWKX{Y3q*bxz__ID+O?fs@F@<Xj<FP`ykAXV}KQFxx"
    "Od7wWJxWxx$tGQ2pV+jDs0j=wcOIf}1g>*n;{=xp)Q6ee_yK)wPYvD0%q8oGu&{Hw1ex=y*2}ey#2bLXTYt;"
    "DR&x@sz^y-TPmtEsj1xFb8fJ*OP&zr|I|ATUI^Dbmgg71LHn*jrrC}i??5E&E?Puw5(WTDy8ngZMfGnDR*Ll"
    "3*sxQl;RoInQAXA-~*z1QXVlHYB4z+`c5p;q0tun-VwlyD%Z0)GqtsTB{>{vIim{0C4V<2c!)YkYoP)@M3Bv"
    "ys3?iAbc*IFhGbT=Bu&?={Mj<ha0GhJ_N?8sxF$90J{sO*sAG?%LRG1C_@JcOm9?uHAykqzYG>j9Rm#gYX|o"
    "u&4#}G9lS>l&vd8nlnM72eqiQ}1NlA1RSLVE$llRgq;&=s4cFq*ljxef*u7mIbbM?X^e;Q71fM*QPILi>)BY"
    "O<d^c+OGZoFSF$Y9PrR>z!c8@m9&&mG2VjCHGeyGH6jKt!p1=U*#kXe3(%O+m)gh?h#sFA4+5pYp`OZ=lhvS"
    "03qchNKt`46VA>19U%-n}NC`ZWrdhnRPBcfH=t4fOCo=FpDi79w*9-r6|-dMH2%qQiWFr>d12{V4$+=B+YxE"
    "tj*1FY8xktDI*HFK9Hz76?FG*!Dppb4s@=C)9K)`va$~xnyxzp0Ui}WGNhconj!GQv>?EzCSr;g(ZSB3fOv>"
    "%0eR9BUyeH;=lFC9KH~88vG3F**aZ%SWKT>+p;H}|h07HdZIuI+34#Hj_zLt>6sg-Vu)Z2hb0|gF^?Kkt3sz"
    "llC(ktI0mzYY4id%ykWvw3xf5XCbAz;2h`dO81+&$9D^(QUXsHm3TDrh<5m8fU<bbe3C^ObE-lIu&p?*ER_#"
    "JTr3ej`G`-<*#@~s&~f=?-H6V}u08J%QPG4jfCEsP)%#Yxti1qHQev+|Vc%Z<V;*p3R+(ebNSqqEV;g>z)%<"
    "gQm<Zz4m`oS~RNiW1-qSr8iLaJ>xL3%TdG^em5HUZJ6UE0yDD^m=qL+R5l%uR@d+kR6qZuhA8lt5@g>`(+Dr"
    ")dBinW@?5qee)2aNw&D^nV2vFXMoT$43*$bge49u(I{G6CvQf_3YaO{xY_ax%$o8yTsL+_TyPL!Qxe4-ON0@"
    "NU{%gD^xz?T;EYTjT!rDf(q`?Lt4Du-dwe!JLKXm=(_9re5e1|BFpLS$WDS?#n**5}jQF*yMslW2SY1*R8?E"
    "sO?&Ac2n+O0CS!j00W;`2<KK0A06eYT+8LKjTnSyAvC|G4W6XYtqlL0rV2qJ*%<g`mZ1{0J)b4}2`04sm`{="
    "@`3TAAZSji?I`afib*?yik&iDbDfSH)2eRJbrpA7&0xR|TK%LL)qrnQGgpq%ISzWav`RQRgMenY)l~e5`|*_"
    "#-#m*xo2pNbnb6i<B?5P&mV3=B|0kC{#%ILYQ*#wc*uhutInUOj0JD-JFLD0z1eUR-%ijib*)9n=dF;mpZ>g"
    "<sQ|PU~X>fxE#;;JW+3iLxwkclC*wV0X&=u#paq$myAt=9}AtEc)*Zx16m?<xuCMqHeZ+H<}52>@)!}djx)0"
    "e0vM(0QW#vD7nZ#z*a!!P#BfB^+!wA_2#peRL_5I{L7w5uD)ceol(h8(G=vc|!42p?xXQ34u}W5XxMb#u$^h"
    "&Q*Mwg#lejlC+stYZX+_9`;zg3qG>6t8_Eu%EKuQP&JKkPG<8UvfluNi<S?UhU0;%NQu9JACT+?OqZC$=zFx"
    "lsMG|cmeJyS#c|2pYh<>B4-*{c34f)h$Nl46KLDWHd{d=(<@zA9Ql2Yn7$D2)cZlb&KUZvDsU@rg4m;1H)LD"
    "yWZ*hjjm-AI=N?uxi#fu@Fw=h@W?8eC}T#8X;qvfE1mi$sPFkd0U>S1ME!z3t5~k!>g3e+5Yz`n9sFAY>L)P"
    "h(AR%5VcCf7_yo``+9<qd&KFWM~Z+0tTVa?3bGwB9!`DB1d{UP-Rsw)Wk6q&xK$hSCILy=8sXH0XCvm{6<pV"
    "@F~JS`!rvh*kFmg*<q50ntKPoj$ku<XgNU%B){w15Rsrb84GLnGbnx!t^!ViPZ1e^s5=T=}h9*k!T<s$_Kz)"
    "g#r=O@WhFGG^#y)`^2-E3wB-0|GCx}mg;Q-$#WcrwiiN4|M?7f}~VJ5f3SsI`wh&HvWV8)72e;iLvPA9I)`|"
    "I(^(ecS|YN&dSPF_h#1Ic#ipCxyd2X$bObtcXlLMp!#mQxX7H*WbANQ?dY`MuQzHvQaQ{-9q{{A2fjZp2W{$"
    "_(4ZT0?oeu)G~cIrNs?l~S1fUbF9xQ1~Afz{GjXTx^w#zkjR`P%6us`nzMfSmnbmmCYfe+isqfi7h+bKN~~R"
    "e+|*MQWXXS3AwS2r3U?zZ$eXVO{QJBBOnjIbYs)~;ATPG^e<^U#ZQNsIvf4xJGvOgiBu#V*q~;}jpT*i8rxi"
    "9j#cg3Fle{{KdVTI?b5f=z3pG(nL!n<@<n7|JZlJPNNBJno2?+X^ym_qMM+Kx)ITZ6T)m9K4BmqasKtGqtdc"
    "04vEj`a*95qeZhsvU?@y_^;tUAlid+=DRY!D<G*<;cP!@YsW`e%K3Sa}j2i5(#HyDoYMoTAY2~I1#%bB#Q)0"
    "2^V^=^^n!FW)2>B7F=wK4P%n_v^x$*2i)tGFA>HM?Zm@7mPr{<sBKM&Sp(a~1O%dzcsz+T?oAm+GE^Yzo4V*"
    "F3f*vd$0Qj2eThNwo9g=IJ^k!0KelCnPq&%XnSH1ZNKQmt_$<cW{*u_6VA!t^W#B-RTmF+{3xbwymC&B7y_y"
    "8P}T?><@v~0zi_@XO6VF(UPl}mW$2A7y?d*p-fJF#gQU2H-Cckq7|n~W{#tcQW&V>0q!Xi3tnf*U7guu<K7U"
    "_4Gk(KZ&era|I-a4p3cxafyV-Dz&WBrxtRcLm!~)t-@$`}UopZnDaKvbhGa5S;I`f^hS@XMYKpa|Op4^Tlt+"
    "UoxROA`m|~VucbjdKUGk0MqDi!U$40G6a#6G)GhpG%q>7uXvg|{9TE0H<DsqeCbG#T#JHjwy&=DdEX<eWXhm"
    "-Z!F@u7Goy;@rOJBaX2&IGAo0w`iMAvn*N{+BiW`R}E*O-e@m?c=uqVOu@Z2e0DxD2V}CbCC@*<>MVYZ%UQ1"
    "2M49lVyO%CfYoH$STQOx+w49VBni_!G_>OH0&7fU*|dSJOia`t{t#7%M^xa=TmYV8<iES0OW1cv`XU!n@s&B"
    "q|~3=m#3$yKNn*v>cS+56FQT-e~g3|3t=J`d|BpO#gSAhG1knQFz5VSyBc5WVigXm<Uc4J$Q3SiwG<|HRUKA"
    "AB0&thnGwmpr`^w*!{q3y&Ro9Cyre>6qbw;%>+1eQhI`acxmH$qlWn{xtE)S#kl>E+?y6OiW#J`yT~<2H00<"
    "LV_rEZ9#8vSN5WbHVKVmI;yECXJGu*+s5F?r_b<lVC*TK2V&C0G&;%}bS@Ac@_g+goKcC#Tzx1O*iaM(D|2F"
    "qZBb+Ddw)>;X7WG@F+uzmKv?Xeql?cIwdfT69#5*DVKM<Wb$4P~<tptA8HYKKKhJ>O2RcFGJ~*F>>)TalWKW"
    "y_Net_>rMq6u>5Vd>o(M}m3YAY%}eQ@VcSEjL%~!1!9r3VO{Zm;mo;34xT5%k>RnZ0$fW;Jcb*qGK5EGwiLk"
    "DmCbJk?WKt4Kz1VdgPF;?iULT-6f0z#7gg_jzUd<<{Oi<A3hF^R>qk-x#Np44#8|I4$G~?TjDO&f#RKAl9^0"
    "jRS8X&wL+w=;421sZVHy}l=ifp)4Ta9WyaeE*6uUNMmJTNNf&h7TJ@)^YI9p?i#rva*JZjiq`XudP)CeaJgm"
    "X2+#x^aD>AL>m+6~kWsO@0hZo0x8<h#VIxP7f;oFX^hv@Al_ugMvWY-F8NA2*or(%@0*CrQfu^%sj);GD8oq"
    "+t@8p!J~JGS~_yACN@=WHof7D2Pa#X?PO^QlGKD1m$w7@rfA5gNPB0uhjAMy6N7lt6)`g89N2X#u`5PnLAS?"
    "%FtL02aVY#o_6jw-8o$^z7Hyr-y$X9X+GsIS2{qCr1Qxsah0?caB0{1`Kp2Aad?Srh>&=_q9|MjEf~ySpr}|"
    "APz(1`mfL)B8`wUgP4;V5VPx;W~uTJ?=&ljAf4JnhC!$d41@~V5-aayB!t;VL6Y^UXVz*rfj44Vs1BBECxON"
    "`X?y#a>sg4NSeZ#7A<D}BIJr#b8-$9$`4c8<31jMhe*gL-UDEbz&-c!*dWq)nEfj<>g9>8pXuz5PRta--%E9"
    "k-o-N1pV?48m=C_?rbk_$b!2|AGBWU>#A->i*mN2Vjklwhq);-)iM($Z>MQf;I-b6#U4;tbhaf=8tLkE5wN@"
    "OxrvjQEhqWTt?@2g<B8A`<Wn;06!WO6e6Ny{0Pw+%B}$kVPfX@!9Veu=6~vlH81veJHVp)*;ie25E#401Yd9"
    "i5(x#;vK1Iuwr5Fej?P3U8l{4p09!IvX90TT}NqJE!nENawv9ggp`%#EN*R<`cz2v*IEQveQ%%@C7A?B6}oQ"
    "K3)t`t+tBL&V9ytAK{MSC{>|5&LkR@gC>6A%&P)fgRdx)3N_c4%+<bNJgZTd<sXXP;E#SrBT_dhQbu{WL)_3"
    "xJPAL60?<N}^d`dFKcDFXk9lX;$z2AyG8ojut^v4|P3!SQ-3%F1rIlo88v^!uI2~Y~FUgf+&na*tVj1|VcBG"
    "G2Mh#I{S1TQ)7=U9u0O-$IBF3%hG@R?@D#^)vA}k$}OLnu(r?3S7OJdS=&^IS`kH}(=z#BZ1SqZ<PT~zD~(l"
    "Cgkf;9Y49gjxEc+rB23-_^nBs`>o`yB=wnqPGcT+1fvNnmeg-Kf`7Qm4cq)H;bo=<uB7=-M+!VI+8qP9%MxI"
    "F7-_$#O;!Ve<;s8<~|c0ZII>==sv0@|u8skWF0`1^)~|rxm0WO!TqfyabNxtJH{AL&i9#0x%_oIdJXa?1r$7"
    "1GG=)*8q~ck*NOwvFegb0$=$y2csTZs!W;{6j?`*+>@=SRv^Km>#U~MSS9q-y#mU<DzNNp0?ZyUvg^?zydDL"
    "rs=^G=DW{ZX^~EFryW1@#ae&uG^&jV_C(kZU-@JbICP;7Qpo=kTC8e(CVZ2fBoQvV4{RbjQW5F%?BgxD^fpf"
    "y(xI4)Gi^5Ck>n#*Qq$l|XPcuI_h&N7b9L{Be18+kQ1AGyIn1cb~hF9zS?@<Yd1jj(F<M59)FtN)BJ%>E7u3"
    "BW9c-9536|3nq$@;0rl+@|8@A3sKA44@3``c^LOUftIPemmvU#=!t9(HIx-Cy1Skn*a%O7$ZAG+c~Z{m=LrX"
    "o#*$m8}=>Xx!@O%N6`E57VJD6oKbkuGA1$r7a(lCm!|=-UgDaznFu5-*(0~ZrxpuTj-3ZUke6#`jN`FwPE2q"
    "mCqN^I=gNcp1}uUk=xP;s?;WJw~&`zc9=jHSDPQ>jVy9!kuO(L@mk9C*mW1Gp{gxdS<Iywu5;+ti9ComAB*%"
    ";qPebHpHYe>B0gxG4_|bPl?f#da0mvcJ{B?4IAMQySmMkb9{gafRLui|wpCrj7$<qMjM}$BwAKTdpbQm5I@M"
    "4;ufz}^!QBxBez?SnUC0`^Anfc0ioh`Yn6Fa|xSWP@m={h8<oIQR7))d6;&Qu1m$2OCfV7<~Pd_?h4YwG9R{"
    "43pUPapORk3Z5rom=X_Xg=x@LKUbI(AsU2^K&-<u!7Gpp#VHXjr^lFG0-WWcu)zUn;ab?GPc-i?jBpC-S*%z"
    "EP}ghc&E9y~32ZdO3`X$`X5RK0N<e|8aM{b{RsZD>}jbleZzhgI4p)=NrBG0##R$0w+MWIHJxM&&g6+S>C=v"
    "#Ekn9tpC{X+!@3N9L2}x_V$?2s?z3n>IR9KVIHX!%I;w_7flA+Z(FpA`83XnC-rgrTia*GT7_4E6N~-0MRKu"
    "EAZz-;DInPg5cX9Ys&#S~eg==-dOR>*JCmGIXoPI4z?CXF37GCXgNnI8WFyti!<JGnZ-KRjn|M<j!rav&N`k"
    "yo(KD1!q<J*-foHO6R3Ct~!2ChR(n=SPhDSTjim8vLmR&XrRtQvL2V(>#EbVwq9~jI3TK4(gxV6_+d-U_3uU"
    "9Vi#;woX`<HQRuU5mP_&=-~;i#FX>l`nGU3Gr?KI<wjaYe}NM&(?k$$v==k+0G(>vhPQ>K;_nMtXKcXxGsXx"
    "{ffe=?CkTnvQtZU3G@j?W%JCdIzh<>EeBo-gp3GP#G_w7ux*af$brKdmG89LMNn>SS<zBT;CdShP_K5E-$fq"
    "e|+)#>AMS4n8$%_`5VS}n*=w_9v=N^rDtR?0ND;$B6<mx?bZ8h3`~a42-ZAz%lB!rmT8kD+coI=?xwFcQ!K2"
    "OCt{gqMFQO^U<!+!`?*wM&%i1Nfe!%u0|o|D4L$<}GDjSXRe&Zx3{+3Ug&@%z3-Su{ebH%y)fd%);wX1^cj-"
    "A;Pav2F($>~-9!6-k2Qc@{$$#f85@Q}g7g~k=07pJ3Z5)V1*Hplyz?F&z_0_+nA(hO97H#<FaNMF-J3DHjZU"
    "mMnjr7|WTQUHx3yuIBx1OlaGJ-tJBmKoQr@^$lUi%(`-A~O+;rz?A+y*NnxXti~s+I_O<+j8hCSsT<yiyp?x"
    "b@_TI?@@E2!5iewR~YtFHER>Lv!Z`GXSg%jGs4a`oPl<IHk|n^2@k|e4kvgVQfAhm@X&O<;N{!0M=Xg*^kcN"
    "_6hKyO{188&cN_A|1wo={3mTeXQ1e>S+deH#*B<iz^p!kv-AUW5QERY^|9lAp{&~AargjEo`a(|qyBP^uEMh"
    "Bkrk=3@LjiFD|Wenu%)}~nM{LE5_DdBmZ0gkJ4Yf?VR%*qFdsBB0D@WNws(JdP-Eg?RE4=t2^EVuZ(kkh6p^"
    "y1obgK=xDW1n5VN=1ySvRpfU=+>xEOW|Mb)kBAGh!a9(RU++LLZK4h#@vpF!hfr4*#eb}DXD-d)#@=$X@Pr@"
    "X+VdwA<hKMhz;7Z=d65V+C2b(3b@2LA%Roc?x5Prg@AVijd&(E{+5pI^?4$KReU{{VcbjHG<R3#6P*Yl8nOe"
    "_o7RPoMrLmmD2DeQNrVGNkwK?Fj1C7J8fWZ};!*fa!e$-A$!GIR%?KX|hw-oX?*pe&6)-Bu{ql^J6)Uxl^l;"
    "U+W;vFLjW+>iF;S<G?0^Veo5=ctMz8=9lPInO+6(PcG(|q&KK79w_qx!E<#n?J-SsAeyI(zt1^rQV$3|K90p"
    "LN@)*W8<b!Q`(Xtm7_N<6Iyt?dPcwc}yusm(p@i^Nsxt%%$VFj%yN(DQ2!2gc`@qc;1MTfv2T^{#QRDTC=YS"
    "Rg6@(ltIa?(RWMyX2fT$ktUE(aCEFS2%0u%@dnpi^%0&B7yE8zB&*%(gzq`FUGiqR1yG|97-Q~n;zo!Y1=R$"
    "0%cD*FG)dl&Aukt<#FuT<odYqCSqa;MXisE3oImg%%^WXU7R-KWR$Awi-jAqD{k03~xg{_l6~N4*OmrS8n!S"
    "u^LXv_$}gLOpiv$G5*N(_7}0C7Mt8Qx`tc`B^YV-!bMHryT=gdSZ}njseA-+<0?f&ts;ErEU(yDyborFi8RZ"
    "hCtz!<f5S1Ynk30FrhG(Gey0d>cHM5_Zil$8|nZae3hOnA}fVR9-!&&k~N~aHfXZzwFl;Wf#DN?hAoL5I+)d"
    "m1S^A`USCblUZatL#cyyZL~rJ!o7eDqN)E@I<v;bAiApC3YiJvo4{*NHA%qwuK-XaNm?Q|V7zkBpkk=Fh7-2"
    "GV*$@-Zra|D)1F{1{S=*1`Z%>%G%Knz;IQpvU5^f)?C=S-npKlJ(0=os?Ba`b1H`h(FS|gn+RDs5FZ9qg+E("
    "KM#$_lKty5Mb&_7m5Cyn?jWf@dD0ut<G{%3x~O4Oh8~wn)%x45O;`O`-ep9gg4e_g9nilL_`C@;I${tc_gK1"
    "sD^xRBs9%PEv00$4VjaQwi?(2iOC&-#H;^v4O0rC4HtXVMdhud@n1o%(xgc^LJ5YR|KQ3a49{ZlIkxlgWBrC"
    "=qgYeeZUoX?6I(ZLmr(Y@rtV+*Lwb;ba)N8PbAAB*cVjvrBzE+yWw3GtmQSQXpe4+SCiMr@6N8LSJ%g9lj-s"
    "6>&fNxYI1UM{_0AM#h(rj_bO1RMdTG>`Fu)3C^==s7~E*;|B<W}YI3uNEdQ#p{Y}VQKzCq-MTlpB%|1hDw)h"
    "VS-@^y+zt1nG-%pOOuasg8ub;wO73h%ZocybH;WX!#y4+L?89tia$|;Za5&0n9$6$J<$RulSL(X4T*d_UAd*"
    "Ge$+@~tZ2I#}o*o?={$N6bD%4@0aID)NdQ~^X;5eb}~u?}a7jnZL|H@?^&ngCXccUU73&x}Qh0=`eby?9;`D"
    "ilHX?egXTmk@r(XRoO&{X!yxF3?<I9ueFNum2L}VSpCyf56e`aQOwf3GBQ3JLZEIVSF-{eEmp#hA;a9dGNeZ"
    "Rd>TR@}=(E9pI+IA;BfDTZSqe;1uj=mI{L#X&vT|7#)xln1`%L0RU&$2H*pT@JOiy7p^;8#(@frP4x=wmDTN"
    "$BhCiS&Vtcg<QYV=V>Oc3T&6wHin)xYt}2)rn#E&asLl$(Nj+GdFo|ewxK%c5*!<|DW?E5IFpv=eS&g=AxfG"
    "-hGJPXy=HyviZLmGT3$~VkgAvP2<}thyuU|W(i8LoW<G!qJN9lZIVR#*bn=d~yv+2$#wLR{InvC}PgYo>K;B"
    "h|U;x!3Fcnm*p){XAllwg8AMGqdXo)!Ts#-ih3W0ZN<p>+uw7-8Waq!K)^%>K4XZwVT1<!$RdJqoi0_L&|HK"
    "I@nXf~A>hbAgrO;_}txQhfij{lu%u)yY7wG5AZIoxVA}7BAJqk7xYz2F+VzEEvc3DdjB}TW*ix9j(`sCU%`h"
    "CU(V=<k1JiwK9(U7A{ll5hfINV#Cb98^<buF2<s#H-s3f*7~ESY?55PRSmyBSuE`axM*)BU=AbQRe@4gf1Hi"
    "gK38g<_`lXSJJUlSruL_U1azi}zsaUfqgi6=H2lLOzq2MQxs@*K+9T$v!^zS#>nRFNj2+qdIsw@TD5ih7zJ8"
    "0fjKSO&1j|DEgyZOoe3QyrygI)cz}`!-JS~*nfNUYav{IhQHB!2m+6IfeEKdb82@`#FlS0%J-Zl~^l|*o5lG"
    "kEUl{KWPrCBjhHSd$^_I@Dp^0ubZ0l>kKAyWf)6us0V@z*TRRf)t!o<Yx<qkaLdM<AA*BgP%Gc^mLZ=0Yxk+"
    "E?d|P`pAhWdI33dPb`Ij4aHj1$;pv`SzPQzHo&h-w<pF*zzPdf9k5)=BKUs_omA8Y(C<mQ|^yEO8c92E$~G$"
    "ZKmhtr6d<Z&warNrc+(}0nu@lI+#iQp-WS~-`anmq5%W<-x=SvT=PF5Btgu5VpZRAf)p=2zNL^4n<}@eN&Bv"
    "GlY)P7HGu)@fnG;_VS?)vHH>`>JN(2Nz~pWvX<d&Rb^_x#ZOzaAV%1%bjnG1Xveg9HFh+3e&o<#$1e?X~1*u"
    "$a8fAoy#mmEisw?#=TWwZTe2-ENzW(;>>C11vQDla}S2Ar5tkqFgLrP#Hd;E^R%7;7@|E%5%!ujRbhw92p_}"
    "h6g-Y`A<=8w}q{L{C=JpBlm5H(dubtqF3O<EL@id(PC0v)b|jMjQ0V;bASXP<pda8u<Fx879wz`q2u0@u!&#"
    "Gm3owZ-E}yL&YM_8ShNVN3@+{1{IQTI6|zMwJMK+J0Yx8+kC{M*e4T3lA#X;*76#;}<7T0|+QtDA1=tT#A3$"
    "cRuVRLK|7|QZWZz&?xJ&xX-GRa(^7a0Qjqp2pi<$(<mfHfF}qd*CTF2#noIQCqN))h4_4P0DL-x8M^xU>U#3"
    "#^5Wu}|Gqk#o=h&Ur>{>zA8>wpb$WSo@WsImxCmo9w#Jt51#&h&3{`Jy)?vwqd2=va(VKj4mU{$;sRUWE-ZW"
    "!>ZUVh6EjP{B>eJiPx0CQDSylE;uO>f2W}SB*ud8u<eaYf}Y|Wx~6$AR?*MLZCuSo_YI3oQEWqFJl#fW8GTY"
    "7+yM%micq{@L<QX3n_sqWEBXk0>Epb=(0de|YawRSIWSbY;na=b+mgwj!<E8Ss{Pius)X$&WnF%@hgb=m$@E"
    "zUU2>N}-!a9CM+h&RQ@0{Hq5xVWToDRDQ<4jqWU%I(CfHtLUaRuIplH{SH=-di-rag}6656_28lcu@*rC!U0"
    "J2kL3P@5agH_N50MxbW(l%)BmzPnoPk~(Rcsz<sH?jYG@ur{lk10Veh^y#3tVt##PX?x5jiLMVdz$5YY;_CG"
    "O(0Z)U-3KTvzeN+rVvxj;Q+$b1JWnb_pn?_*KQM4S0nXS5Jo!i7HpCAZ(SP`}MpvhQIX=65GZ5zQ>8sag{rI"
    "D50Myf`*QakLm($D1@!0?e{JnKhe6Sk9gKDg9p=0cW0}#3`q4#4?PaIta>Y19`_k2?Natj(C?{h%%7Q+M5Yl"
    "Dx%F~3J2k{{x-B`m{?z9H{Z=9?@}r;r{ZtDdd7_WL7D8VO_BYFq|dHU@+2r8x<V^N=bM!AYZOG}d>}4&XNR@"
    "%=&kPP~jjq4~ImHsFX39y^NHtYy*nb$C@Ln(F?rEG)e0#kNI(nmx*nwo=9_ZJa-ro`Sg^h`I3qQKAB*$60FG"
    "XCsb}bURA=9);Q!j)m5)zC@7VhmL4usW*tq$bq=tu4N!D_3)t)w-i&%LYhv=!#zb&IGt)IH;LD)Eo4AVK4nd"
    "hWo5fDo~2HP+f`XLUI27RP<e1Yqjd}*&)S3n0kEAWRVwmqUL`;w@HEr3mWxU@Z2`S^*Czufsz*lwJ7=kfCd0"
    "?~P*eFfF`Lni)1QC-`R6xpUcI{h;fFVGuCD%V`icL~%ehe@6760BU(vhk6NKt7;g{ac!Eb;5?drGb^>2Up?V"
    "I0T{hQViOq@aG8caFnG5yTnI>Kzs>7D%4`vx*O%y(ISsejRLk9`uj%pahR#k=bhGi9!~lkN<@p?yizWljXZ*"
    "1~vQZmQurPcUbJuM46cAD{x9a=efb1$NPoVJ`1wZgVT6Yvwf|`f#O!>D*Ave_>oO9EdDSsDV)dT40Ic_C_cG"
    "R6P=J6XN9q*rX~hvSREIKn(zx0k>%whUz+IK66I5;VdSA2ZCYd(v)2O&aR7S@vKs*kEKKw5pi)P>ZC~L<)^1"
    "cx))`hD|B}<U#MGwC<Qj<{*IG<GeY4Yg4NL)CrA0IU&G9SdpRiH!dh`0__%s3_lO-f4(PY7UsKSKlwW0!H=n"
    "9Wev8ZV$b!t`a+?Aius#x$&PJHV(P#u1S7q4odYl5yECi78Ew$81b@~VOfLXUFp!a$v$@3C&mT`&i^FS(#Y~"
    "TBhw@b4~O<`5^$iro_BT>Y_KWuX|DN-~v#F#R4D_NKMz0WP06kAVku^uF(t9epjTD|j2R+tAk^R^DtzYx5M("
    "20CG@@LUf$>MQd{Q}p=H=2^at#@-^n#9!Lrt0Zu`Yi*=%Y(-f7D?>vA>IQ(Ss{hj7;hX^l;w4EFoJ9%SOJyq"
    "PB71aQvo~<@u!>&C;TdB-F}IWh-tocNVpjo4```n6wI3EfgpEEd>|}03BD}YF!8=>zWYcoO%u>j&d!c?6gJE"
    "wx>#82&E9f&9yuT3?%>CN0X!G&JJ7^~;>MemsUPrEI?~6%`}ZFxZ_o3*;-|52GeVeT+)-G-s06MQ=M@2<#JS"
    "q*??OOWZ|%h}x+;DUNAOP0Ute6F0B;q{h}BCDsJ%Ns{_*(q?D+e$NlOa~CW%$v%Xu5KwW=LCmsB|-ZF^%;k}"
    "6;y@twSr`wW63cDlzQdYQ#&Iz&_iu0%+1Xv0*02J0;<Li=A(#P;g`Fa9PYH=gi8cA2<Ge2ydX`d?6B_3Hk21"
    "pB}#lL%}^81qAt=fC(fEhY77pGx}jyedCpb)>z&)fbxmFDN?jbJ1B1N-bD?8XQX+`Y--s6mU=b)>IbTA<%#f"
    "@k85s6nv{ATl-&>wlomt7F#&?zp#3=ix8In_}^gPXA8hxm#LAvN;Df)705#7f0GM`KZ2Czz^xf4)<z2u+WUe"
    "*U8?5$nOg-k9Kmxj`WLb~-yBR5<R;TL7u#T%R|Q}x%GuPb2z<9#-Ij{S!k{F1tX&_#Z3;P=w*`*@Y%Wcm?sM"
    "%gRoO(SkEMwgn2c8fr5py#kq{?k%7>tNi@#d;f#xJ}w{5%ix>~HK6m)ly1uA++?pJUHm*UOITMp0<#5yYq%4"
    "uC!Wm7K7Tzq}_^&f_Z-wt2?5e?R}5-S<4AxojGhRGTag%qRm7&|NS?MR%L=*%973&@WdK>dd`q5-fb@w{GC*"
    "_xBK0VU2gBDoQ9TO~!qX&)w(gF=m0&v)2sKX3NwhUX5dp<3pcDE1gU0QqQ?xoND_YbXyjPZl4+)wNHE|INwU"
    "6Pz9^ct4#&(04lZgZ4Jn^J<kqZ5~a7VrPrmP!|#@zA0r_Dl{=CKiX~G6~wszFlYw=WJ?Gqq8j-d@Zmu&#HX7"
    "B6`|(Nw}i64J&eY!oUkY^@PXNnys}io1d^e>pjr;X)>>E9QWyVH_m7P<R?cYmu^QaoAc4lp2dXbo_(hC;cPw"
    "61=`v2F^KHaI&R-J-8?$0}K=Rj%T6V`Ju>|ZCe3Z4UA;*sLUZY_I8{DnliJxN--PX^}h?g}4XQp!?*MnTmun"
    "o-(H4qpj?eG6^xoJ9{4uc=CITfh9ipFo~^nv>_G()4Rz~^{40c{5q&OZY7ijw?q4ve6VHpW_8vQFk%o;AQ3d"
    "~g)v9IRVW=$sbI5^p4hjb7!Qn!^TC?5Hw`xH-7Dd-d|>Aohwc?d)`l2c^iG(XDKHk<aau)Udx8WQA7Il=W)b"
    "D0U6SKz#n<btw|fiAb7F!3iNU#Y2g|9utR%K@8ouLceYsu<VdQT!(y$v<NW8w^}cFnn#B;%&fNIk44#N_Ko_"
    "5SypL$^w9;rDvPoyi)?{pJ8{TCKypL{r~IW&<Nkv6ck%g4IGy|`&nR$so@}bRj6{-K##yZ~1tw&14H$PN$v4"
    "G1xrOksB$v6UWfKfGc`?Rbhj`(H{qDsII5aj(4?fB)J&^!rg&{*y&&9up-+vvwC-fq=yv-v2ir>#?ruT>%ci"
    "h5#JiK>24^&_3fe-OpZO<DGU~pUiNqb5GB49t^MAFKNZJRW!k)jxVi=W+IhaLP9u1jr4>4s3cm`U5O%jt#;c"
    "WF9})~Q$j1!JJ!C$*g5Ki#T?`lGN`xleV7FSY$+Br(W4wvu(7+)AH&aG)U<te3@<vDQ=9Htil<^d7%&KswO0"
    "ND6`;8R81@J5+8AgO@enRq}ukw{NGv|Km5il{pyEeIRZcccAt45vL+v@efoIQe6&*9)BKH@)m5!DxyRTthOg"
    "VPR_5VKV6<)PrRuje(x%IIErEjV1MbC(LZ{t<Wuj^OqUo2Lb~5?gBQdAO)zFuC_p16A{!p=Xt3T<Su+7=r{V"
    "+?7uyc7I;!O1J-2uC!C9w}Wt`nhUaGMb>8{${YtvEv{-IM2XJRntz&I0~a<$?e_c<y1|Fug&gLAu9!{TmJe5"
    "AENu$WFY_?yE&ecNySC~T{<mj?d^9aIKdj-*<|o5fwOn@T3DXpe1U>UU#$X)A2Ux*4R;w||)a;rHKmb}=G_x"
    "KF#p5KP>_v-PaMM)f++nx0xN0|#xyvekv|L4#rCRO^)25*c*^%3H@@_~_S4q*9(ikK8SL?W5RyeQ_`IaxFb9"
    "D274!t+oJAdZIe2KyPKgB9L6H>L_Jyt4wf>>)(yVOABVFHqAJu<l6STACHQt->s^5hHAId4_70nf5C6`f$$s"
    "s>x<dgQ4<(cmMJnDCB?1mz5LeS>SlGQI2s98)oQ51nvN0w9z72hHGPQ^R!WD62!^s~vSK3vl=(d?+m6Yh*q="
    "CgQ0-<|oYh9rGwz{UgQu+Lu3P9~zptEpna!bVsjU3Wd%FJvEOBRgq7U61P=<~@E>F@|Z3Tr;(irGh+eNX}gF"
    "(1L%7ZkJW*SNDe!a#32cze0f4Z@qnRoZrj=e{fZKA8ijv=s@1PetWKiwr*$d_|^luUpUl^>-akRFJny4_$Ak"
    "H^V`SlF7~bAnSD+>o}`_M;;3E%@lsTV;QrdT49VPw1*9bzgHn#iU}?US3cf&i1WP&c)4v9tQYP|MA5cLb4Jx"
    "IfJLwKSL56-iQEPze}jD=tq?SSjpl$A(`ibA`Mc9xy5Z&Zr1Q+qb^TZS&snqz4m-&Ys?bXfa<@Vo}Gne5aZF"
    "#%JSo8J&C|r*NEkc`9?<%N!F;8r|};mBnXZM8?&)o6*sWBBJF^8ySpJ!kaSD7w;rl%SJSDPEZ?c`Efw$SPu;"
    "%$j~O`nL3X}(OiKlcux@2OWaqZ<u13Wuj@)pCduMFbx#nwIbc!foZN%*gE4O3?K&m;~vz=F2%5<?<UOSWbP}"
    "~c&eL^B2H)d8KZOB&Ci?WixN|6aC3AB>`Kn5O7QpOLcGCsQrdo)pxP6&AbvaX)X@~@&fM-B=W&~X*iN+?pO0"
    "aM)^yibPzemMNohu?uXLh^q`SJ~wM(?wYW9ZW8ljRFp$2}h`!fvB?EyVw=%3jTfMq`g5y+S!5+_q&tbfG})J"
    "<@IP;$kkk?ko-}L)ds5@LpA2VZ<1W3*-92QPV<3*4wjfcQi3{;>67)}tEN|PR=rQ+EAa`nuTMx&4qfYuue$G"
    "|KD$_Fdz<nT1P1;z5KswE{Cl>>FX!<>UmRq+Q#zA);dB!+IIKSri*q<SbZVvTK!U!(q52`ygP&TkQqb&Npeb"
    "%6w#`*>pifXh)A_BItE6bMh4Uk_i7mFyZ?s%tr#e61XLUy2aQoHV=+vpXXtH)!Fx?douv7JllWFlwb?L_N3%"
    "wLyuM<dXu-?q`Y@s74LH8k;H&UTZz5F7JKAH$@SQr8-Cv&4OS6x+^;-mOYHp_|%l#Z<~V_L6sQ_p-d2VVJe3"
    "D#FO)|iYia#w@A?QyxLBT&V8?RD=|-k;pjfz>Qhs#$$R5C<xvF0V`nr_AH|WUxCJ=kf2tirKoGdP3bEo9?>o"
    "Gy7M#-m9>+UcZC52^I!d;r}eJtnPjv7LESPfij{9i4c&u8zOSi8iep-&mwjyE0em>@2FrMV3IU16P!w|yBrU"
    "J*mhP_tM)a#dHU#mibZNvI=<TNMftF6*HY_R;yQ3sw%IWI9@~e1e0|uxk7zLXemvslQp{!Z03>NIao2u*s5c"
    "IMEh+-2Of?C90ILRgwLiczboLyvXi|7@x6qCCEs+yn=@(7`oF~?_Q+t%_*;$fmES{5`dM++5-3wj`#-$QepE"
    "d99%|S;Y#`ZFIIAmt*01?kbIxKq+KG?Z#d`1e=WvqD+{|_|<2Rk@+Uk-vh{)6x6+YC4hQc`ftdzK%NOU9`Mj"
    "on~cB=2|H={*O`Lg#D9V5d|&Z@Ub*BlrgIT7Zw{KD#PPn1ilIYAg;J*1^5{=dtjSJ<9RqGV`R#wnhc+llK{i"
    "qeCpYLJNQR7)GRk@kqQ7fBg2F!}xcwrUOhK;)UqF6u&badm(;*Xn(Rf>c*r(@cYdLH&YlDFjO;s{|E7v9<Sf"
    "2AM|*^FE8O2)f@eT)wFK*4k0OeJNx;>dVQn+EMz%|EPwwV=(kUoMV<HW<S!Wb&L;c}pM|0A*fD)F5+Ty9n}d"
    "x;wLfMF1F|EzPclk6JBS08jL_88uI@loQbPlGV^erXroU9aQmDzi8oSN`4tzmh{4<|mH-+sQUy9YLLO8&Yc+"
    "}givyE@Cj7r^phR_!j%g+ASZf$Q|*cnG-5{d0(t1M3{YOn<yq2w0E{a=32@+y8epo-{5YqsOJ;u3%{#RtCr&"
    "?3gw)rbghK|uH;1&!&y2i8Z&J+ZyR(Ldo1r%G%i5*9;oIQklXwpvnmWNF`C1rLDmt@y=e4Go|g*?6_8T<jyx"
    "jau6DXngepQO3P=Z{aS}5ysu|v$bfwwtI@O+yOM>l!hv#z$*f-eCM*xr-<uWG?bj3Kqc>7ij%WbMtGY#(Z8a"
    "v=*e7O1Mq?(xJ`Fsy-i8J^IhPpdjnrP5GQ4^%x+%+C!Tl>q@AsfC70VqXkQj{kKGh+bQsC?soTio>$1!t&#I"
    "@ppsX;Omj!l5jIDn4<fo=e#y1DcvYNwD;<`5+4}=)gNAy-%G0kOh+uV&`4q#!bzrH?HC#jAQ?0)Fw;UT7W0;"
    "Sxzx;gm2;M_VM{@da3Pt)Ot-~BJGEw_dm_H(Pi5L}xK2;0v<RcH>BVx8wY(N+y<kZem;RH!tuCIUOO{lFX;v"
    "V6!vypG~DN5V|*=bMA7%o(-`OCMDuiel=coI{!X_m|KKPM;79SiPrM#uV?M5;^Z50c52>Dw2IitzJ&N0i-;B"
    "fvN@8fp9gqIG|O0M>M)(+0S^BzSxffS0foaCGnM;v`!q1wgMbpOw9uN!;0!Dd0=z(Ey~q8myY%1g#dVQDBFz"
    "}fP>Gcgz2`+U$sK{?YAVKrr|=h!vSVjzyc4xj?CIiDno#O^j#KZ$l=HuO#9W+>xIrmf!OM%0$@e5l$fQm!H`"
    "};H)B?bCDm=3KzzOPE)~eWSj*!0bO>XdG?}8^<VqqJd7i8ikr0zYj>#`fMr3N-7wNjpisl8$!%d~h6?B>mR2"
    "EA|LJ{87iX3TKRu4cMk^m2GB`ZIVULYYf5;#!ew75(jbf|e=rWvb$QU`P1U7itDM4?JYfNWZ4m`L4}L<UoHS"
    "&AgZ;ut2LiMCUoW1bGU|9~?ZPg)8jdL+@0Nk{<?&Qx(NqJV@m>gdG_98^pnfBEv8-+%Ms{-v|K@%84Rzdy+0"
    "_s$K;I)i#zw%~19*Og4_63UNE<#Jh8jm?<v`hQ}fEj(wLhi9EsCTO#%a)k5PY8Kv*5}OlaefaxbYuN9S3Q~W"
    "@&LFZIN}<8O^726ezte2PoOXBF?H&J`e7ZUKvK!E2<JI-sLDk37i$U&7(7|!gXXk@@eh3%FVN@^fq=~?Prn%"
    "=YXqt7ciT^x@Ha`xl!_6V*d-(dM2Kw0c>kvo=1-<P$+J$WwmsN7Ra=h*R{b5tIc_d2+XLupY+8Ld(+7#@L_d"
    "eblJ-uaU2w#5vPou-p;pip(d35x0ua&^$c5WAI1*}IYcoi-PwT>Cxwj#}xut*;zm^A7R!`JWvG3R_M?^g%O@"
    "Za!!>ss~g?5x2ev+~*w!;1)q@;lWIh5{gzf4BH&I|j7J(#5(&zzRSM104x<JUdQTytfrjpA!{Y(K;rrf#4pV"
    "U4qWhzr$2rhJhXlpX5@Lukt4||EfC<A)l*U;qknwP(eHlNm_YwW5gEx<?nUo2xz`u4usO40}*Puom!g>ge<-"
    "x+Mea(V**>$`%g%nA6{$i*&-KsL1hCYx;~zzQ8K{yHwRqpj;sA(mzFmN-=prrr9I&JFLq!o1Py6m+*q_{$2r"
    "0VYgWHl+$Er7p{$_5ende>_Crgj=OOL45xClHhy34`jHz~J))1ZN@%cfp_3OjKR$QK?94~px5F6-cuc0piM3"
    "0Fj?GS-_1iATk0SDpvU9dr`d#ZPO4pmSJijiSDe9*(iU70Pw#k5f#hongu-^kbSe6W7TJD--0rQg<_kj1wl{"
    "N!cyu6Voe4?nb~Uxg2CEJ*gg$WmvpOYQ2S_oHG!<4B_&7tChZ#aNi0w;_q?4jy|2gx!rfbs^`ESv~62yB80#"
    "7F)f8^R&L3k_-eeyU~j3revMcc1e7B(yGo1rIQen*U-{?o%7+k%+m14y@btkH9B(9g-Bfd9c>Eto%Y8L#QP6"
    ">K!v#}-Xm&D=-(+%albE6xtHvZEz5j}zL2D<l5NN0g#POIX80!UdTjCJr(;g_^DBQik|9}1b)RtyeJ}}vEzf"
    "BW(Abiy(RM6H(udf=(vN$KlmlFkc;|7#ez{=cwg0a5ez1jc2k|cs4{!h?A}tzq#xcnl{UUYbY6WZOBGZr>gd"
    "!?kHdJ8ol2TL;_gizWaiu;EAPsGTi{J1s5%pM|2e2R?v+&t$!aN{T{6wa#Ft<VJwE<y92wy$mK87&q9gMlBM"
    "Jhjau;+Gm-$V%Pd<AA$2XB(*e78BsyMi?ik?t$56Z}?%G`OVfNK7D1K!wN1TSf7LR2HMBmd9*C-r{9hY*y;h"
    "RZ~rh&8mxf<w<dvtCJrlZ;q!wPA;!bFU|$36~+g7g;5Z4@&1BviV7uz#~_(9*MJHaxIM7n&PCv{?t701K)w9"
    "!<;B~JtH~>r7wQkCz#pGnpZ++Zf=~EQb>YX!<>~96PtX5?*YscJxs%D;Yi?Bip)P)Zc5(7ozQ+GnmtG&Ap79"
    "m>r@C-*e10-fm((AwoA;`Gh<@O!%j4JA_#FOAwFpU^KjY8%FWvL2tJA-ntDf^8`k8m<=jxmJ-=?`APbWWdd+"
    "^V_hT`Oh<MY2vuBMlhe|<N(Qg;QvK6T{G;2i6)XLO8CE+@zO?ey<GzWL<h&D*nyeop<hb23{#!wMGAK^N`nI"
    "gj$W8RC<(<I^{)7xZs+>D9$KEnoPD{&8|lH^V>NH(!={WQUxdpParuJ|l#1l@*I@4O$*uesg?vJ-Nhds|1pW"
    ")rE_<lgs1l3%aCY0qWw_+vCeiHLmqKsVelpTD$9%8@-nm@{PM4<t^Q_1WOQ0f#HLCkXS3TtnySqm2J=f9)Mw"
    "s$ffVf2P|w2^+QBv!WuKCH=Ds}{h(@|<836ekH5cocP*?+%2$7VH&CTHp%x`oTnC0p;05Cn6?dkbMwMV-qq$"
    "3<%p!<kqM*c(4%XZ?>0}_xp<3V~fjZua39ZTsLi|jxMqwK6xc-f5G1CCtLg8%AGXV`4MqrU}=g<SnRmTvq1!"
    "Wa=S@(5Srkh2>_#0BzfeWpWq6Qb4T&<gJG(zG)?whR1rNiu8gNlrxcl&pna?c4UoBmB{HY}1aQwB>OIZ~ga%"
    "nyaErVdcH9hmMOUrB?M)ECR2fb_Rx)op<Ms9a+Ga;WVM=!|<wih!!B++e0;SwX@EmhPZ?F!S3iKxI8lsVv+S"
    "@|<R(29Vvuo#eHX)#B>nC$c1gyYE(-<*pYv&&oN}Bf(^$T;4(<57-~@G{(PpB`3febPwd}x3u)I%pLKhV<H`"
    "Q)?n4af4eI2L*U5*%dm25bqV$;QUd}27TPnJDxd;Z*?fcPksK3LrB_7)BMB#bl8b7S(`KTYt{Yx|*mb$c6Xd"
    "~}LqCWU4?o;rE$~R8=*4RxKP8I>Ip4Y7F1M8KUc$VA-6d2n%C)_IW3v~DxG8LPF}*F5JY37x@`+q53poYSdh"
    "5Le@R4!~z=tCg=+3TCYQlaC|8fJog|YSSe7s`gclPxntKFY;ifC*LqIXu-)26fs)e{Q4x_gY27rT1vj|`;KW"
    "vQ%UT@A(@_I15Z7II4DmPSB<N4<pd7gGvzm(|uPBB3&R@K(s`xR-4z%FYj{%Li}p!tq>J<!TM(4VTuBrt@xl"
    "G|ZlowBL}kuClw&KvP;d8)T8KGcWtf68d%pL+hb+zEx`^wv+f%Yz>OPBTur`6i<NGhv^<Fe^qy6Dbd}}#5<7"
    "ne>@VatN_z@35AAP!s0^-wTK!-S%@C71eByAWGSo{<AZnxG@%l)64h{1a1kyxockjaCG5WdvV&<Z0pr!{lR#"
    "ZpvIaMRu7~}MBplh*#1RJPIdzEC3D&IO_i_b5<$Mc8$T5Gl7O#izz8SuJUaRK7y$4;XRmqneAfk}HsSbfk7V"
    "-g)f#&7_BAcLv%(r5deB@H7sx4{>yam|6BB?4=Fv=w!oqW1wDDY`i36YUYmd;CAEFc7gUq=RDWQHJZc00uq9"
    "Tui+FF}IDu1UUgSsOg1yQNs_r`fHn8}C>^N6B=)^`vELevEyLBTz{dmqlu6&AYy70zwX)Qytn1*KCJ*C|KK7"
    "O3EB^ZmLZ&;KkIGbarN{_Po(t%mFzYh#9s5|3-CW=8V`3RcTmI^Q_<$r8~I=Er;}0_k6*FI;lUR>WT+GDYyK"
    ">fhvMVziORhBz~eLs+I5+=sA)HL}V!}w7<bVO0Y->A?1)N)#wilT54pT(^}=Fhz=MSH!7I3Q+;S#GFn4Ws3o"
    "7mR#lS9nyD^DtFW9+woI^^?J}Y!_=_;UQX;C2i{#{a46oE1q$NJWN(2B=g$Ok_I7jGI>Vr%*kd@(N(<Po^+p"
    "!faLRoHaA?KyaE_9U%eT${T07cBYX<zm~-C_NBOGiz!bQG*qV9JEm&SckcmcpN@1mxY^TK~M*h`|}1e5gTVF"
    "9>vjQcWJUtuCY9OJ{(Wn_Rb(fdv8ka4z!_GQc$lxB3XODOC~ar>slQ?uzLczT%{QQgpXk0b!Bm-L>&)`?8|h"
    "cWqShB@B8->~|-t8aFP%DW{5XyjAk58#T*d%Y90oguBoz#vd)KzI{ak#9v3?s!ag`IVsfcd~S>tc1Y-@cP!?"
    ">urpvv@El3k*I1GKQr>6s0p%<%7U;d>W608|*<>+|R@4N$R7k(T_wi9pPa)AcPz9Fildjmm!wq1p5AdP%Iok"
    "FUHHcm?%2qNdV$Uh{cO`{PsyqYEB_+ghx7`8RT?T*BaV6JzvH<h)sa9Q}X}6vCt#faS&hScddt1p{PtK{j(c"
    "ldtp}MEFhQrixz87}I0NT;8m^e=HH{dM73dwx_UtRp9SJwCH(8sCQCnsoqNvb^?<a1NM(m{KhKv)dc1FXoTk"
    "y;f3xCzWX9O=f##*<w^|2C+U)`0r8yk$+bAtqj&^TgiYEN@E`spt>cL-GZ)99;{lUoq=-Ei1^bv@V8IZEv>6"
    "ES`uZ9h{8&n#XEa$JR*nqt)@@s;k`{*3Y!M)2IaPF5TKwi0X1vEg~*L&&I96$a0xAloAp-6&N2Cj%6y-&01<"
    "`b>L;VM32~Rr2^Lj`_F0tR6-+hoZe@3S#4QNv#%impMX)K*HyU&Hc0kzvi7Whtz_Mli;q+AH`b&^bG+ngBwk"
    "4>6%5H>3-;n~n>)~K(9lp$Vdc(N8x4*cc#Wli2zq0ZgMhC=2t@aj>*1i|TV)~L9Q@y&I?dAmK$>}({s+ZY`z"
    "9eN{%Nho9q;a`(tYi9*s4KTs3>oIANjPdKl#Y#iV`UX;+pE)O<v23D!a`(bOYoyLtep61$Gdkx7od1gV=0J4"
    "8;^c15{SjC>b+os(_sRvza*Z@IjrEZ-<9}9De<$Aq@MUhHH3inAO8t0(D~}hw${U-mGPXs~76R<(NmTAOou8"
    "3@9s*a)>N<B=2+qGm)3K^?;np)W>0oA5269>}@!|xF!>8Bqr!i0ax#E$WP>3tbkJxxgCDrkVPNdbSqU&K;`3"
    "wr9(<fEs^P9K#pnJ1R!Q$af40*fRS1x`2%>_6DY>HSrz8E8Zgl#I?1sT=|Y-Og-4n6eh+m6zj_+TxSje&Y%e"
    "?{ALUe?6J0mrtWM?qj`nuD5@gX_qOTyfP;2@3)@|D*;WePzPe^R<QwtcQtZtKnfx1rato1vVJUbGEI~a(oe?"
    "0@f=v=0^vZDQR3#}5*+PW(LrpmJZ8i}*@-KNR%Sh<|N3&3_EKK3pCY9l`UBw7?+I+qtuN&0C()Y^>LXnC5C7"
    "S((B>yk-2t4%>d_t&9?YpG8gzZVv+zLzq)X*;w&pfp_`XVep>yi(g9(l$6hMRcDud}{RQ{F|~p-_jEMIaND7"
    "rC^~+Hr4N$!b!VYHv{*2SxwO=%StIaTNRs+e3+^#hl-$epadP>7<outiM_Yq@&(o&;*a31G}TT!^+Sb6(Qfi"
    "Ts`x!HB!JNe2#@8BoY8;>>Iaw-{<oI3!4DWn;9r#=mH*^W1OLTAXZJ7rY3q#VU)VpT{{kyR{ZZ!XIXb8L4;9"
    "2yf4bpm{?&FhbW9M}X!BNBdXEwI()k?qYh9`iTF28W*-pVdtf__OSvrMi>@;C#`oM{dPIZi}t8?rq((`yLo2"
    "mM-<(W$6$=2+=dHG3Tb-g^<PVvuuo`Z0ec?|w)c}j=Wyb=8A;_|OoZ;wwV)3+CArzfYAtAS-mC)M_<NY?dTs"
    "pcI@n@{I_O^*~sr1U%ZcgX*p{;kw2sOF~Z=lc#?#ya+5b<`+mPUoR62p!`##X`(}S!bXB6psb0G_7-X;IS8W"
    ">7-FpuJWMvS2`DCE$(~nqa*$7%>n-4obl;AQn-u`uz)^=*s(^SiI8=M#p>~2z<p^Bx}N_%{jh?Hf6GXI%Ic="
    "}%a<VL*sihEx4DXsIm)G}ve&W_Gkex%LSc(R#Ra7bZea;N5c93w+@dth;{*1Gp~p>M0F!hB@L$wzOiRye0Lj"
    ">G-6chuOXc`+#$23}GA!PnzB+>cAz)M`5zar~8vivlMHn#5x5e7C>=IIz0XlPl7hna>#w8?X=VeXt#M*Ah9I"
    "t%*PzV%kX#kGQsk?bueiV835yG(ZDk;)o0lv#Z27?8Fwp3XzZ%<#*7mUP}lq%3TT5q4%08|C+Nm?!-wNq>PJ"
    "{Ph+P{lZU&Ki2;|Bc(zuRsc$vbO+Gw6`N8+nDeSWiwiROtY%T71XK7oWP|xEkD}qBdYF+Qr~*5d((Qr++>b("
    "wF~Kl0L1sB`q0rN=>fl>MePzWO|(5!RscC(<TXC+SJEk~+ffG-r%GthvIFiX(J`W0yuDZ^47a-8G>Taib61p"
    "9QvV`YG^FwM;GbhE$6z=}4FiDDVtdr~z!#5BO|${=qLjpaf;?hn)Bh6EwwKkdQ%xPwH4Lz_eW18FkG%pj*;3"
    "!+^3vl6M<M$0;<;o<zbif#<pcZEMlFSgk7*(pRIepw2858%2SEPRZ4c8dwpL-jIe1rNs#eC16nDRbt+vqM)A"
    "zquWAni+6$0UzY3o1_+>qPAf>d=)g}zIj$MgdGVW+d6er9eQ!2k|ui@KR6wqDj#|FCOYGNYsYdj2y1F0J%YO"
    "?p=Ktn?mA69`hR-K+pu+b@HxXSLgo#Cxm@*Sg1irmm&GuRp{SrXA7h>jneveWYO5a+jP@_xwYoMoa#2I&a@7"
    "dX}b@M76V}hUbBrW-R5PJBwv4X=<T34F4N`pKtL_Yi)knxv>(Wva=`^&{?F+mv&+WgGL_MRDnJu*{&??dyB1"
    "W23<C=Y0v{8vrVbMCgM@&R>6k5w<y)|;_aErWmf3VcQ&d*pN}<!-cxDQG1=EWJli2P*~^iGAsgfWdRLrr8#{"
    "mb<drlgWB#vqH9&Ze=|5p<-A0aW6MLlccf31Y-1iv&^sA0*@8Vc91II}(pSWX<o-|N%fEz*m`$_v&?NXzMjr"
    "c?Ol&5WH-e({3o*%}TTLXiqa`I`FR=}&u)eN%xq)`QJyhvh9Ti9#aM^1=cD<1*_E$f~58PwOBK881sD2zS!-"
    "5@>Z`oMATj3_~@9~9O~?t|y;`BFod#_HxohI-1P4I%|)(NzImkxw5X(_-+E9#y!DVYg|cG+KJ#O;)6ZJyGv1"
    "%rsL9PjG02;f*e;%od%V+v~_C*H)1-JdbNJGov<B!FmWoLAX;KpJ58z@Z#;{+#0u$&1i)gn~*aDHmholx(nW"
    "!8NwJK$O(WOw6&@HBuWcTR&XlJ6hJcr<~dw6ZZr}^ctK$*I6jsYSPL>M>MXU)YDn;!rSnm?0%9!KgZ-JX#FF"
    "T}9=`h%0C~kS`_y-E6i`#xCK&=V6s&aQ#KP2|a<$5k8VhR@g8n@m642u=Su^Gas(Ap@X!23oXD9`u7+kW2HI"
    "QJcJdOyEx<RYx7J*q17%^4c?)EZ<9@Hau273T{d+1Zt4#&F?#E~u@Dv*%X5r9zOQ*1^W{se1uIQ<xBj7FTLB"
    "6}c!o{q@kEG9J0f#+ls4hiDBARfs*R;+wH3s_*nP|{iQw}v5|m(87q!ctz>b6o>c7h*tn30-wYXiaVsL_R>!"
    "SA{x}@eX!fo>aFm=~jm@`NkpaF&QzE5bE?LqNf(3iGi_uCTsfQ4i6-|zMTq>{euO2T|(JaKQpUcnW=$5&R1r"
    "PEF7DEhRoh;ImZ7Tgsc?4v>V5M9QXuTobEFQ()v$p1R9(AY4ExA$84=(HX&Doofb4kz9jHHk4R+*ZdPeR<C_"
    "B&ydW<Mu_q?ze1>Q?W5N(dr?GmFljiI8PL|QQDsvfC&QzRYgsI0*R=M^scXDwlE+=o#j!!03`J^r%>Yk-$Vc"
    "dv7?tS4^D~S{-smljc(%`>#3-Gs5*rqV-y?#sQ+Y~lvBc`m;JHI&o;a?8Yuv`232$>ZbIbp#sXieN4us;>$+"
    "MHw(S`-;21TO8y7-%vaa^edhH89IGfhGr-bHYa&r=3u{h@Yrz?vG~#4rJzP!jb@#iS843vXY|5ES<7Kq83=x"
    "b^|R6L~vYl6SYZWo#Dlq-53H=IG>PCUCy^Welu~;kVGuy11))^L!l2D1@&s<-3#bTy4QMG-p6j0SGxh13>Z_"
    "=UAiYI-}9X2^?BRY!0n-B>l!?TKw@QY81p{63Qm8wmyGJVd8T|_KC~;l|6^3z6C4bYAT$?CT$0*@QJ&je68j"
    "MNnvPf_VWr2@J=MxhWsc}$+6k;IH>H@lX2xuw)Ed-Ri%^LfwJ61)T=|U&%8$&gFbueA5npArnXF+SJnF05HQ"
    "t9tr-^<SNTXV^dP-_71-7E08(&{(H}D}KY}r-biHG_K8GUUwTDZw0+p1L&(d#+#d*leV7j^83-Fiyv0v<<yZ"
    "Qt26D!#)ZYJ$gw05G1VTQ#60lYI*po;mV&(Blk6NV@0ugl1xr<HaHDE7wd^twF%hU+jRdSBnmv$`5y2I!BNg"
    "na|vjzX(Bq`d$bSf%^A>27!#wdi|fEF5&n`lqI~5slNOZ{_$n!s?w9qE6PG&?pQbWX6I>fm+`M4M4TmWNesA"
    "Y=iYLsVE6tag+Ag@+HaeMaViApdRY<_oYKCIPKOP3XzyKUEvD1+le2fPfLlfx{IroOIi!Tv;Sm0f87iPuQfP"
    "1`E2y7>sU@~nO4?|l^NsMaa?==n&^b^o6WyF;)**F=1s@wqg+Zgd+j$c?RqeTNvOFhs8=W%1lYMibDD%zes0"
    "1sb-2hCH%E5BnQp>D@;HS9VBv9BJq!~}c^T2bWK+?C+WY!>B!nE`&NkLvkuJgbwtxop=-=-Z1^d*i90ftlVc"
    "1>hUz7yL5%2e20F44wg!`YyG0$6+n>2s~&Y`s-#%-ch~M}ZAEw#^<MOx1c~T>EXd_-Im9Y>y?X6`hgo#ipu="
    "ZP6l&+-=dQ;)k4XjujB$@Zsu*S<1x?^_kM=>V;(?d;RyZub`%U!^R?g?Z4Nj#0ST?MkX|nOp5!A-HNy!6vpV"
    "g2w9o+jNM>X<9C5MJfcM)RgrE?VPbj21c%KXx{M!6kV}c(PFil}pwj0sKRqz!vDS(&5O{C0^BMSYA>G4nh^q"
    "QARb)!uUx1*Oo1DFpAl4Rlv?G^ErD&{3AXqJ7I%VJBt#robLK!CfE`!m+cY$}@bALVq1e$9jtHq>9ZCwqALh"
    "qK|6HEx*@#T@5aS6l+u-M_1+=VYnV$PLw9`Xp|+vlPHrzv<98cqVi6AF`DbW-YFl5)mUEi4AwxE;PgMlgil$"
    "&`wqaOhkd6RAbl48+^qF)uV3@ekBM7p@^jxn3<d|CD6JZG|3Wo>2YP0Wob-0T(qMxXL=EOq({WIS=_oeQR~7"
    "ooVZqCd@Ln`F7f!#z}n}I*m!$8?$8cW7iyHZn3-!Ee}2)Gv{oQ9@CSDI?@qzvU(`>5fd2x&-AuFvNE<=5!7+"
    "zE7P;r*8TVwSk?7Z$KBSf8-A%lWHl9YBYRzm;Bcm{msvq+^m9&tAAb$aq+NP{yT-rMf_f&dbwWAn?G!3Cp;H"
    "|o<;ZA#2LUuthO|0ux8|$qaN?TPqH9uNBo0#V4d3Ia#jLZm_D-YD5j*t7#@zA@JA1K272T(qd+}LyoH)y}>M"
    "(_^v=#PyMy)rKm@2He0vrddq?#<ww%pXf2&l|T*;Q2ADXiV<2zPQS2SM`Wz{`>xL<-$o_1;l;bzJbQ#M0}6D"
    "&0!dLCemTw-j8rNERl>#_FZ;<eKD4>qk46I5^*|TE~7s)C?HkW&IM~&PO5LNq>YlIpfXuI=_XF_;bQf`9Fsb"
    "x>LuGxj%*vS!d%TYG%9Q+VZ75Bw2&o$zBzDUx5WMt&su2m6F9a*ivuHJQwpSlgoUol`5z*3ud%d=fMCln+;M"
    "Q63o&!t({@zKF)Bp1R=ws3)!IldivtxNCmU*sA6gZHR<i+^`2<~$|}u@q}qx_qPB1%Ze_8_3W;!!&Elgfx7$"
    "m^qDtyJvC7h+)tH95Ad%jm*pPv7hMOO<B88Yez5%fwh&16$ltO;8EE61Q65Eht0j?bSwA+nDPF+yLWhFb@{#"
    "c_K%MyTA+WY!2C>HfNIpoNqhjf&s;`mCqa;6+9nY1oP&Xypt5?AHf^t1OuEgn+w|6ZJ)dy;5fwzP3^4zKk0l"
    "NfVPNrFA3BY(X{#Tkn<_s&5O>5B?DSID&#hO!HXg5%P!O*E==Dc8Nj(c#zO*+=3r2q+o@x;?IFNAjXOH@xoT"
    "*0;cbf91%fr*jQ_2iLC>)T|`eMUY227ybSF4+9KkerPWvNris{cd?ag>$DKnDi%pm7C=yGK?!UpDT$8(>K8S"
    "8Jb&MJB;X};SI&B*WLd9gsotM~Zo>Z76luRPJd1v3dmPqzvK;{Qz=3Ni$22m1q929$v1b0du`mY8r%Z}0<mW"
    ";3)~4A+tO4bX0oPq^!}l!YUREGRj}zSLk@;L0)0rp>h3lG`dBjh=&F*Dkc)$1yONEU9p8+Y`1!U9}w^)f4s#"
    "f$Mz{)VZ;Seq>IbtoEqJ%jCN$iZcLjN)GlrvFZeMNXh2s~63Rbc<>E3^ebTA8;fALaAftO^)NCs`03M=!Kv&"
    "lnH254@zm`2e#V*(fG3%Xg}m2JZ0ypa~+ZC?6DT?$JO^97P}uyq7}Zd3ZXn?L<m?9T4&Wr4IC+iMrPkC{bpy"
    "<6-t3k5)2$?XbI!kpx{so*oMK(-NnT?Ez2ceU>5*OEmhP<b4ONIk2Ou1DgbNM#(jQ-&pDNOwd_~hYHGbdgF`"
    "Lj+glLGe5948;Rq>9wt^$OmSdFmx`p=qWOnFN$|enm@D9BPtndjw?UsoPb$>ntR(T_%T%KojsSPDxO0)JO-c"
    "SoYs}Rk=UawR626M@vfPV4D?9$Ow-B~Iuok43p78NY9=3XXh<=&zA<DM|3?Z%~;IDFT%>-vZ3XIvi^V5HQH-"
    "U*{SL6cR$kv``fAWy2hSAlGO&p0?TsNLEm1R--t6XWcoNG?L(3FNW6*E_VQ3|mwVi{NhQ_Vhk0Dq8c0Xv11v"
    "SqctOJMV?WwlBQbf^v#QrXF)D_02=#lcdeXvHc|KWoh8lGlUKEGG2^;(S{lMhx$zu`+_pR)1-2M2^1s2Heui"
    "vXTxAhf?MY<q|8-W;s*xSw*40q@iFE9a^5okPgIsmMGb`-a`0pHpSFzkk`B->KRJ@jOg<dQ6yDWJ~){*n*x}"
    "0Q&#0+sb&Z<K0UuWeKm=FoC4fI=Nf0cbo`LVX<cOy4Z%g$qap+B*-Pf7-p)=ex84?uyQ(b8O%0o`jv>M!_Ya"
    "))4^>&*VnCRPrGQv6SxUR&F0fbMOR*_b=*2n6UT2>u3tm#{0L*|N7JV^uqWti;_6)&xbwnbCcOwZ6FD&``!V"
    "jDeL__r3w!+X7M7MW~N&{%_WLCi-am{pH0nP)UL(YDli{`G9Qmpe$t!39x%QkY<4VXewouTeWy4en8f#;^}B"
    "x49z>^QZoyWQgh<dn6{p*jor4L1c7%E3$%sLonA7Pw9pAoRFeG{9aVa0cgoLU{<BfWUi9Vep(J!5}lhg5j08"
    "C}+3;?qm)v0h+C3d1LK|tfuoOtHq{h%FW_V5)mh%0#srHuNyAkf)zLr5$7aEsAztgrQd1w3^m*(L?eLXfxyC"
    "`NFvRJz0_@`azJ_}>Xie*I;~8_#rer(pf)~E^jfx|673_BBwb`N@n$Ej$<AV_oX3EZjS^cxO39*uIxBPl{C$"
    "(;*>cOGk@Q@PPC$-sN8;vync<+k505Nry`qw3h3Tt(IsqA6!HXzJWN{KZU=3KjsV<1b&@mE^Q^0^hP&H7`zS"
    "E!<BHeXV3b+9#;0_2;RS~0A^3nF^XdgFG?gFze%{_Cmu1a)UXIRj?kyQ;5$Qjy2GuvVSVlOaGdBxdZ1C*DD+"
    "s1Y|F{+WQ?}iyegLbPy35}v;G11op7^E>(js*loT7fcl(J-J#%s3!CJ8}&j=c~&6o(5K*6iS8Ev!_NGnLG<2"
    "sT3jdA&03o@Fs-b>+=qK!-I2kfTWbNY8V>>AphsJO*T(u+M6o-r0q>vH_1HDYS{hCMW}V(O)(_8P35M;Dl4g"
    "7GrQ8{E_^2*y=Cd_@RPEj5Gyc6)l_W)7*+L`B{grkqZVbd-I0RY1#6YeP?Ka-u9Nl~E?gjE<UTY`xX<`NV^$"
    "2X=^^1M(-4!E?A~N;2iP6#C#C!=`bwPScW=QdcYY1oXDD32jL@Ke<kRUp{vZl5;hwLxP%gOgr>sbOp!1babv"
    "1t=vF+0bM(R7_u7R$gz|V4;YVZd<uf5zZvSc9E4tvrx1)Gs=I+>v-VM%=8cSww>7ga;q7P&4eRwWB*+x687B"
    "ovf39bkn~_Df58BM3m;_WEEj)ifa}V`(CVWuWy3+#Mw?I3W*;jN;Kamzc|I-sU9tfoE16E7Bkj?E*URy(SjY"
    "u4*1DOOr++rhM+bHj9NMkMF9nx?}llhC@rczzuJoPRs1mfEyd?iYzuoku3@c=wtRGR1$-WDa(BD08up;%*MY"
    "hIvn1v-`C`Fq;lb)<O=kAYtVpgD->cZfKbN?>jvP^7!9Z!7qV*)wcY)s&@0xWSGtMqV5WpjpoGzvqg*}DplX"
    "_Ta(#zGJ%bLhn}lcP<l@bn(`(WXVomAP(1NPvxmpjg(ZYe9RI3~qvB<Ah#T!eCE%_iTxy&(-6c-A^)kdujba"
    "zxq_u3wH<tTInINFktT;U%JLC}*7y6_?gzAt^CyypX5D5ounDM)1M!fR+%&xy)BraJ{7Jld0+BIT?Z^*7N@D"
    "bPwIMT>d$-W<@$cypjhl$xrC50D7DIoK50-#0RL!%_TmJ%^OLovqp}4*s=>H1?R3mz`-ZJ5yh}(qB4KVB82v"
    "+p|y1<n7LqLB6uR%A!93_y06>iH-=hz)K==bimwolLd7X23bK%$B*8+9Y4o^rglOIz>yPF?RwVV*_1Bg?e;w"
    "yFi3)<KsIzg{xEg&R=RHX5E4G3EB@%0{gxZAQ}5uXB+g&?Bxp@b4PiI!^4tgHv><tQWw?e}uGsK%`;_QDDB3"
    "4X4{67w?zc{@kL_L~I~UPDD~2^Z`xI-rbI0NKOuRMx5X=feE{WsUP$1g(3y_!g4z;(yQ~L~cAM}nd%RI%R$m"
    "kjy2+Wv3?htglTj`<bm?G$U<`9Mw32v_=D5brE=*?qvo#jw*fP~5;>U<8AWvlxmFKZ+M1w^Q#i*|_KW6AZ_T"
    "OW~E>E?d95pYuBT|!!8ZTkUkivKoFTwZ(S$bxl;p93vDHf~9wQ664e26<UG+RjutEnv5~Os%|5v=_#Aov|MN"
    "{p2sF=i>Cuo5`!w<LgNn7JkMH2ErHMysk4KjU_Y6wJx65VzUNTg`yD@5S?!tpV*eaXrbJ(A3?)QXpG32r|J%"
    "qr@${|<gR?8=~oWCBnLo+S@q20n@Wf+>5vhI5H)Rn?JZZKc%h*zXI3w=b)FTnKN1mq8B{|?(>kOAwnB1HCe+"
    "Nv79@xw8V<$}KLgy0joYovcUoXJ3e~6vM;pA$_&M4-<p4o!cf-K)D55*jUh(8ejC>q~NOPpnA#$5kFMIR7rR"
    "(I#oc=5~c|Ni|e;JjkQo30SIMCBVTm-0Yqy+}LPlAgRMaH@2ZTZ=M_iw5%@FFT7*pi^aL9zK(sANOM;#_67c"
    "MYB>EeRqMt0+rEfIx|H{9dG`=7g!1los}j*><ldNfboGUOg?XAp$aGvRZaDsmf?z>T^mI%%7M;rL<<>d>H~K"
    "4u>VNcOy;l+vDpWS{}c|s5%lZX;fqURjP8~*jI~nB)luBMy;x80L5>ZFF7l2ZCGzV_ON4q^kT=8xx;VS2Lbo"
    "{_G*vZKzw2DIq2s1z>yN`Tl5I!^UxlMLG0;)vWl(L@lD};7;*Im#d_NFA6m^eY|m*an!%*%E!a6H=#1W#-)e"
    "4;q`E#6$5cBcrL3%ha#bFm|9t%e<Wx&q%O<d>+-F0&b7(h~6kBwj4OMuu&P#nV`jylvcDY^US@Dr+R_(U1JT"
    "JP|IAMU*3G%cq5Q>r1kfokrNLr%A1ez=*n;bFCdI~4KuBdVFr4f0(i8=zP5IDZgeASJYS^_5-ec-@}SL)Ibx"
    "*$9TL`itkSNK4VXNqZ!;T*MKwj?MZlRByD)|;qkk@7D(Ga#rET#{3_p{#9N^_Uqc8=74Y)L(~pAKCt)`Z9$@"
    "mFU-^7seU}_WFm@D-6(EX(tDS6>)U11XQIhpChg*$I+8lr>|d6E+^;LcBYn^db=$IO43}CB7urLf2yUjtG?p"
    "cL#hvyyhFv$nS+PYK=63=$a>(!@%ZOqF7IUyZoPa{XZJv|eVeT9#%6$AdBLRiFEoeOx5KiKL)FS!RzoImPG^"
    "EpGY&T585UGWCM*osaZG?w85x&Pz}oFcI$v~nSCZmjSMgzG8a9|&=@5r8Lp(%3TdEV4ktz;hSg^yW^!CX4#;"
    "%e>Gl0NI3NvBw&?+^-72q?6Qw;G4wy}Y$(g|!5|0LHb1z_s1+F)9QV|ZvDG}ysVkxpwVbhD;R+_Ab)(9!Bj7"
    "FAgjawyN1m@Cq!q*jm|HPYT&%NF*8lR(WKpzq4;QM5)3Ty7l$YSI9imjHK6tO7;fMp*b>{%7JxmfdNqnC}vV"
    "ersI(^~u7}Zx5!`Zo-y|ED!<mDK-Uw@NNsJZXoz90V(dc*OY+8TS6buvS%dGEGV-x8z}q(x)`7o5JE+5!;0D"
    "O5NrVTS2%owxFEN;wh;l+W{Il>7G3zsCl_zto=t#T05Nre>7gR^B<~T=2}1$X=r=`F^Bby4FEBG+rTin-19j"
    "w$O8+K@4JFDXc|LGKHR@za5DH@MCGEOk8?*fQQA(^Xr@2$rfjWvr6maDRN&p8j!tzP6l#m^U4;w$Kbc9)g{I"
    "1U}o%l`lWz^~e1Y?SyCv=f23L|R{DDql#s?xN(BXt75g*4>Y{BvHJ&tcy^hrxaBkj1t)3hN80dk9?(wLiq-u"
    "FMv)Hn?@YLM^(-jwX5O8ln({PV-P0x&-!gq#aEeYTedRHiE5vT5HLbJ&(la=L7M41SPS0bqU;%y>J2b`yI1t"
    "nNk~CE>nM1M4#=q<n9NRuY&SbfEU4kS{k{X`X7h}KCp?h2sE`@<{;9OU|}uDtrpWTS<UwaJsxp%z5|sTg9d@"
    "_1A9h@X;#jGA1LS_OVdD@lB``)3He_nRiRXr975O%dAPYV*rramS!1P3FH+88G?$)a)*vqO1~zw9xw*w+_DV"
    "({xtaj$t@_iHU(La2Jlp}d4~K{aRow#eRP^8YZGlopyPo@oUs^7S#l_P0QMr>zu~`Ggg+JbJ4e<aV-D`yM3B"
    "?1IvTwZJ%<Cp=Huyq;!xv~f+%J7d+Ym}n`a$<Y9Dbey(p;jhPGa6Vto`;{tf)3^!PXIigDte5I*ld-=^Dj7&"
    "{|-j12k!f%5%!9B^#+qydg#sO;FrfBfphvJe8BBax|c_T2Q6p1@hu3!xmtcdKxf^!bbHfXQZj!G=y>$sWl=f"
    "gmLnY#LS457a&kx%z|x6B<0|Iw{M>H*}-Re_JF!UQzcMACm1I5vr-{07)K;9gYTu0;e>*Gi)GhvqtA>w74MU"
    "B7@HeMSv}?60Co?#h(J8i7wg;n=5T2%?wtgI?~+`M%33gkS2LYG`TCP!G0@vlw^=|L@@gc0O7f2ixMh8m*sN"
    "g~dJqqYC`(zf$Txrj=!=;?31&L?#{N;w5z$U{tp&E3)YeN$k2GHgZh3_U&M4zK6kagPb7^igpkI5dD^Dn{`L"
    ";#D1oO`Jdk(JSyCh$Fd!$@02^J9`yL1^f9&*tCdZy;3_{uC0t_0VQgJpCBW9#&70l~~GnS3O2EOksX?N?;L-"
    "C^#0x3h*@-fa(sdak3Ff$nUc-;pxCpMLz4(TiaxVLk`sK#9-buhlQ>b6`MAjBXv-Y$bDE;xI$~%F482?X?b-"
    "(X4T`+;G!FNEjNVRUI7goeJ`fqkllF8^FgpORK#-l%Sjxl3<q-qaCy!^6l+)9sk!Yx;tp-9ZEL?pcC5;BFP!"
    "<dL?1H+N<4dktq^sRDT|ltw<qj=*b%X&B1ALFY6|Q#29f0d4M3|y(J3bbY-8f?Hy)S0h1QBw6m1&eQX20W-a"
    "T0Dk<)^4M<7QYyBa~KeC#R`n2a17nc_o*SJCv=RgSpzG90{gA?$y5jEuRXy7zzY1V;1EH`la>^NVB=482ecm"
    "4L=wK)0V^z0RQu_zA$lpJt(euXaLue!BkwcAml+vvh1tTJHd+0Ix=fJbNbY7q2p4HT;2g)J#@d*qzKpiL3$2"
    "ig=?%b>v7DR1gyStIZc&WF1!m%(@`n}$*x1GU34MiNUb9=S4z)hbG9hA(kE^vr}PsUE$WZ3u_v3HW+dDN`+>"
    "sN_yod>j}_6+M=<-~#7sRXCrUHAYrNU2l+=Nd8B=(g<b{lSKwz4z49!@3OzLB9(w2B-If<5)5_l>XQ7%TvJ`"
    "sdPe-bnD1MME+{n$9DUMOAn?y%-MCtpP0`ApqJ#-ngQoG7F{3Ww)En*ne842oCaVFXAD94_96KTdeRl-Owtf"
    "7z_-T8o8zpfM1ITaBw}<rjwHiM5CiPoGngS1&rS#g2wQ!wp<qB-SDWC?r%W{rvfp3ZBW1rJDIQP;G42m+?+4"
    "fK|-bb*usGL)^4D7a0Lc{pdr(-M(Z5jmj$Qnah7*WZ&0M=0;GQx>;jVzU>a)}A=%W^|?o}ko@=$*WLaq@Mal"
    "i_1VISWyYuPw7r;p+pTRA`YGEZ52LnJob!y&}<{hZ#zcV2zrLQPps=f=QfN^@nC8TJXHsG)m@M>U>>#_ChEP"
    "AuXSh9FqdWKa?`81X%;j*^Q;NVFV^H=9bC4%IGw+q!=u@C48cC3n?klFd4W@Q)}R2kh3`7I(;9|B5jkFM)U_"
    "KH;O7(!hrgwNhPwT5B1f_4@?`=R=tDvNU;o;10o`oxn24@8_&*A%A$i=jjqWpX1QS*dJYzg_u=l6-80r#K}2"
    "k-Ii?d173?As#%K_NQCmkE2x7I$(dSifs(VcD!Xz;@U2UgFsa${NtQZA#!PXl<W|(8ab>a@OHfm^@;d8rez("
    "?C<+#Q@vGRp9WCtGCP9&Y9~+es@comSM|dW&OOwYM1BToKEv=n5|E-i(G##L^}@0yJ<=dP+vSj-i!`_6DD(Z"
    "ge!*We{q!uW(ios%-5owwZ&d=eoM{KgPlf?7CmqdZVG38kf#q_8TAC*}b$5?a9kNY5cB+!nT>~4GtBlVrcm$"
    "xuzyyBv5He3d9M3!{0vFFD_r5o*!TS?6>y%^lXBj*T9V7^mN_Do_`MHGBipuy?H6qOt)fs%;yv2RM&T#Cd)_"
    "Fs;Ok?@ipKr-iIUtXXj;e{7N-9fuuFy#G(l<#jIW=MVeI+2l7Cd45boo%zlp9m=?8;I=7or$WLpzz(7{ggqi"
    "}hg#Kou4rc|E0fPr0uo^fa<zlg^U;vTl9VK{hWSO*ScP5CChgUMUN5OOsyMGs)rW&L*T5pezNP{OR2&By$>>"
    "IieKv9}0K`^Y6hj8GyvH^TAoZmQo_(>IVa0&i)LpPIbt;q{^1Vp1Zq@_a?J3gU!LOqmK4RM|1wg44#LGbrLX"
    "CvejAB_Wx8P6czq3{JdKi5b}QIH*lPloL&eHICua>#JehK~HS&a*`ZHmQxshl3E)!)0Ytz_1;q(>7CCFRE;g"
    "^9b;@#R4Z|B>sYodWtVXNuMa7!nUDi?ZRnxWLZ^Qs51y$;3u&XZ_0V**++>6447s;pcF#_SJZ!*2m^VP0u_8"
    "yAg^9h-zkb%z!fD`h88@K@XHGROmaLLNkSxH6j60#nSBD5rb@0#B0a!9aLqtYCZUm1AeA}^#3i{NFnb$$EF3"
    "ySpf+JO9=K(yq(OCK$V*N-;6pFdVg=uJE6L{bR6&^=y^fG@y}r&KH~JnZz5j4s&nns)O5-l&FE|nu;MG6PF;"
    "IxbH=8~xY~5;%5xgkd5hA!!$OF(0*P9$&9SyQ)<y$z*I`aJTBr9ljLO)16xrby?rn0Yt{y0kQ?a^5SoHzaf)"
    "pY>vbRj!>veLBbflZ&4pu4^rbcQr=VWLgSCkazMWn4vs{7IIaLR+UxdB*qu5{>NA-JmqkMk)K{IvCVG*0ZTZ"
    "9eNG-P{I661snVB*AEbnt)lnt{QUI%FKvLH(Qdmoz}RSK`m&PsU07_4F}O!u7R*T)TV|ia<DwGiA{vFO8K<o"
    "_mQz;8G{MO69S8tALyE0&0cjKbu6$s#EL5TRCUHyZZiS$tpH8oTxOjIB2VGu1IG!-&OJXNcXsScDs8s$$AV@"
    "<qt7nv0B4T3~y{(m^&n<K+!3d?qmc|`WjuEC|<RCX4$bGKm1`0%S8@S?Va|9oQb}9w>GEMKu?vr^GeuDi<@{"
    "yRWvKoAy)AHkN2o~&pf+`hy$Ftj_1js%JsCrENTck+`?V(%)1ek@3(BMSMtlG-Pq8m}A-b0@%@khGmDvV!E^"
    "kjv6=zm9=rz@sd4h8P^rKlegwG!xC#Jnn#G*`v#aV-J~CyWJB27zK8!!Zz_661MYMSy&+8dsUj>#~|F(l^}3"
    "QDiGEQsrI;Z40;UfVf%)(S2eQn3<@M&ev0$0uF>?D<JV*=dI`i1RNgVYw7S-Avyvb531%H=8aM@4XyPM7bfM"
    "Yh1+w5kq0vOdZ<B=SCv&0r2`%D$?o_Q0xf+n_d<&kA@E&>J_R7v(=2!mirNL!I-R^k-QwhH=zjnsc_#4qs^o"
    "1_rO2UEE_tN~o^4i{jVm#T==82bv!1cDu2SN|!;lNG4Th-Cb*7DlUd(9lW{OI%1i@%lbP{!uTbPK-M1FCg9k"
    ";!$Oi)%=&^2LcULA^;-4YcPgMAygkGDbx2a%W>)9538hA%@^Q~Nt2->J&dLCW$-4V2czRn8L0v5J#fFy55Q$"
    "ex0_V+g!Z4>%`>)?cBNk_t$z*H!r`TO|>Csb@TiF+YwC;D<CrI2FK}=D0TrtcV~gi1`_|yno6!C8lqel!paF"
    "nmhy^gNQ~u>bE(n)iB1$ee8EDBP8Y}@79d)tO`^kWbFUrk;-yO#rKof7nei~%8Hmp);=I7DZ_nQ5UUkWttQg"
    "%PIcpODyQuPJKu5y^@-rlQFTpz63zo=zNyqQo3>YNFH?b0kt!IXW)&i%BqOH?2n8;q;hDM2GfA5I7L%qy8Z#"
    "8+`0VOJCnWW3UOAtXUgI%+K8sjUTw^cexYizDriDqtA)7fVU}&$Z5&~Wx7!DsLK81s)*k|~VgVekw_^N8pYm"
    "?mbEh}VpHkgADUCv;WT9s!+4;(~}U|@0OlCHor`n<+Bl2h_y7*<l;s@Of1ZRt>;qH#<qtxV5{C!6*J#!hRP6"
    "rB$A;`_6UlfO=0y?A|mdNz4Qjxl^`Bzq|m|5r-deA|Y>SEn@|9Go~zSd+4j;V?296bz?Yd3H>P&Mri2!+@HP"
    ">&HvBR4cHLi<q=JK+8h`pem!}MXX)~XH!k|8wLBFHGhBeP)3%GmHWc{nh(%AS4VOvS9t51w7WoJL|ivI3(u>"
    "yw<{@cV5j<xcxg=|ORL{jM@t*Ee(gt=ov}3hI=Pqe6mzBqb4m)TY)YRGs&0~s3!g<fA<><cWcdb|V82glX^8"
    "GxcpVYu$}zC1;*kBGf4C}=b$wShT@T0U3!jPW<fAO?gT>^1ptpwojynEEN?a>>BRnl@n<5!8c8J9=!kpCxyp"
    "&eivcc0l)*d-xz3SrNeiW}Qw=mVnhLZt|T?%jXvS@>-m4OC>8O^-o2O_y+mlT=Q+y0Rbov@i6YYfS+6F!4s|"
    "6W|7E`fn_EsMcGN2TOek~PA8Xc@nCf{{KX_yv{_#)r&YGKE<T(-dnF3B7G#lZA%@bE}dInXRcj0SAHx>Y#|V"
    "=SeIoVU{#nA?c|@e=Vyz!xNEDJFo>~Ef%FY5V+Je>E|PHl5AiI7*IkpjNJK9^%od}qnw|Z-=LN;P9kzm14S!"
    "CM=Z*BD795LOuHLvP7M$mH4aUHL$1ZqvBEwlT4oq~5aAy6HaIx(cxO-5rJe@j4~)B^AEPnFP7f4laxCnqSg*"
    "_RX>lMj7Kt=ikGW&)$n$*}J(V5Z!RM*QdrarilxAkiKX-S%{UUuz{5tkzq=Qf`)Qy@jwDkrh^jmJ|7{2`o3H"
    "4lt;@vt?#F#$l>5QuX`7?l!7KE*l_pa4PzPvg3+-%p9r(!gnLPT>q{c<Ee<4<2&AN_AJKdv&+F-OnDU*&dQC"
    "RKV0BUf!$LS1d^My@8GfE}We;{4*;%sJ}o+pgnB)(f!8$p>G5%+_mliYaLh$}AnQq^~2pzcDM~@>ri1B;E;i"
    "sSXTw5gIvb`DQ4EKh|iSU~mQbsF3LZ1nX@j>slwd6=gFLufarxDjEYSi&f+JL=Q?-l7qZd1+c|(?j~Ek+%%v"
    "<u-i9JBjWs7>@CLy`PN{<u-Z#)1_k2KcqXe8h1QEA5QsH*IT>CapI%K~9obd($UuP58rD_8vt7ScVWL}R%g%"
    "JLj8P0Elg9SI<ICDF1S{_VSMf2+^H__$tJA;y_4MqlZ{@!)=wPQpERd5yz7A6;3Obu{x?XfqX_63blr;xsrK"
    "ghWul)Z;w$af4&l7AsvRJ+y$j@A?C|^ZP^vywk1ghh-k)N8LEEXm5L*E>1n&t41fsyr0yaM1vYB8r;Y$7#s4"
    "3fhKqU1Es0z+2l>W%@J4R8frt*~txw2#PNJsgTuHqvtatSjV5#J|@uN2L_Eu%0omoyd)jv|mSsB6T?&E7PJi"
    "UkEi>9jW5#aV3@2she`W#)#c61pi-@-GkdP0b_E`N$iw%H{*#RToP;ais;WPwW8xzI<9KCPbiS0(|n1E#fU!"
    ">U~7*9s1pM`gUOo8re}qi@v58kK>#tzjlw)&Z3dArWg|nLJD6?8pa`rtG<b=pO+6MmPA~$I+4h`eyeZFmzO{"
    "`^{8p?45)V~42ZK$BGnZIil0+tX#u*?e5qB5ops`xx3|+In@z7;HhDzobe@Dnr6I*>VikGSp$0&m+WI{s~`p"
    "2BZC>nr;x~NH^N1W<?Sll=kNCcV+be-rI>tL8#e`aN%5sxl!7{r02h=c(ZvI+C;kP1*}=LgF{U8f1RpV~%%+"
    "VC5?&&r(Z4EE{NrJx|jcWKdpcpwxxgi*NU9EcgPG(&nOz3f$@!3v$DE9h7P5Hf(1bDgq>tnkguxEn|to>cec"
    "7ucP;C4FSH9H{_$Gp&1;@4)L(Ki)>R3{2)$5@-Wd3%D`6EmpNe0cNg7rA;Xhzdrt^Y^l%8I9aWm4Qo=?C>&F"
    "77EtU*rg*YFBqrR5+0RgMD5f-NZwnQLR6Q`nb!#Xl3+Uf7R_M)QOPaR!xOg&dum_ho%ZM9NNVR;dqvnS>Gg)"
    "#VsKS>$gyALQ!;t2*wMONft-)5SV>TL%)D)xSnpa7&fZ90?sLX9f{xeR3jhjS?Sjq>Kjy2|Whg_3>TYO-0YS"
    ";|(lhvU}gdRBAb-mo=9d14yV$rb>f^b6z=2?-b(#gs`kdgwUSsn9vQER;L2Z~1xQ>VE%NK;umOpj?+C%2W9A"
    "?KYJ5aEHgmmSggi=WsMRDNDHZS9MNS1c9F)JHstd}1fA-GCD}**b!_!H&#)0IDjh7ZewQ39%=Ay{Xn=+{ZF}"
    "tm@y0{AGw6>!QjO(I-G1x5+ebAR4B6HCUBt0m+nR^+)Zlw|tge`;2E6=Z#bX9irovVsjaVnW1qprRhLw&KJP"
    "6QUx?hD^gM^I(5|5JvILdcnlU1+Oq~l)zTr=m1$981gN7V#q#FBqPkimFPiwwshVW<{|1b&_7<>w7Rvjbr;k"
    "2??3YrRX7qqSD{Zgpw-x1if<e2}sNKh~eU#(Q;<OvO8}&OYxTM+b=h7;~+%@>in}gnGXS%=ij|7?0?s@In)b"
    "6z}o&DZvU_ai--aVnezQMewHSb#V?l^g{U%S|xQ)Rxh+?)+=d9DKF7*`yqvV3fwNS$BF>>o#MUaNO+UqKBEO"
    "D#yS92FjOkWYnr0K-ZmNSCOK5tSAR1#tNkf#9=&&KO*df;E)uI9Z=f*atQVBL~RisKV-`$0=Zd3VCVJK-^`C"
    "SiPn;Dq_8vE5{fX*ian4y@XGY)@}bx=o%l2ra%EYOy$;#D^Vv!IxjyV3Sa7bM(b@q9z9Et6+olYRQk=#UgtA"
    "DAI-Vjw3G&1BQ2jvlBGf;r0J|T4ZAH-sZt7nuAgw!7SH_YhsosxtU37T74sPUyR5LRU_B5sT%+lXeU*sT8pI"
    "D*1$Lu_%LyOGuL5yuaH}hzt>du0l}&WvD4*uc4*!<qp~m7Gdb==XMnh<jeA_vQ!=ZS)ZSGJ#N0dLzF{=bkwQ"
    "24+(ZbWf<K9?L@ud%<*<>%5hXD}foa)9#mm>4UnF`xzYS)3*5(}d$i3B+lwS?>|;5_bW*;xZ_QOQxjpV%w0A"
    "GOx36mCZ|01maA1Lf_30%UP{s`Srk8ze>R?Yo{YAEvQf4L?D{PT83Jcslth@?Ds-)%|62s&@SJ?_xyOum4rt"
    "aP+fk`PA+baO~IbEw$TOmr$dX+)hTTAV%(0eTvjS*;Ifz8$YH(9UrGQ;rrF(TE$q#Pynv=ECt2l-Pu`31{R5"
    "1tgH8DaeV%&v#mcrrOq7e<Wag=t!q2`X;r2&pTf^Q_~-cMfVC*7f`q#07^;!{3bo=?ec5*1Z@<6`ojWjhIEk"
    "aGd{~qlsD?5YFQbwQS%>hcuDYf>QY~4rw#p5EcD@|4EU|<ToJ@Msv?kM~uRH01+AOEjlGK0lnT&Q=64#D$<R"
    "B_t46^dqbTBz^TBSzX8g`FFl_Xbc)Ntr4Mcs`$Dw8(PEhVi=5LC-Y2*J2FyIO@|XY(5HpjIs=b3Q(|6d$#5y"
    "IaM|O9><_S>;N|O|xq@{9;aYhU|Fgw5Wk>z(bTlzZw0F0_{7ImP{EB;J#$jl#peB>P>gItbEEquuNh3wmuyV"
    "fg%t!0(rHo$9fo7OALscgI5>llc-T<DJ(s#B!wCZ7p}zrwo0zpk@CyS&|05CDC5pLv<I=utFS0|5gY}l+@1w"
    "kp~51^90%g>8(D2J{D>hAH+kt^R%y0qjyR*ZP_C3B>Lg`1x@M<w0Iqj3Pfg(?53?FK>j?f2s~GZh7Lrq&VM`"
    "|&I5uZqU;zD!<T#H)Fu;CAi#0J$P3|-vQ{ef>2pRz^kqVIFS+V9Qfu|NTrTHHT&W*Hl1K<V<SD72)syQ2nlm"
    "SP0VB@%{HbC7^<U{6qg`9QK-Vu)*>-z9$C2dB8WWlNY8hKOS!7WkO1!R4cF#n@}^tLd>rg~~QPx)yDo1aw2m"
    ">=Z=W$@Il)OCH*Q5?ZMp}E?+o>S`wp7TRpmg}?oq=1@9kp@P1!3pVwEt}|F=R4Fgj=e@->vHFIbYX{IcW2)i"
    ">j7QMv%2Z|{p0KXe^CG4f9PIIY1+{@yxQMuj&{6<^UPb{#Ml4RM`4eHDPBz&4(Zm4!pv#x)LGQ3FH}WFkC91"
    "44}4}p;+=LE3<Mj(P&ZtVnTcvErr9cDCk<cHK%5v-$%3h2X9gCFEif;0>zdzU2&M#3o<~St377;aBs%#SKg|"
    "RBD^>xMARXpCTd4-1iL-GHN4-WJQrnzwy=(9Qttx#UX~2)>2(R)o{;LY(i&my%gdZ1G@8Pf=#5nL`U=rO$ZM"
    "=%l+7dyH6BS{j$>I(bBBL*0)XezPY2rg#O@_%lKsDfmEIE1#jl){Z*gVwS&R$vaAlH(j--?Dx_0XP>w9no|&"
    "ctJ_1c53MUZt9APiT}@Xl3G@(=uo3Q?<Bayrj~tO9<8u|5I5TY->Xyk&1Ffsi(ft1U_x_`%_jJ_+Cp)x05Mz"
    "cTgIu3R2Qk9N9DiZD2R93WYIF$_(FBWas#hN;jF&t^?6htt(7!j({RsnVF(tb1n=UD|NSoX~@G;dY5EHrD9X"
    "S7-4QK+&BO+IQemM`7?<@&53KLTm~32iiHnUYS>n1m-PIvA1D&O7G_u#eWJaf*V<66H7JwZA8<DQreG+Z#<x"
    "TZ-6%2%Cc#7Tk}AlUvL|Wpf_lus(P*hBBB7@BNv@KIvKZVPu&_d$YPFFHEq8K!b|$V)uO}gEup)>TJ$hTPlm"
    "`CP*r~7qUbaa4=&qm&GzbBeIL?`c#=AT|KjJfP=6IhXJkY9Qy^}p{)#;)VI^ZRO0331ND^yVkGTfCyhwyU>#"
    "~`dgv6F^t?e*~T>)}gGr&8p@kdJ+27m85D`AWUa+PDvqyMSCZum&uXQ|b(T@bO$<pHZE65HUo-8ZxPnABh2w"
    "9FWBNg9bk;KV=B}M>&l`Dw0dB0IRvY%^F~GU=Ay?W@9yG+eikiGP6vdN4OX6>j=Vp<W2FD-+}%tQjiiS8xIT"
    "|4L<5jcD&LeH2xFohDLhEbyRUv79VSkrS2sN^u}z{3iqi_2+f-Qk&3{@MlGr2Fe%buExEQ6Qkto_GfoRBE-J"
    "40{l)bU(OqsFkSYfno#TY0LzZBSKWAp1nGXm9yWt_lA6pyZ$)X&VYprt`8RSfGgvkD3s18YsjsWwr{SzUW)~"
    "<~3Uo!>VDmspovpvP;{h+Fh5BQ3#oVw$e_@i}F-5kLG!#o|$j=-y#8Zf$}&VE8Tor1@cYCF2II(c4-8C2W?W"
    "=(iKaE@2W3Q)dB3QI^A0XMOd7Kwxvg2}8FjiVO;1sp~uH8@eIH{B?B%83dR7nAaPEp1eR3Nmv=XArx1DmveB"
    "jRFcOedih0DlXk7u?LEPu~GLz{a;NA<b|m?Ejm*G2oPIj4dCjY5u+1u*euXX_0eF|)O<SCfk^_?$-{`JHZ%2"
    "cA-76*m3wKC^$?=kw03pz6JyzCX<5iA$eo$e{tPQR?ZP#>P#`6Z6+-wT7y1rFS5*RzN<vJQi_MD4^Po(|794"
    "^lT8idx^}%kjdTUwt?R$x%A=xx{WtBBsecz}m!J!IPIQ-zaq<!wdfH2TnsC=LKWx6b@c1xJilX?$;&BbL%O%"
    "p!H&a$lLSvrLmO=*CVyl1!K?CQR~R-cIrrQ)BwyTqew4kCCYt{`O$+0{@fxyzuG2E|Fr2ZS#-h_3W=!dXuQC"
    "`Vlunw$#!NlG?Ahv8S2%xphWc`uA@ISb;MMyjAWO&^jvHNz1#VHP5B)F_X1EpuvpzGX@0QoL>9gY4G8N35%J"
    "BL^(a)QERBB`aVVTwuCVWj>_SuH9f#*P9hMjO~sw%_K<IfO`^twDxWa-)4QV-Hcjb_HVGn;tMldx(~OTHO7?"
    "W7ij#3r5v`uDDpD(ZNC|-X(%iA2A8quH)GR@r?cVyXn*c!=XF{7#UzOV2#Kh|fiM3v^>9b+X1Ft}y`n^v%Eq"
    "<oNGak~kIs42T-&4mrQv(lA5VS}<Y5HlSytRWue&<t#~JaXw;d1Q7;$Es*+sk9Si6^1WJ&Fsv73YQ(rpJx)L"
    "`u5J)i(L^;n-h;n5R0<Ffc1Kn#>l<u#`sCguQQ0G&FWRD`UpYmb<Pn}f@ZE47$hPt}-+4}pD-0o&wQ$4GaAS"
    "Z)rkCTEkAYk_K|auJ}?F=buj_2tDIqX!Tg#St<DO$mC{^Y+-lMsf5<OW7>$l03JWep0!}Y7#XNsR306+}?oz"
    "!4fm6!iG~2pvD><vc@OulZ`jsm?z&GfGiys61F%Z-tGb367c;3M4A3G2*~RA<Q}^R`M}ro`OsC-r?GVmd-s~"
    "-THRca!-L;@$eWL`KlGkR=gFK_a#@ex16M~?KD?*U^atYThO<2@a6S3%e>F>E^^d>?I#zmWsVc7g^uxv3gfx"
    "nQE?<!+AP1G*r_*4Y2=UfR5_VH-`WWCI1w>MM*}6lJ+}=V6TwnEi%+=4PNi){`Q}$wY1e{r<?kYWcI#2ZR$;"
    "ssHwF7xWxS?s|H)C`6$q&cpf0<lOFDL){ZgO=!dG!n49JJ*qh3$~_qez%%A9=%Amh-F0<+V6HzrIjbVvoTas"
    "H9pODcm~)Jp+Rl*{z}_-md72ACJ%8O|C@m&jaEA|9(^;_u6s-P!P8Uj-$fDo}N}o5lD>IJ6blM*F^(`LA-9S"
    "4X)QMFk%?IetZS|a#YR^=s`Q|0AD*&s&aHbHT*_(%$RjY)~o`PJ`XQku-DssYi`Bq`O-cvPqN{_tk_O#S)@J"
    "f`mtAoaP2ws0Rzx-ZJkwoRvrK1TCI<Z$#UR=Ut_tBNRc7Y61tC=AJlapLHPRp2^dM22*~>oc#<Y0Hpt{xmA{"
    "zg^7X|f)=NA;elvN+>YgcP<9l9IvH<3Gljoo5Z^(n72uw%7GNY0xkKdk(rYt{trIm<^8jm4x%&arC@)A<~SD"
    "K`pOh~dDsk$7d8rV#d+IoU#Yc!m>E)`IeCk5FKKr;Zt4IZ)%564dbgTddSlJ*4qjX|<@Lyy$v{Z$EUBSmQqg"
    "t0IiR{OVYy0l}&WrT>VRUiNWo)A=lS!Rs_X?ONj%sXJ1q?lWFk?qz84ZT|_ryWG|M2FMV7P#KtGv}bShX&3R"
    "I4<01lkRr6k5t~DotxEt8~%hkxmtXCZstqoeDXIk?jcGIrX2u!A<P|K9}+XO74A3f&ZyfnBG5<gljE8hQY~q"
    "0@Q6!}AZ`THXV{o@N=*=OfkSv43n`G&4`l|q1E?BQsS3w#0AavOtz&ztQFR@V)>wBsAk}@w0l6s%Hdj01TnQ"
    "#62$VpN3Ui|AMkkh5EHCH>QV4&XT%NxE86YE3QzuX?y|*a^Gu~#ul|<HS20mA{ND@I2$6D)}S;xkvZ}f{jsI"
    "V@<Abzh0&z!3GC+#<SkSrhdGZq8jmQA4qo`DZ`PeN;0o0LGCWKWhhy|NU$rEiq?DO4zc4ouPTqOO0i&LCyOh"
    "%uX2({-}V%Ov%L?&p~NGOGdfmrnf?^{A=VliF?yK$y13h(4hf?~!=lijG#Jk6PBqOwr8&dnf3Ec2z*lBNz+*"
    "*|Gq?w73GnxV1(&yvw(DzeFuT&-eOcT==q!!CN8iZiLo}>2gE3$<!Uw;2Qq+TJ#NK9`}0u-b_L;bpAm3R`yu"
    "`;QKhxe=`Q}kDkh)_5P!c^#5GL)t*x_Ut>qOGdFDpYv7*X5gKUVpZWNHOB)@w?PJZ`?$sKu+E(@#$ZS(|Pq0"
    "DZtbP4gWXAwMr2(lEzZIS7byGoxbeClf0fP?g@D4%bEY+6lFFSpXhx=I3PcpsDJdd)v1UoNjqEGHH$6+&B`+"
    "K1IMGEon*}CU!eYN2RIP{B#Dvw1XE!qh@78Cww%L)Q9WW9YKqQyNXZ67)5ORzD0oH<Ea))fgMb+<ZRTKd7c*"
    "y`?%-0Z+Dsb0{uIwRHpL1m{sdk^5bq)5l_1EU7q0;q*h$U%7ReQ4c5Su=(DsN+LQ=(b0N_Q1=wS>-C->|Abp"
    "0}n^1BC}^ch^1U8#bK{edy;bGDN2@3vd_w~Ude05dUVG!@>G#s`LVVEV&V?AMcJ~Jx{uk81hhw;rQvz11V%4"
    "%Y^r0>K4ER$ur&f2i)cO0G2D81?ntWE=4<CN+C%C&jqQ7=>~%mJ6KS8V^0G}59PVy@4{^7@;<3ka{DnSb<r<"
    "{WBkLR-ZFJ1itkut4{3?=WFH?-uxZJUWC-xgo<<s6R8g33q3BulNR=t;j&W=k&t&V<cGfmkfIi@qoqSvDC#}"
    "bTUUTHSs0=KmGYscMRPjpW+LV}=7;MT-4jM@jI-Hu7jzv+0gj=}46FN=(FV3F6*c+_VDA|z7jqX-2==hG10n"
    "#xs)=s(MRol7WPZZa&Am13g@&=Az{BKXwpM%J~gu>d%aLd`X<^1KsHheLf7juJRBJ_K&BbP52%F(t>kL6Mq*"
    "%eW1}5<n=N<*zyl@I!lfu1Ddct{2Dvq_);G&SHq1Yu@*Ob{s*x(Dfjg6MMD#Qtfu^+2>lkGwg@&de?40JbUH"
    "DXY;=Q&i?qMc#<D}r_8V>=GT=T7e$*t+V){TuLFnlIl}%-bFZ)%?1IamX=(JlcCATq%)gR@-DXjDo#3`ZTh-"
    "i2zjL_P4Ph4N%@9Tt^C*;a)bgrlE8_g(V#^wPX_fT#z?T%2(`@t5bIYLvH5nRNA@w%kMmh!6j&O^;+u%`u&z"
    "W_h?sTIyu{D7!z}XJ};X~i3pVyBHR&ebvn#NJ^r6&IOD6kh3oFKBxabq#O%Y2(A9rAVTn*)v&iqk7%FuJ%@n"
    "$lQ2f8G}tm#-$5;`^VWmsu)aO|DMF+3B0pYw<F~idj<<fr*uc?1lUI$I#nypX6BzswOJO1Ztv>MPL(XC*_09"
    "YVC=D05|K$wI4Ojs}gaZ3yu0y&s@CLNag>uIn{IX=f}+qtDXNvvjS-Gy7{&DpFJ_^++!K3xlELv|6jD{{+sQ"
    "dZq+Qhi|VF|PL4{RiYX-a#t{>A7o%cSNTXa>$K+cmy#dG1T2`nxD|MQ%*;x48TpfukJf)P{4Af3`opRG5yJ)"
    "Y;4xnO4$`r^Y#kM_|${>sd;xQ?afE7;I0?RC5)et5%Hv}*64X7?~0rWf9Q+}RQuw05;D6fFM$CQQxP~`M4nh"
    "k-keI!zM$u*{fVn?Zjo_%NCf*G_J)`u%yC-E;qBikFs&g<GooIOU2c^G65{2H1f$d(7*$~xF0qg#pSH_X{O="
    "mSEumAZ0tpX8fp4mqnY9x6B3z%<kr41{H+<2vgQmaY3QvhP^sGYYCDL6I7v`lyprOq-XoNuVkRS03P$U03-*"
    "EpkYiLV80$wp>=+en*<ElTL-(m^R04iE3$u$-c8L2{h|iqn162xk$BS;9blx?f=r`m{BY_rZc;RCR^XKa~WV"
    "_LN?!k7&H89b=N|W4Pe0Auy}YzxfNqn2_|C(gvX|{4ic--x1ws0shE+}t7igoGfCIYilwY7RT-RG#|`z5kTb"
    "7NudYwePp-Wh>6G&D|FQQi+-)04x__ld*>gmCKw5Slca|PbRwFTqc4Wyb$??t=bxDwDidcgH2LMHLwEFM&)T"
    "3WOgQ8?7vyYsd6H_G6X!N7H>Z_`+m=fD}JU{gauE-dkx_h(7E5Q`?Ap8e=j;U)S@DSk=HS^5V>@>H-On)Gg5"
    "Dc}YoPu6pEp47{a6)HA(NQ^WhN*@U&4KsOn)Crx1E*sxE|{pYq;gblIVmn_jwW6~ZBa&`)f_jZ${%8SI430%"
    "MNpWElQSh<)!~6<5$9nt>oB>G6+T3PN_fD+I71O;ED{lMQS&*~^tOXap}qx{G=p3(s0L28EjEa*VFnpgkc1E"
    "}NXZ69#jdzo1GbXo5fm1&^CKzCpO9)S5dt1?dP}mKc_{EFsOzv&yH7p1Qn3QL*GbUcnzDs4mnbJI9f3Y4;uS"
    "=m7V&nPeUwef6|Y%1AU`vkWSsA;>ZgVLKz@}scFDEC4xx=y8OkEF(x+IAlsN<gXkI~uJYa+u6Eiqn!&I!ID="
    "z=-6d68%=L2{IAg4>Q@Pwwp9%M7a^seU_rbgopR`2pjT;^D{3S`L?NfC)W1kWt1%p;tBa)C5OSfVNu>1r{-("
    "(#tAh+Cz^40jHC<TNGhS63XGMS|(fn$pR1#X>*u?T|F5zzW)!K$uZsO-(rzg=IK_+<$Q?r8|!=shVE#^7X;%"
    "?>Yy6=)C@}Q~WK7OWEO(>i{zYja;Xk%Z?hj?jrg{1uw`jpS*PiGKpN#a@~UM$K<S9xP4pH!V?=LYz1uP+k0L"
    "jAO6uc57h2}3|}Q`adgV6f{AY(rwvb7$7L!p+NA%~03{yYqx1J?SFIOqad;`FDuWNTXMgYzjXCy80Y0GSj^w"
    "8@52s~|{qup?|0g#&cfZ|8I^BXIozUb%SId5wMCblm7>n0r<vEGdvZcr7dbl^mY5bEo@aRqUYycHs3wlyNg6"
    "+%f6SkVB|8(W)!#pzz6PRm?1|ty1gNu`&2gfuoS3eEJ>EQUs!G+6#L98RiN2fU8Jkvch4OS^FZ2Afh6+tt-#"
    "Wq%pEhz{b%X&}h?M@D!hbM^;wW@VY9Y-_NY(psVvZKDGX+Z0OfPs!{<9xlL%pH7FM-dyLD7Fd+ZB@T6=%U(E"
    "In6?<C#mbuGZJH17m93^15#k=Tv+zFsHMF?68a}RtdrFuEi{pcp|}3m%kwilD9bfgQOac@(~`j``w$~m2Pkm"
    "|8a#By(7<8q2y%eh62+w=H(|{yOyF+t7u$Rz%28m|g_EDCB9e)g$F&?89kD`zvP$VnJX=X^Qc0NXp!k@=A}%"
    "vviXg77$wq`J)rl^%EZKgdeFnT&3tZvy5V%lPL5(dZT*auU9CwdGdyLRMFYA7WY4@r?76uS%6X4$MY?h1t7N"
    "BnJ{f8!R-l1i*)~c2xr5Dt$bC_^qppN^J={j;NcYz}yR9P_IqdOs@?!fbnHSUTnM&8qBh`R?R*HlH82VG3Cp"
    "mB~%qN4bOR1uGmHw^c#yi!1Keu~n?_R(5OnHw-J1Pi|}GhwomKW)yeG3Q<847PgO7aL_vB)oSY`4Cv6%@<_r"
    ">wWrl|K{O;#h`Di>9fOVU(Id8QneLTH6(sab>N%c|AUS;V2E7Re}w~({#<emA9HLNU(e@{=@0QIryL{I3VP}"
    "|>re%J=}NXAWr{@-I2DmZtuQW@o<N)#9&kSUfa<1!7quqh^^6f>qjUAF_GvjHC5l3Cbff!);dHD7IHg(IA*&"
    "85Z~}0vY+zNaiW@_ePuKfkCBxxejxgZib`=+Ma&K_&Fh&oN!r>m%=JVK5^d2d>Dkj+}l(x*X1=N!4iho^2w}"
    "?xurgKi}<$`hs@CMQ=ZOT*S-btnQJW$EKL{wia;sOyWRWv*<tqQJ0*B#(yCm#3Ja*IY{Ih$09pZTh#=fTYgT"
    "(0!hh!a3D#I9b)!#%2wxJnEB=%wn9qsm>KY*ki~b9<4Hj@-4^WzWM5V%-5FCaCNjQ(tP*B&APHh#M~E(lRwJ"
    "U?7-Pr*y=*Bfv8{?si{}MImo-;E0MgoD^9C$>qqJ3v0Sg-iXCCt%BpSn1aizLW#s99L2y_mH}Rsc}a3XKh1L"
    "DJc9`b72YVpD9VcpZ!cj2KE`oeL8fBH3^eF4-K>e~6Deh>ux+=RZB@=;F@!TY`j0He_?P9X4D6|>8{04xaA!"
    "9zM)0Us`}y2kD*qeLQMK=Yo<KkU0e7wv*&Kl+wiPx4-Yq9=DJJeG;yZ;?*<NY1`+}`Ch=zOmR{J;tVX}p03p"
    "5TI(B^no?>P(NE+*D_SyVgR$h76PJuM2)9FBm)bhC}E)3P_=cgNhcO*l3-wT5G?%C4VlQIjk^QGEtUGK(dg0"
    "c=>tX*IT<2^qUK8HXY1sE)n5+QS~B7&ELdoA0lNQ2PR{9lNaVFUA+MbyIr9o6p!!G{~S6F=NSg8{nRvxRu22"
    "7g9*Ya|&J!_s%EaIo4Fj+qmC$VJcx()P`VARs-E&&O<Gx>twg&NkKIwp1A}ED*7WX#MQ<7fw(yN@u#a=_@-&"
    "x@D38FA7T_KjJ!6D0bK)M6rD$LY2ZGa#K6Y=6pXi2;V*Qop(}@+cU^Ik7A3f4;qd9%eS6gfbv2TCOtGRq#4+"
    "IG+F{$ZNOs?t=3aMg9zGQ{PuPsozpcx^?uv^+XYlWXqxV;XOTNRUkI-_-EqrF$_{Mr9s%?>ra{3w#$WufhR)"
    "uWZW`4o5+9G5`ua>SdZ*}5Zs}tn)b~^$&ut;<r$zqzvq)W29o7jDwK3KI$R;6eW&c!5axGP~6+Rh@oEag&st"
    "yEyx$hb-tz|x-=U7f<o&eL!OARdc;<facR-B9^y6bb|-ro*(9t^!wlREG$M2jAj8Cd6`)xpfb+nWYVN#bpU~"
    "F^UXTYofMyq9!pi%Ius)__2y{Q>~1dE;y{J2>@Ynu}#Hh3E&cSSaM-2QHQ!t6LruRPbcYk0&noI${FkqgoFL"
    "-{`pY;nnSIq2UP9E1qvdA?I%xsW)u9V#$17qs3kHAFc0s5cuA&08<Jnd`ai<#NnCq6u#_yy$I}yHNF~~|{G@"
    "-1BG7TKWb2~+<nct_sa!5#V`fX)h__mY)T0vo3n3>^JW*3a?=oH~xxoD#El(a9l^(-K1E$4T0*6}NVZo)@B#"
    "Sl}M8F0AXwgk1B~wILf264)6Px;y8*Cb|W9lb}S0MeTc++Z!y+de|QeQHUPRxArI#5e<@hI3VLAJDWQ2f>~m"
    "cGP~ueiD)yergjmy{wR<}ysm`G!rq(sBSnWqQIw(DH(yUQ1M~L%v_lsZ<%OJ+YPAr@{^7p-I?@@5kI`xh{$E"
    "98#n~3X-6mvIlhPk}mLTbmLMkidI{#T{oT9UE8Mhr$eeogi}Ei7+q32utU6%J#oJui2W|&0p>~VhwaipMi%9"
    "kPwB<NI%nq}6v=C-YZdqd52z3ic?7Y7Cv6yu3-&1)V_T_}6)?+SBQh$}4VGp>3d^H==0dshVEY;P>D}Oh&_k"
    "I3gBTR63#W4nH?mVfZBUGP#oW+Xg@n_PV9kZGH^|qnJo=R>$(Cp94pG=s_7&<-6s`^xV(=WGA`TFa#nEB5?)"
    "w!er}1s7>oI{N7)uR0xm&r!r${$ij><YQm79iB(2T^x_vrlW>f+?j?;#XgTwr*TeX|v)q@*B*DCx|&q65bte"
    "n$lBRvIp7w02_{RgdP6&a1;ISH$h1gCtSxq1i8x^PHE?xJ)uteXxN(j%(4Vs7f;cx*(EU8GvB&uK2*&yKq-f"
    "NpP$N{);%B%N&H}GzSm7-5w<l0LMJ&)QF2A+wxjUuyw4gVurM5F(T*7M8Y9V(G>OzZ^Y$a-s@ljwo_ESZxzv"
    "rS_?vz99)Ph7dm_6(pmBYtw_X$wSlr!oLpUs^ABfayz(}6p?G|_d*Z0^$1|*>jof#jOJu7e+28?Vg|y-sRfJ"
    "_+Am6=oH;|Rb!KEpDi5DWzB3Ys1uTlb5w6$L(eB9yZ{6ZjYb5ig&oQG!BL1oQ`Cv6+)9yco5_KSi>jTD`a0!"
    "N07uo}<fbFWlauzAJf!Rg7*gA1rHYW!Y+mStsE{D4k~YBO<qCyO$^RjI(zfvrz2sWcJng<j=&Kt#dob%4rX8"
    "@);iGEk-FLUyd;narp{*EbS=HzOQtO=H4@<$Pl_vZB-*LQ({m`H9n2?M<KvRBWX)WI4q$b-Ud$f)5d;%o1tX"
    "l#6#E-NDtxIgpxPRhtBp78WMd+IEDpwW%>eF^%X61`yXJENj&=32haNFp|}4#a%dEWvfEh(q#VvV=$mhWlko"
    "uqJ7d<^_D>Y=gb1-SI{G#FDM2kEJ%0a!5nO9Mq<R*gNH`v%WSyE!S5h4ZCSJxwdU<^Og5!*;-(~bvpKP-a8I"
    "-U#~dV3(R3}Lj$W1+FC-oB>huJ79QPC{4b9CO^%52oy6eb;Oy_o0MZhH$s?Ikz8=Dvhz{0+$EIrOl*|xGe$m"
    "jux@@&aPZMcjg>oD6rS1de@T1+6duAt=T0t(1>>p0ai6u)u`>r9xT086WZ>w@#^b*WhgRIJsQ%CvdQS}%hhx"
    "&#wM*pc;W>4>BUk!W>`VCiAh1*>ooQl>xYQ8pQb&6U7mxH8!|3O-vGlZ_br_zA|K)I%Q1WaF|auvJwMc4N2b"
    "Fe_L^oe={L8*@ce(bkH^fN*w18uqq(BS5gDos}hszkN!<E3d9pYoz&LdcN9JlCGl(=C}E<ZIBJ{O;jh1+68X"
    "oZ004=ZWav2$>9=JfB8yL%Dml8!^+kpRTJ{Li47_&#CU-{9^NV+Ge_UjleZ;6SV_fTzwJ`xNF^7mG8dytC5?"
    "szhf09Di;=5Q`m(w_%SbMkt8g`^yIW6MD_4MH=uo?~95O6@owR~*%R1Es8tp=q%y?o12k|nXZ;DZfFL}`Y1b"
    "TaDYDKCVoW_pWeWcn|^ka*VhdBT<*FD<c)wRraKw2%n2^5>i3LOqI<C6!bJyuhWxJ(6Dq0GA}Wmu{;K`0THy"
    "|vPNnE@Zi-;ec{L>39DoAj=oj;VlQV3#;0-xQhj46$vkV(2B_X>u)nk&<&3FOLoA>@LuUdrGTOlu1f`0qOh3"
    "1~E93FVl2Texc+giSmF=Uj!c2wf{^nspz@1p&C<lb8GM_Tr_%ia^RO2@=CxzPO|cuncW)K*CX&8e<yECcE<1"
    "Q^7J_DLSJF#G9$P`tF$_k>`S=u8YQaqYYJ@_Sf>f}<LX%1p7duAg%;cY4h&i&TeHc_6uDLn{EbWA_;nit^L+"
    "Kr262Z&P03Npt^*$lUtg&o{}udJtQa3~QFL{=2SUs>K}Ex|s-jmI5twIdj|QwWBs^hoH^^CH2Shzv1-Z9cd9"
    "asST$pO$>=+FbjkrG&^U0BKl~52^Ey`GeHC`aZXnm38Pb!p3+XN!99@89PExpU*bQ&*1m1Do7a-dZtUXr!7+"
    "7-t#i6@dwKs4znHO_8HPos$~QcU|wjN8lK(FOHdcI;nT)|TUnH?U)z<4&aPyGT)u#)RF39>bgoT^K@FA#apW"
    "KwGB72Zp+;axssW)!j@cqq-YJShpITM8|<eRDWMB5!uA5Q`u_IGfy;k$S&nlJrz<LCF6rr(e!g@mF%iFUUF<"
    "76^eSD9naNWjVBN9sn>97JZtYCt6lm4K{URl@htuD5d%ms%>|n*#STUTculxffaHmOqy*ia6GiF~8T-Y~Y9p"
    "y|?1CqKf)J=!k%CqwLPML*ndcP_iWkP9R&Xij^!QuVX);)pk3#-j>yiD9ZZT*OBK`(72pyLJ>+RI4(_;8YG3"
    "h7g(2<_jBy$nZZDN?+st-*z6n@eZz?5>0(aOj&dzfx+fLh}IVOQcZ)l|2TWh7_eDk)oL^O)iZH|;=N-?Y0#="
    "E5pOEmAVV>JCf=%`_J_1@z8wHI-gY!C-_F2IQOWGO(%{v2}gp2Tshr&sSQ`GB;GX5_!CTue$ND|F|Eta<gJ@"
    "F(J0|@QzB6Lv8O>`b7b{S|M>Rc&_~rJ3)4<0^N{MKP_9^E6kcGCg$L*TU8K)G|5*t-l?KyKf?oM&yhVTI%|}"
    "(FSiZp4;b}i#o4=)6M_{#N#sIV2~eLiE{aERvL{cJ3QJn0k<1h1N8$&gtWpt5{;(v^lzr7`9(hX{TFIf5X6N"
    "W!PqCuR(19V;P$H2Ud?)jc@pmcZ5S(D?EX$|l{HcUaF${$kU`4cHj8UIgRc*n!g}@zVj{0D#wi*i5RdxYy56"
    "7yiGr9HqTl(NBKUe>Xw+(i#*U+;acFv=_N2Q9`sSvwFepK`rN(ErP4e(%S#R!2h1~axre>`h1IXIThl%f#a$"
    "09;JyNU9Tyz8N|7HH6%rC{sTY9(-Au5|?_r#6cF6u<DqF8va+6T`W}L_X#QpfE255|f5WE5%l1IkrUMDa~F0"
    "<uL^DOHer~cxfg;Tmb-yk^f)zbfkn#H@(D{R<x7Ib(A8dIl(f(sk-jGy3zKDqfC8HO;wc-(Du^n^sN-L*3*k"
    "k>KB@9`WB3{wVI?e2u<xZ*KV}9&z@GfkqXXRk@9+2?eOnii_br=9_0CjxEQ=UJv<svDJwG*j)hAnq+4yOp)j"
    "L;o0TI~EC2LJ8Nkn|gXF60jdhOpYL$$1K_PqP2&iyt;3;RZJZuAm0rgsjd8t$mhcFbBfgV<mM3O@;DGd*4;4"
    "iEdy;lSYfgSI;F6v38w*Si{mgcKORT%o)3gY;fWW)F_RF0|27PHb$DO+<?W)?#j#zFV&Jq<{to0euZ^+;Y?n"
    "c3B&6X~R+5hWy~Y~zP|(OO-bIA1083~BYgef6R8yBdJL_Tp5QrhW}woX&o;ptg{Cu=?g>wL~rm#y(|*N1g@y"
    "cmdhZ6hch0iFFq7BZ?sYxT^$mwyaKQQH^)a%bCrmANJ6$FLnCSUg`9WCvTdA@Xhj$(@FM8aoX`LRqEZml|>Q"
    "WN}K;sr(q}IW@MVc_fV}kOa|I>-AuEH&#ExnIM3XjMbPGzhbKQm8d%Z%Dt&k&(SZl2sp)vnc|z9)VEfhxn$~"
    "X|?uf0`(Qw3B+pxztw5jok>fSzgX*t@hRfwa)^|QV6=Q_0xoD8cJ9nf$9N8~FkO2wKX4E0xFCuC}9P9+vT#="
    "Ugzs19oUTY0#4X*ilITvTTAk)fouXoitmd*%>g1kkew%Sl+RkIeQJS|1^cu4pSirm2<i$WS?ypJv2dX4x;c_"
    "Dr{)KB9ClIXrh=c;<PJuf6b2`uB2At=|B{y&I{&{=xdT|FnZ12Q}rgOL`<%*XX(ClYR7&GrX$nnN~bw=xkyS"
    "huSYOhHArG(Su)h+*HHXnCay5$HxmU_0SWTG261Od)6ZUHHWk!7%afYEh_wEd+=A9F{i^1?wI>*7yr>+8Otq"
    "3Pa_#c(9G)K^a?10p4lxT;dr;2B=J-fGF))dB2!r#W_$20CSk$cd4$Lz1Bp(Quf3~reF6-Hkq4Q@Ps7A*<e?"
    "lFtEJ5E;v&nt%guPdaA;vObmbt0n#H7|=(e*c++di7yhUFKKLu%vmw9~W5Esv7vefZ(aoVAmq9OzM0IqWYsl"
    "J6`L(I`J!3jhzHM8}uF~o~66myfv<!zWp2~=oCCKm|fR9(*F+gsojiMJJQ+A2Y7>N}oyqf&k<cP=-y6CcXcc"
    "}9{rMc3gq1RxoK|DY?LvhXv1mvNcMhvD8$rI9Z1Ql}A+*b5U%eQ)F^e+K@>2jHNr&~o_d^bFJC9%bC)Bih4s"
    "FW4*c=}V1Yju5|mxy+Jyy6G-Ad%ZoiO~7}j>|>%Gf^KvcA=E)(T!a#%H96ZbMI?SSnysMT&S(Vdgt$-Bj7h%"
    "qRh2$;Wu8OI9pAzFQNy(;1NF6)l2<Np8Hfu6Bowg;JpcUt)zSIe!RYGZ@a*#B>g4?Fl0Dw;oxrtke?u;C8fg"
    "a{eRpyG?)-9a+!OZ|(Ddr$=RpNYJvux)8k`P}6%6IMqi2T^IluorxH$RYuP0|e1`QYd`SkqgFN5QzpMN+!Ic"
    "@$`b&Y@b_E7hrxswVj4jO;QVR=>;-S_kMkS}@CWCXrGK<gL?b$R~5vsl!UqJ^UVvKNVF(>cC4{Nbu6?)4WJg"
    "Tv#$npJ5%(76vUv+p~+ygd2w4En3TQX}$cW9nX~AMel3U^P9XIlk3>tDjE>AD-11-s!xlT14ieqn{2T&vtY%"
    "__z0i%d1`5t=-}qOg+yR=cj|w;rpwf&M!``{>sx#@<Y|#ux1y2=)Su+IXgOecX$fhVK0)iIMo&snPiHc3qIc"
    "hh1p1c!k7rq+6+>vx{j^BJ-oacTws&SB;@NzSz`k<E#+;FN;WY}A}Y$@Tcxq{cY}+=t8;1&PJFrt6@iiF-&K"
    "7{6$1@73?jDD;N`o+iwl+jR)frg*z`^)I8OMj>d(Ql$p*96b23ax@2jHCfHU`4xRb8#)%<|uigXrEMt0};8k"
    "%u{y)EXsN{HzLB_?n-+{4`u0<x<YNLF)C&+_CDujiZoaPLr`sBs~1F;ML<0?pAEzM)7E7>Jyzg#m%BFUptep"
    "ZA{>Pn^n6<p}a9!!+sxaZi>NlE}&7p7k95gziAav1AoV<_o45b34&C^HX&tvQx8HIg6Khw#*8sS7d*0<A3-C"
    "&xfK#?hDU}O;K9(n1E^?we^&%=cq~NmYy-s3JB!LaR`v4c!PU>_5&#4_v(R-Gq2h~Es8*#S(dNEJnA5^C&>g"
    "|X^a`E!C@?mr%ZiA>|kZu-@}{>up$ymlM_gbn_xBtLHyO}w)9K&#`QG-*oK~eb4Z%GSKkq6$$!-5ymILNO7o"
    "4Z@t--@SQCIJnzQPeW1eb<vJm?_MuSly!!JF-q!WYhMQ1Yw6yl2qg(k*d6^e7>==9`i)BPxs<2JfmS3)}^l8"
    "p%=J53>ljBwr&#C6o!WWgq-nCC4iAT*O_8CroI^?p4Se_Tlj;p54volNe~Ei>yO-Xq2lJ>@U$I)L41^4P4`&"
    "jEZtl_(1#C=;N>Q~4ucLoUNS_ic5(s}0D#2sfjAl@c3luWEOB=9L0Xcu(HfN)ulU4pE7#{=k~$!!x$8g`LM("
    "fD27$-q7BdR~FB(KiOeR`viqAWyGf#cW;po16v1V#HM%IM_+E@=k8wZ7fE5MnODM&6)Q-U6q6*w@b_9=+ZHp"
    "Q)r#WfXcVV$IT|@8zfwAKc^emHzUhjiRbFH{MXI7Vfia{o3YS=6084<vPC81$RYcM5M_D94RWX6jvG#ShyAx"
    "#Hp?gl!QYJ}!3+kfk`lAg@shivcb$a)k;l=d5@W<}A@<8&v_}9)i18Hg)!}3^$WsCu!)Xw=*riUj-`um+*ga"
    "A0ySGl1}{@XM#Oy6Cc|9o;hxVW^F0epjQSwzD<)iQv~)mC-%!*Gx00WNaqz`g_q7_YEL+P(ls5wCE@)4C)}l"
    "5i2iB~3$x7h875tt>}soLW#W#2i_^cV`Ku_4U5c!-o0_wBTqdbXnaqxT}17pHi_o-PAoiqS@Y1di?}hp5VyS"
    "+Z(H)FmEP&kD!`J<LO8E;VD8b%HSEh=$vYK1cFvTfH`)b0zs3BDhf^7Bf1DL(?<mz)~B_EoIQyhJ&1~<peqh"
    "ZvK1ml78{^lD&{gt`d7rD`!t}FzeNX)`Xej!N1#*VkHvgd#)<iJHCg5vW;W2Hpw=$RUa5tq{wko74}2vc`$;"
    "@eSG^0%IsZ~_mKgTVsA;+h#JfBzvuTz*RZhcy|Drq4n&sKwWr}0e9;S3gyvwtR9JtwUycl|-TrCqxBBk5yG8"
    "QgB#jsu?*@FZR@t?Ry=%=ELvQ>$DQS^oJ9Ta6G^Su5&cALJYW<;M`qu>7p)LEIA8{<{b?gWqbfY%*>{NxnC?"
    "mP^Kss3B$`NMFp3uZ^&YVSHixcrX7WjKkGxQulwQFs@|3FfSbNtSUHB4#wIzbz<q<!FH;2bYV!_yIWS?O$g}"
    "c<UVy&Qg__Q|r5*G>+z9MmyX+D#A3HWS@Lj6lK1e!Zw2FOrR(BUetWQsUU@jmhX}kgp6&!S5wydF1wTYBrCi"
    "v==@Gz)6TM|)h+r$#d$>l+C})LVWFn7wn86WIVb5|Mp5J!WY6^~uhZjwZ>n?I-w>dL4_A{!=$dCC!E@b<wyg"
    ")cqP`&Nt?V@G`r_u<SC{Ot3e=^ItD)Z3)>Wa7k5M5pO-mWLpEaSEOF30SAiuq{QXW}^t|NOYWqN1pXPg>|=R"
    "u3pE=My|0XBWO>}ir|aA#tpbod0nJXPqn9}Y>PJdokqx^xjHWP$qzv4kwQ6fA4crl(4Z`bMAR+CKEw<~RL}("
    "{iUlQd3h``&#PURM8<?Q&cAz#DJ3gu5L1LT+=I4Y=)BXRxNN4A?7IZ9upu~?0Y7wIEhBFUPh`q7mJ6w#9)=y"
    "S$5S%mtkH=D}p3ISGCcIon^v9uNxtKI?1w+qt&ui84-2Hs+%13j&z|eh7njdTA0K<Ebu^U!J*AClRn_EZ5UI"
    "x)m(N98RpY@E1wPbTHpWich|$=x^vUM&t;dY?zNze_CxD?^%M9e;1O+=sb)4bLL-DSj%0cl=NUyNy?~`qn2x"
    "x6v!^T*`0prNZUlqB%Osx05Fla!z*;@M3v;YKU6{=LWtdZ1nb87N(Q#UfS<&TgUBeeQ7%+hK591X7#Y9^5g}"
    "K_Yg{I*W1ax)o0`VfuZwqG`G--Np^qJfFPCg40SbB)`))$GfnbcJtlZoEfd3sBk8~krzl*&Hb9*}0(AMTlFn"
    "IKCYds^lj=RsIip>JvBU4X${Yv-DK$}f2nc*;lgQMgrqp<y1;IgNt2as@0w@AvUSW~;LQ>Y$pLfF?`zVQnRx"
    "5EN{kUk+d_0{gYDsKvpjB`~bEfoF|W=`S<vkfVm5+O9$8Hf9y$s5EPSG3|3n3u+0zv92OLU0p(dxj)N1w|<s"
    "+&Of*7u1CxzRkJ<~mqpJjngb*4{gRh2c)|Z-j(19;i@#z^+y2V7rPcQyz?FCjmPOYX5W8UvVA^m`9+a?y>_p"
    "`Msbc{aT6lDNLhdXGV9FtJ+iWc^fVm8(AH!Q<6eV7;sn@6VEL@?smr8jH$2d}Rkao68^<elgg?Ogss_Rkch;"
    "IlP*65mw7W|CSMXGn8SqYW)0yI8rF2un2F8KF-X~R&3)Vd3ieS#_`(hw>}aWM^%d8G-(sWl7y4XXTMXoKifw"
    "lAAe&%8tq)~K_&more{#c-Iu+2Klp`2wRKY!ATeTGjD3rb$36+<ajtRYB-68y{@=cw5sKabx;#w=H@bZwvGj"
    "-%fwj`oRVcA0SN~am^spA@}=A<75!|o21`VL-dz>yEbqr8WfwUWGPe{t(aH!Bdq8)U*bL-^>J^{@fYqles1c"
    "uHp`yWbA8wL-uo_79eHqM_GoI!eK%<z6)UB31){aaP>+BJ3DP!^>(NHAYc$biTIkT#)as2ESH7qB+(#FP4?$"
    "oJ#ee#4X+J}Du_KLW74%Nlt!{dxeijct$pr)`VwzA4K&XJ$1(l8va|ne%#<f+-bXPz{m4vqi`YGiiOv`v$>?"
    "+cBJ0(oln`D+!1IQQxD0c)_P-_QMNaI`aa!HDAoI*r!9xq#M=gd~u)b%W5$GS=>+ux#(=yDf67cJF{tLfnd|"
    "9W|Tb}V6h2aO7yd4pEHMnr$Orw#eq6mW0)_rpC<Rx$H~`%=xe&ck(I{Z{!&w_jCu-NA|r)J#BC<mU!xX0?C0"
    "m%0m+70N9B?hC#Y7RsPv&D=OAot2v<Bv8ZRcYlC8_=n+M9<H_7;(cj+O-lPvFQiA!?)ENjV{+Qq{uK86f{zX"
    "~LA9;B!d8g|<w1h&3bkECrvs@8gme4@qR0LIWL_Oj>beV+@RwWke`gW`C$`DP3j%FUz&rC_3f>5mp^WBnpnx"
    "|=4h2;N7fuW?4r~m|b+THR6&@4H8C%7!JU=#ISPsGq(M=LWH4`A3mjyJyKF|-(u70{We|K^;I(&CB`pe+2&K"
    "7ije}DDU=<570gR{ydN2iDHj|Zcp^W(wj+;dg?slb|(#i@V^s*PjNiKf&}cmLV#wzLgGW_jD8BS&KcdnfuDc"
    "BwoR4)acjFLttIVKeP^H(cr5Y7v*VoPGpWtL<t9O3}yDwbcU61+^Uf5L>i|K+;yqMOWMDz>9g%4sh6fbti3v"
    "?FqwL&!4Rfw1-4Qb+WHMc&@29YWmbk5;GVkIo!M7ck5YpS=nemAUc7;h-v5VGS9YAE~+vmeRmx&SfP}KZm~?"
    "_60>+()tQF_SM;s<>f#Y>rKl<YpS%BP19g_w1mW1iSM>t|zO<5LD3e0s%EE{2-jcfB*PPLK{OQU&tOK(KJg6"
    "CubD#gcKDc4lHk=23qq;Jn&CR|wpw^=GUJn;p*VhO{*y`*z4HKEL9Z(=b7%sF7l#Sb=#823F=C<nA0mHPhd1"
    "c?W>m75g&&C#@3!P#*mk?%CZKh%o_C33`aeZj=!wx|L>eVTvaS#EEw05^ULr~dj_1OnkZsQB_@Aa42CN<h$+"
    "!bSCpyARKWyb@|L&D==llGl~*Ep`KsAS&t&1v5l`ojTq>L}SB@pD1|qOg@jjzvNzhOQ2v!h*a(KUHW`Ipd#7"
    "tH51%OjteNluv24RTEV&zlwFGseOP<!dL-JBd4enlMPEnRA{1_THijpbj|xEo=ndj)cU|$Xqu8~&!Vm1&Z`n"
    "}G{8_-2Zsb7fO{Foqp_xhf(7IP_&h0UUJWaDk7z+<r1m_+eE;AfeHyG#nFMh3*Et`n8k`5TYCKsyXGa}vw+|"
    "7|{aO+R4ylh#a<UyHtgIk-OO@q(?UnICRleV}z0YgH=`Aewv1hB3FWA=TO=XTWNwv9Uu`KHviYv=o^{({cf7"
    "J<HM}5_>9?0q$l-~8dPG_QPYc#ytO=s1!Jp8u#Kvta(8*`_Qu4K_7s9qc-a_i6#ugZ@X`7kfKoBuW!?1H7Zk"
    "ck^Zg0EuVXMS?-KybQtF$3Gybt&w<1QLBl>XyoCz=8rF<)+;eh%w<g^iggC{*y?5Z>0?qTlg1Vgb^+6hnntf"
    "UB}2e2e;HzZkyKK`du0MqR%7SXP-#osQNlg#|gXPPU!faig2^;sSNpQqGk)9ry0IADh{z7mHCZ}5t?Ro8WaV"
    "v>QJJ(+~!p?+~X#mjv^ww<_8ZI)>g+0X$|M$y5{Yt5$mfDLBr7X^{46;de!!=we7A5^Fu(pUE5wkjgtLV_Ny"
    "!TiTNfN<?Y~xm`m|%B8Hv_YA*2i8>sj^P|?oIs9}nfz2S!x?ccrtqPPd>-@jWJ;{OKF53bqeZupBdD-Zm~-u"
    "fpsV>G9yB8sNVf%D{N;EUJ2*EhdmjK2!Th_i;$w{Oj01!Lst<jd^ItzGzE1H<#Q=|SW4yBn~p4h%8I=Df67l"
    "Ks9oVCYKi9CH5a0rNID^7O{O=h6PCFpw>`>mZUE{8zoTG7gP5Rsp{!H}KfDscsq1-gM_d?Ik{Sa0ROvD^@CM"
    "8_8P_qhWeM;&^s{9mlR>k^eBraJwDdt7*@Tp;KBPx6E!U_2YR3Bh)<c6ji{wW@L_o5-}>8CYew-2u#FsF$J_"
    "oN&m|wpi0D?$ImHN6PB#onr~V{TW-<N65g6od6IgZUGr0)1}b?d`F&GJ+s{9NoNDt+Odq$sZHPbQF)51Nf(B"
    "QmQ|!e%bx<)5mi+&;8RYHE)=J9<Jl24I_dxURP~ROa_&p24b9GhPKYrIQ>A4`IqS<j_X8wuJ3vH=B+jX(Ky!"
    "9i&kE6k~R9Q*){iL;a-;W(X2yB79aG*7G^%0wcTHcNu`(;iIv(;KdU<ZSCIE3})bH?0*>{jpk4GaF?fCZOx;"
    "Is2V!TOiqkYMz^L8#{{Pr)KxCXx#F2>y<Y)#2WQA0#$la<v5;G4^TE@vQcI^{l`ExDUNLM*BY%;57@@ooD_1"
    "uLAEPdxwMX65-7u=J7Z%?*J)-;*@nb;v!Ks9aZ)tmk`b4=vMv~Ao?p{xekL)4a{Yx9WR=TL-s|{-1irk=NJF"
    "W;N0J0IDd=b{4Iv_w;0YZ7sKgEXlrc4B7xo+(ycUsb(EW-WZypiUlCDBnanNLhmKB9+FL*hy7-CsL{-yzHca"
    "uQ0s+mt{#OeI+&aD$xX--AKOz*ccJx={{k$Q5Rk8Bf;d%^m*c-gQWPEf<k3cyN>pT{)uA&|7-(vrM!+U;<{r"
    "fHU@3+{$|K_oOHTq&3Cn8?nj?Has@vRvOtkyZQ!UBCcGq$ynW$b;L#2=+1uxH*F{p^5hSAf_l!{j3o;HKiIt"
    "E+cn{POPA%jrBUUo!p(@X>gf$~0PLG4Qg5NIsPael%(txzz(?s}0kQsJ{eVJ4{=5r>AcNar#zVma8bv0=|}A"
    "mF=#$Km-h##=}C4$Haa+9*c>bhAS%50KFKd+zqY@2R3PEnx(V&R`mx5<U)o~p>?G~3L9V1L}ER-v?Ipj%k!T"
    "H7lV@@&qjwo4$iJdr>Ad6e?Gh%jNV_Ij>lrSdVTQf8}$G&9?#2i*?ajCyEM;=viIuMH(!7A^6u4m+-+|0?fL"
    "P*+uYyQGW~iy7OnBP+wG19|2{Z+e|3JLZwQk(EZSRIW|YXZonc{}FcA|O<}!aHCP|onMA2%`CXUr9EFCJ3r<"
    "P2BgG*G%IEXK}@ImncR?c{Q2nyf@R>c{Q-@w(w)Mn`yx$LO=sZjJzgtsAVJi8j;M9wlO7LabrIRsLGQXfMVi"
    "1Z5+3>TZ?>8!ww5Gu_52&;wQr8<a~DoTWfm;i|y;w|{qZpGWoc>ec6W9twrYF4ly2WLN@T%4c19h_a&XpW5v"
    "0Ch&1H=Z^S{FmY8c;p&)Cp~hpoqgXPrpJRH4&R?%>F(h9*tqdwxvD$xVR$)iZ1`#P^Wo{q@#ymE@ap{~az5="
    "<JVso|fc~&NN2ANXUS18}j^16IzkPRQkKJFh6$bW%;!>>@wts=tIRp!hif;(Gl8f^plJQ1tvK13%(XN9ns*e"
    "5P9$69?w8FJBHrXl{z_<>#gSdv^A*>dAes=m-;P8sm+e8ZBR}t9+>Q{m@0|W|zP}uT*mLt=-n8C4D1k?go*~"
    "EfasxU-PI0N*b*7*FV8J<AwbHhk<hLu?$RkzHhA4jVclc##(k60y)%1EWFB>7VytjksTt)964BS4ht?Vn(g+"
    "D)ffDXFDmj`D;qd+3TEurd{dq{uWb=2Ul=n+8Ftp3Hp`e_H{2Pq|TzgFiJa#YZWZg`i79tZxxc04a#eLgX^b"
    ";poTH_zjgp-6CkQ(6vL`S7gbZ9MKusLX*bYkA6zd3sU-8AZbRcJPGJ12t>GyN1&>k@FR<;(-II4BPYtMzD2I"
    "d8tWr_UsN|>yPFe+JluP!Z45f29HU~V2eCk~RAoTJce%4FQ`{yT8KB8GEKM3lLyzZ94>WQKr^oSFH5GKI`mR"
    "(72(RyIxhdy>lRW0UvN(yidao)6!1Mi9Ux~9!q_PB|(_<b)s2#H@#4LovHU&z)8S+Vq3mH!5j4^?FNYJGSOS"
    "?Ub(+_Av1F^<hzhNp?X-*9!pe&N|D>gumjAdWQvqN<NtqQ)*Y>p`C+u3iw80L?JWboVv@h{>Y`tk7WNN@j-7"
    "48GiMO9S1PVMxIq1QX(OxHemQ>}P^9N41ahbPpg@wQ+ie1uuH17GB}4Xss!(f$WOW~zLNA2Cx0*9eT#mN`L~"
    "bQzbtDyc@Nk@`gzV3rrm74I@m-vo*#xF=>w2121y;9`<R8&b^h5pdk|wjS=OrtwSuY*1$SmK-~9p34O)U#lF"
    "6(he9_n#&Lt3O(Q`Lt4|$6@omV>Oe3y&%jH<tRGC>PJT<<SBi~Y>q45O3Jmv<&waS}XSDgmJ>U6YBzv*UQap"
    "?+Hj1@zK(3%Psi6XJ073d-ALGxX)nZw+VAR@OiHyUocA&b}=YRF6_xWGvM5KHI>{6E$KX9$TJ~*iH$udcEiz"
    "kPm3uSUCZO}buVY6ABfcin$<Llwxbe_din9`f;gPUtT$M83(o;X7kPN`R<{DkFkXw8z6YpB;NH0MwSKr<hPc"
    "^SZ=BKk;T-JQkgO}Ga~A$^ncR}cC%sBT<d0tDE3g?Moqi9hz~+u8{I3^1rWNI5F$@G4=Pq7UCz&-H_FA66I?"
    "p-<QJ_#Qs@_u|#f7WzSG!NWedy=UvN;CLrrw`?g9*<vA+`Baj2PvCElZ^apis}jeH76X2YP<Z7{_~%v(dOXS"
    "rf(F~WGg9);aavMqE(s-UdFC58<I>c)*5Oy_C`lG0(mq;tC6mCeViJfKRWT>};??Ve?g5UC1u{75{$cSEb$Q"
    "ZRkgJ#GsmyF+#=1Q~)e8@OusOsbpANPXmF0%?v93|2VxFb2d&gtizvHpit}NSu9hr-dQ49_Z$ZIJ`&49?1q8"
    "w!(Rn3l?I>zF!Q7Loht(ay@$(|EyC$NL-noLuP1t7B-It@uw;C&|UdJEZTApRmZ{7J6^PY>vkzCN;85`E`%b"
    "ac^*10F}Bq7~7wRKE47Nl>3UOZbE;Hbp5HWP@nw*b^=~@en-H%Bqlgtqr$cC5}Q5Yf8Z<s4g`erj~?2K(cG9"
    "YB#Ayyb=6<el7I8LY+J3Py??=d_=LYPdjTEv9K;CNTGUR4RLU614Q%s_8x0h6>@kE)($Y8L1&oNw>T9p(!?="
    "9Hr7-%5qz*o-xag|_ox6Z?RKr!0~FVdMqysYP*<ktUqi)qPztkr(_KWvy_=@q&=~m=8{v3uP1^`%PvM2>(i?"
    "BOq9xktvH+FCc0;bB-M>*-j<S4&K+K3im=Ve_Ucg~7A3=T8bQ&k9M{H+YLtO>bsQz{Jtr1`9x5#H}jWpv<ZI"
    "05@)tx6B!KNRK`gQNygM*vK0vEczg*k7GAhP+g;b8~|)crVGdXuQ`2?KvqEml%18HWN?O6g>eEOB1cHlo+CA"
    "gO)^jTHW$UYp;Q&qVPOy|Z|Va8_^^4=Ie$YVj{PbR*DTanrYAD8^9sU$;R=n`RM3EUn5}=MQ?5+7Hm{+4l_d"
    "!u$8t)!w7t8LZHhliqvS_dVSQAm!{OP};#FjMG-7NS0dwN-0LBpjLq_+(p?cHbociy?)ih*g*`uQ85a*C*T{"
    "^0ap(J{R19cTp!%D18nyC6}FN{>`~Dce-f`u9HRRk2lNW@OpHc=Q;kOC?H-NbjYgy4o~m4gYUCvzK!Z<l*}^"
    "w&@8WdT;YpUS^;i<H6&GbZ6|?vg*lJkc7~=8`;PiQAE}Pv=|K0sT4PGCGNLI_;R*^_k8uv?Jk<UdIzYrMr8^"
    "HD}0c>W~{MW$MetkTQ&KNZgkJ^F_2Hz9yDf`$nAXuC4_ub(dEHb^530XCDo62u{sayV07S|)HxYk%I>3XEX4"
    "EMNX6g91hdHD#`T8?ciJV{CZN*uB$_wban#Zu&11z{3rX;&#>Qt40NAY_AtV?4s#afxIze|!en_&EmBz_*}>"
    "iD?Fan=ZeXXVGfvv<4b+9*I0G=Q0=NJWT0w9pwS*xRem5oXf7A+Tm6Ez6(5;uyOEt+Pc+ngKAF7ra7^cu3)q"
    "3|E7FCQTen(az0ZX^-IM6x?1Tfaf`6OgD-lb2os5}i1-!)wg5$TG}LwYQHtBJ#EX`B7EVjx0=$Lh0oEm5H65"
    "mhQ1Gmb(`hLd668u;ENJtMi`7!*cQIwBjsvm8y{Q_3ZJDo<IK3UFt#Oo1i<k6g@$&V->+d=Tf9Sma=MHShf9"
    "~L0bmF2@$Yq#^rR=~HI@SXg(HP*;@um5yjHe%4?J-B3&=M#pC~=WRD+ui8V3aTc9hmV&xL}yl6_tA{Z`2i6b"
    "1BMo2H`A-S90iIo~>@@A{?d=YId80*8?9Sk-!As8~{rQ+k#*Vh;UIa#!)n2&0?4e|2|Ch1z5sv37Hh!yAWN)"
    "*(xvRLJ6CgnkY)S#92B|Wo3ewpf<P0U|r$pj<R$r$88a1K=?wzF=7pIBJPx3o_`poN<$8?erZk$_b`10{%~Y"
    "=A)y~8AjmhJ8CI&|rM8~KNKL~uOJj%$v<u!rI7*yQ$@THk#o+L2a6E=LMW8qji+N`qr%|>RQ*_RmR;Ib6SEk"
    "4!toJopl_JfsF|M(R{St^4%ShnMo5zni6SF%^U!!9`645G08-cn`j)RRXg#?ftOQS+iD{YoYPGj(q2O>)%^b"
    "o~)0gZj#6&mZtMF8t|nZ&S*^e$V^C7xZlE<o<Um_bzHH(ha9W(#;p@T(+Yyx^wPtE&f7Dh7-L#pGU|e+aD6r"
    "Q-v)KZWtra?kSkHcmlz{0Qse+=}w)Qe(V6V8@Rk9Q4~R>gOE!i|QCG3_g~jBW^41D8r6~aSpLNsa(Tb;d3G^"
    "H5SffvZ1^?9D1AOC;blQ1tn%EoEm=|ATdl~N19QkNMBf2RMQcoQ<IH2JHJ9zpl6E%0sB76K%#cS+cd*%(hAk"
    "Ojne><<keu48)we74%<q0NO4(ax!mVnOA@prAX1*5b@`X~w8+IoCfQm=S_SULgB8FcH(QlJ$U00HVL6=-_n<"
    "l8WkuX|oXeTaA+Bv?$7dBU3+Tl-fV+!0lJsn@1-f4h)5$8jmEa%b2Zlv`pNp`7H8I=^>7_c5Di_b<DGhFhQP"
    "v=pl0+sVoD_&@lhd~SjDT9aYMLdS$5eE~$y8l>6ed(fw2@F&e-%)5alUf4N~dL(8IS`s(Kt-L2w6b|J~4aE`"
    "g{rYL?A9QIB+g;Ej6^}MA&hg_D<TVeEf&-qdewC>iF9}C|Pc{2P|gDVgTubTZ&dXkVM2YU;-{Plta2O`B9kX"
    "v1FZ|7XM=sL!?qh*pd-VidI2MFkEYCWFG}A^Q~^u0{!sQ;gx7%d2(%A!uEi7h^KS1fYcuth+4GYh_R<%VUne"
    "~Z8jt4Qv7uOL9{|GaRX~hLQaItMUk!YsX7zdU3-kxIB}<d@5t1kCm)kllRkr@-moqh;z~k51KpdTuhs8V?Z="
    "Uv&E&KkK^9EUs_3rMf#5$_4?n^v_VpgRzP9H~Ey|m~Sv%a;%}$FBPnDBWE}RtyN(YS+h(j6@l>ID>aI0pd6M"
    "Ce%GosW^WkAs?rZ*Q~=~LuCiLGseAB&Gv{}H~NtqT0aYRQOoF>WkdtJbT-i1vbBnmb;FFhb*nku1ZQB8m`zq"
    "!-<7oE>Nd)6-P2yPgIe@{W?Q#W^3N8E%M8AFtM6(!YR7Ci8M*uq7T!18jQRucJwLS)J7aL%y)5n%6?PHId?k"
    "o~DCLG2&vCY9(vL>Q^hvZ+nhJ#J-^k$ZObHe*;uPT^)|J>NIn8Hc;wndK)!$sBXlVX<j-&%OFBAW_c2Rlq1z"
    "tF&25Y2KhxY2=ybDFK8a190}nmAU<^u)^SP_?&$E6G&xF4QAUc<nUEHPZ>LQ}PeRMI&{|2SgX{{d@piJ=3=^"
    "G*dy^Xa@$hQUgCBXQF&Cv2>ui-oLdK|&=1}w3RXCNdt+2N+E$G~*aPdSI1-xfYyE`-ksB~RyKSkvS_N0RNQG"
    "o=?P*4j+oKk8v;!CKFBkf~gfS~CrN6OKV;?Y2pxZ^iNeFX)Ctm!y#+GuJ^JZC^Zg!x3e`^&-U;OGirF5c8)O"
    "7;RQv~I(KAQJ~H4zh2oqih})YTn3tQ1(Y?5<N3x8Dgg~lsJae1F>3a<1kV26O)Y?kF<`g_Q#l2O7%{t`k*es"
    "Noiy*PN3BJ+28{;g)^e&ACxZlJfnkZ4OmH*-*%%(cbcUsDQKJ{FP*xQG<*W`!4Z=?yj1Ai4;SZewIx*u+f6&"
    "%J01LRCI0pN<V^VXlvqdN{7kH)E*-ujeH;LO+l`}2Yaf0ZTnymyI1(q9;_UtDsW`tl9$bh&{}sBHtz+#9YIk"
    "R{oX*1}X}O>b%M=FBG{T7hTAr=10oj5V15g-OI_!I(m(JG3jU~xy^R!Klk)Wj~bE1S%C2xH4ZI{*9U~Q{o6s"
    "(cYHn(|YqRsszdmC&lC_pVfl2uf$rK-DK625}To<+QX_`ML^Us#s6DCnitxgS$#K5+-N8H&*7dyFP4-*S2Qc"
    "3=&~8@QLTAZ!eTK><)oBrpO%<%4R;$%Aem2Z+?b#j7;_+e&UZ9Dac<#`}T32*fh$l0JhU9o;1mp><za-11G8"
    "9kg|`ezAROHAFZ>A(k1mu*|wn%hUpGXipR74PoZa&qUZIQi00Ld&LAgzc?VPr3NPu`~QSWirjCDzS!s8v0rK"
    "7=|aWaU2$tZDs7xO2BA~;?W!wr?md?nQL^PZW=&LDuzAc}aYsMcLXYPkfu>O$)kR)VQR!7+yz8l?Jp!Q47!d"
    "2O7B^#NwkXM}1>}(RyRZ<87*oT*!n+0ZLo0KqS;j}|gy?Jv2oM<CD_{)q6Q#^kF}Ce-z<c-kJ;#>aRg7j?KB"
    "8S3XX#jrgjJ!m3$1!GO*UJ<^d^V}<C{QIC1A-Y3Y5PpP7%F1KO5M37CUIhG)p5GnQ=Q11t!>A<HFgL6R^4fg"
    "T}0*g$!BKq&ZT{GokZ;9xp*(U=k$|ym30s(gG6Y0oWj2)4>rG*rf>WI@1Ir#vq^(hE%jb<tj*`f&4<{Oy3T%"
    "W{Xu(vNkAyX5J?tpDG_?RylA$3PGElIVbkr)l}WqhiAvGSkP75K`N+qi<04c$FbVdKL%>&uqen<*{bIX(2k?"
    "MicyuYrwk{WP3UH}7M#~34bu3w)-Bqe%Jf%Tt;aG3w;BYa*Ed-%l|&dLSUm}+AHC=a`AkmEE>DgJ#z}!j&Qh"
    "k1QPBFK^ZuVZul5UYWwv2Y<g$a%B5LDs6r&QVk6g@=Q=#1vJB=vp8`5zD$In~lE5e_N(llf>!}4jcDv@O$Rq"
    "{Wo6;=<?MngI&Yr)YZLC;`>)fl$z>2ZVQ%_AS55{l5MS@Q-rsXJ>hN~}nv1j+zRb562Mf6iqIUiK`BryJ)z7"
    "$899QrqtU4P0~RC>CE;`X(z4ww9PSXs~v60zO2sG>u5r0?)XL7ct}4Mc$LzV*;hcF}r8G+BfV+114biq;l>+"
    "thMJTvU=mvB~@?t)37<OTaWqC!`x7)!Hh&x-2+Cg8p|3GeAI_vT@6P%0*y}r68jo*qX+L&_Mln?Axq0RU0Do"
    "68n@`zw}>;M`obNC(hQunSnSa-FK&U@Omhb|Po0LS-t(`7EsuIEjH<E-0#gDd-)iztX#yO}vQBGKFA(u!A)^"
    "?r_8RF(tWS9KA={0C1dLHC;gn%-F!TfggGGj$GQ7=13x%EX?*J7J2*<YD762{t@J_0TAEozzh^>jia9Xa=Jr"
    "+(09kJU28E<4TNbwN*sxcTS>|#a#K<bN<%80+%ZE2I0VPHgG5S9RUDkfVHcBJCy!I-0XB5lv{&;zVt4hifU3"
    "@&ItixBKR2<QU4QlFQ@Jyd~2ku4<7hLfpXG6xOGK6FWkQ}j)y6#2A}Im~G7LeM*5My%)}H!4lhi6$j0GPPCZ"
    "5f@c6s=rNkRD@T*^0)7f6{eo3R{2lhO7vW8oEm+#sk`PoU+duTFxQf6#uFl$CDv8&m**nm95I)g?-~lERIb-"
    "wHN<IFeJWO`JimGyQQA|Vy^O7@wgynfEolQ<UN$pQa+D!EApyXYm0-eMWhi?;EE$SnkI-8Cahb_AxRUa}{DE"
    "lB1yqny;6WiY3~~30$lEVSWWeT~gU|R&9F0Y`_Nq2qoG&RqM{nvj=uK_cwy;#omT}4yZQH|AdmQWUk6G%0?J"
    "Ki8({nuF+_4^78l3p<_Y+>M(u#+{)<0(~h8cl*BQj9_*ANHCQM1OT=EKN_)#J1?OXAyk$-p}Jf8eB<Y~bX9B"
    "kDbbnEnCTf#UR5V+G@j_h)A(XFrbN;q)YMm2*TGj$K500v(2Iyi`F><?Bv>(+uo$6%@tI+wV$_k=sm%kQv5Z"
    "04mrW0Y1D=B16oNjfBP?I)Ga&mZf?V4#L9WQsHM9WD=&4axTO37+9T1?IMLc<Qslzb_x+=$z2vlg;<2?3Vey"
    "HTpyo^H{?oArKmYCK<D9vTRPs?8m+-&Un?AnN|+DN@zMF)cc%kDqXRMi^Xd7~U+^QeJO1JD<P>0P#7h*(CC4"
    "?<<lL?#1coqDOEi{5VyWhifWj0!M;W-7rjGsO;uYVvJ#Yj>i5e`8R#v}Xjr!>F9D7ng8YH5D52`%#Ri|}TQn"
    "5`{NtG~t-1Zyxze{Q8wdba0g$cF7Kr3RL3w>tV!tiZjAM3Sm{C-y-Ki>0SV*FH0z^^uNW+*`KK<Y?dt#43~%"
    "w>nAh;M)@>HT5KN@FFM&grL{ny^J_s12eSHdj@s8|rxeP+Fv=K-ABk%G~(o#d7>3oi(4593t(Vui9MYVcM;I"
    "QJ<CO_-MdAH%_7+hYN&Jl!F+FAlf{Eij5dKKmnL2eIa@UBJU!d#_&Vu{WqOgUm%4KR7f<@Ol+GZ<|z1oB$G0"
    "1j9-HY5<5BXnE(C);ckqis<%M3WyD<3-W%Ys;Hy(fOKDY3M`^a+DWL8xCI1BK0}2HsdmJ2I4uIE0jYew`rz="
    "RDc=c_2n7%#y_tC}R>f&SotcPD)IuCdTMmUF~GW#ggRy6795vn8&@OsF@wJte{_m}_#^d2m-G%I1JgSnlSG8"
    "bc|N-Q^H^aE&}f;_e=h!-5T>ID37a(1c1KeQWPJ@;(zK~Vq+69-a;A0><*B-5anUZSFli!%UAA3y;j&5zS*v"
    "Wk>Jj-0&k`2ScHr7B}opc&igh+BzXG<sLyzQf>&Til)ux<bSBJG)vK;D}X*8~k+#nXP~-$B|qtGvttf5$j~@"
    "ROX#ksyeJw(aO1K;k#29IyD8cx~M+E0n_7))tew<!?S|1CN#D{G^u^FSrTbqw*?H~0N^vN{@KVf1g<2gPFmv"
    "(l$5L|uTvUok9t5wn?zqc?yNXDyBu6xiIcOdb83zjX3;_$Q%S{9ShmH_ho|ocm!h@L5{VY_ejvVg>sqvUkG0"
    "y}xX37qrqwnM2?dD5sP(z8-6SkZa$%}SHkp@vkpp>G=w|Q=%d9BCl(BbJfM3a+&4pY}<7JH20@)ITIq+Y=lV"
    "se1yzO~EstwRr-&^AtM|~17!1hzzy0uW~`F;AwJ}XpL`udZ7m%>KqU+Wj>WFP()xEXq~Z_BCf+2Dhna;GP<u"
    "cybI5IkR^N!PMZTQ;;EO}fh!Ff^E)NX<Wn$Flltvb|cEBs|>XtS!5&f+f6L(Zu2#>lUQ50f$N)S>$6HN#L=9"
    "pY$IOt`4Aht<17mOSz;4TU3@unyq`*3Fj^lPg<8w`Er9{K$vXER#SK>7rI-lxTYPY1H5570JbzCno7)~U~;@"
    "CYECqm+iHVCW%jhKRZ$Gb*8R?PPTn0KUJW#<o46cYv31%P`)7j>`vJ+Z5q1dP{r%}_Agbk|Q~(m^xsD1634f"
    "L7g+R5q|4&`3(ft4$ZF?FeN2%`5(dpsITZnuBrzo{Rg1Z=nB~XKX?`Z?BZGCUWUYThHpJ`iM46fc^p#2-yRq"
    "|E}K!+u|%-z;UCp4iJZ$VN4az17;ao`nb!MAE|*GeoyCCb6H0T);;Rx6z}ZzvkHVl9{MzA}wfc8~QGvxDMvl"
    "6|Ud9X#Bf2nyUBEQT}{oV)k54h93OT4+0QNIU!3?a)5GoM{0$6ON0Amek3aXsIQs)>h^GQk|;$&^_TaZylNU"
    ">>xcHoXX(x`&dX<ZM5A3iv8$-K%zP}O;!b1$E=VcX$_cGU4hiO;CM=-P9e*H4Z<+$04*j}Rn|%)s8FQVV2~<"
    "vUXNLbbh}+q*|9LVW72Oht|COxS{a8X!3aXp#9=CAm?yBlRS7qAETQ03`Zp(vz`(sjj)lMik@5^Ptpvmh^f("
    "<-tsUT{$F#&S0wp$}Kk*iVbMWtpOk#N_BOUa&K~WrWUO4Gk^y7;cWMqMagdmgdix)UR6h0OvmWC;iH@h>9^X"
    "Uq@$4WIrOz}*gYZ|_Bj&EL@*h_aOvsCU|cpR<@_w2(!N`jtOsrKq|P6==XW6}-GlGZ0>PN6jP&SD-$T8m{}S"
    ";gOg2dh-}f*pUu*D*dgO)zY+HTKEvG+U3`2z}cGdYq79^{Gc*wotr?Vhkcr%NJ}`kb@CV(>c<)gQE_%8&%W@"
    "k${D#HHm@o-Bz$ky-pxp^^>})DHHg+P6U~5d6-~I@HZW0V?pAt3DAT^L4c*w8DX4o`63Mf+FTkSCG8d`<Kz;"
    "ii=g0rglh5_awV)daQyL`eGJ4{_s9TrRu~Qw=54CE<<$!t_XVu<7c7v?8hsI_(F<X0p%!^7Et3SzK!7J>PP@"
    "jF2%Nquz{6fM9IuNuoHmf$_l13`YW_l?J?K?z!(1BkY_cjls#7xz_EDg8oKg+}nyN~kU?mu~OZ76LRRY7QEG"
    "PvaBJ#|9ju7=3m&Tx8hf1*q;}ZHEiXScpmp@hBqtn5NH?GmB#|GtIBGD~CgTm`CTv8-7tAEQ}7Lqp+RP&SEY"
    "?+C%qS<J*$1leb)N)QF1O>L*jOUXuf$pPsJys<PG=%Mh5`}9zQ;A^BR^Le#u+JKcr8Zo9+-=)R01o$9f~oMm"
    "dVxGB>_@;>fe1Qz7Z@an3f~PHqu~lasEBy3dyfTkatW9tx96>RShM3Qhc_?{no&@q*QgV}>l>N7+qle8Ig=6"
    "6m#cdpRCpwvt)cFhLH^22+}ED}P#>f99EuGdayB#h7a-w%HQ^B}3~5RvK~E&Pr`Z&!V#b!@l+`()0S0C7u~L"
    ";s_;x6o(3rtOiuw(=PE?99ms(lWQ(}$oPDiL^bvB-*dP=ulY@J^<NaE2u{v;4~#P_RX;g-Q_j#0c14@&WSft"
    "cgdtwIH}_@GB8SO>Eu759*n^SX|SC2kP&*$GB&w#K6ykiizFQ8>bmngj)cWq7+1v69X3FH#~=H{}?XG%E?U;"
    "~VYvCV7}n=QQ?fVRQx#D#EjdCN!7$rVP`?RK*1)u+t!G#xRq(LIJa^9v>j*N#~=8Ei1rA!)%{xb1OTfYIlDh"
    ")Gc28wr=tA*)&q~{rztC`j!tqQ(hZPzSe6N{`YOm>^HAqesMaDUpL|b7)wcZT+6XEz_@Qc0WE0Pi}3oYkSi0"
    "=Lz=4Fb1;JXh4@y%uiLgc$UQ$+VU1?))=H&Uf@t$>jV=Z_0bf12GO$t&SaRar!0v6T*NX>-QWDsd?TIj@cmm"
    "p*fUOr)$HehIbRVPI+p0?NvJ2yqO2?sMi|uc^TkH4Q5R7}32aq8TXBFgRAE-W!_IhTyRChW2h#ww26n*@RW-"
    "Vt|eb5CwRM!o1p;XnUfEUQw?z~IS;;?UoHr|H$N9ALY`luDh=VyZe(1UokQH~jkHvx(5=<J4A!Nl1#C<2P1f"
    "!C{AgK@d(aYT?la}e*f;!qZ0B$+BscED`w{6TwhDSXE7^3i$75HZp*08X<}nx(R%u3yT$Ly8>Oq#9`eK!h)m"
    "g@E#Hm{*fZt$Yy(LV>VlSGx~sV#eBcbusw2_a_&FV*zz**I6EQ;LOD6e^r(fUU(W85TJ^t5l8*7qO0s}HbOt"
    "fOJxL+$&BNQh|z>2<;WnSzz&{IILH?nX6sCKkzfw8nXM9@PGI?=6)Ph{jYDVfzCs8YonW3oWwP6xfGT&NK|4"
    "L+Mo%Iyp3Fg}7FME&8hxGr9?3k0V&&R-i*1&~oXH9Fh-QFykPN~U%HQfVlu}MhrR_jMlp;SAEpCkmM+c$Nf="
    "|}rrcj#*DSWVE8ax+Rav+@?jZc*^Yt4w60wOb{jc2WMNINZJh-;ik@YMo4S%JlUQISHY1~@uW<RDT&fwGrL`"
    "BFiDj^agB^bt-t@d&8UFbo58yrfj)N0b>`cOWA$;w8e}3%hY+1!MBk;`L9??<k|dyQ<;*sHM2842Ac*f2bRk"
    "w&;1fSqYgsRO*#a(-c$gv`98KrO&K>%K~ZYIcI&>H<{gJz9MOb&xm`9t@6e6X*dz~pYO23+jmk$E9yFa=W60"
    "7mz5MfR5<O^-kk5woxi9$<^mCBcaTB_E6P1LK@Yk&o&H_^(4=A^3IQ@7mqn*T#u=qCWJ-Xuc~%J=sg5@_XuK"
    "B{>r4v?c#084fc2&5T<tcrc*ND6(pyg52ps^9#4sRup_YmS>cO$&Z&I7T%x2(9p)aA1LIR2=-&2B#!(OK5g|"
    "HT;r-5l$%3I*xspya+eDj^PT33&ScWY2$D;~wq*FlxOX``WDm9wWF)~H5aV63e!Zx~64AqA)%X&oR|5-S=X4"
    "-St-ID?=gE1*m3jX#6F88bw}rzVh0sKKddQAYE$H4Fx<Ej5Fu6toD;t<<n|6c^KY3H;btEZl;KdujnIriso%"
    "F0y>1`!eQaEtLkQPi9_GkeZ6{IBOD=4{b<eCQ@Mk%1lI{#n6VvI1x$}Bd(sR>WH$B0qfS~zn#XIstv_xJ3zB"
    "2Bxa?Qo3>DT$u!GziFm&4p^`cbh&6Wb>5s`Vf%;XaViB;E!BSqD(uPnauHICrp`zq9Hw_J#;Be6wI-flt2ZJ"
    "5aUQjTgp|eI7k7X|J;%o&3MC6!M333#&ib^pMn}|gmbug@K%Hk$e+(8BjR$NhhYGpG(dQqjx?z~99UWOV5@~"
    "%b?xLxL(5uGM~v{!qt4z=3X{=P3>HG;9C8jV!HvJz$DfT@i@YSV>P1kAmP6Yq`iVwaY#$N;p&JH87@hAk~yB"
    "FePS8T^V9o(c&*8Zi|p(+bk1{(&0R_g@y7)N0uGyC_+^C{&F|*p?XihL!l}Req4JuGd>bqj?mzH152nF9ne~"
    "Aevk7f_@d5@N0s8GlTlwn?C+7e;}5WEP?}+j75g^=s@Cg%CBnS*#ldwLtD%~he(!E7J8G?g%QD?BT~Z*#BdF"
    "^I`Tn-Vo2;Hca2@jYG?tmOyH@Du^;$CC_24v(P!y{7?b4agH*eq^x>3WSP7));KmS5eMHzlT3<JXYc^y_6k!"
    "@cz%dZ%+Q|~gR1x3GqU<ceNrm}wSjPxR06A7j7X!@&-VZz$bFApV1S6l9Fu{Lhh5ndJ{jrchCY<HwN9I7#f3"
    "7BMW9Y9SLSYssrV)rs)qkv3akQ<_65&gmB^m?F3!HAC*I$V%;3|=kcz1FvUU88NR4?cas7i`Bv}~;RC&#GLV"
    "BQ5WoKPu*uLu*0;v$?RWO&f71O@dClAyu2^dwF^Dp^_9^MR2W#Yf0b7#*FTT^*jB4K79}XD3&RlypyzZUEI2"
    "0v-#lDjP50=;ZW=R$Kg@%70n~6`)+obRH*gc~*31kmcUSLI?b9U#PmfS<x+I32gbxqIIq6tnjyS)cWq*Z@>P"
    "=`F2(R7S7f?e7>ph6o~8U7sI{I37I;bKhwW2<8-;gB3`$7xacl7RO)3tj}uvc(LX|QI_~!jNLuEw^mVDmtmv"
    "-8xLn3j3tuyUZ$6xz|8#ivqXD7Rjh0Z)itY%eu5|zxa5L2GsLb#uta+^JFv9TB+4f3B^+Q*YC8l-IJ%Dd@o3"
    "CEOK$*Q}G1uTeZ7CvLU=1KQ0h>o*8TMcm^WQH5RaFDcJ#%3(&B2i1!a#g;Hie{n*zlf!ZgnH(&K9P8?6)ccD"
    "Et&JR*R9UI22<z>8tNXU;p8ofRl>7<&W86fhQ-2uQAuWf^IRK5el3{agH)#^9F+SG4vKl?*NcPM_IOvQQ0@q"
    "n8)}|A`0b#?G+pffxP|%2%cY3wd*k7fZ=1aabgaxjw*v{Q>w-NLVSdPkVM2qMtodU&=z8=@=09gVZI^RX;yQ"
    "AocU~woKww~7-ITLmNqbbmOaP2pUR#BZZD(4pW&JnT`ZvXJ1miw74fIxo+@9Q%kFHIBxpEVnnym|yAC^lKkW"
    "S9gU&zSn2+6;y;0}p#lIBu@b$Of^{&It3@*6;?wbeLe>i-e*o=B9=Xd9qC;v_+A_7-QJdI2D4C2WM@xUkE2D"
    "`p>%c->^4#XeT@&otu*9X>aq*s0Q`Va7nyA<J@@4g*<`}KFu^-%4FHjpk)j)h-v!S3{eaf;}S;3Jp9CQaQ|Y"
    "PDj<zzhe3#E;T2Wez3X1O4jIJ8l4aMg8C29n}rmDv(+VZ0?f{@W4RNtd8Bqc}B|$>&?GAy!r{W50J|A8)HrM"
    "1Iolm&fpMjo4WJ1xNy{Am<j90E`kdM*kY~W-YShh^<KVwIoxY|Rb=OtN{CxQSc2b8lL3do1`#G|e;4pi>6Fa"
    "1-tv05cMOT?7o(rfFRzAsz^P?*Q19*u))>=j24wJBOYnNQM>jF@h8uxCQt+4F9(Y=H$l9SR?(e%N(ZhpVR%#"
    "k9QFo$cTCLp&1)FN$QOV-|w}JDXed}G_M5UPP9(GB4g!NQgL|gzCD)-u(GRxKA%JdeXv{kB09lo&=&2<iP+c"
    "{YtFCj;Dxcdj$|2a5lSBpSu^<FK3&Tvos9#y(7{O_AqyE~VkEHhu}g54|d7lZgcR;NX4naf%HsSoBymUNI=C"
    "C>ZYQ-lQc$zm^M*hruxV96+bMH}mE;g7bBgEig)NRTGJK)SmdPKqo6457WldRNPwdqz2rD)x~M1F?XEvXd>T"
    "Y7o&)gVt=XeSS7gfg0s+zpdoUz-YmU2fA)}oL1ohs^_c~tw2;oW({4~0};o?#{wk;J(qDoRcf`O(|Vh)jvo$"
    "AU(M1>{W|rlxC9!A!opL#TVFPvFS4k0kbU#bH+Hpd3eW@|zAFBr<w4DPwn{zUb~^dO`!#mE79AQcr0-g?+WI"
    "MZ6zUfw*=+~0=e+qEOke?J%lqpzTy~;7TlNo6Pa7`?qo{v5xO#tb{MaQwHe6E3=_-%Q4J_t<nsrjS?kw~8E>"
    "0x*H|n=6<EZH1Y1aSxwf71FEBfDk;~(kia#i$Sd4F2*DOvj|&0uNuzx$^5-8cR_3z_#1dI#Pg%f)i$dzqKzV"
    "%f{rL7IUq4brSw#Zgeif0z9~`1`&fl%Vp<Y`9l$ma;zqn4!q0{rj5ac?hDS?33LTz|=R+yiCh{gQWrb4Ha7H"
    "fUDhk;%n~DzSkz}&EEZWZe-T-?A?Fk)OkctqjOfj16sdUmGB97ZtVdPoT2u;C+`OJi)5bL7hPT*pTEDVUlyn"
    "KRmX#$F$2`S%zs8(3CVdmf~4p|7LeBEDdTG~nn1F*dK|rrrPq?i+efu8%=2(#Ct0e(E^xwX-X8l5Q>cU=G2m"
    "I*#+rvGu|{&b$ArjV@)v}@tDk!w8an=|o6B3^dCgnyiw0KPgP#XySEGx;;c?qTDTiPH3}esFLdI&Uejt5lEy"
    "7Q&gTRbKr#23?W8A~LhXZ_d@^&yfJ$ZX_<wHVsf_g5)kN%&h^HuthAPU&Cm7(b3ahE}2=r;%d{9SwNYhqKqX"
    "Z3=iF8K)p2wK=s^RvKe;R4IzPgO)<w_!VffA#MD)w2c;>|oVWWr=$pUSGUMY(!l27snfbotDzPae;IXUOlo#"
    "8b_$2f$zG2)UEJhuU>znn0Kzu&re6hvgI2qrWdlr(-pnOfWk<6jZ^-}DxSrd9oegmzk1FxgZ$qH3c~H9pA}u"
    "GF0s7T3x0`&dc>`FdGh05PEJqTjW7F-ZUXxF)pFpWU^E(EQB;eydnWwbw*MmDg_zpfs$A1(LN=I^s1+)kaf("
    "JID1SHJQs&je%YWkd&#>leeU{~Jr&s0OzY=3mbH^A<Xo_y!0m6g6cnq99A&VI*5dtq0XH_ru3xvbFRVr2}^a"
    "wCO|H(tUEY#43NuozNPD;99Hr#`e?*G4g_~+rDta-3)_p@*zi>mp)n;E~&od4p3kK~^`SeD68`RwX@Ub#RQv"
    "~2lKvnnj0tkP*B!*sP|*C9u{OtW+r=L-t}U`N%x%|SpYIATm(1knDKMh<=@6OI19?zsoZO}iUm>2{(rgUZ{;"
    "JO&%4?X<_xOs}S_4)@r4qH@<Y^N7v*(L(v-;1NDc31T^MFz6y{5cr~pW(S>Jm{_~#uFzrRR0Iaj_@3r3Q;~l"
    "ZgBox`YaVs56c5}A0XMp^s(FeEzlXvUN^>@mz?{9t6j$t8R+nLwRUk?h6jhHU*E<KD?>eep1jNq=ly3cP4?`"
    "MFc_35LVj8Z1)f0ILARRZo#VV7`=BfJLvw{fuUJiax;n0x;D}A_!D7`NH92bhxlG>rMQMJ|Zq=fuqDfun#US"
    "VGPAFP_xongve-i{v0MKsnW4*@?nH8Q9Q8N`%f85^#x4K=LEVE_f51wI|-38Ts>Bp!-;Q3YB4*6>KsGz?Q;N"
    "PQxsTP*Kpg>t7G%QH^}40Ty&^21WvDXtzk>q<_`q4+jj(QnjOPWA=_`zhvIE{ek8nu>fB_HV(=<3zwm6zoa("
    "_ZYGq;1)=2We+4es+ho=i&1FWHKs2gQGhKz2MZU!?utvGLea~9;f7#iu?tB5J{x=(i*TA}T)}E%k(snq)SxM"
    "f4sxA>!@?3uXaJ)iv9+{KJs6xs^)gG9K}8#zIXBu=Ec${Li&NBy%6Qd@I3eto-v(AXPcAk}8k{Z{o`ws<$4;"
    "-W8d2#P5RbWna8?depfcRdnhn9XW10qwv6f5PU`;(bh=Jxcn+aw<4RnwtQYAtl2TTX&gmgOWo5LVaq{yAowu"
    "2B8UZ_w>G0Q&|%aCi;FiC4jHMFTj_L;RYDbUc8Rc>W3Xe`iqA>e`!iL#Ow6<5OGqKoVn(5%vi{})0w5|zn;W"
    "EUwo{eZ_dp0YdcQi47`P+lsi+lVCSgy*{TmZNNnd!CEBl@iPFV4P-|VJyokg(;?OWADA9q+Jg?6h~~8mT@vn"
    "&HezU8K(3Xarp+qRe_Aiu*&c*<b_Wgk4T8X9RjHx%$$*e3S%evtWAjluHP8FK^6d9!_uzl%gJC6N>SjkPd{T"
    "pW|faje129iD6R&sYzYfZQ#(^_M|exSaLn{0$mGH<t!VQrb3CUmT3~GetkS6Ddm?8g4LiQZGiq7;LR2cDCnU"
    "&rFK6wi%v9I2@cVB-3TmVn-=mpwM=Y$6zA2z-SKs|I45QmicGHssaha`|di0rfyS=xu98&`q-9e>`QPJ3QJX"
    "qA_Hm6s^^;&8Q9T!D$k(M~cs3cCus)~QoEm$8ysYm;;E5LFgLj6FXn=bu|&dD{Thhdnt+DTEihIk7ZE}+b#I"
    "*dT>qQ)XhS(NZEZ6|05)TlNdY`g(Md!^E7#Zo4if(haRA`zt`ePIy~o<0>(mVP+D0PpPS`6Y!jSqB;I<g1gT"
    "zl=`Kt_BxBAD)gb2S?{;$3PhSx_huwHRcN?<jl(HHC2n|?@8cP%X!nH*I(}N=u3UNdFn@DrFZ0b1$n5r)lte"
    "tanu#>awcu1P?e+8lQ#?j^Ov#!1=Oj(%{qgvQ|YqP7<;o*!D(irTb3h-!t|Lf=|je1#MVuDCTHiW*fwRQdU3"
    "bImDHChRFUZE5&}n9R6<jyY+<dukcBK+$5d*WA|6vgrDaCcX?}&&<WX-^JaH>4?Gw4*v_gyLI#F+`%7R$OhN"
    "70(&ch6t96QT;m(hQ*U=Vf<H{@IwwC5pH>9uZM9?0<#w=!J;+Iji@-QeQqlgsl9EdNtW$u_D?NozaVFsTnoi"
    "B(E;W0>hdHCi~g4d+&6iBSC@ZTaC)`S_Z+K3|nK5ofvO)X;#DviaC6^o%tzfJ>n!E-D03)@ru%v7FXEhbC$^"
    "+%sJe_fESI9DWy4U?S$o7!aWDb|ab?KFvog+=!b7iQj+^8Y=?3Rs%$xf}UDWJ+KlXv7#>K7^SRKpn0k6)?!Y"
    "05z74}BkCzAt@}8Cm4c0G7oMbdvM6J$9zzx?K8j*Az_O(xYIBWL(Gsjb(~9#3Nt*-pJ*rf`1YKTjjip(c4$E"
    "C~MYgVzESs7vm8i%J`HOG-YpsClt=1x=W<GOufi-wfEj?*H9j3Onu4;lpB=l5Mi(Q_7`0qQ(dAPPBZ)qe(;U"
    "r8UlhZlInv3ylKWm0oEpg8gSUr6RX`my;SCHm$*g>B^XLjus#Rx&RxUmM60*D1f5UmO;K!p{2HD--;WDT{W#"
    "ErM18k&CtDGLyXqUav{n^1;y2GqcmK$_jf5t{reoO_<F2?PjpQ)0-t3dk3sC=Ad7zRjSdpJAiK_Hzt1<SI0v"
    "sH+O%2`=ln&>V^g2*CI;_Q`3KAQpAmK<rV-!yGy68^b-3uZY2Of>eGyF(-!Wq(WWMQ!|4ki_Yd_poWZYHrt7"
    "xla1$5P4JI3fJV&HHerX8-&t72-IbM&rUhCK8pczaSN((;i=xbyg}t@7N>P^DKq1vBv=%^<br0=)(B_gd%nG"
    "gR;K_6N!`0w|3r<_FL)lIRBSVTF67r3DI@y$q+N*l@lx=FG6oE>;Rq_ZUoK=-3)iGrz(W?Q~1ZBIcbpDB|V5"
    "DHGumYW_D;cLlvr(ns^MeU+AtE@jk)^`nym}IE>t3_GsT7*k1%y)Z_%yeu+%t>3KK|c!Em_O8UsuJ{&D>hor"
    "#?NQ6j8mD6FDXiS<PnjGgVfVgL}a?W=qvt3e~0<&qF;G3TE9&hGt^|^%WItn20=_au&EsE2uVxrza{#w{S5y"
    "8;HzSMA>SR$d0@Ex0bUTk<wsP4?VWlz}bp}VUJ{;*(th!CqQ4pD;4#@uH=wI_;b6l{1Oy*+*+uK2{!tA3HXW"
    "Ml<e=rYvDDm>K*N_J|JMdRDOl!tnV#U97RoIP)(?aEU2^D_avq2>sDa@_3s~ip<Pr;!t{g-REA|ON-UyF(`;"
    "RDr(A)@lZY7VL4eic*FsbpE@hjw?@vY@)D($(^#VYH(sDUZ^lwhC<U`w=1Ueh;83n-H>Q5DZ&`2=lS|~i2wr"
    "CS=i572aiyIdMZ7kL!3}FeBlvGxX{>9(;cHd-k&M;0ftQNoyb2%&e*EjwkKO0+xjp+qlTl|N0pE-WG0$G-o7"
    "_8paI`?o7zsOf<fgi0+sqc2EUB40^sQfWHi(@FioMi#gP*f`ya*y`eBNPK;g;wr3IAp{r93$ou!Mv#P%Q2BM"
    "k6#|m!zH+wzGhi0Kgk?;(gdX&hfdeV{6q{l2UQ~vv3{gx4l^SHV?nMg?ef^zO<C$LaoACPMrtT5jraFIh;@z"
    "BM!i<-rqq#4+Dw@#CXG_*Sy>*%DddM?)VXuM=?-@>SE7KBN%Ce}KgGqU*esGb{aB0NaDe!hz9kU#)l#pDSUX"
    "8Mw9Mv6K!p-c5Ooh9I@c8BA%}Z6wP@A2P}QMZwHB~AUE4jwRMQsF-crDRy@j?x{kg;TmH_ENXP{fHr;!JOZM"
    "iPh@EH{{kHaMXyTntD1`*A0we-M`VeiXo(b=78B^+uA8-f9L2}1)Bh(@)Z%g;2(tvVyumUh1PX2mxY1-A0r!"
    "YVfMC{ss6nc!{Og1$zdNWNOYkA_O^K-)kY`ZVK+4ASK?1UUhk1@&RS(DDT7osFwwzwk9i^vb2|ggN9oz!~aT"
    "R&B=wL1hvZOkZ1C=cF<u;9JLJOALD?QG<dRe}<e@t{dX**Gwq?AdyDvV<#}rA-65LpvjHxeMe=!88ktc9ku4"
    "w6G~T8X$8k4P(T}1U8}9vmf2gi0+`ks0mWc&bpG?;0>Wk6_1dDY7-QLHk)qV}?<Z_%K<_F$S=c+$zu&B##T~"
    "3{9*<tXecf+i-ZhfXOQT^uNg*%ZPHU!%%1qc%fYr0-;aBLiUW^UBZm4JO7%}zNYP?PR5AmLDlFGyVW8d=_*$"
    "v-bu>hcjW+MPY6r0DKgEU9DG|kR1Uq5onn+vPQJF2B}nP&iWkN7(4dWUEK(8hF9tkpg%75AEwxei{1mbi&X>"
    "Q<FS*Qq{MF+qgQ$P~k`EjJo#SuY8yan(fx)EWQ#KSCDZD8WtfMupH3d|3yO>~?%uk^>_pRLawfB3Z9T;eLI~"
    "wneW&%{aEQs0cb^DpgDSSD4yNu^yZH4a#0FRf_FMXT{n&M{S^OMU<L%tsUBzUE1cA4eo)ed?fT+pa#pA-?xt"
    "g<IV3IWv+2gRNMnvm}n;f%u(yfUI6p~Y>g4PSf%I(23})WL2*G<(4Amm$Ywyg9wZb&Iz;r)WlJJaGEkrb@fb"
    "AjlM~(=Qlf>|fQjLC_aK3GjDZ&ebh6PYlubQxtJB`sRyCGDx3t|oljXM!?chvS$KDzV-&^+G@w4lSdZH!z_+"
    "{W(>~<IX`dgE|Z;`(E7S)RYgL-;ml4VIC(5-bBCWe;98!_W8n+$Frz`%jeCynfA%%js2ff=C!Q$kh6v+OpW2"
    "2k;71=K6I%P_a9zKx+<7y&bezFTGR-1UnRjY+sxd0MDg<6<5zS<xbB0YY4q;k?YVYzEjv9)H4e%>gsnLdYi4"
    "T_8sj#2oTAFvaiEjHCL*JWQj6z$*hUl)eD`+jz=HMAv>hB<&dE?t&huM~&4ij5SS(R^{XA$0DF`PoY93Gz@_"
    "E<ts|*Z(W}Jcye|nUW%hKPdZ1)wL1Qb+)T1Ck51I3WAf}qs*V@=M8T~6NHw}L^t`~Za`-(3e1RxeUKR3qpoy"
    "&NRD*>>RmU{Mk&0|7kz@c{Tq-Y~n_?h8K|+t}&qU6{yI7%ZA?U1fS`wdF97Qr^Mjl?zJr^Li{WHKrr93vUgh"
    "FILMO+4;H-u%fB(d!%P5^MCe{}&iUkEU*Jxn2hB$-09)j;({3X;ZPV9|ms0__fPB7%XgHA_3US(p@Uj|T=Pq"
    "c3pjsJ((g0=OFGsDPq~Web+*YJu!YftcmtLiRW+0QYXixu0e!(V#%<c7<bn7Hq<yu9QGztNb`n6*!}g{%M)~"
    "x|6f3pwcbHSmh+`F&A22AAHtfIa)&C+K4V`(Lb&u2c=gv&kF^zMe9RpyqKt4X3NUu7Jru7cy$}!b6-RD=(g9"
    "{;g#IZb24IW*Y=ujsAp7eou2A++EFK-e)ejoTy-_!P1eyj8}3>A8-}DO?jgOG8uTgv?FYB8eXFR{J_0Irs}0"
    "|Q*?zDC-XCkhdz!W%yfN*qqODqL8!x9eBcRaimPUIv{QwNrY`ABuBKJC2#?eF{L`(Zm%oSQ4eF%gVCDBGuq9"
    "JXpl!*iymJu*0r?STFc*VaPVa*rzbb3ad0^xgLxVXZ_B)(l`t3nfde-VDi{Bb_U8tY;JiMwSs&61|*c^p`<F"
    "H5v5)V(8_{%s{!@(V@NKQ%&rUY5%dnxJ@x9)Z8DBoIr%e-sA%=d5h0S5%;VI7Q%5#}cbE0g|LzRPF?VDKk|G"
    "h}ivH5<Lo2OgBA<n6)3M7DUfFYesPzm!lE4&DDfd7+U3QVd?;fJq|eIfDG>UJ;N6_>fHPu@NlZc@HoejVolQ"
    "v)-d@Am3C|U13>xGgvb^rxBihwQT^I2mPripay{tPtM>K5&1a9TGb(f&4o(bdu6wt#e7}VFKMv0;WEwG<?{M"
    "$o@ex_2z($BA&#T)wM!<X<<3=uvWtJA;F5y+@MIm9G7QL4*!)4r6q6D%bUQV(SnYHY3{yp%t;J5Uy9`Fc;t6"
    "$-@Gc<Od8|IitQ1tJis01=Fb*?r`N%^xFyfekueMwG&hb?K+Uw^wp7Br1Ymx(@q*B7r34tB_O*5e<^DX=?44"
    "HLmb4=4}w`vKu}(b5l|WYMPHFI+SRC!-buTy|~B(uOZM;4}L;+-ocNrGd-Dvz(sx3XEpJW3d7|d~nb`R%LjI"
    "0_#};=r{Zc=F-Qu@Rc}xcOu?joEDnEOjjxBav-Aow4~TYco3-ncsW!t$)Z>mUEc=s%os0!u2u0ym1Na8J=7y"
    "Z%fO-zg#kvnP7L^3hr+;~kvd9?mj2Wel;csz@;Xk-o2umGIo8)>xu$f}0*a;<Doc7sfgm+g<@x>y^sqDiXKa"
    "kF;t>`~eVV7Mh0NorQ2;Q@8^XO@a!vJ;$kE|OCQq`m2Pah^w!d&SL!x=S+NSISkfrt!JX8l0NSH}hDhX~CG6"
    "#IDnd)yxMt?%Bb|Un@9Gni0uEY!R!^QbqVKfcyw*t$!7#e-q2cL~-@n3-r6?TztJWCyB5ah=R3yKuOPuD6Er"
    "T}--pU9$t|F!(*=sPfQF&;@xzRuXsuzDP1aFHoJ$x5YOIGcWlu{b%q99&$9le4Sy#*xr(O=ST1E!yJe!_)VJ"
    "OVRp1_&)GfC_O(DN9SihoSqzAsW#f;_?+v_p@R&Y>|=qgNJd@f3Fd1(a8+hO)iY`*um@HAUVL-lq4BTUwI~w"
    "-2T!ofL4+wK6K7il!hKG=$&7o-f_;Kl@pu5tMo(L===7)2b(gO53Ql=UWm1}<U`^_&Vk0o-2xU6J6It=~iWD"
    "6;5ly;V$3$=5Y&L_Wr-mWn_IB98_1()Y_qkKo2R9~sfLtPM#ji*W1pc+G?4qh^chpf7%@6m(J$eH2qu?h{Bb"
    "kn{Cte*Kz~8L?z&Ee)3uf^ctyG&mV82lX0`_hmYFw}t^@DE=aPZrT6q&z8GGs?*+C`bCo5cjo1w-YG$L%bJ2"
    "!0b}{KVocIRy{zF3ev&e|O+#mN0+%{z)>%5id>W0CKX?EA1%gS-?~9xT^PU3-)<@I|uEjuU2cvt~Ugl1y4<!"
    "s%6@HOg*LRSUufM`D|Jcgkje7Px_Vo%=DRcU;RVvmUt#mnbKEO6)Mx_AH~7bdaZ^C*K4_{J^L$hoKawQTug!"
    "J27+zEJc(s47IHCxBX~wdmXYWM0#K0*8FOn$U{qU-3$a|W>)7VlAnyl-MM4fI7e(?qx7`H*9CiFxU`C_zDyC"
    "qkGwxb?1f~jS^8(?kKW~Moo-{0xf(jR!`drjeOwaN4aym!<mwPpQx#K(0^DLXJ@;fy3c`wXwL=aus^xn#&|N"
    "o1-_UE>(EB&va%1lkF6k|6{+hxjjZHwJ?w~@7yvYCwOAwdEXQ9}R?fKoIX{qH;9dE5tZLDNoV`b%Pwz~z0Md"
    "+xdC@qPTXj3clg04&xSFFp+IoPaISx|w7}6Ns)OEunKU2|tH1X)ox4KV&pUAp25cRrodPhVw3sbbpW9Pa8_="
    "@Z*g4=}u<7An(U8{JgW~n9g=4i`V(|X9K7fegX$8J!QED{xbE*2Kr#GIZ}u)49el>M~`y&JWU?W-hK+BKW0Q"
    "KabW7HcXnn;`|u>6D<7HM8!?{vk5ORQ;nU9+kEl(6auspn4*CE&ZF-;lUTY?3*@>Tmm&!Si9ZMb1!Yt{5-Uh"
    "mOEveu%k9|P0*0$0775mYzdGX@Y!(P1Tl=p62UwYr7mYfTh*rL5g56%}gQ}xvOUG!o&>@B8sCds8IrI^D40`"
    "g^U2+L7(s2}2UY|e_SqQ2aDAPKTTLsJ7T(`V<iv(q<c|Me`ico(O$i|Ex)*p7t38odb6`Sm+gMh6+A#I7C$Y"
    "L(UV*&#J9P~MZjP_QZHNAJg3on$u|`2FahXa?P9hUzxA3;RowYP)j{fwpsdIy-waJAGynZAE8szbK1wG~DY-"
    "mKnf(8!fp@^i}jy5-2n&5m;GNVM_I=8e1DNp$|=4^xs!k=ggePwd#hN^gtDH;J7bUrVbl;&<!GX1-}Asw&xW"
    "VuxC7ki{dx`(ebX!`}p6x$v1y~{`@_~73TBeoj?>e)viFgYVw`&-Nq>AuNtZ)qG0TMKd(?6?)G-=2V00{4`z"
    "X+-S!9}-KNPxno$3q{0-5X^?g43cG(D<F`gISax<$zBX^a~yT%LezVe=2!~HB60fUpSj>t+8c`)_*UR+HX(4"
    "fE0u55N-pSOP!eD{a5|2cd6<C(txIr^2Ym*le;S3e>6e;dhAr+;%qzol?>a~#5q$|DWLqMXpzqrYI2o^^a0a"
    "y|-Aot^i+gs1L?J(P_U;w5uyYFj2fYOS?=CTXVK$;oHs<L}AEr^Op<4nv&=2rc$z<12VCPYnKAgl5NKVqq3W"
    "^UD6GrDOJS^TEDfFsfw|I2L=~J49#>Puo`{BrdV&%_1qwe34}Jmb?-!{_w+=@PNzQVu(!0%^~SAbtKefyJk~"
    "X3Zb{feoKHoQQVar!2M+@B=pJ}&MwjiU@=+id#~d>XUx=3A~D)D6Z_@&G373vSLM$S6nfr60LrAH4@nGMP5t"
    "gwT&LfosEyPj5P#G`*PFA#HE?VLb0b$}nPVUaVaLr3bs%1wSduIUlP*c?b(L*-!7O}Tb+c2DwTEr`&h%Z*aL"
    "F5+m`Q#24jB)e_aGj4#IZl_I8w@q#-Rkpdc8?+V@j#$=Xea`f(s?ENdqtd30*88`_&|*92mwOUs4`JfI3MCo"
    "9twnhn>%Y?|@<J7P-r{dBx1GEHGquArsL~esK5elbfJ(D%&uvLMb|al8eg$SC)A#Fdn{qxZ2{_`M1Og+$$6U"
    "EzT1Rg0i48L02SHSl^H6fO;Qf1Qqhn1#2*7+3L`LlBNn|8PRmokiZ<@+M$C-Vj*UMC*4G4g!l*Y2=La^uw#?"
    "jCWpxTATyQzI)HBwP2=Zk2IW`*p*Q=uOMP~=>lWFhH+1srES~@2&x>#W^vA=~q3EZ@zzuLV+H5CY%n|?PE*w"
    "gE^h@{iRs)^>c8Ad2-nHG|BzXQN5A5=Hv~s#7Xx*4<PFJs{d?#nI##xiz)^Lnreo=jer<@^sX<RsOKBSV!g{"
    "#2CXx9B&s9`nc{4UINv_Nv^FEe+R@5bb>)yMQ2>-H~fV?A;n&x92xZXSX>Hi=l?7Dd@Cq>1^_)4p6Uqjo6Qt"
    ";{=m$N<q%N6ok7`O4Of9xkoap4MU3$>3>yPmKZD74>ME)C1TV+dX(Z>vUbEO(%QjfLW0(u`um<(;~|<_Kqj{"
    "#rBTa>(<!Y2gXqlW2bpo*bJJzD-`V9Bv`JNGUs`>$69O`q{s3fr*&>!La9<&1w@Ydftku-8*(PVQA;g!NoqS"
    "IbvSB03f-WZ3B^hdSO;mMDa%D3SCmk>e=(U^bbiDI+eZ_gkEf8(n>03)yyj-KwPLWNNV${vFS#Lq>84j4pQp"
    "X1dhMF%uTY+)M!K=g$I;)jBB4qr{)X#bZmJj?acc&FdLIJ1YrfP#uMRW3`_R7)O}4!9Mn5>0;#!k!a$i<SO|"
    "B)IoUBI6#J}yNsBl0zdIE2dP+C)lvc>|&P^=YmavflCf3BVVA$_3khG3BghyCQKE~Wgg7<k;?6_xj)LH)9c^"
    "95yNi<PCmJMpElq3$Ju)4ABhd3m!h7u;HvWJ?CijmR>Z%HdAuV8<Zq`UvSQV~SzP_>*7F9@EriHX^0Cl+6qJ"
    "aD-Kw=JD++i6dAU6Z$Wo_e0mJ&|M{y@^MGBG_<B<j;UGuEzM4Q;auAgRv^7X=HT;B57l0^`86G{RuI0RBksj"
    ">Z{+uJ!5%c(Gt^<XC@C&J7|cHwC@L(Y6?OstG)Kf(XLd?kU9Z<=h2!O$=$O`1y6{`GZB;bdBzXMd@2354IXk"
    "TP<=^-G?EDBYSN|$AHulH5FFRa(*)=y7jNN0lG>SK%$9QK9J99dQfDJlwH8{cl-#vdnq7UA^c;A_eYpSwUIW"
    "@PMg*A=&7v4=s5Z-rd%#7R(sI3uA2){J+{)qm{!enIpKj9wA)|}OE#a?|n+#R;0wNQW3R@5Z=j+(Ss0%Ow-N"
    "sc25ogp-&qk2w03v}h2eirjH&#u$;!#YnV3N)a7RRjk{kY_9fR2{Dw#CnC<iiITWB|(u}q=muKf~0|+w*LAr"
    "E+E=pzg`X*79E%lu?3+85mi&>#HB?vU|>>eYUQgb5<|uCiVQaejG%f`?GZtS#q$FGP==Z<+VnkXahK+(K~)M"
    "MgS?<st?+MCxqix6kyj2U1OvI_HS?1$P&+O+4lxE9_6%*QcpLKq*!v@CX!$aT8L^}w=S|As@ebfF!@rjW0mz"
    "NGCYxL0EGGi{MR8ie+4!;4)-nt+;Bp3W<N(IenJ`q!oq2(MPq7D7>)yvXVaJ!dhCt@A8YqPz5*xt42}xR&rd"
    "lVG5msG>R7l+Pc+#8_=sb@adOogF2OR{Z>JzM+!{32|ce^x<$}E)QHN8%?J~flCfe#3E3g~7Wp;%)yFR;KMy"
    "|?Aiii(StuL)!uqkKTFSyILK25g_`Na2#r>NJ1&aw?onTyOGhW!Z2xJEAE%)a{Dp%(In#eI7TPUc0b|zs9ni"
    "0B!lcWHWw`<N@M&Q|8GydMQ^RUH^v2;=Y2hNVOqgG@=D?3DpSW8TH=RF&C37Kys{jO9r5TDCWPClJ<ygEWyx"
    "kGeuQ9YHPre3J+doJRTdEtc-?B00;5<19IV5dyxM}c$+GarIU2#@dHZXi55YU4Y5gX3V_tf<s9wZLux+mvXl"
    "U;iBgM5oO<9}pfP6OpTE6$dGXWY^v%WW_0`*ppQK_V2={V6ySRLFc{Mv@(v~f-#p;Hl9!%+SdH(YCY;pefpK"
    "o5jnSrjwI3Nd)f!`WMw#BC*%fN-;B`XOSuzN|7)#*er8wj^i>CKYt27zR({5r05!_tCD87zZH#DxQvUd6@8C"
    "|2RnwwoAYRm+c2FsWeVHP&55wK)2TQdl+&-bO&)%CJa?7~oA{&5D+m)vj<Vjh6-xW6*<@If@i4a}>4EGKt+s"
    "5G{sd213R3&5xq%DwPv}GN6Fbnbq5}rYWh{6>b!fK=cTBC0cXO1+cRUe(88#EY2@xm$Qrio1HG+o-HnB#D#)"
    "ekAX=Xb1B~_@;@2ZoA~$N{*eq>r;*b)7k)!7Z0QLlzB`03@pZZ&YYz&2LMjG;J}^}7QhRO0=N{O}KX6hk^}%"
    "IS7GM!XBg+wkIx%R-gTrCM8!)Zc@s{t;3Uz|k$BRW%?b0N>TP&8m%?Mp0Ue#s3YtpC!BI(J798{#|YXS#H1V"
    "~^9Kw(N}yfCW)p2I?wa!>h)gpy+fsCAy*t_YmsE)(pI<#-I>ksIOdU=}!52PfMqz0Q7?LtOxD8Ox$7cQ+dax"
    "<a{{5W-|XLz$!qeh``zbIep}UFO(<B;P!5X5(Nzz)%_&33(wc_KL_c%b)!oCtyajHb~p1q*&f%^do#&tJ<Zm"
    "OF2QU;PK{3kfGNPb(8Yp!y+sX@f4U<d+Dr=BM3COYfxliR%y{75I1p^<iG+Z(nvaoMIdx!)B{QI_83CV@lB*"
    "o1gpqlD#!HVvtODjsT7wGh~d`*ZX-dnM0t9XHQ6nutV-jWojKH!D`+Gkpb%2Ln}(?l96YfTBTn>oQ%0-wMoc"
    "G>IMOU+J{5fWN@B+bIZTa9<&%@jW#ffb${Lx1jus^;u2NWtd!cfWvGVvpGiVpiX?jaUG~{G<4{2Rc*j=(&L?"
    "Xc2>m&8J*ej-#T?S9NHvDhR+T~;iFVSHibISZ^cDrp9z&AjT5}I?~!noJ_QqXMxiK>{=VW;8C-4@uJF@epb)"
    "CaHqz2ql&0AK|`0gdwz`AK$7Ds!N14uLhS5;hnf&MSjf0C~a9fLez2C$m)eu*_dDfveOETOfvX7o3~~5-BMI"
    "k7(AcK9`;ANw8G{#Bb4U=zf!uIL6!;*yLKEp$J8g_}Yp*wcBmT5%yZiFfw;L^b5sUdhXX6NtzU6Qqy$`LuJB"
    "I8KMV>{R;&}HSdDPH;w#`=_Xf2KaB>0MQ#PF^{^jHF|UsGg+)^iydNEU32WAuh66Tjx#s`9!rGFw&V6(gq}P"
    "Iw)AJY-<61U7Lg7uzze6VY*3K~w*FSHT(edsBiFiSB?%8g~z@?!KbT>}{-#I1c^rv5GPivxB>47-3GMeBACr"
    "&FZwiDjYCM?b3e3O@}!TjhLiJM&?FR9zCrm=Jiw!28ibXR*A&<{<gI@kouYyO=T-0C*2$cVdceJTI`^pDYS)"
    "su-P{VXy1n%Cs3=ts5`m#@S#bXmmPdIQ7mYx^Om>@KZShdht~W}};LUFwBFV6+IU6xM2qgIYXP?pdCLpD<eC"
    "ezv^J;9-QLT9EZt)YNTOGyF9SCwp0g{H94zfSIlNCN0t`&d1T!eJN%Hvz4lW={dVdF_B}2xR!Pm4KFM1vO2@"
    "vs3|428wMy}?9dv6N5L5LZvV#RCSZwRoNt7*i5@wxkow*{MBhZ^HC;;`PWnHcSVdyooa5VM($7;P-b<N%gV="
    "g(e6h)~r66bw=_@CxTOll>?XH4fijGs6{Te=G^xO4~x|26fR>#X|VC~cVfqn^e5vvF5d4ahjL<!P$2ep?Ea6"
    "{B#NV{acaGN)5I@B5^m5>ov-n#?iVliDe`ecRsIc9A*Mk0vycW%H+hGlf@PrBw?uTe-&5YVq^l;C&}gh2)eB"
    "o~frQ1i_NK4bGNU;LsPyF)H}=*W0pyn6e?*(tmsm#^QR&*=8+{yM4l1uqw{pshP8E&xRe@L;_ke<Ix2-sax*"
    "_@(+!7!97`mDeBShGixPrj#d*&Jm&<bA<ay!*%AkJ#x$u&0fO$EHLLIOF>3*)KrYY7<7)WiRs<-bB{gYHB@~"
    "q5q9<h@3qVS{U_u~V41<l8jx|px)l0lSIS4~3zROmSkl?4UE2#xXKQb(@}{DOKD9aM>e;3r>jVef&8}{3;|y"
    "vm3=K8Ao9jD2dcY$V5khYfWvP|+PBt1c+vPfTnWoXw7xJZ$0>~sGzQ9!R3HwOciyJ#9ikg*G^`JL=+=a+7Zu"
    "P8CR<rG;)Viw_aIF*<h?~|GQ6W?z__d1xhJ*tXI@^lkwqY}zt4?B^pIzBCMXd9a(@y#KToz-Q&V|l0P$(C!k"
    "z^K_KcJuO+0XhaV}dV2_PGdz=a+R+DSToXv%^g*?eYoplR?U%)~1^u4S%JUPVQlj{%Wn&o7=wDo;*B_9)XFb"
    "(c@FsuypNS?T|5frvbxG6!BD}G2%8QYi>;RZj@Iw7!o=XKkZibN%zn(eaGIUv<Jf>`pb`c7Ghhf2GZQr7uy5"
    "Rnm*=p4i_fgp=*_x0Z#^>y!69zWiqoprkJ9Q8EK(A82P8OtJ&+T*(vJ?K&2wAN|B*;p2iCF8QrEajJMK4ADe"
    "^j7za2!#dJ6=k4;aLUBxCu=GHun#5JW@suCOK=wF<$kA6e6P|@4VlRUjk^QexCWL5rbI7MS5xvV$%zuTH|TM"
    "#?A>kZ{k2#&(|**t<&U@#NRb)MMvnra89;;Lki8J#wg#YuqqI%eo{s9GjqFkI#w0HJ*)w6H<7Zq)V?Axoptm"
    "grovTO%Q;mWl70{}MZ|9X?f^f7n(;>uF4KzG|fDNH5W1YSkPX`?!m9R30aSokCN2*kyuRlL<e4W!&6Ouvg%u"
    "D{C3`cFvj#a^A6B+L^u1y@8IzI(TH+iQ}9vMOwQdE5^@96FVbGdcWh~THm)M6ckvArjg%gr=;6m>w!|Mwsnc"
    "ncyzz;&NEME=<;{j3gp7xqGhCrv*+b;R5AB!OviWhTkZ`1)kzhSM^%?@$#d|Vsl4NT+;Nc&ohfhUr8&PDop>"
    "Fx>;z0x+IrFJ`-$6keNlyi%4ExYV03IHT`4t=nhb*~UuY1tovHp~q^mHM|BtxADgAe(jZSUj4t;hXABb8unr"
    "x(jyOm8alA8(ny_Xv9r^vkBiOk^`Q`*fMKsuIgg*T8nHZCN(YCO4mE4-(-g>!ctEdxRCrQEYAK!${S-K?A;F"
    "m&V7pA54aVnoMamP>;jmv}eZf$mV4T2%}JbK5GVpp&)rnMQ^J(TBv9%`i=X*~XPcJvW>5)(K^c(qooJn&NS^"
    "v?&2gkJQ)@Jo9Diep4bYUuGNwU2n>=7MAFYLrvh&;aS)f8uB4@PXa<hOwUV#kQ|HNY!Bk(c7uZO6iEz}!%Lj"
    "2!|C%J+Ft>%AXR4G6+tr*t_AsV=pf@fnQ?WCJF04Or9znKZ=FQTE(6gj<-3b^BCDf|+24P-oSi!1NZ$MqSye"
    "V9a52@W2&y>AuG@a?WgM-T(7~~J4Lv9F{V9c*88qw_@_^L1?`+q#Id;jgjfGH`P|=YHk|wTqtGdaW9bNb$8Z"
    "}=C<EGUm=5;G138rQPj04vijl~kB4WGY~DiY_~j2#Mv*YQgvY?w40@GxRqF<UX#OaX{CRo)3pQ&Cb1R#QeF3"
    "I@=SY&1~<V-7b67knHoCHk;5_X%dk;KZY-C{N0*lX0dIU^tsN8f&P-)dg#0U{_GG5fh*wpp=S2%%C0!_-L%6"
    "_tHq%F*Ccc<#S9@fd#tHY6LOBM(8HB?Fzh<Cgj=DU*-e|{%SB>j=<Nj-gsJ&g<0$*!^&}8qPBz*oTRA?c^EK"
    "98&r8TBO>mA^S8Wi?<@N=h}6|W55KGG)N9NLB=)Zn0Foj^gX(B0B^*m~FBT5|usg=0jBa)W_0|gOV|&B>Kq{"
    "|>pJcFfIeNy<`ndT$!%-`d04DJ^zM+H(LvQ9fgv^fePBh&Rd?-LIh^vG1oQ#tSXKTZSNLFTU<C=USX1j=P)u"
    "n>il$A;k&saFHE~-}Q;ksF0hezRlD6o@UueP-P$p8OM^%M@i$zzW5-QJAMp&)^G3ai;{zi!~oNt8=7(pb-A<"
    "2}QSJo}KQ+h>p%I%qNP*eszCo?oE0-jvxIc6=y;XNKJNU0iecj&d=sC(+}xQS^+n+{GU3PWW*6v^S2o<7r;B"
    "GZK5!0hA_g>LllNeOpo$?nK9a36XKI(NDj%U%Y<w!no0+Kko?olAe~9Lz33cMI@b5o1@58IDX7HoRrY<cIyF"
    "LOR&RLTcUh8xd{{5ZO<AHq+VFJAScII3GFk|DCllu_IwmYtXg;fF-hm+XUh2e=uL5#)=fq#a8up~n7ip<M81"
    "<5qu4@K+EfohHOrKUdqm%?k0bn?brni*kgI^jJkWcj9-MmjbR@G$t0Y6UnIA5uH^y4j9L}87hWWASH0sTb{A"
    "Bdk?K_h|b^5Z0LL5oA!2^8MIlbG*wnz5PZ4uf(e~IixvOoE@gEL;@SzUqW(4pUykvgNwn{v0_jN2ux7@ry?P"
    "Tgnj4xU!2UE1WD83TIMd%tcM4TXNn@<Z|x%!{L^{|Ca|DlP"
)
course_bytes = zlib.decompress(base64.b85decode(COURSE_ARCHIVE))
if (
    hashlib.sha256(course_bytes).hexdigest()
    != "f7de1fc6ed4824959f88b9563502b5cbafc6c80d1bd39db6601b30fbd02ae9b6"
):
    raise ValueError("Embedded course files failed their integrity check")
course_files = json.loads(course_bytes)
if "COURSE_START_DIRECTORY" not in globals():
    COURSE_START_DIRECTORY = Path.cwd().resolve()
if "COURSE_RUNTIME_DIRECTORY" not in globals():
    COURSE_RUNTIME_DIRECTORY = tempfile.TemporaryDirectory(prefix="lucy-practical-runtime-")
COURSE_ROOT = Path(COURSE_RUNTIME_DIRECTORY.name)
for course_relative, course_content in course_files.items():
    course_target = COURSE_ROOT / course_relative
    if not course_target.resolve().is_relative_to(COURSE_ROOT.resolve()):
        raise ValueError("Invalid embedded relative path")
    course_target.parent.mkdir(parents=True, exist_ok=True)
    course_target.write_text(course_content, encoding="utf-8")
for course_import_path in (COURSE_ROOT, COURSE_ROOT / "src"):
    if str(course_import_path) not in sys.path:
        sys.path.insert(0, str(course_import_path))
COURSE_WORK = COURSE_START_DIRECTORY / "practical-work" / "ch19-b"
COURSE_WORK.mkdir(parents=True, exist_ok=True)
os.chdir(COURSE_WORK)
ROOT = COURSE_ROOT
print("Python", sys.version.split()[0], "Pydantic", pydantic.__version__)
print("Offline teaching files ready:", len(course_files))
print("Save your work here:", COURSE_WORK)

</details>


## Commit to a prediction before the examples


In [ ]:
prediction_notes = {
    "prediction": "Write the expected behavior before running the worked example.",
    "reason": "Name the input and rule behind that prediction.",
    "falsifier": "Name an observation that would prove the explanation wrong.",
    "revision": "After execution, explain what changed in your understanding.",
}

### Reading the Python vocabulary used in this notebook

You need basic assignments, `if`, loops, functions, lists and dictionaries. The less familiar
features used by the supplied code are introduced here. A **library** is reusable code that
Python can import. The **standard library** ships with Python; Pydantic is an additional package.
An import makes a name available, but does not mean that you have completed the exercise.

**JSON** is text for exchanging structured values. A Python dictionary is an in-memory object;
the JSON representation is a string. Use `json.dumps` to encode and `json.loads` to decode.
Decoding proves that text has valid JSON syntax, not that its fields match our business contract.
Predict which of the following two decoded objects could describe a stock count.

In [ ]:
import json

intro_data = {"sku": "MANGO", "count": 3}
intro_text = json.dumps(intro_data, sort_keys=True)
print(type(intro_data).__name__, type(intro_text).__name__, intro_text)
print(json.loads(intro_text))
print("Also valid JSON:", json.loads('["not", "a", "stock", "record"]'))
assert json.loads(intro_text) == intro_data

The first result is a dictionary; the second is a list. Before indexing a decoded object,
check the shape that your function promises to accept. An **exception** interrupts the normal
path. `raise ValueError(...)` refuses an invalid value; `try`/`except` lets a caller inspect that
expected refusal. Catch the expected class, rather than turning every programming error into
apparent success. `finally` runs cleanup even when an earlier operation raises.

An **annotation**, such as `count: int`, documents the expected type. It does not by itself
enforce the type at runtime. A **class** defines a kind of object; an instance holds one object's
data. `@dataclass` asks Python to generate routine construction and comparison methods from
annotated fields. `frozen=True` prevents ordinary reassignment of the instance's fields; it does
not make every object nested inside those fields immutable. A **method** is a function attached
to a class; `self` refers to the instance receiving the call.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class IntroObservation:
    operation: str
    count: int


intro_observation = IntroObservation("count-mango", 3)
print(intro_observation.operation, intro_observation.count)
assert intro_observation == IntroObservation("count-mango", 3)

A **callback** is a function passed to another function. This is how the classroom harness
invokes *your* implementation. The argument `candidate` below is a function object; parentheses
perform the call. Predict the two answers before execution, then trace the result to the callback.

In [ ]:
def intro_apply(candidate, value):
    return {"input": value, "observed": candidate(value)}


def intro_double(value):
    return value * 2


print(intro_apply(intro_double, 3))
print(intro_apply(lambda value: value + 2, 3))
assert intro_apply(intro_double, 3)["observed"] == 6

`lambda value: value + 2` is a small anonymous function. A **closure** is a function that retains
access to values from its surrounding scope. It can bind a tool to a shop snapshot. A shallow
copy duplicates only the outer container; `copy.deepcopy` also copies nested containers used
in these fixtures. A **set** stores distinct values; `required <= allowed` asks whether every
required item is allowed. `frozenset` is the corresponding immutable set. A tuple groups ordered
values; `(value,)` is a one-item tuple, including the comma.

**Paths and cleanup.** `Path` represents a filesystem location. `path / "file.json"` constructs
a child path; `read_text` and `write_text` read and write text. A context manager, used with
`with`, manages entry and exit. A temporary-directory context removes its contents on exit.
Save your submission outside temporary runtime directories. Reopening a file is different from
reusing a Python variable: the former tests retained bytes, while the latter only tests this kernel.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as intro_folder:
    intro_path = Path(intro_folder) / "observation.json"
    intro_path.write_text(json.dumps(intro_data), encoding="utf-8")
    intro_reopened = json.loads(intro_path.read_text(encoding="utf-8"))
    assert intro_reopened == intro_data
    print("Read from a file:", intro_reopened)

**Retrieval check:** explain JSON versus a dictionary, annotation versus validation, class versus
instance, and defining a callback versus invoking it. Change the callback above so an incorrect
implementation visibly changes the observed output. This distinction will matter when grading
your connected work. Reference: Python's [JSON](https://docs.python.org/3.14/library/json.html),
[dataclasses](https://docs.python.org/3.14/library/dataclasses.html), and
[pathlib](https://docs.python.org/3.14/library/pathlib.html) documentation.

### Pydantic: turn an input dictionary into a checked object

Pydantic is an additional Python library for validating data. Its **model** is a class describing
fields, not a neural network. Inherit from `BaseModel`, declare annotated fields, then call
`model_validate` on incoming data. A field without a default is required. A field with a default
can be omitted. The result is an instance whose values you read with dot notation.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class IntroCourseRequest(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    name: str = Field(min_length=1)
    quantity: int = Field(gt=0, le=1000)
    note: str = ""


intro_request = IntroCourseRequest.model_validate({"name": "mango", "quantity": 4})
print(intro_request.name, intro_request.quantity, repr(intro_request.note))
assert intro_request.note == ""

The annotation says the field's type. `Field` supplies constraints: `gt=0` means greater than
zero, `le=1000` means at most 1000, and `min_length=1` excludes an empty name. `ConfigDict` sets
model-wide behavior. `strict=True` rejects conversions for this integer field, including `"4"`,
`4.0` and `True`; `extra="forbid"` rejects undeclared keys. Pydantic can otherwise convert some
compatible inputs, so choose this boundary deliberately rather than assuming every accepted
input arrived in the expected type.

Predict which rule refuses each payload. `ValidationError` reports a failed contract. Its
`errors()` entries contain `loc`, the field location, and `type`, the failure category. Catching
that expected exception lets the notebook inspect the failure and continue.

In [ ]:
intro_bad_requests = [
    {"name": "mango", "quantity": "4"},
    {"name": "mango", "quantity": True},
    {"name": "", "quantity": 4},
    {"name": "mango", "quantity": 0},
    {"name": "mango", "quantity": 4, "approved": True},
    {"quantity": 4},
]
for intro_bad_request in intro_bad_requests:
    try:
        IntroCourseRequest.model_validate(intro_bad_request)
    except ValidationError as intro_error:
        print([(item["loc"], item["type"]) for item in intro_error.errors(include_input=False)])
    else:
        raise AssertionError("An invalid input crossed the declared contract")

Use `model_dump()` for a Python dictionary, `model_dump_json()` for JSON text, and
`model_validate_json()` to parse and validate JSON. `model_json_schema()` describes the contract;
it is neither an instance's current values nor an invocation of the business handler.

In [ ]:
intro_serialized = intro_request.model_dump_json()
intro_schema = IntroCourseRequest.model_json_schema()
assert IntroCourseRequest.model_validate_json(intro_serialized) == intro_request
print("Actual values:", intro_request.model_dump())
print("Quantity contract:", intro_schema["properties"]["quantity"])
assert intro_schema["properties"]["quantity"]["exclusiveMinimum"] == 0

Four is valid input to this schema even if the shop needs six. Pydantic checks the declared
shape and constraints; the handler still needs authoritative stock, price and permission.
Ordinary assignments to an existing instance are not automatically revalidated unless configured
for assignment validation. This lesson validates new input at the boundary and uses the resulting
values. Explain these limits before relying on a model object in a transaction or tool call.

Chapter 2's full introduction expands this pattern with a separate data-repair checkpoint.
This notebook contains the required pattern here so prior Pydantic experience is not needed.
References: [models](https://docs.pydantic.dev/latest/concepts/models/),
[fields](https://docs.pydantic.dev/latest/concepts/fields/), and
[strict mode](https://docs.pydantic.dev/latest/concepts/strict_mode/).

### SQLite from the first row to an atomic change

A dictionary disappears when its process ends. A database can retain records so a later process
can resume from evidence. **SQLite** is an embedded database: Python's `sqlite3` library opens
a local database file without starting a separate database server. **SQL** is the language used
to define, select and change its records. A **table** has named columns and rows. A **primary
key** identifies a row; a **query** asks for rows satisfying a condition.

Start with a deliberately small preference table. Read the SQL as instructions: create the
table, insert one named value, then select the value for one session. `?` is a parameter
placeholder; the values are passed separately so they are data, not SQL instructions.
`fetchone()` returns one row or `None`; it does not guarantee that a matching row exists.

In [ ]:
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as intro_sql_folder:
    intro_db_path = Path(intro_sql_folder) / "example.sqlite"
    intro_db = sqlite3.connect(intro_db_path, autocommit=True)
    intro_db.execute("CREATE TABLE preference (id INTEGER PRIMARY KEY, session TEXT, value TEXT)")
    intro_db.execute("INSERT INTO preference (session,value) VALUES (?,?)", ("lucy", "09:00"))
    intro_row = intro_db.execute(
        "SELECT value FROM preference WHERE session=?", ("lucy",)
    ).fetchone()
    print("Matching row:", intro_row)
    assert intro_row == ("09:00",)
    assert (
        intro_db.execute(
            "SELECT value FROM preference WHERE session=?", ("another-session",)
        ).fetchone()
        is None
    )
    intro_db.close()
    intro_reopen = sqlite3.connect(intro_db_path, autocommit=True)
    assert intro_reopen.execute("SELECT count(*) FROM preference").fetchone()[0] == 1
    intro_reopen.close()
print("The row survived closing and reopening its connection.")

Index `[0]` selects the first column of a returned tuple. `sqlite3.Row` is an alternative row
factory that also permits named-column access. `dict(row)` then produces an ordinary dictionary.
The book's `Database` wrapper supplies that configuration and its schema; the wrapper is course
code, while `sqlite3` is the standard library. You will use the public connection and transaction
methods explained at the exercise boundary, rather than needing to reconstruct the wrapper.

Now consider a budget. Moving five cents from reserved to spent requires two values to change
together. A **transaction** makes a group of local changes commit together or roll back together.
The example explicitly controls SQL transactions with `autocommit=True` and SQL statements.
`BEGIN IMMEDIATE` starts a write transaction; `COMMIT` keeps its changes; `ROLLBACK` discards them.
Predict the row after the deliberately raised exception. Catching an error alone would not undo
the first update; the rollback is the operation that restores the prior state.

In [ ]:
intro_ledger = sqlite3.connect(":memory:", autocommit=True)
intro_ledger.execute(
    "CREATE TABLE budget (id INTEGER PRIMARY KEY, reserved INTEGER, spent INTEGER)"
)
intro_ledger.execute("INSERT INTO budget VALUES (1,5,0)")
intro_ledger.execute("BEGIN IMMEDIATE")
try:
    intro_ledger.execute("UPDATE budget SET reserved=0 WHERE id=1")
    raise ValueError("injected failure before the matching spend update")
except ValueError:
    intro_ledger.execute("ROLLBACK")
assert intro_ledger.execute("SELECT reserved,spent FROM budget").fetchone() == (5, 0)
intro_ledger.execute("BEGIN IMMEDIATE")
intro_ledger.execute("UPDATE budget SET reserved=0,spent=5 WHERE id=1")
intro_ledger.execute("COMMIT")
print(
    "After a complete change:", intro_ledger.execute("SELECT reserved,spent FROM budget").fetchone()
)
intro_ledger.close()

The literal `:memory:` creates a temporary database inside this connection; it is useful for the
small experiment, but the earlier file example establishes persistence. Neither example proves
that a remote supplier rolls back when the local transaction rolls back. An external operation
has its own state and evidence.

**SQL you will meet later.** `UPDATE ... SET ... WHERE ...` changes selected rows. `AND` combines
conditions. `ORDER BY` makes an ordering explicit; absent that clause, do not rely on row order.
`count(*)` counts rows; `sum(amount)` totals a column and can be `NULL` on an empty input;
`coalesce(sum(amount),0)` uses zero for that empty aggregate. A `UNIQUE` constraint rejects
duplicate identities. `GROUP BY status` computes one aggregate per status.

An **invariant** is a condition that must remain true across operations, such as nonnegative
reserved money. A **snapshot** is a consistent view at one point; two separate reads can describe
different moments unless their transaction contract binds them. In the book, `with db.immediate()`
groups related writes. It is a course-defined context manager with the commit/rollback purpose
you just observed. Do not assume that an arbitrary `with connection` has identical behavior under
every SQLite autocommit setting.

**Your prediction:** two workers both read ten remaining cents outside a transaction and each
approve seven. Why can both believe the next order fits? Explain what must be checked together
with the write. Then change the example's initial reserved amount and repeat the failure.
Reference: Python's [SQLite tutorial and transaction control](https://docs.python.org/3.14/library/sqlite3.html).

## Acceptance assembles independent evidence for an operating day

Lucy needs to know what the assistant actually accomplished: work received, drafts prepared,
orders confirmed, money reserved, deliveries still uncertain and recovery actions still needed.
A final paragraph saying “everything went well” is an output, not independent acceptance evidence.
An **acceptance criterion** names an observable condition required before we accept the system
for a stated use. A **business-day scenario** exercises several mechanisms in sequence.

The book's final scenario combines tools, memory, scheduling, authority, supplier ambiguity,
worker recovery, evaluation and operation. The report must keep evidence sources distinct so
agreement is meaningful. Comparing one summary field with another field computed from the same
buggy total is not independent reconciliation.

### Reconcile orders with the spending ledger

An approved, sending or unknown order retains reserved exposure. A confirmed or delivered order
contributes to known spent exposure under this report's contract. A rejected order contributes
neither. Predict the reserved and spent totals below before computing them, then compare with a
separately supplied ledger. The list of uncertain operations belongs beside the totals.

In [ ]:
intro_orders = [
    {"id": "one", "status": "CONFIRMED", "amount": 1500},
    {"id": "two", "status": "UNKNOWN", "amount": 1100},
    {"id": "three", "status": "REJECTED", "amount": 400},
]
intro_reserved = sum(
    row["amount"] for row in intro_orders if row["status"] in {"APPROVED", "SENDING", "UNKNOWN"}
)
intro_spent = sum(
    row["amount"] for row in intro_orders if row["status"] in {"CONFIRMED", "DELIVERED"}
)
intro_ledger = {"reserved": 1100, "spent": 1500}
intro_uncertain = [row["id"] for row in intro_orders if row["status"] == "UNKNOWN"]
print("Orders:", intro_reserved, intro_spent, "ledger:", intro_ledger, "unknown:", intro_uncertain)
assert (intro_reserved, intro_spent) == (1100, 1500)
assert intro_uncertain == ["two"]

If the ledger said 1000 reserved, the report should expose the 100-cents disagreement. It must
not choose whichever source makes the day look more successful. **Unknown** is a state with a
reason; it is not zero. The same applies to historical usage after a stale restore: missing
records cannot justify a confident zero-cost claim.

### A report needs one consistent read boundary

Suppose one query reads an order before confirmation and a later query reads its ledger after
confirmation. The combined report can appear inconsistent even though each database state was
internally consistent. A read transaction provides a snapshot for related local queries. The
actual `operating_report` opens its own read snapshot, queries the work/order/ledger tables,
obtains stock through the real tool path and retains scope information.

In [ ]:
intro_before = {"order_state": "UNKNOWN", "reserved": 1100, "spent": 1500}
intro_after = {"order_state": "CONFIRMED", "reserved": 0, "spent": 2600}
intro_mixed = {"order_state": intro_before["order_state"], "reserved": intro_after["reserved"]}
print("Mixed-time report:", intro_mixed)
assert intro_mixed == {"order_state": "UNKNOWN", "reserved": 0}

The mixed dictionary illustrates the problem; it is not itself a concurrent database test.
The connected runtime task supplies the real read transaction and multiple sources. A snapshot
of local SQLite also does not freeze a remote supplier. Label which evidence is local and which
comes from an independently observed external fixture.

### Design a failure schedule, not just a happy-path demo

A **failure schedule** states where an interruption occurs: before durable intent, after supplier
acceptance but before reply, after a lease replacement, or after a backup but before another
external order. Each location changes what evidence is available. Predict the expected retained
state before running. If a failure occurs elsewhere, diagnose that difference rather than
declaring the planned experiment passed.

Unit A constructs the operating report from the real data sources. Unit B corrupts one total
and asks you to trace the discrepancy back to its query. The transfer includes empty work,
uncertain delivery, unknown historical usage, changed supplier totals and a reordered catalog.
The final practical asks you to use earlier concepts to defend an acceptance decision.

### Retain an operational explanation

Your submission should let another person answer: which source supports each claim, which
operation remains uncertain, which budget is still reserved, and what action is allowed next?
Include an explicit next check for each uncertainty. Do not invent a purchase, refund or resend
merely to make the report terminal. The notebook's offline acceptance is scoped to the authored
scenario and fixtures; live model quality, phone delivery, OS containment and host operation
retain their separate evidence requirements.

Before the main task, draw the day as events with two columns for local and supplier evidence.
Circle every point where a retry could create a second effect. Then name the identity, authority
and reconciliation mechanism that prevents or detects it. This is cumulative understanding:
the final chapter should make earlier boundaries easier to explain, not hide them inside a demo.

## Choose an explicit starting point for this independent notebook

This Unit B runs without Unit A. By default it prepares a **supplied reference starting point**
and labels its provenance. It is not evidence that you built Unit A. To investigate your own
successful implementation, set `LEARNER_HANDOFF` to its saved path before running the cell.
An invalid selected file refuses; it is never silently replaced with the reference.

`SourceTask` supplies copied-source execution and handoff validation; `RuntimeLab` supplies the
controlled failure experiment. Their public operations are introduced beside the main exercise.
The artifact stores identity and observations; no variables from another kernel are required.


In [ ]:
LEARNER_HANDOFF = None

<details><summary>Prepare and validate the supplied starting artifact</summary>


In [ ]:
import json
import runpy
import shutil
import textwrap
from pathlib import Path

COURSE_INPUT = COURSE_WORK / "ch19-unit-a-handoff-v1.json"
if LEARNER_HANDOFF is not None:
    learner_input = Path(LEARNER_HANDOFF).expanduser().resolve()
    if not learner_input.is_file():
        raise FileNotFoundError("The selected learner handoff does not exist")
    if learner_input != COURSE_INPUT.resolve():
        shutil.copy2(learner_input, COURSE_INPUT)
    HANDOFF_ORIGIN = "LEARNER_SELECTED"
else:
    source_task_class = runpy.run_path(
        str(COURSE_ROOT / "book/always_on/exercises/source_tasks_v1.py")
    )["SourceTask"]
    reference_task = source_task_class(COURSE_ROOT, 16)
    try:
        reference_task.install(textwrap.dedent(reference_task.fragment))
        reference_observation = reference_task.visible("SUPPLIED_REFERENCE_START")
        if reference_observation["status"] != "PASS":
            raise RuntimeError("The supplied starting point did not pass its connection check")
        reference_task.save(COURSE_INPUT, reference_observation)
    finally:
        reference_task.close()
    HANDOFF_ORIGIN = "SUPPLIED_REFERENCE"
print("Starting evidence:", HANDOFF_ORIGIN)
print("The core task below validates the selected artifact before using it.")

</details>


## Understand the supplied execution interface

The course runtime is provided so your implementation can be connected to real callers and
storage. `SourceTask(ROOT, chapter)` makes a private copy. `install(source)` replaces only the
declared function; `visible()` invokes the real chapter probe; `save(path, result)` retains a
successful implementation and its evidence. `load(path)` checks the saved identities and hashes.
`inject_failure()` changes the declared boundary; `repair(fragment)` replaces that broken fragment.
`close()` removes the scratch copy after you retain evidence. These methods are supplied harness
operations, not additional packages you must discover or install.

`RuntimeLab` provides the same copied-source failure experiment without the complete-function
construction layer. Its `run` method records exit status, observations and the compared expectation.
A subprocess log from an unfinished learner implementation is feedback about that implementation;
it is not a successful connection. A syntax error in the notebook cell itself is a separate issue
to fix. The task below names which interface it uses.

For direct-function units, the visible driver calls your callback without installing a source
string. In either case, trace where your code is invoked. Supplied fixtures, database wrappers and
replay models are labeled infrastructure; your own implementation and changed-case explanation
are the evidence of learning.


## Main practical: construct, connect and challenge



Forcing totals to match hides a real accounting discrepancy. This time you begin with your Unit A implementation and its saved evidence. Lucy receives a reassuring report even though one cents of reserved allowance has disappeared from its ledger.

## Verify the handoff

The starting-point cell has selected the Unit A artifact explicitly. A selected learner handoff must validate; the default reference start is labeled separately. Run the setup and keep the runtime and implementation hashes in your submission.

In [ ]:
import json
import os
import runpy
from pathlib import Path

ROOT = COURSE_ROOT

SourceTask = runpy.run_path(str(ROOT / "book/always_on/exercises/source_tasks_v1.py"))["SourceTask"]
REFERENCE_LESSON = 16
HANDOFF = Path("ch19-unit-a-handoff-v1.json")
handoff_status = "MISSING"
if HANDOFF.is_file():
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        handoff = task.load(HANDOFF)
        handoff_status = "VERIFIED"
        print("IMPLEMENTATION", handoff["implementation_sha256"])
    finally:
        task.close()
print("UNIT_A_HANDOFF", handoff_status)

## Reproduce and diagnose

Predict the consequence of this injected boundary before executing it:

```text
matching = True
```

The controlled mutation changes the same implementation you submitted. It refuses if the declared mutation boundary no longer occurs exactly once; inspect an alternative implementation with the instructor before adapting the experiment.

In [ ]:
baseline = broken = None
if handoff_status == "VERIFIED":
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        task.load(HANDOFF)
        baseline = task.visible("YOUR_BASELINE")
        if baseline["status"] != "PASS":
            raise ValueError("Saved Unit A code no longer satisfies the visible contract")
        task.inject_failure()
        broken = task.run("INJECTED_FAILURE", expected=task.spec["expected_broken"])
        print("BEFORE", baseline["observation"])
        print("AFTER", broken["observation"])
    finally:
        task.close()
else:
    print("HANDOFF_REQUIRED: complete Unit A before performing Unit B")

State a diagnosis using those two observations. Name a test that would prove your diagnosis wrong. Create a real approval, corrupt a retained ledger amount, then call the actual report and inspect both structured evidence and displayed exception text.

## Repair the boundary

Return the complete replacement for the injected fragment. Do not edit the oracle or print a desired observation. Repair the actual source. The starter keeps the defect so the learner outcome remains incomplete.

In [ ]:
def repair_fragment():
    return "matching = (spent, reserved) == tuple(totals)"

<details><summary>Hint 1 — the consequence</summary>

Lucy receives a reassuring report even though one cents of reserved allowance has disappeared from its ledger.

</details>

<details><summary>Hint 2 — the evidence</summary>

Compare the two observations, then trace the changed field to `operating_report` in `src/reference_organizations/store/operating_report.py`. Distinguish a schema refusal from a business-rule or authority refusal.

</details>

<details><summary>Hint 3 — the design</summary>

Read within one transaction and always release it; compare spent and reserved separately with order-state totals; preserve usage completeness and uncertainty in the report.

</details>

In [ ]:
def connect_repair(fragment):
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        task.load(HANDOFF)
        task.inject_failure()
        task.repair(fragment)
        return task.visible("YOUR_REPAIR")
    finally:
        task.close()


repair_result = None
if handoff_status == "VERIFIED":
    repair_result = connect_repair(repair_fragment())
    print("REPAIR", repair_result["status"], repair_result["observation"])
else:
    print("REPAIR_NOT_ATTEMPTED: missing Unit A evidence")

## Transfer under a changed constraint

Test an empty account, exact matching reservation, one-cents mismatch, paused operation and incomplete usage history. Treat observed_at as a run observation.

Create a fresh task, load your handoff, inject the defect and apply your repair. Then change only the copied probe to exercise the new condition. Keep the actual observation and a prediction written beforehand. Explain why a visible-case lookup or a blanket refusal could pass the original example but fail this transfer.

The instructor's holdout applies your repair to a new copied runtime and checks both the positive case and the missing protection. An exact exception or changed state must cause a failure; no broad error is accepted as successful refusal.

## Exit ticket

Submit the original handoff, baseline and broken observations, repair, transfer probe and results. State what Lucy would experience before and after the fix. Identify the guarantee that still requires separate evidence: Local consistency is not an external account audit, daily cash profit or classroom learning evidence.

In [ ]:
passed = repair_result is not None and repair_result["status"] == "PASS"
exercise_report = {
    "unit": "ch19-b",
    "attempted": int(repair_result is not None),
    "completed": int(passed),
    "failed": int(repair_result is not None and not passed),
    "skipped": int(repair_result is None),
    "connection": "PASS" if passed else "NOT_READY",
    "handoff": handoff_status,
}
print("EXERCISE_REPORT=" + json.dumps(exercise_report, sort_keys=True))

## Changed-constraint construction: Reconcile an independently supplied order ledger

**Allow twenty minutes.** Spend three minutes predicting, ten implementing and tracing, five
on a new case of your own, and two explaining the surviving limitation. This is dedicated work,
not an invitation to run a supplied answer. Both units revisit the same invariant after different
core experiences; in Unit B, attempt this task from memory before consulting Unit A.

Implement transfer_check(orders, ledger). Orders contain unique id, status and nonnegative integer amount. Sum reserved for APPROVED/SENDING/UNKNOWN and spent for CONFIRMED/DELIVERED; DRAFT/REJECTED contribute neither. Raise ValueError on any other status. Return {matches: bool, reserved: int, spent: int, unknown: sorted IDs}. Ledger contains reserved/spent totals. Do not replace missing evidence with invented money.

Write your expected values before running the table. Keep one accepted case and one refusal.
Your function is passed directly into the driver below. The driver copies inputs and checks
they remain unchanged; it does not replace your implementation with the reference answer.

<details><summary>Hint 1 — identify the authoritative inputs</summary>
Name the source field for each output value. Which input changes while the rule remains the same?
</details>
<details><summary>Hint 2 — choose the boundary cases</summary>
Start with exact empty, exact equality and one value on each side of the boundary where valid.
Do not add a special case for a visible product name or operation identity.
</details>


In [ ]:
def transfer_check(orders, ledger):
    reserved = spent = 0
    unknown = []
    for row in orders:
        status = row["status"]
        if status in {"APPROVED", "SENDING", "UNKNOWN"}:
            reserved += row["amount"]
        elif status in {"CONFIRMED", "DELIVERED"}:
            spent += row["amount"]
        elif status not in {"DRAFT", "REJECTED"}:
            raise ValueError("unrecognized order state")
        if status == "UNKNOWN":
            unknown.append(row["id"])
    return {
        "matches": ledger == {"reserved": reserved, "spent": spent},
        "reserved": reserved,
        "spent": spent,
        "unknown": sorted(unknown),
    }

In [ ]:
import copy
import json

TRANSFER_CASES = [
    (
        "uncertain and confirmed",
        [
            [
                {"id": "a", "status": "CONFIRMED", "amount": 1500},
                {"id": "b", "status": "UNKNOWN", "amount": 1100},
            ],
            {"reserved": 1100, "spent": 1500},
        ],
        {"matches": True, "reserved": 1100, "spent": 1500, "unknown": ["b"]},
    ),
    (
        "ledger disagreement",
        [[{"id": "b", "status": "UNKNOWN", "amount": 1100}], {"reserved": 1000, "spent": 0}],
        {"matches": False, "reserved": 1100, "spent": 0, "unknown": ["b"]},
    ),
    (
        "empty day",
        [[], {"reserved": 0, "spent": 0}],
        {"matches": True, "reserved": 0, "spent": 0, "unknown": []},
    ),
    (
        "unrecognized state",
        [[{"id": "c", "status": "MAYBE", "amount": 5}], {"reserved": 0, "spent": 0}],
        {"raises": "ValueError"},
    ),
]


def same_transfer_value(actual, expected):
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return actual.keys() == expected.keys() and all(
            same_transfer_value(actual[key], value) for key, value in expected.items()
        )
    if isinstance(expected, list):
        return len(actual) == len(expected) and all(
            same_transfer_value(a, e) for a, e in zip(actual, expected, strict=True)
        )
    return actual == expected


def run_transfer(candidate, cases):
    observations = []
    for label, arguments, expected in cases:
        supplied = copy.deepcopy(arguments)
        before = copy.deepcopy(supplied)
        raised = None
        try:
            actual = candidate(*supplied)
        except NotImplementedError:
            raised = "NotImplementedError"
            actual = {"unfinished": True}
        except Exception as error:
            raised = type(error).__name__
            actual = {"raises": raised}
        expects_error = isinstance(expected, dict) and set(expected) == {"raises"}
        correct = (
            raised == expected["raises"]
            if expects_error
            else (raised is None and same_transfer_value(actual, expected))
        )
        passed = correct and same_transfer_value(supplied, before)
        observations.append(
            {"case": label, "expected": expected, "observed": actual, "passed": passed}
        )
        print("PASS" if passed else "NEEDS_WORK", label, "expected", expected, "observed", actual)
    return observations


transfer_observations = run_transfer(transfer_check, TRANSFER_CASES)
TRANSFER_PASSED = all(row["passed"] for row in transfer_observations)
print("TRANSFER_STATUS", "PASS" if TRANSFER_PASSED else "NEEDS_WORK")

### Design a counterexample and retrieve the mechanism

Add one new case with an independently calculated expected outcome to `TRANSFER_CASES` and rerun
the driver. Change one condition at a time. Then deliberately replace your candidate with a
constant answer in a temporary copy and show a case that rejects it. Restore your implementation.
Explain why that counterexample is stronger than repeating the original example with a new name.

Without viewing the worked example, write the invariant in words and trace one observed value
back to its input. Identify which part is a local fixture result and which claim would need a
live provider, host or external-system observation. Keep a first attempt even if you used a hint.


## Instructor explanation and additional transfer cases

Totals come from order rows before comparison with the independent ledger. Retain unknown identities even when money agrees. Unknown status vocabulary must be investigated rather than treated as zero.

Ask for the learner's first prediction and attempt before revealing this version. Passing these
cases verifies behavior on these inputs; it does not establish independent student mastery.
The original core holdouts also run against the connected implementation below.


In [ ]:
INSTRUCTOR_TRANSFER_CASES = [
    (
        "rejection contributes nothing",
        [[{"id": "r", "status": "REJECTED", "amount": 500}], {"reserved": 0, "spent": 0}],
        {"matches": True, "reserved": 0, "spent": 0, "unknown": []},
    ),
    (
        "delivered is retained spending",
        [[{"id": "d", "status": "DELIVERED", "amount": 275}], {"reserved": 0, "spent": 275}],
        {"matches": True, "reserved": 0, "spent": 275, "unknown": []},
    ),
]
instructor_transfer = run_transfer(transfer_check, INSTRUCTOR_TRANSFER_CASES)
assert TRANSFER_PASSED and all(row["passed"] for row in instructor_transfer)

In [ ]:
# Instructor holdout appended to a submitted Chapter 16 Unit B.

# ruff: noqa: F821
import json

task = SourceTask(ROOT, REFERENCE_LESSON)
try:
    task.load(HANDOFF)
    task.inject_failure()
    task.repair(repair_fragment())
    outcome = task.transfer(ROOT / "book/always_on/exercises/ch16/holdouts/runtime-transfer-v1.py")
    assert outcome["status"] == "PASS", outcome
finally:
    task.close()
print("HOLDOUT_RESULT=" + json.dumps({"unit": "ch19-b", "status": "PASSED"}, sort_keys=True))

## Save your evidence and explain the result

Fill the prediction notes and your explanation before saving. Include the exact observed value,
the input or retained row that caused it, your code's invocation point, one failed hypothesis,
and the strongest claim the evidence still cannot support. A completed code cell alone does not
earn explanation credit. Do not label reference-start behavior as your own Unit A construction.

Keep this edited notebook, the Markdown if used for notes, saved handoff files, and the JSON record
below. Your work folder survives scratch cleanup and can be reopened in a new kernel. An instructor
can ask for an unseen case after the visible checks; keep your implementation general.


In [ ]:
explanation_notes = {
    "causal_trace": "Explain the input, learner invocation and observed result.",
    "failed_hypothesis": "Describe a prediction the evidence changed.",
    "remaining_limit": "Name the guarantee not established by this experiment.",
}
course_submission = {
    "unit": "ch19-b",
    "planned_minutes": 90,
    "starting_evidence": globals().get("HANDOFF_ORIGIN", "INDEPENDENT_UNIT_A"),
    "prediction": prediction_notes,
    "explanation": explanation_notes,
    "core_report": exercise_report,
    "transfer": transfer_observations,
    "explanation_review": "HUMAN_REVIEW_REQUIRED",
}
submission_path = COURSE_WORK / "ch19-b-submission-v1.json"
submission_path.write_text(
    json.dumps(course_submission, indent=2, sort_keys=True), encoding="utf-8"
)
print("Saved evidence:", submission_path)
print(
    "COURSE_REPORT="
    + json.dumps(
        {
            "unit": "ch19-b",
            "transfer_passed": TRANSFER_PASSED,
            "starting_evidence": course_submission["starting_evidence"],
            "edition": "instructor",
        },
        sort_keys=True,
    )
)

## Keep building with Prof Rod

Found this material through a colleague, classroom or shared download? [Get the complete book at profrod.ai/book](https://profrod.ai/book) and [join the Prof Rod learner community](https://profrod.ai/community). Bring one result, one question or one failure you learned from. Share this resource with another learner and keep its source links with it so they can find the full course and future updates.
